<a href="https://colab.research.google.com/github/chavezaltamirano-ui/Growth-Models-in-Comparative/blob/main/5Growth_Models_in_Comparative_Perspective.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Opción 1: Montar Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
!ls "/content/drive/MyDrive/tesis_china_eeuu/data_clean"

bitacora_reparacion.csv
china_us_panel_1990_2025.csv
china_us_panel_analisis_v1_1990_2023.csv
china_us_panel_analisis_v2_1990_2023.csv
china_us_panel_analisis_v3_1990_2023.csv
china_us_panel_analisis_v4_1990_2023.csv
china_us_panel_analisis_v5_1990_2023.csv
china_us_super_panel_1990_2025.csv
china_us_super_panel_reparado.csv
cobertura_series_nuevas.csv
diccionario_variables_v1.csv
generar_cuadros_apa.py


In [8]:
import pandas as pd

RUTA = "/content/drive/MyDrive/tesis_china_eeuu/data_clean/china_us_panel_analisis_v5_1990_2023.csv"

df = pd.read_csv(RUTA)

print("Filas y columnas:", df.shape)
print(df.head())

Filas y columnas: (68, 146)
  country  year     rgdpna      rkna    rtfpna        hc          pop  \
0      CN  1990  1857144.8  0.036908  0.351905  1.956077  1153.582724   
1      CN  1991  2029162.4  0.039276  0.366076  1.991197  1170.788528   
2      CN  1992  2317807.2  0.043118  0.396038  2.026947  1184.574237   
3      CN  1993  2639603.8  0.048899  0.421519  2.063338  1197.308575   
4      CN  1994  2983718.2  0.055067  0.446154  2.100384  1209.003096   

   gdp_const_usd  gdp_growth_pct  inv_gfcf_gdp  ...  sub_inversor_alt  \
0   1.041178e+12        3.922520     23.942156  ...         -2.586862   
1   1.138795e+12        9.375632     26.033374  ...         -2.194105   
2   1.301616e+12       14.297623     30.615923  ...         -1.333442   
3   1.482880e+12       13.926103     37.310754  ...         -0.076066   
4   1.676881e+12       13.082712     34.694508  ...         -0.567431   

   sub_financiero_alt  IIC_alt  IIC_alt_lag1  IIC_alt_lag2  d_IIC_alt  q2002  \
0             

Después podrás acceder a tus archivos de Drive desde:

Empezamos. Estos son los pasos 1 y 2, ambos de diagnóstico.

Paso 1. Auditoría de cobertura

In [9]:
# -*- coding: utf-8 -*-
import pandas as pd

RUTA = "/content/drive/MyDrive/tesis_china_eeuu/data_clean/china_us_panel_analisis_v5_1990_2023.csv"
df = pd.read_csv(RUTA)
print("Panel:", df.shape)

col_pais = [c for c in df.columns if c.lower() in
            ("pais", "country", "iso", "iso3", "country_code")][0]

col_anio = [c for c in df.columns if c.lower() in
            ("anio", "year", "ano", "año")][0]

print("Columnas clave: {} / {}".format(col_pais, col_anio))
print("Paises:", df[col_pais].unique(), "| Anios:",
      df[col_anio].min(), "-", df[col_anio].max())

FOCALES = ["gdp_pc", "inv_gfcf_gdp", "trade_open", "inflation",
           "priv_credit_gdp", "bis_pvt_credit_gdp", "pub_debt_gdp",
           "hh_debt_gdp", "corp_debt_gdp", "ind_agr_ratio", "iic",
           "agr_va_per_worker_rec", "ind_va_per_worker_rec"]

print("\nCOBERTURA POR VARIABLE Y PAIS")
print("{:<34} {:>10} {:>16} {:>16}".format("variable", "pais", "n", "rango"))
print("-" * 80)

for c in df.columns:
    if c in (col_pais, col_anio):
        continue

    marca = "*" if any(f in c.lower()
                       for f in [x.lower() for x in FOCALES]) else " "

    for p, g in df.groupby(col_pais):
        s = g[[col_anio, c]].dropna()
        n = len(s)

        if n == 0:
            rango = "sin datos"
        else:
            rango = "{}-{}".format(
                int(s[col_anio].min()),
                int(s[col_anio].max())
            )

        if n < 30:
            print("{}{:<33} {:>10} {:>16} {:>16}".format(
                marca, c[:33], p, n, rango))

print("\n(* = variable usada en alguna especificacion; "
      "solo se listan las series con menos de 30 observaciones)")

Panel: (68, 146)
Columnas clave: country / year
Paises: ['CN' 'US'] | Anios: 1990 - 2023

COBERTURA POR VARIABLE Y PAIS
variable                                 pais                n            rango
--------------------------------------------------------------------------------
 bis_gov_credit_gdp                        CN               28        1994-2021
 bis_hh_credit_gdp                         CN               17        2005-2021
 bis_nfc_credit_gdp                        CN               17        2005-2021
 bis_tot_credit_gdp                        CN               28        1994-2021
 agr_va_per_worker                         US                1        2015-2015
 ind_va_per_worker                         US                1        2015-2015
 srv_va_per_worker                         US                1        2015-2015
*corp_debt_gdp                             CN                0        sin datos
*hh_debt_gdp                               CN                0        sin datos

In [10]:
# -*- coding: utf-8 -*-
"""Verificacion independiente de la cobertura de las series
de valor agregado por ocupado y de sus componentes."""

import io, json, time
import pandas as pd
import requests

PAISES = "CHN;USA"
ANIOS = "1990:2023"

INDICADORES = {
    # Valor agregado por ocupado, dolares constantes de 2015
    "NV.AGR.EMPL.KD": "VA por ocupado, agricultura",
    "NV.IND.EMPL.KD": "VA por ocupado, industria",
    "NV.SRV.EMPL.KD": "VA por ocupado, servicios",
    # Componentes para reconstruir
    "NV.AGR.TOTL.ZS": "VA agricultura (% PIB)",
    "NV.IND.TOTL.ZS": "VA industria (% PIB)",
    "NV.SRV.TOTL.ZS": "VA servicios (% PIB)",
    "SL.AGR.EMPL.ZS": "Empleo agricultura (% total)",
    "SL.IND.EMPL.ZS": "Empleo industria (% total)",
    "SL.SRV.EMPL.ZS": "Empleo servicios (% total)",
    "SL.TLF.TOTL.IN": "Fuerza laboral total",
    "SL.EMP.TOTL.SP.ZS": "Tasa de ocupacion (% pob 15+)",
    "NY.GDP.MKTP.KD": "PIB constante 2015 USD",
}

def bajar(ind):
    url = ("https://api.worldbank.org/v2/country/{}/indicator/{}"
           "?date={}&format=json&per_page=2000").format(PAISES, ind, ANIOS)
    for intento in range(4):
        try:
            r = requests.get(url, timeout=60)
            r.raise_for_status()
            js = r.json()
            if len(js) < 2 or js[1] is None:
                return pd.DataFrame(columns=["country", "year", ind])
            f = pd.DataFrame([{"country": d["countryiso3code"],
                               "year": int(d["date"]),
                               ind: d["value"]} for d in js[1]])
            return f.dropna(subset=[ind])
        except Exception as e:
            if intento == 3:
                print("  FALLO {}: {}".format(ind, e))
                return pd.DataFrame(columns=["country", "year", ind])
            time.sleep(2 + 2 * intento)

frames = {}
print("Descargando {} indicadores...".format(len(INDICADORES)))
for ind in INDICADORES:
    frames[ind] = bajar(ind)

print("\nCOBERTURA DIRECTA DESDE EL BANCO MUNDIAL (1990-2023)")
print("{:<22} {:<32} {:>4} {:>5} {:>12}".format(
    "indicador", "descripcion", "pais", "n", "rango"))
print("-" * 82)
for ind, desc in INDICADORES.items():
    f = frames[ind]
    for pais in ("CHN", "USA"):
        g = f[f["country"] == pais]
        n = len(g)
        rango = "{}-{}".format(int(g["year"].min()), int(g["year"].max())) if n else "sin datos"
        print("{:<22} {:<32} {:>4} {:>5} {:>12}".format(
            ind, desc[:32], pais[:2], n, rango))

# Que limita la ventana 1997-2021 de la reconstruccion
print("\nINTERSECCION DE LOS COMPONENTES DE LA RECONSTRUCCION")
COMP = ["NV.AGR.TOTL.ZS", "NV.IND.TOTL.ZS", "SL.AGR.EMPL.ZS",
        "SL.IND.EMPL.ZS", "SL.TLF.TOTL.IN", "NY.GDP.MKTP.KD"]
for pais in ("CHN", "USA"):
    base = None
    for ind in COMP:
        g = frames[ind]
        g = g[g["country"] == pais][["year", ind]]
        base = g if base is None else base.merge(g, on="year", how="inner")
    if base is not None and len(base):
        print("  {}: n={} rango {}-{}".format(
            pais, len(base), int(base["year"].min()), int(base["year"].max())))
        faltan = sorted(set(range(1990, 2024)) - set(base["year"]))
        if faltan:
            print("     anios ausentes:", faltan)
    else:
        print("  {}: interseccion vacia".format(pais))

# Guardar para inspeccion posterior
sal = None
for ind in INDICADORES:
    f = frames[ind]
    sal = f if sal is None else sal.merge(f, on=["country", "year"], how="outer")
sal = sal.sort_values(["country", "year"])
sal.to_csv("/content/verificacion_va_por_ocupado.csv", index=False)
print("\nGuardado: /content/verificacion_va_por_ocupado.csv  ->", sal.shape)

Descargando 12 indicadores...

COBERTURA DIRECTA DESDE EL BANCO MUNDIAL (1990-2023)
indicador              descripcion                      pais     n        rango
----------------------------------------------------------------------------------
NV.AGR.EMPL.KD         VA por ocupado, agricultura        CH    33    1991-2023
NV.AGR.EMPL.KD         VA por ocupado, agricultura        US     1    2015-2015
NV.IND.EMPL.KD         VA por ocupado, industria          CH    33    1991-2023
NV.IND.EMPL.KD         VA por ocupado, industria          US     1    2015-2015
NV.SRV.EMPL.KD         VA por ocupado, servicios          CH    33    1991-2023
NV.SRV.EMPL.KD         VA por ocupado, servicios          US     1    2015-2015
NV.AGR.TOTL.ZS         VA agricultura (% PIB)             CH    34    1990-2023
NV.AGR.TOTL.ZS         VA agricultura (% PIB)             US    25    1997-2021
NV.IND.TOTL.ZS         VA industria (% PIB)               CH    34    1990-2023
NV.IND.TOTL.ZS         VA industr

In [12]:
import os
print(os.path.exists("/content/outputs/Cuadros_resultados_APA_v3.docx"))

False


In [13]:
!find /content -name "Cuadros_resultados_APA_v3.docx"

/content/drive/MyDrive/tesis_china_eeuu/outputs/Cuadros_resultados_APA_v3.docx


In [15]:
!pip install python-docx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 11.4 MB/s eta 0:00:00


In [16]:
from docx import Document
print("python-docx instalado correctamente")

python-docx instalado correctamente


In [22]:
# -*- coding: utf-8 -*-
"""
Elimina la especificacion M8 de Estados Unidos de los cuadros de resultados,
marca el guion en el cuadro 4 y documenta la razon. Genera la version v4.

Ejecutar primero con APLICAR = False. Cambiar a True tras revisar.
"""

import os
import re
import shutil
import pandas as pd
from docx import Document

APLICAR = True  # <-- cambiar a True tras revisar la vista previa

SAL = "/content/drive/MyDrive/tesis_china_eeuu/outputs"
ORIGEN = os.path.join(SAL, "Cuadros_resultados_APA_v3.docx")
DESTINO = os.path.join(SAL, "Cuadros_resultados_APA_v4.docx")

ETIQ_US = ["estados unidos", "ee. uu.", "ee.uu.", "eeuu", "usa",
           "united states", "us"]

NOTA_M8 = ("La especificación M8 no se estima para Estados Unidos. El indicador "
           "de razón de productividad entre industria y agricultura requiere el "
           "valor agregado sectorial, disponible únicamente a partir de 1997 "
           "debido al cambio de la Standard Industrial Classification al North "
           "American Industry Classification System; el Bureau of Economic "
           "Analysis desaconseja de manera expresa el empalme de ambas series. "
           "La muestra resultante, de veintitrés observaciones con doce grados "
           "de libertad, es insuficiente para la inferencia, y el coeficiente de "
           "largo plazo se desplaza de 0.2742 a −1.9773 según el tratamiento de "
           "los componentes deterministas.")


def es_us(txt):
    t = txt.strip().lower().rstrip(".")
    return any(e.rstrip(".") == t for e in ETIQ_US)


def contiene_us(txt):
    t = txt.strip().lower()
    return any(e in t for e in ["estados unidos", "ee. uu.", "ee.uu.",
                                "eeuu", "usa", "united states"])


def fila_texto(fila):
    return [c.text.strip() for c in fila.cells]


def escribir_parrafo(parrafo, texto):
    if parrafo.runs:
        parrafo.runs[0].text = texto
        for r in parrafo.runs[1:]:
            r.text = ""
    else:
        parrafo.add_run(texto)


def escribir_celda(celda, texto):
    p = celda.paragraphs[0]
    escribir_parrafo(p, texto)
    for extra in celda.paragraphs[1:]:
        escribir_parrafo(extra, "")


# ------------------------------------------------------------- comprobacion
if not os.path.exists(ORIGEN):
    raise SystemExit("No se encuentra el archivo de origen:\n  " + ORIGEN)

doc = Document(ORIGEN)
print("Origen : {} ({:,} bytes)".format(ORIGEN, os.path.getsize(ORIGEN)))
print("Tablas : {} | Parrafos: {}\n".format(len(doc.tables), len(doc.paragraphs)))

# --------------------------------------------------------- 1. filas a borrar
plan = []
for i, tabla in enumerate(doc.tables):
    objetivo = []
    for fila in tabla.rows[1:]:
        cabeza = fila_texto(fila)[:3]
        if any(es_us(c) for c in cabeza) and any(re.match(r"^M8\b", c) for c in cabeza):
            objetivo.append(fila)
    if objetivo:
        plan.append((i, objetivo))
        print("Tabla {:>2}: {} fila(s) a eliminar".format(i, len(objetivo)))
        for fila in objetivo:
            print("     -> " + " | ".join(fila_texto(fila))[:105])

print("\nTotal de filas a eliminar: {}".format(sum(len(o) for _, o in plan)))

# ------------------------------------------------- 2. guion en el cuadro 4
tabla4 = doc.tables[0]
enc = fila_texto(tabla4.rows[0])
print("\nCUADRO 4 (tabla 0) - encabezado:")
print("  " + " | ".join(enc))

col_us = [k for k, h in enumerate(enc) if contiene_us(h)]
fila_m8 = [f for f in tabla4.rows[1:]
           if any(re.match(r"^M8\b", c) for c in fila_texto(f)[:2])]

marca = None
if col_us and fila_m8:
    marca = (fila_m8[0], col_us[0])
    print("  Fila M8 actual: " + " | ".join(fila_texto(fila_m8[0]))[:105])
    print("  Se colocara un guion en la columna {} ('{}'), valor actual: '{}'".format(
        col_us[0], enc[col_us[0]], fila_m8[0].cells[col_us[0]].text.strip()))
else:
    print("  AVISO: no se identifico la celda de Estados Unidos en la fila M8.")
    print("  Filas de la tabla 0:")
    for f in tabla4.rows[1:]:
        print("    " + " | ".join(fila_texto(f))[:105])

# --------------------------------------------------------- 3. notas y conteos
CAMBIOS = [("las quince especificaciones", "las trece especificaciones"),
           ("las catorce especificaciones", "las trece especificaciones"),
           ("nueve especificaciones estadounidenses", "siete especificaciones estadounidenses"),
           ("ocho especificaciones estadounidenses", "siete especificaciones estadounidenses")]

notas = []
for p in doc.paragraphs:
    nuevo = p.text
    for viejo, bueno in CAMBIOS:
        nuevo = nuevo.replace(viejo, bueno)
    if nuevo != p.text:
        notas.append((p, nuevo))

print("\nNotas de conteo a corregir: {}".format(len(notas)))
for _, n in notas:
    print("  " + n[:120])

# ------------------------------------------- 4. nota del cuadro 4, acotada
ancla = None
for j, p in enumerate(doc.paragraphs):
    if re.match(r"^\s*Cuadro\s*4\b", p.text):
        ancla = j
        break

limite = len(doc.paragraphs)
if ancla is not None:
    for k in range(ancla + 1, len(doc.paragraphs)):
        if re.match(r"^\s*(Cuadro|Tabla|Anexo)\s", doc.paragraphs[k].text):
            limite = k
            break

nota4 = None
if ancla is not None:
    for k in range(ancla + 1, limite):
        if doc.paragraphs[k].text.strip().lower().startswith("nota"):
            nota4 = k

print("\nBloque del cuadro 4: parrafos {} a {}".format(ancla, limite - 1))
if nota4 is not None:
    print("  Nota existente en el parrafo {}:".format(nota4))
    print("  " + doc.paragraphs[nota4].text[:120])
    print("  La explicacion se anexara al final de esa nota.")
else:
    print("  Sin nota previa: se creara un parrafo nuevo tras la tabla.")

# --------------------------------------------------------------- aplicacion
if not APLICAR:
    print("\n=== VISTA PREVIA. Nada se ha modificado. ===")
    print("Cambiar APLICAR = True y volver a ejecutar.")
else:
    for _, objetivo in plan:
        for fila in objetivo:
            fila._element.getparent().remove(fila._element)

    if marca is not None:
        escribir_celda(marca[0].cells[marca[1]], "—")

    for p, nuevo in notas:
        escribir_parrafo(p, nuevo)

    if nota4 is not None:
        p = doc.paragraphs[nota4]
        escribir_parrafo(p, p.text.rstrip() + " " + NOTA_M8)
    elif ancla is not None:
        base = doc.paragraphs[limite - 1]
        nuevo_p = base.insert_paragraph_before(NOTA_M8)
        base._element.addnext(nuevo_p._element)
        try:
            nuevo_p.style = base.style
        except Exception:
            pass

    doc.save(DESTINO)
    print("\nGuardado: {} ({:,} bytes)".format(DESTINO, os.path.getsize(DESTINO)))
    print("El destino ya reside en Drive, no se duplica el respaldo.")

    print("\nCSV derivados:")
    for archivo in ["cuadro7_cointegracion.csv", "cuadro9_corto_plazo.csv",
                    "cuadro10_diagnosticos.csv", "cuadro8_largo_plazo.csv"]:
        ruta = os.path.join(SAL, archivo)
        if not os.path.exists(ruta):
            print("  {}: no encontrado".format(archivo))
            continue
        d = pd.read_csv(ruta)
        cmod = [c for c in d.columns
                if d[c].astype(str).str.match(r"^M8\b", na=False).any()]
        cpais = [c for c in d.columns
                 if d[c].astype(str).str.strip().str.lower().isin(ETIQ_US).any()]
        if not cmod or not cpais:
            print("  {}: sin columnas de modelo/pais reconocibles".format(archivo))
            continue
        m = (d[cpais[0]].astype(str).str.strip().str.lower().isin(ETIQ_US) &
             d[cmod[0]].astype(str).str.match(r"^M8\b", na=False))
        salida = ruta.replace(".csv", "_sinM8US.csv")
        d[~m].to_csv(salida, index=False)
        print("  {}: {} -> {} filas | {}".format(
            archivo, len(d), int((~m).sum()), os.path.basename(salida)))

    print("\nLISTO.")

Origen : /content/drive/MyDrive/tesis_china_eeuu/outputs/Cuadros_resultados_APA_v3.docx (52,095 bytes)
Tablas : 7 | Parrafos: 37

Tabla  2: 1 fila(s) a eliminar
     -> Estados Unidos | M8 | 2, [1, 2, 1] | 4.469 | 2.88 / 4.01 | 0.0073 | — | 23 | F alto, ECT invalido
Tabla  4: 3 fila(s) a eliminar
     -> Estados Unidos | M8 | Razon de productividad industria/agricultura | 0.6985 | 1.0250 | -0.7012 | 1.0131 |
     -> Estados Unidos | M8 | Formacion bruta de capital fijo (% del PIB) | 2.7350*** | -1.9741*** | -0.7012 | 0.
     -> Estados Unidos | M8 | Apertura comercial (% del PIB) | 0.0161 | -0.0176 | -0.7012 | -0.0009 | 0.866 | 24
Tabla  5: 1 fila(s) a eliminar
     -> Estados Unidos | M8 | 23 | 12 | 0.096 | 0.039 | 0.623 | 0.796 | 1.77 | 0.907
Tabla  6: 4 fila(s) a eliminar
     -> Estados Unidos | M8 | Constante | 7.3273 | 16.8005 | 0.44 | No identificado
     -> Estados Unidos | M8 | Razon de productividad industria/agricultura | -1.9773 | 7.777 | -0.25 | No identif
     -> Estados 

In [23]:
# -*- coding: utf-8 -*-
import os, re
from docx import Document

RUTA = "/content/drive/MyDrive/tesis_china_eeuu/outputs/Cuadros_resultados_APA_v4.docx"
doc = Document(RUTA)
print("{} ({:,} bytes)\n".format(os.path.basename(RUTA), os.path.getsize(RUTA)))

for i, t in enumerate(doc.tables):
    resto = sum(1 for f in t.rows[1:]
                if any(re.match(r"^M8\b", c.text.strip()) for c in f.cells[:2])
                and any("stados" in c.text for c in f.cells[:2]))
    print("Tabla {:>2}: {:>3} filas x {} col | M8-EEUU restantes: {}".format(
        i, len(t.rows), len(t.columns), resto))

print("\nFila M8 del cuadro 4:")
for f in doc.tables[0].rows[1:]:
    if f.cells[0].text.strip().startswith("M8"):
        print("  " + " | ".join(c.text.strip() for c in f.cells))

print("\nNota del cuadro 4:")
print(doc.paragraphs[4].text)

Cuadros_resultados_APA_v4.docx (51,952 bytes)

Tabla  0:  10 filas x 6 col | M8-EEUU restantes: 0
Tabla  1:   9 filas x 5 col | M8-EEUU restantes: 0
Tabla  2:  14 filas x 9 col | M8-EEUU restantes: 0
Tabla  3:  23 filas x 6 col | M8-EEUU restantes: 0
Tabla  4:  37 filas x 9 col | M8-EEUU restantes: 0
Tabla  5:  14 filas x 10 col | M8-EEUU restantes: 0
Tabla  6:  28 filas x 7 col | M8-EEUU restantes: 0

Fila M8 del cuadro 4:
  M8 | PIB per capita real (log) | Razon industria/agricultura, FBCF, apertura | Cambio estructural sectorial | 33 | —

Nota del cuadro 4:
Nota. Los tamanos de muestra corresponden a las observaciones disponibles antes de la generacion de rezagos y primeras diferencias. El guion indica que la especificacion no se estima para ese pais por indisponibilidad de la serie o por insuficiencia de grados de libertad. Elaboracion propia con datos de Penn World Table 10.01 (Feenstra, Inklaar y Timmer, 2015, https://doi.org/10.1257/aer.20130954), World Development Indicators de

In [24]:
# -*- coding: utf-8 -*-
"""
Verificacion de integridad del panel de analisis contra las fuentes
originales. Detecta discrepancias de valor, de escala y de desfase temporal.
"""

import os
import io
import time
import warnings
import numpy as np
import pandas as pd
import requests

warnings.filterwarnings("ignore")

PANEL = "/content/drive/MyDrive/tesis_china_eeuu/data_clean/china_us_panel_analisis_v5_1990_2023.csv"
SAL = "/content/drive/MyDrive/tesis_china_eeuu/outputs"
TOL = 1e-4          # tolerancia relativa para considerar coincidencia

df = pd.read_csv(PANEL)
df = df[["country", "year"] + [c for c in df.columns if c not in ("country", "year")]]
print("Panel: {} filas x {} columnas".format(*df.shape))

ISO = {"CN": "CHN", "US": "USA"}
df["iso"] = df["country"].map(ISO)

# =====================================================================
# BLOQUE A. World Development Indicators
# =====================================================================
WDI = {
    "NE.GDI.FTOT.ZS": ["inv_gfcf_gdp", "gfcf_gdp", "gcf_gdp"],
    "NE.TRD.GNFS.ZS": ["trade_open", "trade_gdp", "openness"],
    "FP.CPI.TOTL.ZG": ["inflation", "inflation_cpi", "infl"],
    "FS.AST.PRVT.GD.ZS": ["priv_credit_gdp", "dom_credit_priv_gdp"],
    "NY.GDP.PCAP.KD": ["gdp_pc", "gdp_pc_const", "rgdp_pc"],
    "NY.GDP.MKTP.KD.ZG": ["gdp_growth", "growth", "g_gdp"],
    "NV.AGR.TOTL.ZS": ["agr_va_gdp"],
    "NV.IND.TOTL.ZS": ["ind_va_gdp"],
    "SL.AGR.EMPL.ZS": ["agr_emp_share"],
    "SL.IND.EMPL.ZS": ["ind_emp_share"],
}


def bajar_wdi(ind):
    url = ("https://api.worldbank.org/v2/country/CHN;USA/indicator/{}"
           "?date=1990:2023&format=json&per_page=2000").format(ind)
    for intento in range(4):
        try:
            r = requests.get(url, timeout=60)
            r.raise_for_status()
            js = r.json()
            if len(js) < 2 or js[1] is None:
                return None
            f = pd.DataFrame([{"iso": d["countryiso3code"],
                               "year": int(d["date"]),
                               "fuente": d["value"]} for d in js[1]])
            return f.dropna(subset=["fuente"])
        except Exception:
            time.sleep(2 + 2 * intento)
    return None


def comparar(serie_panel, serie_fuente, etiqueta, indicador, iso):
    m = serie_panel.merge(serie_fuente, on="year", how="inner").dropna()
    if len(m) == 0:
        return {"variable": etiqueta, "fuente": indicador, "pais": iso,
                "n_comun": 0, "veredicto": "sin solape"}
    a, b = m["panel"].values, m["fuente"].values
    dif = np.abs(a - b)
    rel = dif / np.where(np.abs(b) > 1e-9, np.abs(b), np.nan)
    corr = np.corrcoef(a, b)[0, 1] if len(m) > 2 else np.nan
    razon = np.nanmedian(np.where(np.abs(b) > 1e-9, a / b, np.nan))

    if np.nanmax(rel) < TOL:
        veredicto = "identico"
    elif corr > 0.999 and abs(razon - 100) < 1:
        veredicto = "ESCALA x100"
    elif corr > 0.999 and abs(razon - 0.01) < 0.001:
        veredicto = "ESCALA /100"
    elif corr > 0.999:
        veredicto = "proporcional (razon {:.4f})".format(razon)
    else:
        # probar desfase temporal
        mejor, lag = corr, 0
        for k in (-2, -1, 1, 2):
            sf = serie_fuente.copy()
            sf["year"] = sf["year"] + k
            mm = serie_panel.merge(sf, on="year", how="inner").dropna()
            if len(mm) > 3:
                c = np.corrcoef(mm["panel"], mm["fuente"])[0, 1]
                if c > mejor:
                    mejor, lag = c, k
        veredicto = ("DESFASE de {} anio(s)".format(lag) if lag
                     else "DISCREPANCIA")
    return {"variable": etiqueta, "fuente": indicador, "pais": iso,
            "n_comun": len(m), "corr": round(float(corr), 6) if corr == corr else None,
            "dif_max": round(float(np.nanmax(dif)), 6),
            "dif_media": round(float(np.nanmean(dif)), 6),
            "veredicto": veredicto}


res = []
print("\n" + "=" * 78)
print("BLOQUE A. World Development Indicators")
print("=" * 78)
for ind, candidatos in WDI.items():
    columna = next((c for c in candidatos if c in df.columns), None)
    if columna is None:
        print("  {:<20} sin columna equivalente en el panel".format(ind))
        continue
    fuente = bajar_wdi(ind)
    if fuente is None:
        print("  {:<20} descarga fallida".format(ind))
        res.append({"variable": columna, "fuente": ind, "pais": "-",
                    "veredicto": "descarga fallida"})
        continue
    for pais, iso in ISO.items():
        sp = df[df["country"] == pais][["year", columna]].rename(
            columns={columna: "panel"}).dropna()
        sf = fuente[fuente["iso"] == iso][["year", "fuente"]]
        if len(sp) == 0:
            continue
        r = comparar(sp, sf, columna, ind, iso)
        res.append(r)
        print("  {:<24} {:<4} n={:>3} corr={:<9} difmax={:<12} {}".format(
            columna[:24], iso[:2], r.get("n_comun", 0),
            r.get("corr", "-"), r.get("dif_max", "-"), r["veredicto"]))

# =====================================================================
# BLOQUE B. Penn World Table 10.01
# =====================================================================
print("\n" + "=" * 78)
print("BLOQUE B. Penn World Table 10.01")
print("=" * 78)
PWT_URLS = ["https://www.rug.nl/ggdc/docs/pwt1001.xlsx",
            "https://dataverse.nl/api/access/datafile/354095"]
pwt = None
for u in PWT_URLS:
    try:
        print("  Descargando {} ...".format(u))
        r = requests.get(u, timeout=180)
        r.raise_for_status()
        pwt = pd.read_excel(io.BytesIO(r.content), sheet_name="Data")
        print("  Descarga correcta: {} filas".format(len(pwt)))
        break
    except Exception as e:
        print("  Fallo: {}".format(str(e)[:90]))

if pwt is None:
    print("  NO VERIFICADO: no fue posible descargar la Penn World Table.")
    res.append({"variable": "PWT", "fuente": "PWT 10.01", "pais": "-",
                "veredicto": "descarga fallida"})
else:
    pwt = pwt[pwt["countrycode"].isin(["CHN", "USA"])]
    PWT_VARS = {"rgdpna": ["rgdpna", "rgdp_na"], "pop": ["pop", "poblacion"],
                "emp": ["emp", "empleo"], "hc": ["hc", "capital_humano"],
                "rnna": ["rnna", "capital_stock"], "ck": ["ck"]}
    for var, candidatos in PWT_VARS.items():
        columna = next((c for c in candidatos if c in df.columns), None)
        if columna is None or var not in pwt.columns:
            continue
        for pais, iso in ISO.items():
            sp = df[df["country"] == pais][["year", columna]].rename(
                columns={columna: "panel"}).dropna()
            sf = pwt[pwt["countrycode"] == iso][["year", var]].rename(
                columns={var: "fuente"}).dropna()
            if len(sp) == 0 or len(sf) == 0:
                continue
            r = comparar(sp, sf, columna, "PWT:" + var, iso)
            res.append(r)
            print("  {:<24} {:<4} n={:>3} corr={:<9} {}".format(
                columna[:24], iso[:2], r.get("n_comun", 0),
                r.get("corr", "-"), r["veredicto"]))

# =====================================================================
# BLOQUE C. Bank for International Settlements
# =====================================================================
print("\n" + "=" * 78)
print("BLOQUE C. Bank for International Settlements, credito total")
print("=" * 78)
BIS_COLS = [c for c in df.columns if c.startswith("bis_")]
if not BIS_COLS:
    print("  El panel no contiene columnas del BIS.")
else:
    try:
        url = "https://data.bis.org/static/bulk/WS_TOTAL_CREDIT_csv_col.zip"
        print("  Descargando archivo masivo del BIS (puede tardar) ...")
        r = requests.get(url, timeout=600)
        r.raise_for_status()
        import zipfile
        z = zipfile.ZipFile(io.BytesIO(r.content))
        nombre = [n for n in z.namelist() if n.endswith(".csv")][0]
        bis = pd.read_csv(z.open(nombre), low_memory=False)
        print("  Descarga correcta: {} filas, {} columnas".format(*bis.shape))
        print("  Columnas disponibles:")
        for c in bis.columns[:25]:
            print("     ", c)
        bis.to_csv("/content/bis_total_credit_bruto.csv", index=False)
        print("\n  Guardado en /content/bis_total_credit_bruto.csv para el cotejo")
        print("  Columnas del panel a verificar: {}".format(", ".join(BIS_COLS)))
        res.append({"variable": ", ".join(BIS_COLS), "fuente": "BIS",
                    "pais": "-", "veredicto": "descargado, cotejo pendiente"})
    except Exception as e:
        print("  NO VERIFICADO: {}".format(str(e)[:120]))
        res.append({"variable": ", ".join(BIS_COLS), "fuente": "BIS",
                    "pais": "-", "veredicto": "descarga fallida"})

# =====================================================================
# RESUMEN
# =====================================================================
out = pd.DataFrame(res)
ruta = os.path.join(SAL, "verificacion_fuentes.csv")
out.to_csv(ruta, index=False)

print("\n" + "=" * 78)
print("RESUMEN")
print("=" * 78)
print(out["veredicto"].value_counts().to_string())
problemas = out[~out["veredicto"].isin(["identico"])]
if len(problemas):
    print("\nCasos que requieren revision:")
    print(problemas.to_string(index=False))
else:
    print("\nTodas las series verificadas coinciden con la fuente.")
print("\nGuardado: {}".format(ruta))

Panel: 68 filas x 146 columnas

BLOQUE A. World Development Indicators
  inv_gfcf_gdp             CH   n= 34 corr=1.0       difmax=0.0          identico
  inv_gfcf_gdp             US   n= 34 corr=1.0       difmax=0.0          identico
  trade_gdp                CH   n= 34 corr=1.0       difmax=0.0          identico
  trade_gdp                US   n= 34 corr=1.0       difmax=0.0          identico
  FP.CPI.TOTL.ZG       sin columna equivalente en el panel
  FS.AST.PRVT.GD.ZS    sin columna equivalente en el panel
  NY.GDP.PCAP.KD       sin columna equivalente en el panel
  NY.GDP.MKTP.KD.ZG    sin columna equivalente en el panel
  NV.AGR.TOTL.ZS       sin columna equivalente en el panel
  NV.IND.TOTL.ZS       sin columna equivalente en el panel
  SL.AGR.EMPL.ZS       sin columna equivalente en el panel
  SL.IND.EMPL.ZS       sin columna equivalente en el panel

BLOQUE B. Penn World Table 10.01
  Descargando https://www.rug.nl/ggdc/docs/pwt1001.xlsx ...
  Fallo: 404 Client Error:  for url

In [25]:
# -*- coding: utf-8 -*-
import pandas as pd

PANEL = "/content/drive/MyDrive/tesis_china_eeuu/data_clean/china_us_panel_analisis_v5_1990_2023.csv"
df = pd.read_csv(PANEL)

print("Panel: {} filas x {} columnas\n".format(*df.shape))
print("{:<38} {:>4} {:>11} {:>4} {:>11}".format(
    "columna", "nCN", "rangoCN", "nUS", "rangoUS"))
print("-" * 74)
for c in df.columns:
    if c in ("country", "year"):
        continue
    linea = "{:<38}".format(c[:38])
    for p in ("CN", "US"):
        g = df[df["country"] == p][["year", c]].dropna()
        if len(g):
            linea += " {:>4} {:>11}".format(
                len(g), "{}-{}".format(int(g["year"].min()), int(g["year"].max())))
        else:
            linea += " {:>4} {:>11}".format(0, "-")
    print(linea)

Panel: 68 filas x 146 columnas

columna                                 nCN     rangoCN  nUS     rangoUS
--------------------------------------------------------------------------
rgdpna                                   34   1990-2023   34   1990-2023
rkna                                     34   1990-2023   34   1990-2023
rtfpna                                   34   1990-2023   34   1990-2023
hc                                       34   1990-2023   34   1990-2023
pop                                      34   1990-2023   34   1990-2023
gdp_const_usd                            34   1990-2023   34   1990-2023
gdp_growth_pct                           34   1990-2023   34   1990-2023
inv_gfcf_gdp                             34   1990-2023   34   1990-2023
inflation_cpi_pct                        34   1990-2023   34   1990-2023
trade_gdp                                34   1990-2023   34   1990-2023
gov_cons_gdp                             34   1990-2023   34   1990-2023
unemployment_pct 

In [26]:
# -*- coding: utf-8 -*-
"""
Verificacion de integridad del panel contra las fuentes, con los nombres
reales de las columnas. Prioriza la variable dependiente.
"""

import os
import io
import time
import warnings
import numpy as np
import pandas as pd
import requests

warnings.filterwarnings("ignore")

PANEL = "/content/drive/MyDrive/tesis_china_eeuu/data_clean/china_us_panel_analisis_v5_1990_2023.csv"
SAL = "/content/drive/MyDrive/tesis_china_eeuu/outputs"
ISO = {"CN": "CHN", "US": "USA"}

df = pd.read_csv(PANEL)
print("Panel: {} filas x {} columnas".format(*df.shape))

res = []


def comparar(sp, sf, etiqueta, fuente, iso):
    """sp y sf con columnas year/panel y year/fuente."""
    m = sp.merge(sf, on="year", how="inner").dropna()
    if len(m) < 3:
        return {"variable": etiqueta, "fuente": fuente, "pais": iso,
                "n": len(m), "veredicto": "solape insuficiente"}
    a, b = m["panel"].values.astype(float), m["fuente"].values.astype(float)
    dif = np.abs(a - b)
    denom = np.where(np.abs(b) > 1e-9, np.abs(b), np.nan)
    rel = dif / denom
    corr = float(np.corrcoef(a, b)[0, 1])
    razon = float(np.nanmedian(a / denom))
    disp = float(np.nanstd(a / denom))

    if np.nanmax(rel) < 1e-4:
        v = "identico"
    elif abs(razon - 100) < 1 and disp < 0.5:
        v = "ESCALA x100"
    elif abs(razon - 0.01) < 1e-3 and disp < 0.01:
        v = "ESCALA /100"
    elif disp < 0.005:
        v = "PROPORCIONAL constante (razon {:.4f})".format(razon)
    else:
        # desfase evaluado sobre diferencias, no sobre niveles
        da = pd.Series(a, index=m["year"].values).diff().dropna()
        mejor, lag = -2.0, 0
        for k in (-2, -1, 0, 1, 2):
            sf2 = sf.copy()
            sf2["year"] = sf2["year"] + k
            mm = sp.merge(sf2, on="year", how="inner").dropna()
            if len(mm) > 5:
                x = pd.Series(mm["panel"].values).diff().dropna()
                y = pd.Series(mm["fuente"].values).diff().dropna()
                if x.std() > 0 and y.std() > 0:
                    c = float(np.corrcoef(x, y)[0, 1])
                    if c > mejor:
                        mejor, lag = c, k
        v = ("DESFASE de {} anio(s) (corr dif={:.4f})".format(lag, mejor)
             if lag != 0 and mejor > 0.99 else "DISCREPANCIA")
    return {"variable": etiqueta, "fuente": fuente, "pais": iso, "n": len(m),
            "corr": round(corr, 6), "razon_mediana": round(razon, 6),
            "disp_razon": round(disp, 6),
            "dif_max": round(float(np.nanmax(dif)), 6), "veredicto": v}


# =====================================================================
# 1. Coherencia interna de la variable dependiente
# =====================================================================
print("\n" + "=" * 78)
print("1. COHERENCIA INTERNA DE LA VARIABLE DEPENDIENTE")
print("=" * 78)
for p in ("CN", "US"):
    g = df[df["country"] == p].dropna(subset=["rgdpna", "pop", "gdp_pc_pwt"])
    calc = g["rgdpna"] / g["pop"]
    d1 = float(np.nanmax(np.abs(calc - g["gdp_pc_pwt"]) /
                         np.abs(g["gdp_pc_pwt"])))
    d2 = float(np.nanmax(np.abs(np.log(g["gdp_pc_pwt"]) - g["ln_gdp_pc_pwt"])))
    print("  {}: gdp_pc_pwt = rgdpna/pop  -> dif rel max {:.2e}".format(p, d1))
    print("  {}: ln_gdp_pc_pwt = log(...) -> dif abs max {:.2e}".format(p, d2))
    res.append({"variable": "gdp_pc_pwt", "fuente": "interna", "pais": p,
                "veredicto": "coherente" if d1 < 1e-6 else "INCOHERENTE",
                "dif_max": d1})

# =====================================================================
# 2. Penn World Table: version y cotejo anio por anio
# =====================================================================
print("\n" + "=" * 78)
print("2. PENN WORLD TABLE")
print("=" * 78)
PWT_URLS = ["https://dataverse.nl/api/access/datafile/354095",
            "https://dataverse.nl/api/access/datafile/354098"]
pwt = None
for u in PWT_URLS:
    try:
        r = requests.get(u, timeout=240)
        r.raise_for_status()
        pwt = pd.read_excel(io.BytesIO(r.content), sheet_name="Data")
        print("  Descargada de {}".format(u))
        break
    except Exception as e:
        print("  Fallo en {}: {}".format(u, str(e)[:70]))

if pwt is None:
    print("  NO VERIFICADO: descarga imposible.")
else:
    print("  Cobertura de la version descargada: {}-{} | {} paises".format(
        int(pwt["year"].min()), int(pwt["year"].max()),
        pwt["countrycode"].nunique()))
    pwt = pwt[pwt["countrycode"].isin(["CHN", "USA"])]
    for var in ("rgdpna", "pop", "hc", "rkna", "rtfpna"):
        if var not in df.columns or var not in pwt.columns:
            continue
        for p, iso in ISO.items():
            sp = df[df["country"] == p][["year", var]].rename(
                columns={var: "panel"}).dropna()
            sf = pwt[pwt["countrycode"] == iso][["year", var]].rename(
                columns={var: "fuente"}).dropna()
            r = comparar(sp, sf, var, "PWT", iso)
            res.append(r)
            print("  {:<10} {:<4} n={:>3} corr={:<10} razon={:<10} {}".format(
                var, iso[:2], r["n"], r.get("corr", "-"),
                r.get("razon_mediana", "-"), r["veredicto"]))

    print("\n  Razon panel/PWT por anio (rgdpna), para detectar empalmes:")
    for p, iso in ISO.items():
        sp = df[df["country"] == p][["year", "rgdpna"]].dropna()
        sf = pwt[pwt["countrycode"] == iso][["year", "rgdpna"]].rename(
            columns={"rgdpna": "src"}).dropna()
        m = sp.merge(sf, on="year")
        m["razon"] = m["rgdpna"] / m["src"]
        muestra = m[m["year"].isin([1990, 1995, 2000, 2005, 2010,
                                    2015, 2018, 2019])]
        print("    {}: {}".format(iso, ", ".join(
            "{}:{:.4f}".format(int(a), b)
            for a, b in zip(muestra["year"], muestra["razon"]))))
        fuera = sorted(set(sp["year"]) - set(sf["year"]))
        if fuera:
            print("       anios del panel ausentes en la PWT: {}".format(fuera))

# =====================================================================
# 3. World Development Indicators, nombres reales
# =====================================================================
print("\n" + "=" * 78)
print("3. WORLD DEVELOPMENT INDICATORS")
print("=" * 78)
WDI = {
    "inv_gfcf_gdp": "NE.GDI.FTOT.ZS",
    "trade_gdp": "NE.TRD.GNFS.ZS",
    "inflation_cpi_pct": "FP.CPI.TOTL.ZG",
    "credit_priv_gdp": "FS.AST.PRVT.GD.ZS",
    "gdp_const_usd": "NY.GDP.MKTP.KD",
    "gdp_growth_pct": "NY.GDP.MKTP.KD.ZG",
    "gov_cons_gdp": "NE.CON.GOVT.ZS",
    "unemployment_pct": "SL.UEM.TOTL.ZS",
    "urban_pop_pct": "SP.URB.TOTL.IN.ZS",
    "remittances_gdp": "BX.TRF.PWKR.DT.GD.ZS",
    "gdp_pc_ppp_current": "NY.GDP.PCAP.PP.CD",
    "gni_pc_ppp_current": "NY.GNP.PCAP.PP.CD",
}


def bajar_wdi(ind):
    url = ("https://api.worldbank.org/v2/country/CHN;USA/indicator/{}"
           "?date=1990:2023&format=json&per_page=2000").format(ind)
    for i in range(4):
        try:
            r = requests.get(url, timeout=60)
            r.raise_for_status()
            js = r.json()
            if len(js) < 2 or js[1] is None:
                return None
            f = pd.DataFrame([{"iso": d["countryiso3code"],
                               "year": int(d["date"]),
                               "fuente": d["value"]} for d in js[1]])
            return f.dropna(subset=["fuente"])
        except Exception:
            time.sleep(2 + 2 * i)
    return None


for col, ind in WDI.items():
    if col not in df.columns:
        continue
    fuente = bajar_wdi(ind)
    if fuente is None:
        print("  {:<22} descarga fallida".format(col))
        continue
    for p, iso in ISO.items():
        sp = df[df["country"] == p][["year", col]].rename(
            columns={col: "panel"}).dropna()
        sf = fuente[fuente["iso"] == iso][["year", "fuente"]]
        r = comparar(sp, sf, col, ind, iso)
        res.append(r)
        print("  {:<22} {:<3} n={:>3} corr={:<10} difmax={:<12} {}".format(
            col[:22], iso[:2], r["n"], r.get("corr", "-"),
            r.get("dif_max", "-"), r["veredicto"]))

# =====================================================================
# 4. Deuda: base del Fondo Monetario Internacional
# =====================================================================
print("\n" + "=" * 78)
print("4. DEUDA (Global Debt Database, FMI)")
print("=" * 78)
for col in ("pub_debt_gdp", "hh_debt_gdp", "corp_debt_gdp", "priv_debt_gdp"):
    if col in df.columns:
        for p in ("CN", "US"):
            g = df[df["country"] == p][["year", col]].dropna()
            print("  {:<16} {:<3} n={:>3} {}".format(
                col, p, len(g),
                "{}-{}".format(int(g["year"].min()), int(g["year"].max()))
                if len(g) else "sin datos"))
print("  Cotejo automatico no disponible: la Global Debt Database no expone")
print("  API publica estable. Verificar manualmente los valores extremos.")

# =====================================================================
# RESUMEN
# =====================================================================
out = pd.DataFrame(res)
ruta = os.path.join(SAL, "verificacion_fuentes_v2.csv")
out.to_csv(ruta, index=False)

print("\n" + "=" * 78)
print("RESUMEN")
print("=" * 78)
print(out["veredicto"].value_counts().to_string())
mal = out[~out["veredicto"].isin(["identico", "coherente"])]
if len(mal):
    print("\nCasos a revisar:")
    print(mal[["variable", "fuente", "pais", "n", "razon_mediana",
               "veredicto"]].to_string(index=False))
else:
    print("\nTodo coincide con las fuentes.")
print("\nGuardado: {}".format(ruta))

Panel: 68 filas x 146 columnas

1. COHERENCIA INTERNA DE LA VARIABLE DEPENDIENTE
  CN: gdp_pc_pwt = rgdpna/pop  -> dif rel max 3.46e-16
  CN: ln_gdp_pc_pwt = log(...) -> dif abs max 1.78e-15
  US: gdp_pc_pwt = rgdpna/pop  -> dif rel max 1.25e-16
  US: ln_gdp_pc_pwt = log(...) -> dif abs max 1.78e-15

2. PENN WORLD TABLE
  Descargada de https://dataverse.nl/api/access/datafile/354095
  Cobertura de la version descargada: 1950-2019 | 183 paises
  rgdpna     CH   n= 30 corr=0.993414   razon=0.878991   DISCREPANCIA
  rgdpna     US   n= 30 corr=0.999915   razon=1.098176   PROPORCIONAL constante (razon 1.0982)
  pop        CH   n= 30 corr=0.999543   razon=0.984369   PROPORCIONAL constante (razon 0.9844)
  pop        US   n= 30 corr=0.996958   razon=1.007114   DISCREPANCIA
  hc         CH   n= 30 corr=1.0        razon=1.0        identico
  hc         US   n= 30 corr=1.0        razon=1.0        identico
  rkna       CH   n= 30 corr=0.999999   razon=0.728507   PROPORCIONAL constante (razon 0.72

In [28]:
# -*- coding: utf-8 -*-
"""
Identificacion de la fuente real de las columnas rgdpna, pop, rkna y rtfpna
del panel, mediante cotejo contra todas las variantes de la PWT y del WDI.
"""

import io
import time
import warnings
import numpy as np
import pandas as pd
import requests

warnings.filterwarnings("ignore")

PANEL = "/content/drive/MyDrive/tesis_china_eeuu/data_clean/china_us_panel_analisis_v5_1990_2023.csv"
ISO = {"CN": "CHN", "US": "USA"}
df = pd.read_csv(PANEL)

r = requests.get("https://dataverse.nl/api/access/datafile/354095", timeout=240)
pwt = pd.read_excel(io.BytesIO(r.content), sheet_name="Data")
pwt = pwt[pwt["countrycode"].isin(["CHN", "USA"])]
print("PWT descargada: {}-{}".format(int(pwt["year"].min()), int(pwt["year"].max())))

CANDIDATAS = {
    "rgdpna": ["rgdpe", "rgdpo", "cgdpe", "cgdpo", "rgdpna", "rnna", "cn"],
    "pop": ["pop", "emp", "avh"],
    "rkna": ["rnna", "rkna", "cn", "ck"],
    "rtfpna": ["rtfpna", "ctfp", "rtfpna", "labsh"],
}


def evaluar(a, b):
    """Devuelve razon mediana, dispersion de la razon y deriva extremo a extremo."""
    m = pd.concat([a, b], axis=1).dropna()
    if len(m) < 10:
        return None
    x, y = m.iloc[:, 0].values, m.iloc[:, 1].values
    razon = x / np.where(np.abs(y) > 1e-12, y, np.nan)
    return (float(np.nanmedian(razon)), float(np.nanstd(razon)),
            float(razon[-1] / razon[0]), len(m))


print("\n" + "=" * 84)
print("COTEJO CONTRA TODAS LAS VARIANTES DE LA PWT")
print("=" * 84)
print("Una razon estable (deriva proxima a 1.000) identifica la fuente real.\n")

for col, opciones in CANDIDATAS.items():
    if col not in df.columns:
        continue
    print("Columna del panel: {}".format(col))
    for p, iso in ISO.items():
        sp = df[df["country"] == p].set_index("year")[col].dropna()
        mejor = None
        for var in opciones:
            if var not in pwt.columns:
                continue
            sf = pwt[pwt["countrycode"] == iso].set_index("year")[var].dropna()
            ev = evaluar(sp, sf)
            if ev is None:
                continue
            razon, disp, deriva, n = ev
            marca = "  <== COINCIDE" if abs(deriva - 1) < 0.02 else ""
            print("   {} vs PWT:{:<8} n={:>3} razon={:>9.4f} "
                  "disp={:>8.5f} deriva={:>7.4f}{}".format(
                      iso[:2], var, n, razon, disp, deriva, marca))
    print()

# ---------------------------------------------------- contraste con el WDI
print("=" * 84)
print("COTEJO DE LAS TASAS DE CRECIMIENTO CONTRA EL BANCO MUNDIAL")
print("=" * 84)


def wdi(ind):
    url = ("https://api.worldbank.org/v2/country/CHN;USA/indicator/{}"
           "?date=1990:2023&format=json&per_page=2000").format(ind)
    js = requests.get(url, timeout=60).json()
    return pd.DataFrame([{"iso": d["countryiso3code"], "year": int(d["date"]),
                          "v": d["value"]} for d in js[1]]).dropna()


wdi_pc = wdi("NY.GDP.PCAP.KD")
for p, iso in ISO.items():
    sp = df[df["country"] == p].set_index("year")["gdp_pc_pwt"].dropna()
    g_panel = np.log(sp).diff().dropna() * 100
    sw = wdi_pc[wdi_pc["iso"] == iso].set_index("year")["v"].dropna()
    g_wdi = np.log(sw).diff().dropna() * 100
    sq = pwt[pwt["countrycode"] == iso].set_index("year")
    g_pwt = np.log(sq["rgdpna"] / sq["pop"]).diff().dropna() * 100
    m = pd.concat([g_panel.rename("panel"), g_wdi.rename("wdi"),
                   g_pwt.rename("pwt")], axis=1).dropna()
    print("\n{}  crecimiento medio anual del producto por habitante, {}-{}".format(
        iso, int(m.index.min()), int(m.index.max())))
    print("   panel {:.3f}%   WDI {:.3f}%   PWT {:.3f}%".format(
        m["panel"].mean(), m["wdi"].mean(), m["pwt"].mean()))
    print("   correlacion panel-WDI {:.4f} | panel-PWT {:.4f}".format(
        m["panel"].corr(m["wdi"]), m["panel"].corr(m["pwt"])))
    print("   diferencia media panel-WDI {:+.4f} pp | panel-PWT {:+.4f} pp".format(
        (m["panel"] - m["wdi"]).mean(), (m["panel"] - m["pwt"]).mean()))

# --------------------------------------------- los cuatro anios extendidos
print("\n" + "=" * 84)
print("OBSERVACIONES POSTERIORES A LA COBERTURA DE LA PWT")
print("=" * 84)
for p, iso in ISO.items():
    g = df[(df["country"] == p) & (df["year"] >= 2018)][
        ["year", "rgdpna", "pop", "gdp_pc_pwt", "gdp_growth_pct"]]
    g = g.copy()
    g["crec_pwt_pc"] = np.log(g["gdp_pc_pwt"]).diff() * 100
    print("\n{}".format(iso))
    print(g.to_string(index=False))

PWT descargada: 1950-2019

COTEJO CONTRA TODAS LAS VARIANTES DE LA PWT
Una razon estable (deriva proxima a 1.000) identifica la fuente real.

Columna del panel: rgdpna
   CH vs PWT:rgdpe    n= 30 razon=   0.9109 disp= 0.16209 deriva= 2.0233
   CH vs PWT:rgdpo    n= 30 razon=   0.9235 disp= 0.15457 deriva= 1.9621
   CH vs PWT:cgdpe    n= 30 razon=   0.8970 disp= 0.16423 deriva= 2.0731
   CH vs PWT:cgdpo    n= 30 razon=   0.9049 disp= 0.15936 deriva= 2.0239
   CH vs PWT:rgdpna   n= 30 razon=   0.8790 disp= 0.17568 deriva= 2.1410
   CH vs PWT:rnna     n= 30 razon=   0.3810 disp= 0.06066 deriva= 0.6297
   CH vs PWT:cn       n= 30 razon=   0.3408 disp= 0.06256 deriva= 0.6665
   US vs PWT:rgdpe    n= 30 razon=   1.1041 disp= 0.01012 deriva= 0.9726
   US vs PWT:rgdpo    n= 30 razon=   1.1060 disp= 0.00321 deriva= 0.9981  <== COINCIDE
   US vs PWT:cgdpe    n= 30 razon=   1.0957 disp= 0.00585 deriva= 1.0012  <== COINCIDE
   US vs PWT:cgdpo    n= 30 razon=   1.0970 disp= 0.00376 deriva= 1.0128  

In [29]:
# -*- coding: utf-8 -*-
"""Confirmacion del origen real de la variable dependiente."""

import numpy as np
import pandas as pd
import requests

PANEL = "/content/drive/MyDrive/tesis_china_eeuu/data_clean/china_us_panel_analisis_v5_1990_2023.csv"
ISO = {"CN": "CHN", "US": "USA"}
df = pd.read_csv(PANEL).sort_values(["country", "year"])


def wdi(ind):
    url = ("https://api.worldbank.org/v2/country/CHN;USA/indicator/{}"
           "?date=1990:2023&format=json&per_page=2000").format(ind)
    js = requests.get(url, timeout=60).json()
    f = pd.DataFrame([{"iso": d["countryiso3code"], "year": int(d["date"]),
                       "v": d["value"]} for d in js[1]]).dropna()
    return f.sort_values("year")          # <-- orden corregido


pc = wdi("NY.GDP.PCAP.KD")        # PIB por habitante, dolares constantes
gdp = wdi("NY.GDP.MKTP.KD")       # PIB total, dolares constantes
poblacion = wdi("SP.POP.TOTL")    # poblacion total

print("=" * 76)
print("TASAS DE CRECIMIENTO DEL PRODUCTO POR HABITANTE")
print("=" * 76)
for p, iso in ISO.items():
    g = df[df["country"] == p].set_index("year")
    gp = np.log(g["gdp_pc_pwt"]).diff().dropna() * 100
    sw = pc[pc["iso"] == iso].set_index("year")["v"]
    gw = np.log(sw).diff().dropna() * 100
    m = pd.concat([gp.rename("panel"), gw.rename("wdi")], axis=1).dropna()
    print("\n{}  {}-{}".format(iso, int(m.index.min()), int(m.index.max())))
    print("   media panel {:.4f}%   media WDI {:.4f}%   diferencia {:+.4f} pp".format(
        m["panel"].mean(), m["wdi"].mean(),
        (m["panel"] - m["wdi"]).mean()))
    print("   correlacion {:.6f}   diferencia absoluta maxima {:.4f} pp".format(
        m["panel"].corr(m["wdi"]), (m["panel"] - m["wdi"]).abs().max()))

print("\n" + "=" * 76)
print("RAZON ENTRE EL PANEL Y EL BANCO MUNDIAL, POR ANIO")
print("=" * 76)
for p, iso in ISO.items():
    g = df[df["country"] == p].set_index("year")
    sg = gdp[gdp["iso"] == iso].set_index("year")["v"]
    sp = poblacion[poblacion["iso"] == iso].set_index("year")["v"]
    m = pd.DataFrame({"rgdpna": g["rgdpna"], "pib_wdi": sg,
                      "pop": g["pop"], "pop_wdi": sp / 1e6}).dropna()
    m["r_pib"] = m["rgdpna"] / m["pib_wdi"]
    m["r_pop"] = m["pop"] / m["pop_wdi"]
    sel = m[m.index.isin([1990, 1995, 2000, 2005, 2010, 2015, 2019, 2023])]
    print("\n{}".format(iso))
    print("  producto: " + ", ".join("{}:{:.6f}".format(int(a), b)
                                     for a, b in zip(sel.index, sel["r_pib"])))
    print("  poblacion: " + ", ".join("{}:{:.6f}".format(int(a), b)
                                      for a, b in zip(sel.index, sel["r_pop"])))
    print("  deriva del producto extremo a extremo: {:.4f}".format(
        m["r_pib"].iloc[-1] / m["r_pib"].iloc[0]))
    print("  deriva de la poblacion extremo a extremo: {:.4f}".format(
        m["r_pop"].iloc[-1] / m["r_pop"].iloc[0]))

print("\n" + "=" * 76)
print("COTEJO CONTRA LA COLUMNA gdp_const_usd DEL PROPIO PANEL")
print("=" * 76)
for p in ("CN", "US"):
    g = df[df["country"] == p].dropna(subset=["rgdpna", "gdp_const_usd"])
    razon = g["rgdpna"] / g["gdp_const_usd"]
    print("  {}: razon mediana {:.6f}  dispersion {:.2e}  deriva {:.6f}".format(
        p, razon.median(), razon.std(), razon.iloc[-1] / razon.iloc[0]))

TASAS DE CRECIMIENTO DEL PRODUCTO POR HABITANTE

CHN  1991-2023
   media panel 7.8996%   media WDI 7.9118%   diferencia -0.0122 pp
   correlacion 0.999379   diferencia absoluta maxima 0.2196 pp

USA  1991-2023
   media panel 1.5417%   media WDI 1.5634%   diferencia -0.0217 pp
   correlacion 0.996050   diferencia absoluta maxima 0.2973 pp

RAZON ENTRE EL PANEL Y EL BANCO MUNDIAL, POR ANIO

CHN
  producto: 1990:0.000002, 1995:0.000002, 2000:0.000002, 2005:0.000002, 2010:0.000002, 2015:0.000002, 2019:0.000002, 2023:0.000002
  poblacion: 1990:1.016207, 1995:1.012681, 2000:1.005493, 2005:1.004838, 2010:1.010358, 2015:1.011794, 2019:1.011206, 2023:1.008418
  deriva del producto extremo a extremo: 0.9884
  deriva de la poblacion extremo a extremo: 0.9923

USA
  producto: 1990:0.000001, 1995:0.000001, 2000:0.000001, 2005:0.000001, 2010:0.000001, 2015:0.000001, 2019:0.000001, 2023:0.000001
  poblacion: 1990:1.015024, 1995:1.007240, 2000:0.997596, 2005:1.000677, 2010:1.005445, 2015:1.013397, 201

In [31]:
# -*- coding: utf-8 -*-
"""
Cotejo de las series de deuda del panel contra la Global Debt Database
del Fondo Monetario Internacional, via la interfaz DataMapper.
"""

import numpy as np
import pandas as pd
import requests

PANEL = "/content/drive/MyDrive/tesis_china_eeuu/data_clean/china_us_panel_analisis_v5_1990_2023.csv"
SALIDA = "/content/drive/MyDrive/tesis_china_eeuu/outputs/verificacion_deuda_fmi.csv"
BASE = "https://www.imf.org/external/datamapper/api/v1"
ISO = {"CN": "CHN", "US": "USA"}

df = pd.read_csv(PANEL).sort_values(["country", "year"])

CANDIDATAS = {
    "pub_debt_gdp":  [("GG_DEBT_GDP", "Deuda del gobierno general"),
                      ("CG_DEBT_GDP", "Deuda del gobierno central")],
    "hh_debt_gdp":   [("HH_ALL", "Hogares, todos los instrumentos"),
                      ("HH_LS", "Hogares, prestamos y titulos")],
    "corp_debt_gdp": [("NFC_ALL", "Sociedades no financieras, todos los instrumentos"),
                      ("NFC_LS", "Sociedades no financieras, prestamos y titulos")],
    "priv_debt_gdp": [("Privatedebt_all", "Deuda privada, todos los instrumentos"),
                      ("PVD_LS", "Deuda privada, prestamos y titulos")],
}


def bajar(indicador):
    """Descarga un indicador del DataMapper para China y Estados Unidos."""
    url = "{}/{}/CHN/USA".format(BASE, indicador)
    js = requests.get(url, timeout=90).json()
    filas = []
    for iso, serie in js.get("values", {}).get(indicador, {}).items():
        if iso not in ("CHN", "USA"):
            continue
        for anio, valor in serie.items():
            if valor is None:
                continue
            filas.append({"iso": iso, "year": int(anio), "v": float(valor)})
    return pd.DataFrame(filas).sort_values(["iso", "year"])


cache, registro = {}, []

print("=" * 88)
print("COTEJO CONTRA LA GLOBAL DEBT DATABASE DEL FONDO MONETARIO INTERNACIONAL")
print("=" * 88)
print("Una diferencia absoluta maxima proxima a cero identifica la definicion empleada.\n")

for col, opciones in CANDIDATAS.items():
    if col not in df.columns:
        print("La columna {} no existe en el panel.\n".format(col))
        continue
    print("-" * 88)
    print("Columna del panel: {}".format(col))
    for pais, iso in ISO.items():
        sp = df[df["country"] == pais].set_index("year")[col].dropna()
        if sp.empty:
            print("   {}: sin observaciones en el panel.".format(iso))
            registro.append({"columna": col, "pais": iso, "indicador": "",
                             "n": 0, "estado": "ausente en el panel"})
            continue
        print("   {}: n={} ({}-{})".format(iso, len(sp), int(sp.index.min()),
                                           int(sp.index.max())))
        for ind, etiqueta in opciones:
            if ind not in cache:
                try:
                    cache[ind] = bajar(ind)
                except Exception as e:
                    cache[ind] = pd.DataFrame()
                    print("      fallo la descarga de {}: {}".format(ind, e))
            fuente = cache[ind]
            if fuente.empty or iso not in set(fuente["iso"]):
                print("      {:<16} sin cobertura para este pais".format(ind))
                continue
            sf = fuente[fuente["iso"] == iso].set_index("year")["v"]
            m = pd.concat([sp.rename("panel"), sf.rename("fmi")], axis=1).dropna()
            if len(m) < 5:
                print("      {:<16} solapamiento insuficiente (n={})".format(ind, len(m)))
                continue
            dif = (m["panel"] - m["fmi"]).abs()
            rel = (dif / m["fmi"].abs().replace(0, np.nan)).max() * 100
            corr = m["panel"].corr(m["fmi"])
            marca = "   <== COINCIDE" if dif.max() < 0.01 else (
                "   <== compatible" if dif.max() < 0.5 else "")
            print("      {:<16} n={:>3}  dif abs max={:>10.6f}  "
                  "dif rel max={:>8.4f}%  corr={:.6f}{}".format(
                      ind, len(m), dif.max(), rel, corr, marca))
            registro.append({"columna": col, "pais": iso, "indicador": ind,
                             "definicion": etiqueta, "n_comun": len(m),
                             "dif_abs_max": dif.max(), "dif_rel_max_pct": rel,
                             "correlacion": corr,
                             "anio_ini": int(m.index.min()),
                             "anio_fin": int(m.index.max())})
    print()

# ------------------------------- cobertura disponible y no aprovechada
print("=" * 88)
print("COBERTURA DISPONIBLE EN LA FUENTE FRENTE A LA INCORPORADA AL PANEL")
print("=" * 88)
for col, opciones in CANDIDATAS.items():
    for pais, iso in ISO.items():
        sp = df[df["country"] == pais].set_index("year")[col].dropna() \
            if col in df.columns else pd.Series(dtype=float)
        for ind, etiqueta in opciones:
            fuente = cache.get(ind, pd.DataFrame())
            if fuente.empty or iso not in set(fuente["iso"]):
                continue
            sf = fuente[(fuente["iso"] == iso) & (fuente["year"].between(1990, 2023))]
            if sf.empty:
                continue
            print("{:<15} {} {:<16} panel n={:>3} | fuente n={:>3} ({}-{})".format(
                col, iso, ind, len(sp), len(sf),
                int(sf["year"].min()), int(sf["year"].max())))

pd.DataFrame(registro).to_csv(SALIDA, index=False)
print("\nResultado guardado en {}".format(SALIDA))

COTEJO CONTRA LA GLOBAL DEBT DATABASE DEL FONDO MONETARIO INTERNACIONAL
Una diferencia absoluta maxima proxima a cero identifica la definicion empleada.

----------------------------------------------------------------------------------------
Columna del panel: pub_debt_gdp
   CHN: n=29 (1995-2023)
      GG_DEBT_GDP      n= 29  dif abs max=  0.004968  dif rel max=  0.0213%  corr=1.000000   <== COINCIDE
      CG_DEBT_GDP      sin cobertura para este pais
   USA: n=34 (1990-2023)
      GG_DEBT_GDP      n= 34  dif abs max=  0.004457  dif rel max=  0.0066%  corr=1.000000   <== COINCIDE
      CG_DEBT_GDP      n= 34  dif abs max= 39.394135  dif rel max= 70.3689%  corr=0.955678

----------------------------------------------------------------------------------------
Columna del panel: hh_debt_gdp
   CHN: sin observaciones en el panel.
   USA: n=34 (1990-2023)
      HH_ALL           n= 34  dif abs max=  0.004906  dif rel max=  0.0070%  corr=1.000000   <== COINCIDE
      HH_LS            n= 34 

In [32]:
# -*- coding: utf-8 -*-
"""
Especificacion M10: deuda privada (prestamos y titulos, FMI) y apertura
comercial, estimada de forma simetrica para China y Estados Unidos.
Construye el panel derivado v6 sin modificar el maestro ni la version v5.
"""

import os
import itertools
import numpy as np
import pandas as pd
import requests
import statsmodels.api as sm
from statsmodels.stats.diagnostic import (acorr_breusch_godfrey, acorr_ljungbox,
                                          het_breuschpagan)
from statsmodels.stats.stattools import durbin_watson, jarque_bera

PANEL_V5 = "/content/drive/MyDrive/tesis_china_eeuu/data_clean/china_us_panel_analisis_v5_1990_2023.csv"
PANEL_V6 = "/content/drive/MyDrive/tesis_china_eeuu/data_clean/china_us_panel_analisis_v6_1990_2023.csv"
SAL = "/content/drive/MyDrive/tesis_china_eeuu/outputs"
os.makedirs(SAL, exist_ok=True)

DEP = "ln_gdp_pc_pwt"
NUEVA = "priv_debt_ls_gdp"
REGRESORES = [NUEVA, "trade_gdp"]
ETIQUETAS = {NUEVA: "Deuda privada, prestamos y titulos (% del PIB)",
             "trade_gdp": "Apertura comercial (% del PIB)"}
MAX_ORDEN = 3          # orden maximo explorado
MIN_ORDEN = 1          # orden minimo impuesto
HAC = 2                # rezagos de Newey-West
VC_I0, VC_I1 = 3.23, 4.32   # valores criticos al 5% con dos regresores

# ------------------------------------------------ descarga de la serie
print("=" * 84)
print("DESCARGA DE LA DEUDA PRIVADA (PVD_LS) DESDE EL FONDO MONETARIO INTERNACIONAL")
print("=" * 84)
js = requests.get("https://www.imf.org/external/datamapper/api/v1/PVD_LS/CHN/USA",
                  timeout=90).json()
filas = []
for iso, serie in js["values"]["PVD_LS"].items():
    if iso not in ("CHN", "USA"):
        continue
    for anio, valor in serie.items():
        if valor is not None:
            filas.append({"country": {"CHN": "CN", "USA": "US"}[iso],
                          "year": int(anio), NUEVA: float(valor)})
deuda = pd.DataFrame(filas)
deuda = deuda[deuda["year"].between(1990, 2023)].sort_values(["country", "year"])
for p in ("CN", "US"):
    s = deuda[deuda["country"] == p]
    print("  {}: n={} ({}-{})  media={:.2f}  min={:.2f}  max={:.2f}".format(
        p, len(s), int(s["year"].min()), int(s["year"].max()),
        s[NUEVA].mean(), s[NUEVA].min(), s[NUEVA].max()))

# ------------------------------------------------ construccion del panel v6
df = pd.read_csv(PANEL_V5)
if NUEVA in df.columns:
    df = df.drop(columns=[NUEVA])
df = df.merge(deuda, on=["country", "year"], how="left")
if "d2020" not in df.columns:
    df["d2020"] = (df["year"] == 2020).astype(int)
df = df.sort_values(["country", "year"]).reset_index(drop=True)
df.to_csv(PANEL_V6, index=False)
print("\nPanel derivado guardado: {}".format(PANEL_V6))
print("  dimensiones {} x {} (la v5 y el maestro quedan intactos)".format(*df.shape))

# ------------------------------------------------ utilidades de estimacion
def construir(base, p, ordenes):
    """Arma la matriz del modelo de correccion de error irrestricto."""
    X = pd.DataFrame(index=base.index)
    X["y_1"] = base[DEP].shift(1)
    for r in REGRESORES:
        X["{}_1".format(r)] = base[r].shift(1)
    for i in range(1, p):
        X["dy_{}".format(i)] = base[DEP].diff().shift(i)
    for r, q in zip(REGRESORES, ordenes):
        for i in range(0, q + 1):
            X["d{}_{}".format(r, i)] = base[r].diff().shift(i)
    X["d2020"] = base["d2020"]
    y = base[DEP].diff()
    datos = pd.concat([y.rename("dy"), X], axis=1).dropna()
    return datos


def estimar(datos):
    yv = datos["dy"]
    Xv = sm.add_constant(datos.drop(columns="dy"))
    return sm.OLS(yv, Xv).fit()


def estrellas(p):
    return "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.10 else ""


coint, largo, corto, diag = [], [], [], []

for pais, nombre in (("CN", "China"), ("US", "Estados Unidos")):
    g = df[df["country"] == pais].sort_values("year").reset_index(drop=True)
    base = g[["year", DEP] + REGRESORES + ["d2020"]].dropna().reset_index(drop=True)
    print("\n" + "=" * 84)
    print("{}  M10  observaciones disponibles: {} ({}-{})".format(
        nombre, len(base), int(base["year"].min()), int(base["year"].max())))

    # ------------------------------------ seleccion de ordenes por Akaike
    mejor = None
    for p in range(MIN_ORDEN, MAX_ORDEN + 1):
        for ordenes in itertools.product(range(MIN_ORDEN, MAX_ORDEN + 1),
                                         repeat=len(REGRESORES)):
            datos = construir(base, p, ordenes)
            k = datos.shape[1]          # incluye la constante al sumar dy
            if len(datos) - k < 8:
                continue
            res = estimar(datos)
            if mejor is None or res.aic < mejor[0]:
                mejor = (res.aic, p, ordenes, datos, res)
    if mejor is None:
        print("  Sin grados de libertad suficientes para estimar.")
        continue
    aic, p, ordenes, datos, res = mejor
    print("  Ordenes seleccionados: p={}  q={}  AIC={:.3f}  n={}  gl={}".format(
        p, list(ordenes), aic, int(res.nobs), int(res.df_resid)))

    # ------------------------------------ prueba de limites
    restricciones = ["y_1 = 0"] + ["{}_1 = 0".format(r) for r in REGRESORES]
    F = float(res.f_test(", ".join(restricciones)).fvalue)
    phi = res.params["y_1"]
    ect_p = res.pvalues["y_1"]
    if F > VC_I1 and phi < 0 and ect_p < 0.10:
        veredicto = "Cointegracion"
    elif F < VC_I0:
        veredicto = "Sin cointegracion"
    elif phi >= 0:
        veredicto = "F alto, ECT invalido"
    else:
        veredicto = "No concluyente"
    vm = np.log(0.5) / np.log(1 + phi) if -2 < phi < 0 else np.nan
    print("  F = {:.3f}   valores criticos {:.2f} / {:.2f}".format(F, VC_I0, VC_I1))
    print("  ECT = {:.4f}{}  (p={:.4f})   vida media = {}".format(
        phi, estrellas(ect_p), ect_p,
        "{:.1f} anios".format(vm) if vm == vm else "no definida"))
    print("  Veredicto: {}".format(veredicto))
    coint.append({"Pais": nombre, "Modelo": "M10", "Orden": "{}, {}".format(p, list(ordenes)),
                  "F": round(F, 3), "Valores criticos": "{} / {}".format(VC_I0, VC_I1),
                  "ECT": round(phi, 4), "Significancia ECT": estrellas(ect_p),
                  "Vida media": round(vm, 1) if vm == vm else None,
                  "n": int(res.nobs), "Veredicto": veredicto})

    # ------------------------------------ largo plazo por metodo delta
    print("\n  Largo plazo")
    print("  {:<48} {:>10} {:>9} {:>7}".format("variable", "coef", "EE", "t"))
    cov = res.cov_params()
    for r in REGRESORES:
        th = res.params["{}_1".format(r)]
        b = -th / phi
        d = np.zeros(len(res.params))
        d[list(res.params.index).index("{}_1".format(r))] = -1.0 / phi
        d[list(res.params.index).index("y_1")] = th / (phi ** 2)
        ee = float(np.sqrt(d @ cov.values @ d))
        t = b / ee if ee > 0 else np.nan
        pv = 2 * (1 - sm.distributions.ECDF([0])(0)) if False else \
            2 * sm.stats.stattools.stats.t.sf(abs(t), res.df_resid)
        print("  {:<48} {:>10} {:>9.4f} {:>7.2f}".format(
            ETIQUETAS[r][:48], "{:.4f}{}".format(b, estrellas(pv)), ee, t))
        largo.append({"Pais": nombre, "Modelo": "M10", "Variable": ETIQUETAS[r],
                      "Coeficiente": round(b, 4), "Error estandar": round(ee, 4),
                      "t": round(t, 2), "Significancia": estrellas(pv),
                      "Valido": veredicto == "Cointegracion"})

    # ------------------------------------ corto plazo con Newey-West
    rhac = sm.OLS(datos["dy"], sm.add_constant(datos.drop(columns="dy"))).fit(
        cov_type="HAC", cov_kwds={"maxlags": HAC})
    rho = sum(rhac.params.get("dy_{}".format(i), 0.0) for i in range(1, p))
    print("\n  Corto plazo (Newey-West, {} rezagos)".format(HAC))
    for r, q in zip(REGRESORES, ordenes):
        suma = 0.0
        for i in range(0, q + 1):
            nom = "d{}_{}".format(r, i)
            if nom not in rhac.params:
                continue
            b, pv = rhac.params[nom], rhac.pvalues[nom]
            suma += b
            print("    {:<44} rezago {}  {:>10}  (t={:.2f})".format(
                ETIQUETAS[r][:44], i, "{:.4f}{}".format(b, estrellas(pv)),
                rhac.tvalues[nom]))
        mult = suma / (1 - rho) if abs(1 - rho) > 1e-8 else np.nan
        print("    {:<44} multiplicador acumulado: {:.4f}".format("", mult))
        corto.append({"Pais": nombre, "Modelo": "M10", "Variable": ETIQUETAS[r],
                      "Suma de coeficientes": round(suma, 4),
                      "Multiplicador": round(mult, 4) if mult == mult else None,
                      "R2 ajustado": round(rhac.rsquared_adj, 3), "n": int(rhac.nobs)})

    # ------------------------------------ diagnosticos
    e = res.resid
    Xd = sm.add_constant(datos.drop(columns="dy"))
    bg = acorr_breusch_godfrey(res, nlags=2)[3]
    lb = float(acorr_ljungbox(e, lags=[2], return_df=True)["lb_pvalue"].iloc[0])
    bp = het_breuschpagan(e, Xd)[3]
    jb = jarque_bera(e)[1]
    dw = durbin_watson(e)
    print("\n  Diagnosticos: BG={:.3f}  LB={:.3f}  BP={:.3f}  JB={:.3f}  DW={:.2f}".format(
        bg, lb, bp, jb, dw))
    diag.append({"Pais": nombre, "Modelo": "M10", "n": int(res.nobs),
                 "gl": int(res.df_resid), "Breusch-Godfrey": round(bg, 3),
                 "Ljung-Box": round(lb, 3), "Breusch-Pagan": round(bp, 3),
                 "Jarque-Bera": round(jb, 3), "Durbin-Watson": round(dw, 2),
                 "R2 ajustado": round(res.rsquared_adj, 3)})

for nombre, datos_out in (("cuadro12_m10_cointegracion.csv", coint),
                          ("cuadro12_m10_largo_plazo.csv", largo),
                          ("cuadro12_m10_corto_plazo.csv", corto),
                          ("cuadro12_m10_diagnosticos.csv", diag)):
    pd.DataFrame(datos_out).to_csv(os.path.join(SAL, nombre), index=False)
    print("\nGuardado: {}".format(os.path.join(SAL, nombre)))

DESCARGA DE LA DEUDA PRIVADA (PVD_LS) DESDE EL FONDO MONETARIO INTERNACIONAL
  CN: n=34 (1990-2023)  media=127.42  min=74.57  max=199.23
  US: n=34 (1990-2023)  media=145.79  min=118.85  max=170.67

Panel derivado guardado: /content/drive/MyDrive/tesis_china_eeuu/data_clean/china_us_panel_analisis_v6_1990_2023.csv
  dimensiones 68 x 148 (la v5 y el maestro quedan intactos)

China  M10  observaciones disponibles: 34 (1990-2023)
  Ordenes seleccionados: p=3  q=[2, 3]  AIC=-187.958  n=30  gl=16
  F = 5.372   valores criticos 3.23 / 4.32
  ECT = -0.0737**  (p=0.0309)   vida media = 9.1 anios
  Veredicto: Cointegracion

  Largo plazo
  variable                                               coef        EE       t
  Deuda privada, prestamos y titulos (% del PIB)    0.0138***    0.0015    9.27
  Apertura comercial (% del PIB)                    0.0343***    0.0062    5.57

  Corto plazo (Newey-West, 2 rezagos)
    Deuda privada, prestamos y titulos (% del PI rezago 0     -0.0005  (t=-1.60)
   

In [33]:
# -*- coding: utf-8 -*-
"""
Estimacion por minimos cuadrados dinamicos (Stock y Watson, 1993) como
validacion alternativa de los coeficientes de largo plazo.
"""

import os
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats

PANEL = "/content/drive/MyDrive/tesis_china_eeuu/data_clean/china_us_panel_analisis_v6_1990_2023.csv"
SAL = "/content/drive/MyDrive/tesis_china_eeuu/outputs"
DEP = "ln_gdp_pc_pwt"
P_ADELANTOS = 1
HAC = 2

df = pd.read_csv(PANEL)

COL = {
    "fbcf":      "inv_gfcf_gdp",
    "apertura":  "trade_gdp",
    "inflacion": "inflation_cpi_pct",
    "credito":   "credit_priv_gdp",
    "credbis":   "bis_pvt_credit_gdp",
    "deuda":     "pub_debt_gdp",
    "razon":     "ratio_ind_agr_rec",
    "iic":       "IIC",
    "deudapriv": "priv_debt_ls_gdp",
}

ETIQUETAS = {
    "fbcf": "Formacion bruta de capital fijo (% del PIB)",
    "apertura": "Apertura comercial (% del PIB)",
    "inflacion": "Inflacion (% anual)",
    "credito": "Credito al sector privado (% del PIB)",
    "credbis": "Credito al sector privado, BIS (% del PIB)",
    "deuda": "Deuda publica (% del PIB)",
    "razon": "Razon de productividad industria/agricultura",
    "iic": "Indice de intensidad de la acumulacion",
    "deudapriv": "Deuda privada, prestamos y titulos (% del PIB)",
}

MODELOS = [
    ("CN", "China", "M1",  ["fbcf", "apertura", "inflacion"]),
    ("CN", "China", "M3",  ["credito", "fbcf", "apertura"]),
    ("CN", "China", "M4",  ["credbis", "fbcf", "apertura"]),
    ("CN", "China", "M5b", ["deuda", "apertura"]),
    ("CN", "China", "M8",  ["razon", "fbcf", "apertura"]),
    ("CN", "China", "M9",  ["iic", "apertura"]),
    ("CN", "China", "M10", ["deudapriv", "apertura"]),
    ("US", "Estados Unidos", "M10", ["deudapriv", "apertura"]),
]

ARDL = {
    ("China", "M1", "fbcf"): -0.0616, ("China", "M1", "apertura"): 0.1289,
    ("China", "M1", "inflacion"): 0.1984,
    ("China", "M3", "credito"): 0.0110, ("China", "M3", "fbcf"): 0.0437,
    ("China", "M3", "apertura"): 0.0218,
    ("China", "M4", "credbis"): 0.0075, ("China", "M4", "fbcf"): 0.0328,
    ("China", "M4", "apertura"): 0.0259,
    ("China", "M5b", "deuda"): -0.0049, ("China", "M5b", "apertura"): 0.1043,
    ("China", "M8", "razon"): -0.3305, ("China", "M8", "fbcf"): 0.0401,
    ("China", "M8", "apertura"): 0.0586,
    ("China", "M9", "iic"): 0.1473, ("China", "M9", "apertura"): 0.1182,
}


def estrellas(p):
    return "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.10 else ""


faltan = [c for c in COL.values() if c not in df.columns]
if faltan:
    print("Columnas ausentes en el panel: {}".format(", ".join(faltan)))
    print("\nCatalogo completo:")
    for c in sorted(df.columns):
        print("   ", c)
    raise SystemExit("Ejecuta primero el script A para generar el panel v6.")

filas = []
print("=" * 84)
print("MINIMOS CUADRADOS DINAMICOS (STOCK Y WATSON, 1993)")
print("Adelantos y rezagos: {} | Newey-West: {} rezagos".format(P_ADELANTOS, HAC))
print("=" * 84)

for pais, nombre, modelo, claves in MODELOS:
    g = df[df["country"] == pais].sort_values("year").reset_index(drop=True)
    regs = [COL[k] for k in claves]
    base = g[["year", DEP] + regs].dropna().reset_index(drop=True)

    X = pd.DataFrame(index=base.index)
    for k, c in zip(claves, regs):
        X[k] = base[c]
    for k, c in zip(claves, regs):
        d = base[c].diff()
        for j in range(-P_ADELANTOS, P_ADELANTOS + 1):
            X["d_{}_{:+d}".format(k, j)] = d.shift(-j)

    datos = pd.concat([base[DEP].rename("y"), X], axis=1).dropna()
    Xv = sm.add_constant(datos.drop(columns="y"))
    if len(datos) - Xv.shape[1] < 5:
        print("\n{} {}: omitido, {} observaciones para {} parametros".format(
            nombre, modelo, len(datos), Xv.shape[1]))
        continue

    res = sm.OLS(datos["y"], Xv).fit(cov_type="HAC", cov_kwds={"maxlags": HAC})
    anios = base.loc[datos.index, "year"]
    print("\n{}  {}  n={}  gl={}  R2aj={:.3f}  ({}-{})".format(
        nombre, modelo, int(res.nobs), int(res.df_resid), res.rsquared_adj,
        int(anios.min()), int(anios.max())))
    print("  {:<48} {:>11} {:>9} {:>7} {:>10}".format(
        "variable", "DOLS", "EE", "t", "ARDL"))
    for k in claves:
        b, ee, t, pv = res.params[k], res.bse[k], res.tvalues[k], res.pvalues[k]
        ref = ARDL.get((nombre, modelo, k), np.nan)
        print("  {:<48} {:>11} {:>9.4f} {:>7.2f} {:>10}".format(
            ETIQUETAS[k][:48], "{:.4f}{}".format(b, estrellas(pv)), ee, t,
            "{:.4f}".format(ref) if ref == ref else "-"))
        filas.append({
            "Pais": nombre, "Modelo": modelo, "Variable": ETIQUETAS[k],
            "Coeficiente DOLS": round(b, 4), "Error estandar": round(ee, 4),
            "t": round(t, 2), "Significancia": estrellas(pv),
            "Coeficiente ARDL": round(ref, 4) if ref == ref else None,
            "Diferencia": round(b - ref, 4) if ref == ref else None,
            "Mismo signo": bool(b * ref > 0) if ref == ref else None,
            "n": int(res.nobs), "gl": int(res.df_resid),
            "Periodo": "{}-{}".format(int(anios.min()), int(anios.max())),
        })

out = pd.DataFrame(filas)
ruta = os.path.join(SAL, "cuadro11_dols.csv")
out.to_csv(ruta, index=False)
print("\n" + "=" * 84)
print("Guardado: {}  ({} filas)".format(ruta, len(out)))

comparables = out[out["Coeficiente ARDL"].notna()]
if len(comparables):
    print("\nConcordancia de signo con el cuadro 8: {} de {} coeficientes".format(
        int(comparables["Mismo signo"].sum()), len(comparables)))

MINIMOS CUADRADOS DINAMICOS (STOCK Y WATSON, 1993)
Adelantos y rezagos: 1 | Newey-West: 2 rezagos

China  M1  n=31  gl=18  R2aj=0.926  (1992-2022)
  variable                                                DOLS        EE       t       ARDL
  Formacion bruta de capital fijo (% del PIB)        0.1533***    0.0179    8.56    -0.0616
  Apertura comercial (% del PIB)                    -0.0174***    0.0054   -3.19     0.1289
  Inflacion (% anual)                               -0.0311***    0.0080   -3.88     0.1984

China  M3  n=31  gl=18  R2aj=0.991  (1992-2022)
  variable                                                DOLS        EE       t       ARDL
  Credito al sector privado (% del PIB)              0.0173***    0.0009   18.78     0.0110
  Formacion bruta de capital fijo (% del PIB)        0.0568***    0.0064    8.90     0.0437
  Apertura comercial (% del PIB)                     0.0092***    0.0021    4.36     0.0218

China  M4  n=29  gl=16  R2aj=0.989  (1992-2020)
  variable         

In [34]:
# -*- coding: utf-8 -*-
"""
Sensibilidad de los minimos cuadrados dinamicos al numero de adelantos y
rezagos y al ancho de banda de la matriz de covarianzas robusta.
Andrews (1991), regla de conexion autorregresiva de primer orden,
nucleo de Bartlett.
"""

import os
import numpy as np
import pandas as pd
import statsmodels.api as sm

PANEL = "/content/drive/MyDrive/tesis_china_eeuu/data_clean/china_us_panel_analisis_v6_1990_2023.csv"
SAL = "/content/drive/MyDrive/tesis_china_eeuu/outputs"
DEP = "ln_gdp_pc_pwt"
os.makedirs(SAL, exist_ok=True)

df = pd.read_csv(PANEL)

COL = {
    "fbcf":      "inv_gfcf_gdp",
    "apertura":  "trade_gdp",
    "inflacion": "inflation_cpi_pct",
    "credito":   "credit_priv_gdp",
    "credbis":   "bis_pvt_credit_gdp",
    "deuda":     "pub_debt_gdp",
    "razon":     "ratio_ind_agr_rec",
    "iic":       "IIC",
    "deudapriv": "priv_debt_ls_gdp",
}

ETIQUETAS = {
    "fbcf": "Formacion bruta de capital fijo",
    "apertura": "Apertura comercial",
    "inflacion": "Inflacion",
    "credito": "Credito al sector privado (BM)",
    "credbis": "Credito al sector privado (BIS)",
    "deuda": "Deuda publica",
    "razon": "Razon industria/agricultura",
    "iic": "Indice de intensidad de la acumulacion",
    "deudapriv": "Deuda privada (FMI)",
}

MODELOS = [
    ("CN", "China", "M1",  ["fbcf", "apertura", "inflacion"]),
    ("CN", "China", "M3",  ["credito", "fbcf", "apertura"]),
    ("CN", "China", "M4",  ["credbis", "fbcf", "apertura"]),
    ("CN", "China", "M5b", ["deuda", "apertura"]),
    ("CN", "China", "M8",  ["razon", "fbcf", "apertura"]),
    ("CN", "China", "M9",  ["iic", "apertura"]),
    ("CN", "China", "M10", ["deudapriv", "apertura"]),
    ("US", "Estados Unidos", "M10", ["deudapriv", "apertura"]),
]


def ancho_andrews(X, e):
    """Ancho de banda de Andrews (1991) con nucleo de Bartlett,
    obtenido por conexion desde ajustes autorregresivos de orden uno
    sobre las condiciones de momento h_t = x_t * e_t."""
    h = X * e[:, None]
    T, k = h.shape
    num, den = 0.0, 0.0
    for j in range(k):
        v = h[:, j]
        v = v - v.mean()
        if np.allclose(v, 0):
            continue
        v0, v1 = v[:-1], v[1:]
        denom = float(v0 @ v0)
        if denom <= 0:
            continue
        rho = float(v0 @ v1) / denom
        rho = np.clip(rho, -0.97, 0.97)
        s2 = float(np.mean((v1 - rho * v0) ** 2))
        num += 4 * (rho ** 2) * (s2 ** 2) / ((1 - rho) ** 6 * (1 + rho) ** 2)
        den += (s2 ** 2) / ((1 - rho) ** 4)
    if den <= 0 or num <= 0:
        return 2, np.nan
    alfa = num / den
    banda = 1.1447 * (alfa * T) ** (1.0 / 3.0)
    m = int(max(0, min(np.floor(banda), T - 2)))
    return m, banda


def estrellas(p):
    return "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.10 else ""


def armar(base, claves, regs, adelantos):
    X = pd.DataFrame(index=base.index)
    for k, c in zip(claves, regs):
        X[k] = base[c]
    for k, c in zip(claves, regs):
        d = base[c].diff()
        for j in range(-adelantos, adelantos + 1):
            X["d_{}_{:+d}".format(k, j)] = d.shift(-j)
    datos = pd.concat([base[DEP].rename("y"), X], axis=1).dropna()
    return datos


filas = []
print("=" * 100)
print("SENSIBILIDAD DE LOS MINIMOS CUADRADOS DINAMICOS")
print("Configuraciones: adelantos y rezagos p = 1 y 2; ancho de banda fijo = 2 y regla de Andrews")
print("=" * 100)

for pais, nombre, modelo, claves in MODELOS:
    g = df[df["country"] == pais].sort_values("year").reset_index(drop=True)
    regs = [COL[k] for k in claves]
    base = g[["year", DEP] + regs].dropna().reset_index(drop=True)

    print("\n" + "-" * 100)
    print("{}  {}".format(nombre, modelo))

    resultados = {}
    for adelantos in (1, 2):
        datos = armar(base, claves, regs, adelantos)
        Xv = sm.add_constant(datos.drop(columns="y"))
        gl = len(datos) - Xv.shape[1]
        if gl < 4:
            print("  p={}: omitido, solo {} grados de libertad".format(adelantos, gl))
            continue
        mco = sm.OLS(datos["y"], Xv).fit()
        m_and, banda = ancho_andrews(Xv.values, mco.resid.values)
        for etiqueta, m in (("fijo2", 2), ("andrews", m_and)):
            res = sm.OLS(datos["y"], Xv).fit(
                cov_type="HAC", cov_kwds={"maxlags": m, "use_correction": True})
            resultados[(adelantos, etiqueta)] = (res, m, gl, banda, len(datos))
        print("  p={}: n={}  gl={}  ancho de Andrews = {:.2f} -> {} rezagos".format(
            adelantos, len(datos), gl, banda, m_and))

    if not resultados:
        continue

    print("\n  {:<34} {:>12} {:>22} {:>22}".format(
        "variable", "coeficiente", "p=1  EE (fijo/Andrews)", "p=2  EE (fijo/Andrews)"))
    for k in claves:
        b1 = resultados[(1, "fijo2")][0].params[k] if (1, "fijo2") in resultados else np.nan
        b2 = resultados[(2, "fijo2")][0].params[k] if (2, "fijo2") in resultados else np.nan
        def ee(cfg):
            return resultados[cfg][0].bse[k] if cfg in resultados else np.nan
        def tt(cfg):
            return resultados[cfg][0].tvalues[k] if cfg in resultados else np.nan
        def pp(cfg):
            return resultados[cfg][0].pvalues[k] if cfg in resultados else np.nan
        print("  {:<34} {:>12} {:>10.4f}/{:<10.4f} {:>10.4f}/{:<10.4f}".format(
            ETIQUETAS[k][:34],
            "{:.4f}".format(b1) if b1 == b1 else "-",
            ee((1, "fijo2")), ee((1, "andrews")),
            ee((2, "fijo2")), ee((2, "andrews"))))
        print("  {:<34} {:>12} {:>10} {:<10} {:>10} {:<10}".format(
            "", "t y signif.",
            "{:.2f}{}".format(tt((1, "fijo2")), estrellas(pp((1, "fijo2")))),
            "{:.2f}{}".format(tt((1, "andrews")), estrellas(pp((1, "andrews")))),
            "{:.2f}{}".format(tt((2, "fijo2")), estrellas(pp((2, "fijo2")))) if (2, "fijo2") in resultados else "-",
            "{:.2f}{}".format(tt((2, "andrews")), estrellas(pp((2, "andrews")))) if (2, "andrews") in resultados else "-"))

        for (adel, etq), (res, m, gl, banda, n) in resultados.items():
            filas.append({
                "Pais": nombre, "Modelo": modelo, "Variable": ETIQUETAS[k],
                "Adelantos y rezagos": adel,
                "Ancho de banda": "Fijo en 2" if etq == "fijo2" else "Andrews",
                "Rezagos empleados": m,
                "Ancho de Andrews": round(banda, 2) if banda == banda else None,
                "Coeficiente": round(res.params[k], 4),
                "Error estandar": round(res.bse[k], 4),
                "t": round(res.tvalues[k], 2),
                "Significancia": estrellas(res.pvalues[k]),
                "n": n, "gl": gl,
                "R2 ajustado": round(res.rsquared_adj, 3),
            })

out = pd.DataFrame(filas)
ruta = os.path.join(SAL, "cuadro11b_dols_sensibilidad.csv")
out.to_csv(ruta, index=False)
print("\n" + "=" * 100)
print("Guardado: {}  ({} filas)".format(ruta, len(out)))

# ------------------------------------------- resumen de estabilidad
print("\nESTABILIDAD DEL SIGNO Y DE LA SIGNIFICANCIA ENTRE LAS CUATRO CONFIGURACIONES")
print("-" * 100)
for (pais, modelo, var), sub in out.groupby(["Pais", "Modelo", "Variable"], sort=False):
    signos = set(np.sign(sub["Coeficiente"]))
    signif = set(s != "" for s in sub["Significancia"])
    razon = sub["Error estandar"].max() / max(sub["Error estandar"].min(), 1e-12)
    estado = ("estable" if len(signos) == 1 and len(signif) == 1
              else "sensible a la configuracion")
    print("  {:<16} {:<5} {:<34} razon EE max/min = {:>6.2f}   {}".format(
        pais, modelo, var[:34], razon, estado))

SENSIBILIDAD DE LOS MINIMOS CUADRADOS DINAMICOS
Configuraciones: adelantos y rezagos p = 1 y 2; ancho de banda fijo = 2 y regla de Andrews

----------------------------------------------------------------------------------------------------
China  M1
  p=1: n=31  gl=18  ancho de Andrews = 11.16 -> 11 rezagos
  p=2: n=29  gl=10  ancho de Andrews = 10.13 -> 10 rezagos

  variable                            coeficiente p=1  EE (fijo/Andrews) p=2  EE (fijo/Andrews)
  Formacion bruta de capital fijo          0.1533     0.0235/0.0264         0.0393/0.0373    
                                      t y signif.    6.53*** 5.80***       4.04*** 4.25***   
  Apertura comercial                      -0.0174     0.0071/0.0090         0.0188/0.0210    
                                      t y signif.    -2.43** -1.94*          -1.26 -1.13     
  Inflacion                               -0.0311     0.0105/0.0116         0.0281/0.0183    
                                      t y signif.   -2.96*** -2.

Paso 4. Prueba directa de la hipótesis mediadora
El script estima cuatro cosas. Primero, si la formación bruta de capital fijo sostiene por sí sola una relación de nivel con el producto por habitante chino. Segundo, si la sostiene acompañada únicamente de apertura, sin ninguna variable financiera. Tercero, la regresión auxiliar que constituye la prueba propiamente mediadora: si la apertura y el crédito determinan el nivel de largo plazo de la propia inversión. Y cuarto, la misma primera especificación para Estados Unidos, por simetría. Cada relación se estima con las dos técnicas.

In [35]:
# -*- coding: utf-8 -*-
"""
Paso 4. Prueba de la hipotesis mediadora.
M11  producto por habitante ~ formacion bruta de capital fijo
M12  producto por habitante ~ formacion bruta de capital fijo + apertura
MA   formacion bruta de capital fijo ~ apertura + credito al sector privado
Estimacion por modelo de correccion de error irrestricto y por minimos
cuadrados dinamicos con ancho de banda de Andrews (1991).
"""

import os
import itertools
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats
from statsmodels.stats.diagnostic import (acorr_breusch_godfrey, acorr_ljungbox,
                                          het_breuschpagan)
from statsmodels.stats.stattools import durbin_watson, jarque_bera

PANEL = "/content/drive/MyDrive/tesis_china_eeuu/data_clean/china_us_panel_analisis_v6_1990_2023.csv"
SAL = "/content/drive/MyDrive/tesis_china_eeuu/outputs"
os.makedirs(SAL, exist_ok=True)

MAX_ORDEN, MIN_ORDEN, HAC_FIJO = 3, 1, 2

# Pesaran, Shin y Smith (2001), tabla CI(iii), caso III, nivel del 5%.
# Se reportan la lectura estricta por numero de regresores y la empleada
# hasta ahora en el articulo, desplazada una fila.
VC_ESTRICTOS = {1: (4.94, 5.73), 2: (3.79, 4.85), 3: (3.23, 4.35)}
VC_ARTICULO  = {1: (3.79, 4.85), 2: (3.23, 4.32), 3: (2.88, 4.01)}

ETIQUETAS = {
    "ln_gdp_pc_pwt": "PIB per capita real (log)",
    "inv_gfcf_gdp": "Formacion bruta de capital fijo (% del PIB)",
    "trade_gdp": "Apertura comercial (% del PIB)",
    "credit_priv_gdp": "Credito al sector privado (% del PIB)",
}

ESPECIFICACIONES = [
    ("CN", "China", "M11", "ln_gdp_pc_pwt", ["inv_gfcf_gdp"]),
    ("CN", "China", "M12", "ln_gdp_pc_pwt", ["inv_gfcf_gdp", "trade_gdp"]),
    ("CN", "China", "MA",  "inv_gfcf_gdp",  ["trade_gdp", "credit_priv_gdp"]),
    ("US", "Estados Unidos", "M11", "ln_gdp_pc_pwt", ["inv_gfcf_gdp"]),
]

df = pd.read_csv(PANEL)
if "d2020" not in df.columns:
    df["d2020"] = (df["year"] == 2020).astype(int)


def estrellas(p):
    return "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.10 else ""


def ancho_andrews(X, e):
    """Ancho de banda de Andrews (1991), nucleo de Bartlett."""
    h = X * e[:, None]
    T, k = h.shape
    num = den = 0.0
    for j in range(k):
        v = h[:, j] - h[:, j].mean()
        if np.allclose(v, 0):
            continue
        v0, v1 = v[:-1], v[1:]
        if float(v0 @ v0) <= 0:
            continue
        rho = np.clip(float(v0 @ v1) / float(v0 @ v0), -0.97, 0.97)
        s2 = float(np.mean((v1 - rho * v0) ** 2))
        num += 4 * rho ** 2 * s2 ** 2 / ((1 - rho) ** 6 * (1 + rho) ** 2)
        den += s2 ** 2 / (1 - rho) ** 4
    if den <= 0 or num <= 0:
        return HAC_FIJO, np.nan
    banda = 1.1447 * (num / den * T) ** (1 / 3)
    return int(max(0, min(np.floor(banda), T - 2))), banda


def construir(base, dep, regs, p, ordenes):
    X = pd.DataFrame(index=base.index)
    X["y_1"] = base[dep].shift(1)
    for r in regs:
        X["{}_1".format(r)] = base[r].shift(1)
    for i in range(1, p):
        X["dy_{}".format(i)] = base[dep].diff().shift(i)
    for r, q in zip(regs, ordenes):
        for i in range(0, q + 1):
            X["d{}_{}".format(r, i)] = base[r].diff().shift(i)
    X["d2020"] = base["d2020"]
    return pd.concat([base[dep].diff().rename("dy"), X], axis=1).dropna()


coint, largo, diag, dols = [], [], [], []

for pais, nombre, modelo, dep, regs in ESPECIFICACIONES:
    g = df[df["country"] == pais].sort_values("year").reset_index(drop=True)
    base = g[["year", dep] + regs + ["d2020"]].dropna().reset_index(drop=True)
    k = len(regs)
    ci_e, cs_e = VC_ESTRICTOS[k]
    ci_a, cs_a = VC_ARTICULO[k]

    print("\n" + "=" * 92)
    print("{}  {}   dependiente: {}".format(nombre, modelo, ETIQUETAS[dep]))
    print("   regresores: {}".format(", ".join(ETIQUETAS[r] for r in regs)))
    print("   observaciones: {} ({}-{})".format(
        len(base), int(base["year"].min()), int(base["year"].max())))

    # -------------------------------------------- modelo de correccion de error
    mejor = None
    for p in range(MIN_ORDEN, MAX_ORDEN + 1):
        for ordenes in itertools.product(range(MIN_ORDEN, MAX_ORDEN + 1), repeat=k):
            datos = construir(base, dep, regs, p, ordenes)
            if len(datos) - datos.shape[1] < 8:
                continue
            res = sm.OLS(datos["dy"], sm.add_constant(datos.drop(columns="dy"))).fit()
            if mejor is None or res.aic < mejor[0]:
                mejor = (res.aic, p, ordenes, datos, res)

    if mejor is None:
        print("   Sin grados de libertad suficientes.")
        continue
    aic, p, ordenes, datos, res = mejor
    F = float(res.f_test(", ".join(["y_1 = 0"] + ["{}_1 = 0".format(r) for r in regs])).fvalue)
    phi, pect = res.params["y_1"], res.pvalues["y_1"]

    def veredicto(ci, cs):
        if F > cs and phi < 0 and pect < 0.10:
            return "Cointegracion"
        if F < ci:
            return "Sin cointegracion"
        if phi >= 0:
            return "F alto, ECT invalido"
        return "No concluyente"

    ve, va = veredicto(ci_e, cs_e), veredicto(ci_a, cs_a)
    vm = np.log(0.5) / np.log(1 + phi) if -2 < phi < 0 else np.nan
    print("\n   Ordenes p={} q={}  AIC={:.3f}  n={}  gl={}".format(
        p, list(ordenes), aic, int(res.nobs), int(res.df_resid)))
    print("   F = {:.3f}".format(F))
    print("     criterio estricto  {:.2f} / {:.2f}  ->  {}".format(ci_e, cs_e, ve))
    print("     criterio empleado  {:.2f} / {:.2f}  ->  {}".format(ci_a, cs_a, va))
    print("   ECT = {:.4f}{} (p={:.4f})   vida media = {}".format(
        phi, estrellas(pect), pect,
        "{:.1f} anios".format(vm) if vm == vm else "no definida"))

    coint.append({"Pais": nombre, "Modelo": modelo, "Dependiente": ETIQUETAS[dep],
                  "Orden": "{}, {}".format(p, list(ordenes)), "F": round(F, 3),
                  "VC estrictos": "{} / {}".format(ci_e, cs_e),
                  "Veredicto estricto": ve,
                  "VC del articulo": "{} / {}".format(ci_a, cs_a),
                  "Veredicto del articulo": va,
                  "ECT": round(phi, 4), "Signif. ECT": estrellas(pect),
                  "Vida media": round(vm, 1) if vm == vm else None,
                  "n": int(res.nobs), "gl": int(res.df_resid)})

    cov = res.cov_params()
    print("\n   Largo plazo (metodo delta)")
    for r in regs:
        th = res.params["{}_1".format(r)]
        b = -th / phi
        d = np.zeros(len(res.params))
        d[list(res.params.index).index("{}_1".format(r))] = -1 / phi
        d[list(res.params.index).index("y_1")] = th / phi ** 2
        ee = float(np.sqrt(d @ cov.values @ d))
        t = b / ee if ee > 0 else np.nan
        pv = 2 * stats.t.sf(abs(t), res.df_resid)
        print("     {:<46} {:>12} EE={:.4f} t={:.2f}".format(
            ETIQUETAS[r][:46], "{:.4f}{}".format(b, estrellas(pv)), ee, t))
        largo.append({"Pais": nombre, "Modelo": modelo, "Variable": ETIQUETAS[r],
                      "Coeficiente": round(b, 4), "Error estandar": round(ee, 4),
                      "t": round(t, 2), "Significancia": estrellas(pv),
                      "Interpretable": ve == "Cointegracion"})

    e = res.resid
    bg = acorr_breusch_godfrey(res, nlags=2)[3]
    lb = float(acorr_ljungbox(e, lags=[2], return_df=True)["lb_pvalue"].iloc[0])
    bp = het_breuschpagan(e, sm.add_constant(datos.drop(columns="dy")))[3]
    jb = jarque_bera(e)[1]
    dw = durbin_watson(e)
    print("   Diagnosticos: BG={:.3f} LB={:.3f} BP={:.3f} JB={:.3f} DW={:.2f}".format(
        bg, lb, bp, jb, dw))
    diag.append({"Pais": nombre, "Modelo": modelo, "n": int(res.nobs),
                 "gl": int(res.df_resid), "Breusch-Godfrey": round(bg, 3),
                 "Ljung-Box": round(lb, 3), "Breusch-Pagan": round(bp, 3),
                 "Jarque-Bera": round(jb, 3), "Durbin-Watson": round(dw, 2),
                 "R2 ajustado": round(res.rsquared_adj, 3)})

    # -------------------------------------------- minimos cuadrados dinamicos
    X = pd.DataFrame(index=base.index)
    for r in regs:
        X[r] = base[r]
    for r in regs:
        d = base[r].diff()
        for j in (-1, 0, 1):
            X["d_{}_{:+d}".format(r, j)] = d.shift(-j)
    dd = pd.concat([base[dep].rename("y"), X], axis=1).dropna()
    Xv = sm.add_constant(dd.drop(columns="y"))
    gl = len(dd) - Xv.shape[1]
    if gl >= 4:
        mco = sm.OLS(dd["y"], Xv).fit()
        m, banda = ancho_andrews(Xv.values, mco.resid.values)
        rd = sm.OLS(dd["y"], Xv).fit(cov_type="HAC",
                                     cov_kwds={"maxlags": m, "use_correction": True})
        print("\n   Minimos cuadrados dinamicos  n={}  gl={}  R2aj={:.3f}".format(
            int(rd.nobs), gl, rd.rsquared_adj))
        print("     ancho de Andrews = {:.2f} -> {} rezagos ({:.0%} de la muestra)".format(
            banda, m, m / len(dd)))
        for r in regs:
            print("     {:<46} {:>12} EE={:.4f} t={:.2f}".format(
                ETIQUETAS[r][:46],
                "{:.4f}{}".format(rd.params[r], estrellas(rd.pvalues[r])),
                rd.bse[r], rd.tvalues[r]))
            dols.append({"Pais": nombre, "Modelo": modelo, "Variable": ETIQUETAS[r],
                         "Coeficiente DOLS": round(rd.params[r], 4),
                         "Error estandar": round(rd.bse[r], 4),
                         "t": round(rd.tvalues[r], 2),
                         "Significancia": estrellas(rd.pvalues[r]),
                         "Ancho de Andrews": round(banda, 2),
                         "Rezagos": m, "Proporcion de la muestra": round(m / len(dd), 3),
                         "n": int(rd.nobs), "gl": gl,
                         "R2 ajustado": round(rd.rsquared_adj, 3)})
    else:
        print("\n   Minimos cuadrados dinamicos omitidos: solo {} grados de libertad".format(gl))

for nombre_archivo, contenido in (
        ("cuadro13_mediacion_cointegracion.csv", coint),
        ("cuadro13_mediacion_largo_plazo.csv", largo),
        ("cuadro13_mediacion_diagnosticos.csv", diag),
        ("cuadro13_mediacion_dols.csv", dols)):
    pd.DataFrame(contenido).to_csv(os.path.join(SAL, nombre_archivo), index=False)
    print("\nGuardado: {}".format(os.path.join(SAL, nombre_archivo)))


China  M11   dependiente: PIB per capita real (log)
   regresores: Formacion bruta de capital fijo (% del PIB)
   observaciones: 34 (1990-2023)

   Ordenes p=1 q=[1]  AIC=-168.601  n=32  gl=26
   F = 10.997
     criterio estricto  4.94 / 5.73  ->  Cointegracion
     criterio empleado  3.79 / 4.85  ->  Cointegracion
   ECT = -0.0374*** (p=0.0008)   vida media = 18.2 anios

   Largo plazo (metodo delta)
     Formacion bruta de capital fijo (% del PIB)       0.0988*** EE=0.0220 t=4.49
   Diagnosticos: BG=0.093 LB=0.033 BP=0.937 JB=0.000 DW=1.13

   Minimos cuadrados dinamicos  n=31  gl=26  R2aj=0.866
     ancho de Andrews = 18.16 -> 18 rezagos (58% de la muestra)
     Formacion bruta de capital fijo (% del PIB)       0.1521*** EE=0.0152 t=10.03

China  M12   dependiente: PIB per capita real (log)
   regresores: Formacion bruta de capital fijo (% del PIB), Apertura comercial (% del PIB)
   observaciones: 34 (1990-2023)

   Ordenes p=3 q=[1, 3]  AIC=-180.490  n=30  gl=17
   F = 3.729
     

Vamos con Johansen. El script estima el sistema chino principal y tres sistemas de contraste, selecciona el orden del vector autorregresivo por criterios de información, aplica las pruebas de traza y de máximo valor propio con la corrección de muestra pequeña de Reinsel y Ahn, y cuando encuentra al menos un vector estima el modelo vectorial de corrección de error para reportar los coeficientes de largo plazo normalizados, las velocidades de ajuste de cada ecuación y los diagnósticos del sistema.

Las velocidades de ajuste son la pieza clave para tu argumento: si la ecuación del producto tiene ajuste significativo y las de la inversión también, tienes confirmación de que la inversión es endógena dentro del sistema y no un regresor exógeno.



In [36]:
# -*- coding: utf-8 -*-
"""
Paso 4b. Procedimiento de Johansen (1988, 1991) sobre los sistemas chinos
y estadounidense, para determinar el numero de vectores de cointegracion
y contrastar el supuesto de vector unico del enfoque uniecuacional.
"""

import os
import warnings
import numpy as np
import pandas as pd
from statsmodels.tsa.vector_ar.vecm import (coint_johansen, select_order, VECM)

warnings.filterwarnings("ignore")

PANEL = "/content/drive/MyDrive/tesis_china_eeuu/data_clean/china_us_panel_analisis_v6_1990_2023.csv"
SAL = "/content/drive/MyDrive/tesis_china_eeuu/outputs"
os.makedirs(SAL, exist_ok=True)

MAX_ORDEN = 3          # orden maximo del vector autorregresivo en niveles
DETERMINISTA = "ci"    # constante restringida al espacio de cointegracion

ETIQUETAS = {
    "ln_gdp_pc_pwt": "PIB per capita real (log)",
    "inv_gfcf_gdp": "Formacion bruta de capital fijo",
    "credit_priv_gdp": "Credito al sector privado (BM)",
    "bis_pvt_credit_gdp": "Credito al sector privado (BIS)",
    "priv_debt_ls_gdp": "Deuda privada (FMI)",
    "trade_gdp": "Apertura comercial",
}

SISTEMAS = [
    ("CN", "China", "S-A", "Sistema principal",
     ["ln_gdp_pc_pwt", "inv_gfcf_gdp", "credit_priv_gdp", "trade_gdp"]),
    ("CN", "China", "S-B", "Contraste con credito del BIS",
     ["ln_gdp_pc_pwt", "inv_gfcf_gdp", "bis_pvt_credit_gdp", "trade_gdp"]),
    ("CN", "China", "S-C", "Contraste con deuda privada del FMI",
     ["ln_gdp_pc_pwt", "inv_gfcf_gdp", "priv_debt_ls_gdp", "trade_gdp"]),
    ("US", "Estados Unidos", "S-A", "Sistema principal",
     ["ln_gdp_pc_pwt", "inv_gfcf_gdp", "credit_priv_gdp", "trade_gdp"]),
]

df = pd.read_csv(PANEL)
NIVELES = ["90%", "95%", "99%"]

rangos, vectores, ajustes, diagnos = [], [], [], []

for pais, nombre, clave, descripcion, variables in SISTEMAS:
    g = df[df["country"] == pais].sort_values("year").reset_index(drop=True)
    faltantes = [v for v in variables if v not in g.columns]
    if faltantes:
        print("\n{} {}: faltan columnas {}".format(nombre, clave, faltantes))
        continue
    base = g[["year"] + variables].dropna().reset_index(drop=True)
    Y = base[variables].astype(float)
    T, k = Y.shape

    print("\n" + "=" * 96)
    print("{}  {}  {}".format(nombre, clave, descripcion))
    print("   variables: {}".format(", ".join(ETIQUETAS[v] for v in variables)))
    print("   observaciones: {} ({}-{})".format(
        T, int(base["year"].min()), int(base["year"].max())))

    # ------------------------------------------- orden del sistema
    try:
        sel = select_order(Y, maxlags=MAX_ORDEN, deterministic=DETERMINISTA)
        ordenes = {"AIC": sel.aic, "BIC": sel.bic, "HQIC": sel.hqic, "FPE": sel.fpe}
        print("   orden sugerido  " + "  ".join(
            "{}={}".format(a, b) for a, b in ordenes.items()))
        p = max(1, int(sel.aic))
    except Exception as exc:
        print("   seleccion de orden no disponible ({}), se emplea p=2".format(exc))
        ordenes, p = {}, 2
    kdiff = max(1, p - 1)
    print("   orden empleado en niveles p={}  (rezagos en diferencias={})".format(p, kdiff))

    # ------------------------------------------- pruebas de rango
    jo = coint_johansen(Y.values, det_order=0, k_ar_diff=kdiff)
    factor = (T - k * p) / T          # correccion de Reinsel y Ahn (1992)
    print("\n   Prueba de la traza")
    print("   {:<10} {:>10} {:>10} {:>26} {:>12}".format(
        "hipotesis", "traza", "corregida", "valores criticos 90/95/99", "veredicto"))
    r_traza = 0
    for i in range(k):
        est, ajus = jo.lr1[i], jo.lr1[i] * factor
        cv = jo.cvt[i]
        rechaza = ajus > cv[1]
        if rechaza and r_traza == i:
            r_traza = i + 1
        print("   r <= {:<5} {:>10.3f} {:>10.3f} {:>10.2f} {:>7.2f} {:>7.2f} {:>12}".format(
            i, est, ajus, cv[0], cv[1], cv[2],
            "rechaza" if rechaza else "no rechaza"))
        rangos.append({"Pais": nombre, "Sistema": clave, "Prueba": "Traza",
                       "Hipotesis": "r <= {}".format(i),
                       "Estadistico": round(est, 3),
                       "Corregido": round(ajus, 3),
                       "VC 90%": cv[0], "VC 95%": cv[1], "VC 99%": cv[2],
                       "Rechaza al 5%": bool(rechaza)})

    print("\n   Prueba del maximo valor propio")
    r_max = 0
    for i in range(k):
        est, ajus = jo.lr2[i], jo.lr2[i] * factor
        cv = jo.cvm[i]
        rechaza = ajus > cv[1]
        if rechaza and r_max == i:
            r_max = i + 1
        print("   r = {:<6} {:>10.3f} {:>10.3f} {:>10.2f} {:>7.2f} {:>7.2f} {:>12}".format(
            i, est, ajus, cv[0], cv[1], cv[2],
            "rechaza" if rechaza else "no rechaza"))
        rangos.append({"Pais": nombre, "Sistema": clave, "Prueba": "Maximo valor propio",
                       "Hipotesis": "r = {}".format(i),
                       "Estadistico": round(est, 3),
                       "Corregido": round(ajus, 3),
                       "VC 90%": cv[0], "VC 95%": cv[1], "VC 99%": cv[2],
                       "Rechaza al 5%": bool(rechaza)})

    print("\n   Rango por traza: {}   |   rango por maximo valor propio: {}".format(
        r_traza, r_max))
    print("   Valores propios: " + ", ".join("{:.4f}".format(v) for v in jo.eig))
    r = min(r_traza, r_max) if min(r_traza, r_max) > 0 else max(r_traza, r_max)

    if r == 0:
        print("   Sin vectores de cointegracion: no se estima el modelo vectorial.")
        continue
    if r >= k:
        print("   Rango completo: el sistema seria estacionario en niveles, resultado")
        print("   incompatible con las pruebas de raiz unitaria; se acota a r={}.".format(k - 1))
        r = k - 1

    # ------------------------------------------- modelo vectorial
    print("\n   Modelo vectorial de correccion de error con r={}".format(r))
    mod = VECM(Y, k_ar_diff=kdiff, coint_rank=r, deterministic=DETERMINISTA)
    res = mod.fit()

    beta = np.asarray(res.beta)
    for j in range(r):
        pivote = beta[0, j]
        print("\n   Vector {} normalizado en el producto por habitante".format(j + 1))
        if abs(pivote) < 1e-10:
            print("     el producto no aparece en este vector, se omite la normalizacion")
            continue
        for i, v in enumerate(variables):
            if i == 0:
                continue
            coef = -beta[i, j] / pivote
            print("     {:<40} {:>10.4f}".format(ETIQUETAS[v], coef))
            vectores.append({"Pais": nombre, "Sistema": clave, "Vector": j + 1,
                             "Variable": ETIQUETAS[v],
                             "Coeficiente normalizado": round(coef, 4)})

    print("\n   Velocidades de ajuste por ecuacion")
    alfa = np.asarray(res.alpha)
    try:
        talfa = np.asarray(res.tvalues_alpha)
        palfa = np.asarray(res.pvalues_alpha)
    except Exception:
        talfa = palfa = np.full_like(alfa, np.nan, dtype=float)
    for i, v in enumerate(variables):
        for j in range(r):
            sig = ("***" if palfa[i, j] < 0.01 else "**" if palfa[i, j] < 0.05
                   else "*" if palfa[i, j] < 0.10 else "")
            print("     {:<40} vector {}  alfa={:>9.4f}{:<3} (t={:>6.2f})".format(
                ETIQUETAS[v], j + 1, alfa[i, j], sig, talfa[i, j]))
            ajustes.append({"Pais": nombre, "Sistema": clave, "Ecuacion": ETIQUETAS[v],
                            "Vector": j + 1, "Alfa": round(float(alfa[i, j]), 4),
                            "t": round(float(talfa[i, j]), 2), "Significancia": sig,
                            "Endogena": bool(palfa[i, j] < 0.10)})

    fila = {"Pais": nombre, "Sistema": clave, "n": T, "k": k, "p": p,
            "Rango traza": r_traza, "Rango maximo valor propio": r_max,
            "Rango empleado": r}
    try:
        nor = res.test_normality()
        fila["Normalidad (p)"] = round(float(nor.pvalue), 3)
        print("\n   Normalidad conjunta: p = {:.3f}".format(nor.pvalue))
    except Exception as exc:
        print("\n   Normalidad no disponible: {}".format(exc))
    try:
        bla = res.test_whiteness(nlags=max(kdiff + 2, 4))
        fila["Ausencia de autocorrelacion (p)"] = round(float(bla.pvalue), 3)
        print("   Ausencia de autocorrelacion del sistema: p = {:.3f}".format(bla.pvalue))
    except Exception as exc:
        print("   Prueba de autocorrelacion no disponible: {}".format(exc))
    diagnos.append(fila)

for archivo, contenido in (("cuadro14_johansen_rango.csv", rangos),
                           ("cuadro14_johansen_vectores.csv", vectores),
                           ("cuadro14_johansen_ajustes.csv", ajustes),
                           ("cuadro14_johansen_resumen.csv", diagnos)):
    pd.DataFrame(contenido).to_csv(os.path.join(SAL, archivo), index=False)
    print("\nGuardado: {}".format(os.path.join(SAL, archivo)))


China  S-A  Sistema principal
   variables: PIB per capita real (log), Formacion bruta de capital fijo, Credito al sector privado (BM), Apertura comercial
   observaciones: 34 (1990-2023)
   orden sugerido  AIC=2  BIC=0  HQIC=0  FPE=2
   orden empleado en niveles p=2  (rezagos en diferencias=1)

   Prueba de la traza
   hipotesis       traza  corregida  valores criticos 90/95/99    veredicto
   r <= 0         64.682     49.463      44.49   47.85   54.68      rechaza
   r <= 1         30.639     23.430      27.07   29.80   35.46   no rechaza
   r <= 2         10.561      8.076      13.43   15.49   19.93   no rechaza
   r <= 3          2.070      1.583       2.71    3.84    6.63   no rechaza

   Prueba del maximo valor propio
   r = 0          34.043     26.033      25.12   27.59   32.72   no rechaza
   r = 1          20.078     15.353      18.89   21.13   25.86   no rechaza
   r = 2           8.492      6.494      12.30   14.26   18.52   no rechaza
   r = 3           2.070      1.583  

La prueba de exogeneidad débil sigue a Johansen (1992): compara la verosimilitud del sistema completo con la del sistema parcial en el que se condiciona sobre la variable candidata, y el estadístico se distribuye como una ji cuadrada con tantos grados de libertad como el rango.

In [37]:
# -*- coding: utf-8 -*-
"""
Paso 4c. Johansen con ficticia de 2020, estimacion con rango uno impuesto
en los cuatro sistemas y contraste de exogeneidad debil por razon de
verosimilitud (Johansen, 1992).
"""

import os
import warnings
import numpy as np
import pandas as pd
from scipy import linalg, stats
from statsmodels.tsa.vector_ar.vecm import coint_johansen, select_order, VECM

warnings.filterwarnings("ignore")

PANEL = "/content/drive/MyDrive/tesis_china_eeuu/data_clean/china_us_panel_analisis_v6_1990_2023.csv"
SAL = "/content/drive/MyDrive/tesis_china_eeuu/outputs"
os.makedirs(SAL, exist_ok=True)

MAX_ORDEN = 3
RANGO_IMPUESTO = 1

ETIQUETAS = {
    "ln_gdp_pc_pwt": "PIB per capita real (log)",
    "inv_gfcf_gdp": "Formacion bruta de capital fijo",
    "credit_priv_gdp": "Credito al sector privado (BM)",
    "bis_pvt_credit_gdp": "Credito al sector privado (BIS)",
    "priv_debt_ls_gdp": "Deuda privada (FMI)",
    "trade_gdp": "Apertura comercial",
}

SISTEMAS = [
    ("CN", "China", "S-A", "Sistema principal",
     ["ln_gdp_pc_pwt", "inv_gfcf_gdp", "credit_priv_gdp", "trade_gdp"]),
    ("CN", "China", "S-B", "Contraste con credito del BIS",
     ["ln_gdp_pc_pwt", "inv_gfcf_gdp", "bis_pvt_credit_gdp", "trade_gdp"]),
    ("CN", "China", "S-C", "Contraste con deuda privada del FMI",
     ["ln_gdp_pc_pwt", "inv_gfcf_gdp", "priv_debt_ls_gdp", "trade_gdp"]),
    ("US", "Estados Unidos", "S-A", "Sistema principal",
     ["ln_gdp_pc_pwt", "inv_gfcf_gdp", "credit_priv_gdp", "trade_gdp"]),
]

df = pd.read_csv(PANEL)
if "d2020" not in df.columns:
    df["d2020"] = (df["year"] == 2020).astype(int)


# --------------------------------------------------------------------------
# Regresion de rango reducido de Johansen con constante restringida al
# espacio de cointegracion, exogenas opcionales y sistema parcial opcional.
# --------------------------------------------------------------------------
def rango_reducido(Y, kdiff, D=None, ecuaciones=None):
    Y = np.asarray(Y, dtype=float)
    T, k = Y.shape
    dY = np.diff(Y, axis=0)                      # filas 1..T-1
    ini = kdiff + 1
    idx = np.arange(ini, T)
    n = len(idx)

    Z0 = dY[idx - 1, :]                          # diferencia contemporanea
    Z1 = np.column_stack([Y[idx - 1, :], np.ones(n)])
    partes = [dY[idx - 1 - j, :] for j in range(1, kdiff + 1)]
    if D is not None:
        partes.append(np.asarray(D, dtype=float)[idx].reshape(n, -1))

    if ecuaciones is not None:
        cond = [i for i in range(k) if i not in ecuaciones]
        if cond:
            partes.append(Z0[:, cond])           # diferencias contemporaneas
        Z0 = Z0[:, list(ecuaciones)]

    Z2 = np.column_stack(partes) if partes else np.empty((n, 0))
    if Z2.shape[1] > 0:
        P = Z2 @ linalg.pinv(Z2)
        R0, R1 = Z0 - P @ Z0, Z1 - P @ Z1
    else:
        R0, R1 = Z0, Z1

    S00 = R0.T @ R0 / n
    S11 = R1.T @ R1 / n
    S01 = R0.T @ R1 / n
    M = linalg.pinv(S11) @ S01.T @ linalg.pinv(S00) @ S01
    val = np.real(linalg.eigvals(M))
    val = np.sort(np.clip(val, 0.0, 1 - 1e-12))[::-1]
    return val[:min(k, Z0.shape[1])], n


# --------------------------------------------------------------------------
# Validacion de la implementacion contra la rutina estandar
# --------------------------------------------------------------------------
print("Validacion de la regresion de rango reducido")
g0 = df[df["country"] == "CN"].sort_values("year")
Y0 = g0[["ln_gdp_pc_pwt", "inv_gfcf_gdp", "credit_priv_gdp", "trade_gdp"]].dropna().values
propio, _ = rango_reducido(Y0, kdiff=1, D=None)
estandar = coint_johansen(Y0, det_order=0, k_ar_diff=1).eig
dmax = float(np.max(np.abs(np.sort(propio)[::-1] - np.sort(estandar)[::-1])))
print("   propios: " + ", ".join("{:.6f}".format(v) for v in propio))
print("   rutina : " + ", ".join("{:.6f}".format(v) for v in estandar))
print("   discrepancia maxima = {:.3e}   {}".format(
    dmax, "correcta" if dmax < 1e-8 else "REVISAR"))

NIV = ["90%", "95%", "99%"]
rangos, vectores, ajustes, exog, resumen = [], [], [], [], []

for pais, nombre, clave, descripcion, variables in SISTEMAS:
    g = df[df["country"] == pais].sort_values("year").reset_index(drop=True)
    if any(v not in g.columns for v in variables):
        continue
    base = g[["year"] + variables + ["d2020"]].dropna().reset_index(drop=True)
    Y = base[variables].astype(float)
    D = base[["d2020"]].astype(float)
    T, k = Y.shape

    print("\n" + "=" * 98)
    print("{}  {}  {}   n={} ({}-{})".format(
        nombre, clave, descripcion, T, int(base["year"].min()), int(base["year"].max())))

    try:
        sel = select_order(Y, maxlags=MAX_ORDEN, deterministic="ci")
        p = max(1, int(sel.aic))
    except Exception:
        p = 2
    kdiff = max(1, p - 1)
    print("   orden en niveles p={}  (rezagos en diferencias={})".format(p, kdiff))

    # ---------------------------------------- rango con y sin ficticia
    ref = coint_johansen(Y.values, det_order=0, k_ar_diff=kdiff)
    lam_sin, n_ef = rango_reducido(Y.values, kdiff, None)
    lam_con, n_ef = rango_reducido(Y.values, kdiff, D.values)
    factor = (T - k * p) / T

    print("\n   Prueba de la traza   (n efectiva = {}, factor de correccion = {:.3f})".format(
        n_ef, factor))
    print("   {:<8} {:>11} {:>11} {:>11} {:>11} {:>22}".format(
        "hipotesis", "sin d2020", "con d2020", "corregida", "VC 95%", "veredicto (corregida)"))
    r_sel = 0
    for i in range(k):
        tr_sin = -n_ef * np.sum(np.log(1 - lam_sin[i:]))
        tr_con = -n_ef * np.sum(np.log(1 - lam_con[i:]))
        tr_adj = tr_con * factor
        cv = ref.cvt[i]
        rech = tr_adj > cv[1]
        if rech and r_sel == i:
            r_sel = i + 1
        print("   r <= {:<3} {:>11.3f} {:>11.3f} {:>11.3f} {:>11.2f} {:>22}".format(
            i, tr_sin, tr_con, tr_adj, cv[1], "rechaza" if rech else "no rechaza"))
        rangos.append({"Pais": nombre, "Sistema": clave, "Hipotesis": "r <= {}".format(i),
                       "Traza sin d2020": round(tr_sin, 3),
                       "Traza con d2020": round(tr_con, 3),
                       "Traza corregida": round(tr_adj, 3),
                       "VC 90%": cv[0], "VC 95%": cv[1], "VC 99%": cv[2],
                       "Rechaza al 5%": bool(rech),
                       "Rechaza sin corregir": bool(tr_con > cv[1])})
    print("   rango seleccionado por la traza corregida: {}".format(r_sel))
    print("   nota: los valores criticos son los del sistema sin ficticia;")
    print("         con intervencion son orientativos (Johansen, Mosconi y Nielsen, 2000).")

    # ---------------------------------------- modelo vectorial con rango impuesto
    r = RANGO_IMPUESTO
    print("\n   Modelo vectorial con rango impuesto r={} y ficticia de 2020{}".format(
        r, "" if r_sel >= 1 else "  (impuesto pese a que la traza no rechaza)"))
    res = VECM(Y, exog=D, k_ar_diff=kdiff, coint_rank=r, deterministic="ci").fit()

    beta = np.asarray(res.beta)
    piv = beta[0, 0]
    if abs(piv) > 1e-10:
        print("   Vector normalizado en el producto por habitante")
        for i, v in enumerate(variables[1:], start=1):
            coef = -beta[i, 0] / piv
            print("     {:<38} {:>10.4f}".format(ETIQUETAS[v], coef))
            vectores.append({"Pais": nombre, "Sistema": clave, "Variable": ETIQUETAS[v],
                             "Coeficiente normalizado": round(coef, 4),
                             "Rango impuesto": r})

    alfa = np.asarray(res.alpha)
    try:
        talfa, palfa = np.asarray(res.tvalues_alpha), np.asarray(res.pvalues_alpha)
    except Exception:
        talfa = palfa = np.full_like(alfa, np.nan, dtype=float)

    print("\n   Velocidades de ajuste y exogeneidad debil")
    print("   {:<38} {:>10} {:>8} {:>10} {:>8} {:>16}".format(
        "ecuacion", "alfa", "t", "LR", "p", "lectura"))
    for i, v in enumerate(variables):
        lam_p, _ = rango_reducido(Y.values, kdiff, D.values, ecuaciones=[
            j for j in range(k) if j != i])
        lr = n_ef * float(np.sum(np.log(1 - lam_p[:r]) - np.log(1 - lam_con[:r])))
        lr = max(lr, 0.0)
        pv = stats.chi2.sf(lr, r)
        sig = ("***" if palfa[i, 0] < 0.01 else "**" if palfa[i, 0] < 0.05
               else "*" if palfa[i, 0] < 0.10 else "")
        lectura = "endogena" if pv < 0.05 else "debilmente exogena"
        print("   {:<38} {:>10.4f}{:<3} {:>5.2f} {:>10.3f} {:>8.3f} {:>16}".format(
            ETIQUETAS[v], alfa[i, 0], sig, talfa[i, 0], lr, pv, lectura))
        ajustes.append({"Pais": nombre, "Sistema": clave, "Ecuacion": ETIQUETAS[v],
                        "Alfa": round(float(alfa[i, 0]), 4),
                        "t": round(float(talfa[i, 0]), 2), "Significancia": sig,
                        "LR exogeneidad debil": round(lr, 3),
                        "p": round(float(pv), 4), "Lectura": lectura})

    # exogeneidad debil conjunta de todos los regresores
    lam_c, _ = rango_reducido(Y.values, kdiff, D.values, ecuaciones=[0])
    lr_c = max(n_ef * float(np.sum(np.log(1 - lam_c[:r]) - np.log(1 - lam_con[:r]))), 0.0)
    gl_c = r * (k - 1)
    pv_c = stats.chi2.sf(lr_c, gl_c)
    print("\n   Exogeneidad debil conjunta de los tres regresores:")
    print("     LR = {:.3f}  gl = {}  p = {:.4f}  ->  {}".format(
        lr_c, gl_c, pv_c,
        "se sostiene, el enfoque uniecuacional es valido" if pv_c >= 0.05
        else "se rechaza, el enfoque uniecuacional pierde eficiencia"))
    exog.append({"Pais": nombre, "Sistema": clave,
                 "Contraste": "Exogeneidad debil conjunta de los regresores",
                 "LR": round(lr_c, 3), "gl": gl_c, "p": round(float(pv_c), 4),
                 "Se sostiene": bool(pv_c >= 0.05)})

    fila = {"Pais": nombre, "Sistema": clave, "n": T, "n efectiva": n_ef, "k": k, "p": p,
            "Rango por traza corregida": r_sel, "Rango impuesto": r}
    for etiqueta, prueba in (("Normalidad (p)", "norm"),
                             ("Ausencia de autocorrelacion (p)", "auto")):
        try:
            out = (res.test_normality() if prueba == "norm"
                   else res.test_whiteness(nlags=max(kdiff + 2, 4)))
            fila[etiqueta] = round(float(out.pvalue), 3)
        except Exception:
            fila[etiqueta] = None
    print("   Diagnosticos del sistema: normalidad p={}  autocorrelacion p={}".format(
        fila.get("Normalidad (p)"), fila.get("Ausencia de autocorrelacion (p)")))
    resumen.append(fila)

for archivo, contenido in (("cuadro14_johansen_rango_v2.csv", rangos),
                           ("cuadro14_johansen_vectores_v2.csv", vectores),
                           ("cuadro14_johansen_ajustes_v2.csv", ajustes),
                           ("cuadro14_johansen_exogeneidad.csv", exog),
                           ("cuadro14_johansen_resumen_v2.csv", resumen)):
    pd.DataFrame(contenido).to_csv(os.path.join(SAL, archivo), index=False)
    print("\nGuardado: {}".format(os.path.join(SAL, archivo)))

Validacion de la regresion de rango reducido
   propios: 0.659886, 0.619302, 0.294737, 0.177252
   rutina : 0.654878, 0.466034, 0.233076, 0.062626
   discrepancia maxima = 1.533e-01   REVISAR

China  S-A  Sistema principal   n=34 (1990-2023)
   orden en niveles p=2  (rezagos en diferencias=1)

   Prueba de la traza   (n efectiva = 32, factor de correccion = 0.765)
   hipotesis   sin d2020   con d2020   corregida      VC 95%  veredicto (corregida)
   r <= 0        82.832      81.501      62.324       47.85                rechaza
   r <= 1        48.321      47.000      35.941       29.80                rechaza
   r <= 2        17.417      15.959      12.204       15.49             no rechaza
   r <= 3         6.243       5.176       3.958        3.84                rechaza
   rango seleccionado por la traza corregida: 2
   nota: los valores criticos son los del sistema sin ficticia;
         con intervencion son orientativos (Johansen, Mosconi y Nielsen, 2000).

   Modelo vectorial con 

Opto por la constante irrestricta, que es la convención de la rutina estándar, la que corresponde a los valores críticos que ya están en los cuadros, y la coherente con la especificación de deterministas que usaste en todos los modelos uniecuacionales, constante sin tendencia.

In [38]:
# -*- coding: utf-8 -*-
"""
Paso 4c corregido. Johansen con constante irrestricta en todos los
componentes, ficticia de 2020, rango uno impuesto y contraste de
exogeneidad debil por razon de verosimilitud (Johansen, 1992).
"""

import os
import warnings
import numpy as np
import pandas as pd
from scipy import linalg, stats
from statsmodels.tsa.vector_ar.vecm import coint_johansen, select_order, VECM

warnings.filterwarnings("ignore")

PANEL = "/content/drive/MyDrive/tesis_china_eeuu/data_clean/china_us_panel_analisis_v6_1990_2023.csv"
SAL = "/content/drive/MyDrive/tesis_china_eeuu/outputs"
os.makedirs(SAL, exist_ok=True)

MAX_ORDEN = 3
RANGO_IMPUESTO = 1
DET = "co"          # constante fuera del espacio de cointegracion

ETIQUETAS = {
    "ln_gdp_pc_pwt": "PIB per capita real (log)",
    "inv_gfcf_gdp": "Formacion bruta de capital fijo",
    "credit_priv_gdp": "Credito al sector privado (BM)",
    "bis_pvt_credit_gdp": "Credito al sector privado (BIS)",
    "priv_debt_ls_gdp": "Deuda privada (FMI)",
    "trade_gdp": "Apertura comercial",
}

SISTEMAS = [
    ("CN", "China", "S-A", "Sistema principal",
     ["ln_gdp_pc_pwt", "inv_gfcf_gdp", "credit_priv_gdp", "trade_gdp"]),
    ("CN", "China", "S-B", "Contraste con credito del BIS",
     ["ln_gdp_pc_pwt", "inv_gfcf_gdp", "bis_pvt_credit_gdp", "trade_gdp"]),
    ("CN", "China", "S-C", "Contraste con deuda privada del FMI",
     ["ln_gdp_pc_pwt", "inv_gfcf_gdp", "priv_debt_ls_gdp", "trade_gdp"]),
    ("US", "Estados Unidos", "S-A", "Sistema principal",
     ["ln_gdp_pc_pwt", "inv_gfcf_gdp", "credit_priv_gdp", "trade_gdp"]),
]

df = pd.read_csv(PANEL)
if "d2020" not in df.columns:
    df["d2020"] = (df["year"] == 2020).astype(int)


# --------------------------------------------------------------------------
# Regresion de rango reducido con constante IRRESTRICTA, exogenas opcionales
# y sistema parcial opcional (para el contraste de exogeneidad debil).
# --------------------------------------------------------------------------
def rango_reducido(Y, kdiff, D=None, ecuaciones=None):
    Y = np.asarray(Y, dtype=float)
    T, k = Y.shape
    dY = np.diff(Y, axis=0)
    idx = np.arange(kdiff + 1, T)
    n = len(idx)

    Z0 = dY[idx - 1, :]
    Z1 = Y[idx - 1, :]                            # sin columna de unos
    partes = [np.ones((n, 1))]                    # constante irrestricta
    partes += [dY[idx - 1 - j, :] for j in range(1, kdiff + 1)]
    if D is not None:
        partes.append(np.asarray(D, dtype=float)[idx].reshape(n, -1))

    if ecuaciones is not None:
        cond = [i for i in range(k) if i not in ecuaciones]
        if cond:
            partes.append(Z0[:, cond])
        Z0 = Z0[:, list(ecuaciones)]

    Z2 = np.column_stack(partes)
    P = Z2 @ linalg.pinv(Z2)
    R0, R1 = Z0 - P @ Z0, Z1 - P @ Z1

    S00 = R0.T @ R0 / n
    S11 = R1.T @ R1 / n
    S01 = R0.T @ R1 / n
    M = linalg.pinv(S11) @ S01.T @ linalg.pinv(S00) @ S01
    val = np.real(linalg.eigvals(M))
    val = np.sort(np.clip(val, 0.0, 1 - 1e-12))[::-1]
    return val[:min(k, Z0.shape[1])], n


# --------------------------------------------------------------------------
# Validacion: valores propios y estadisticos de la traza
# --------------------------------------------------------------------------
print("Validacion de la regresion de rango reducido (constante irrestricta)")
_ok = True
for _pais, _vars, _kd in (("CN", ["ln_gdp_pc_pwt", "inv_gfcf_gdp",
                                  "credit_priv_gdp", "trade_gdp"], 1),
                          ("US", ["ln_gdp_pc_pwt", "inv_gfcf_gdp",
                                  "credit_priv_gdp", "trade_gdp"], 2)):
    _Y = df[df["country"] == _pais].sort_values("year")[_vars].dropna().values
    _prop, _n = rango_reducido(_Y, kdiff=_kd)
    _ref = coint_johansen(_Y, det_order=0, k_ar_diff=_kd)
    _d1 = float(np.max(np.abs(_prop - _ref.eig)))
    _tr = np.array([-_n * np.sum(np.log(1 - _prop[i:])) for i in range(len(_prop))])
    _d2 = float(np.max(np.abs(_tr - _ref.lr1)))
    _ok &= (_d1 < 1e-8 and _d2 < 1e-6)
    print("   {}  k_ar_diff={}  dif. valores propios = {:.3e}   dif. traza = {:.3e}   {}".format(
        _pais, _kd, _d1, _d2, "correcta" if (_d1 < 1e-8 and _d2 < 1e-6) else "REVISAR"))
if not _ok:
    raise SystemExit("La implementacion no reproduce la rutina estandar. Detener.")
print("   Implementacion validada.\n")

rangos, vectores, ajustes, exog, resumen = [], [], [], [], []

for pais, nombre, clave, descripcion, variables in SISTEMAS:
    g = df[df["country"] == pais].sort_values("year").reset_index(drop=True)
    if any(v not in g.columns for v in variables):
        continue
    base = g[["year"] + variables + ["d2020"]].dropna().reset_index(drop=True)
    Y = base[variables].astype(float)
    D = base[["d2020"]].astype(float)
    T, k = Y.shape

    print("=" * 98)
    print("{}  {}  {}   n={} ({}-{})".format(
        nombre, clave, descripcion, T, int(base["year"].min()), int(base["year"].max())))

    try:
        sel = select_order(Y, maxlags=MAX_ORDEN, deterministic=DET)
        p = max(1, int(sel.aic))
    except Exception:
        p = 2
    kdiff = max(1, p - 1)
    print("   orden en niveles p={}  (rezagos en diferencias={})".format(p, kdiff))

    ref = coint_johansen(Y.values, det_order=0, k_ar_diff=kdiff)
    lam_sin, n_ef = rango_reducido(Y.values, kdiff, None)
    lam_con, n_ef = rango_reducido(Y.values, kdiff, D.values)
    factor = (T - k * p) / T

    print("\n   Prueba de la traza   (n efectiva = {}, factor de correccion = {:.3f})".format(
        n_ef, factor))
    print("   {:<8} {:>11} {:>11} {:>11} {:>11} {:>22}".format(
        "hipotesis", "sin d2020", "con d2020", "corregida", "VC 95%", "veredicto (corregida)"))
    r_sel, r_sin_corr = 0, 0
    for i in range(k):
        tr_sin = -n_ef * np.sum(np.log(1 - lam_sin[i:]))
        tr_con = -n_ef * np.sum(np.log(1 - lam_con[i:]))
        tr_adj = tr_con * factor
        cv = ref.cvt[i]
        rech, rech_sc = tr_adj > cv[1], tr_con > cv[1]
        if rech and r_sel == i:
            r_sel = i + 1
        if rech_sc and r_sin_corr == i:
            r_sin_corr = i + 1
        print("   r <= {:<3} {:>11.3f} {:>11.3f} {:>11.3f} {:>11.2f} {:>22}".format(
            i, tr_sin, tr_con, tr_adj, cv[1], "rechaza" if rech else "no rechaza"))
        rangos.append({"Pais": nombre, "Sistema": clave, "Hipotesis": "r <= {}".format(i),
                       "Traza sin d2020": round(tr_sin, 3),
                       "Traza con d2020": round(tr_con, 3),
                       "Traza corregida": round(tr_adj, 3),
                       "VC 90%": cv[0], "VC 95%": cv[1], "VC 99%": cv[2],
                       "Rechaza al 5%": bool(rech),
                       "Rechaza sin corregir": bool(rech_sc)})
    print("   rango por la traza corregida: {}   |   sin corregir: {}".format(
        r_sel, r_sin_corr))
    print("   nota: valores criticos del sistema sin intervencion; con ficticia son")
    print("         orientativos (Johansen, Mosconi y Nielsen, 2000).")

    r = RANGO_IMPUESTO
    print("\n   Modelo vectorial con rango r={} y ficticia de 2020{}".format(
        r, "" if r_sel >= 1 else "   (impuesto: la traza no rechaza)"))
    res = VECM(Y, exog=D, k_ar_diff=kdiff, coint_rank=r, deterministic=DET).fit()

    beta = np.asarray(res.beta)
    piv = beta[0, 0]
    if abs(piv) > 1e-10:
        print("   Vector normalizado en el producto por habitante")
        for i, v in enumerate(variables[1:], start=1):
            coef = -beta[i, 0] / piv
            print("     {:<38} {:>10.4f}".format(ETIQUETAS[v], coef))
            vectores.append({"Pais": nombre, "Sistema": clave, "Variable": ETIQUETAS[v],
                             "Coeficiente normalizado": round(coef, 4)})

    alfa = np.asarray(res.alpha)
    try:
        talfa, palfa = np.asarray(res.tvalues_alpha), np.asarray(res.pvalues_alpha)
    except Exception:
        talfa = palfa = np.full_like(alfa, np.nan, dtype=float)

    print("\n   Velocidades de ajuste y exogeneidad debil")
    print("   {:<38} {:>11} {:>7} {:>9} {:>8} {:>20}".format(
        "ecuacion", "alfa", "t", "LR", "p", "lectura"))
    for i, v in enumerate(variables):
        lam_p, _ = rango_reducido(Y.values, kdiff, D.values,
                                  ecuaciones=[j for j in range(k) if j != i])
        lr = max(n_ef * float(np.sum(np.log(1 - lam_p[:r]) - np.log(1 - lam_con[:r]))), 0.0)
        pv = stats.chi2.sf(lr, r)
        sig = ("***" if palfa[i, 0] < 0.01 else "**" if palfa[i, 0] < 0.05
               else "*" if palfa[i, 0] < 0.10 else "")
        print("   {:<38} {:>11.4f}{:<3} {:>5.2f} {:>9.3f} {:>8.3f} {:>20}".format(
            ETIQUETAS[v], alfa[i, 0], sig, talfa[i, 0], lr, pv,
            "endogena" if pv < 0.05 else "debilmente exogena"))
        ajustes.append({"Pais": nombre, "Sistema": clave, "Ecuacion": ETIQUETAS[v],
                        "Alfa": round(float(alfa[i, 0]), 4),
                        "t": round(float(talfa[i, 0]), 2), "Significancia": sig,
                        "LR exogeneidad debil": round(lr, 3),
                        "p": round(float(pv), 4),
                        "Lectura": "endogena" if pv < 0.05 else "debilmente exogena"})

    lam_c, _ = rango_reducido(Y.values, kdiff, D.values, ecuaciones=[0])
    lr_c = max(n_ef * float(np.sum(np.log(1 - lam_c[:r]) - np.log(1 - lam_con[:r]))), 0.0)
    gl_c = r * (k - 1)
    pv_c = stats.chi2.sf(lr_c, gl_c)
    print("\n   Exogeneidad debil conjunta de los tres regresores:")
    print("     LR = {:.3f}  gl = {}  p = {:.4f}  ->  {}".format(
        lr_c, gl_c, pv_c,
        "se sostiene: el enfoque uniecuacional es valido" if pv_c >= 0.05
        else "se rechaza: el enfoque uniecuacional pierde eficiencia"))
    exog.append({"Pais": nombre, "Sistema": clave,
                 "Contraste": "Exogeneidad debil conjunta de los regresores",
                 "LR": round(lr_c, 3), "gl": gl_c, "p": round(float(pv_c), 4),
                 "Se sostiene": bool(pv_c >= 0.05)})

    fila = {"Pais": nombre, "Sistema": clave, "n": T, "n efectiva": n_ef, "k": k, "p": p,
            "Deterministas": "constante irrestricta",
            "Rango traza corregida": r_sel, "Rango traza sin corregir": r_sin_corr,
            "Rango impuesto": r}
    try:
        fila["Normalidad (p)"] = round(float(res.test_normality().pvalue), 3)
    except Exception:
        fila["Normalidad (p)"] = None
    try:
        fila["Ausencia de autocorrelacion (p)"] = round(
            float(res.test_whiteness(nlags=max(kdiff + 2, 4)).pvalue), 3)
    except Exception:
        fila["Ausencia de autocorrelacion (p)"] = None
    print("   Diagnosticos del sistema: normalidad p={}  autocorrelacion p={}\n".format(
        fila["Normalidad (p)"], fila["Ausencia de autocorrelacion (p)"]))
    resumen.append(fila)

for archivo, contenido in (("cuadro14_johansen_rango_v3.csv", rangos),
                           ("cuadro14_johansen_vectores_v3.csv", vectores),
                           ("cuadro14_johansen_ajustes_v3.csv", ajustes),
                           ("cuadro14_johansen_exogeneidad_v3.csv", exog),
                           ("cuadro14_johansen_resumen_v3.csv", resumen)):
    pd.DataFrame(contenido).to_csv(os.path.join(SAL, archivo), index=False)
    print("Guardado: {}".format(os.path.join(SAL, archivo)))

Validacion de la regresion de rango reducido (constante irrestricta)
   CN  k_ar_diff=1  dif. valores propios = 2.245e-13   dif. traza = 2.166e-11   correcta
   US  k_ar_diff=2  dif. valores propios = 4.422e-13   dif. traza = 2.994e-11   correcta
   Implementacion validada.

China  S-A  Sistema principal   n=34 (1990-2023)
   orden en niveles p=2  (rezagos en diferencias=1)

   Prueba de la traza   (n efectiva = 32, factor de correccion = 0.765)
   hipotesis   sin d2020   con d2020   corregida      VC 95%  veredicto (corregida)
   r <= 0        64.682      63.728      48.733       47.85                rechaza
   r <= 1        30.639      29.571      22.613       29.80             no rechaza
   r <= 2        10.561      10.947       8.371       15.49             no rechaza
   r <= 3         2.070       2.463       1.884        3.84             no rechaza
   rango por la traza corregida: 1   |   sin corregir: 1
   nota: valores criticos del sistema sin intervencion; con ficticia son
    

In [40]:
# -*- coding: utf-8 -*-
"""
Paso 4d. Diagnostico de residuos por ecuacion, valores criticos simulados
para la traza con variable de intervencion, y raices de la matriz
companera de cada sistema.
"""

import os
import time
import warnings
import numpy as np
import pandas as pd
from scipy import linalg, stats
from statsmodels.tsa.vector_ar.vecm import coint_johansen, select_order, VECM

warnings.filterwarnings("ignore")

PANEL = "/content/drive/MyDrive/tesis_china_eeuu/data_clean/china_us_panel_analisis_v6_1990_2023.csv"
SAL = "/content/drive/MyDrive/tesis_china_eeuu/outputs"
os.makedirs(SAL, exist_ok=True)

MAX_ORDEN, RANGO, DET = 3, 1, "co"
NSIM, SEMILLA = 5000, 20260731

ETIQUETAS = {
    "ln_gdp_pc_pwt": "PIB per capita real (log)",
    "inv_gfcf_gdp": "Formacion bruta de capital fijo",
    "credit_priv_gdp": "Credito al sector privado (BM)",
    "bis_pvt_credit_gdp": "Credito al sector privado (BIS)",
    "priv_debt_ls_gdp": "Deuda privada (FMI)",
    "trade_gdp": "Apertura comercial",
}

SISTEMAS = [
    ("CN", "China", "S-A", ["ln_gdp_pc_pwt", "inv_gfcf_gdp", "credit_priv_gdp", "trade_gdp"]),
    ("CN", "China", "S-B", ["ln_gdp_pc_pwt", "inv_gfcf_gdp", "bis_pvt_credit_gdp", "trade_gdp"]),
    ("CN", "China", "S-C", ["ln_gdp_pc_pwt", "inv_gfcf_gdp", "priv_debt_ls_gdp", "trade_gdp"]),
    ("US", "Estados Unidos", "S-A", ["ln_gdp_pc_pwt", "inv_gfcf_gdp", "credit_priv_gdp", "trade_gdp"]),
]

df = pd.read_csv(PANEL)
if "d2020" not in df.columns:
    df["d2020"] = (df["year"] == 2020).astype(int)


def valores_propios(Y, kdiff, D=None):
    """Regresion de rango reducido con constante irrestricta."""
    Y = np.asarray(Y, dtype=float)
    T, k = Y.shape
    dY = np.diff(Y, axis=0)
    idx = np.arange(kdiff + 1, T)
    n = len(idx)
    Z0 = dY[idx - 1, :]
    Z1 = Y[idx - 1, :]
    partes = [np.ones((n, 1))] + [dY[idx - 1 - j, :] for j in range(1, kdiff + 1)]
    if D is not None:
        partes.append(np.asarray(D, dtype=float)[idx].reshape(n, -1))
    Z2 = np.column_stack(partes)
    P = Z2 @ linalg.pinv(Z2)
    R0, R1 = Z0 - P @ Z0, Z1 - P @ Z1
    S00, S11, S01 = R0.T @ R0 / n, R1.T @ R1 / n, R0.T @ R1 / n
    M = linalg.pinv(S11) @ S01.T @ linalg.pinv(S00) @ S01
    val = np.sort(np.clip(np.real(linalg.eigvals(M)), 0.0, 1 - 1e-12))[::-1]
    return val[:k], n


def traza(val, n):
    return np.array([-n * np.sum(np.log(1 - val[i:])) for i in range(len(val))])


# validacion
_Y = df[df["country"] == "CN"].sort_values("year")[
    ["ln_gdp_pc_pwt", "inv_gfcf_gdp", "credit_priv_gdp", "trade_gdp"]].dropna().values
_v, _n = valores_propios(_Y, 1)
_r = coint_johansen(_Y, det_order=0, k_ar_diff=1)
_d = max(float(np.max(np.abs(_v - _r.eig))), float(np.max(np.abs(traza(_v, _n) - _r.lr1))))
print("Validacion: discrepancia maxima = {:.3e}  {}".format(
    _d, "correcta" if _d < 1e-6 else "REVISAR"))
if _d >= 1e-6:
    raise SystemExit("Implementacion no validada.")

residuos, criticos, raices, resumen = [], [], [], []

for pais, nombre, clave, variables in SISTEMAS:
    g = df[df["country"] == pais].sort_values("year").reset_index(drop=True)
    if any(v not in g.columns for v in variables):
        continue
    base = g[["year"] + variables + ["d2020"]].dropna().reset_index(drop=True)
    Y, D = base[variables].astype(float), base[["d2020"]].astype(float)
    T, k = Y.shape
    try:
        p = max(1, int(select_order(Y, maxlags=MAX_ORDEN, deterministic=DET).aic))
    except Exception:
        p = 2
    kdiff = max(1, p - 1)
    anios = base["year"].values[kdiff + 1:]

    print("\n" + "=" * 96)
    print("{}  {}   n={} ({}-{})   p={}  rezagos en diferencias={}".format(
        nombre, clave, T, int(base["year"].min()), int(base["year"].max()), p, kdiff))

    res = VECM(Y, exog=D, k_ar_diff=kdiff, coint_rank=RANGO, deterministic=DET).fit()

    # ------------------------------------------------ 1. residuos por ecuacion
    E = np.asarray(res.resid)
    print("\n   1. Diagnostico de residuos por ecuacion")
    print("   {:<38} {:>9} {:>9} {:>9} {:>26}".format(
        "ecuacion", "asimetria", "curtosis", "JB (p)", "mayores residuos tipificados"))
    for i, v in enumerate(variables):
        e = E[:, i]
        z = (e - e.mean()) / e.std(ddof=1)
        jb = stats.jarque_bera(e)
        orden = np.argsort(-np.abs(z))[:3]
        detalle = ", ".join("{} ({:+.2f})".format(int(anios[j]), z[j]) for j in orden)
        print("   {:<38} {:>9.2f} {:>9.2f} {:>9.3f}   {}".format(
            ETIQUETAS[v], stats.skew(e), stats.kurtosis(e, fisher=False),
            jb.pvalue, detalle))
        for j in orden:
            residuos.append({"Pais": nombre, "Sistema": clave, "Ecuacion": ETIQUETAS[v],
                             "Anio": int(anios[j]), "Residuo tipificado": round(float(z[j]), 2),
                             "Asimetria": round(float(stats.skew(e)), 2),
                             "Curtosis": round(float(stats.kurtosis(e, fisher=False)), 2),
                             "Jarque-Bera (p)": round(float(jb.pvalue), 4)})

    # ------------------------------------------------ 2. valores criticos simulados
    val_obs, n_ef = valores_propios(Y.values, kdiff, D.values)
    tr_obs = traza(val_obs, n_ef)
    pos = np.asarray(D).ravel()
    rng = np.random.default_rng(SEMILLA)
    t0 = time.time()
    print("\n   2. Valores criticos simulados con la ficticia en su posicion real")
    print("      ({} replicas, caminatas aleatorias independientes)".format(NSIM))
    print("   {:<9} {:>9} {:>10} {:>10} {:>10} {:>10} {:>16}".format(
        "hipotesis", "traza", "VC sim 90", "VC sim 95", "VC sim 99", "p simulada", "veredicto 5%"))
    for i in range(k):
        m = k - i
        sims = np.empty(NSIM)
        for s in range(NSIM):
            W = np.cumsum(rng.standard_normal((T, m)), axis=0)
            vs, ns = valores_propios(W, kdiff, pos)
            sims[s] = -ns * np.sum(np.log(1 - vs))
        q90, q95, q99 = np.percentile(sims, [90, 95, 99])
        pval = float(np.mean(sims >= tr_obs[i]))
        rech = tr_obs[i] > q95
        print("   r <= {:<4} {:>9.3f} {:>10.2f} {:>10.2f} {:>10.2f} {:>10.4f} {:>16}".format(
            i, tr_obs[i], q90, q95, q99, pval, "rechaza" if rech else "no rechaza"))
        criticos.append({"Pais": nombre, "Sistema": clave, "Hipotesis": "r <= {}".format(i),
                         "Traza con d2020": round(float(tr_obs[i]), 3),
                         "VC simulado 90%": round(q90, 2), "VC simulado 95%": round(q95, 2),
                         "VC simulado 99%": round(q99, 2),
                         "VC tabulado 95%": float(coint_johansen(
                             Y.values, det_order=0, k_ar_diff=kdiff).cvt[i][1]),
                         "p simulada": round(pval, 4), "Rechaza al 5%": bool(rech)})
    print("      tiempo de simulacion: {:.0f} s".format(time.time() - t0))

    # ------------------------------------------------ 3. raices de la companera
    A = np.asarray(res.var_rep)           # (k_ar, k, k)
    kar = A.shape[0]
    C = np.zeros((k * kar, k * kar))
    C[:k, :] = np.hstack([A[j] for j in range(kar)])
    if kar > 1:
        C[k:, :-k] = np.eye(k * (kar - 1))
    mod = np.sort(np.abs(linalg.eigvals(C)))[::-1]
    unit = int(np.sum(mod > 0.95))
    print("\n   3. Raices de la matriz companera (modulos)")
    print("      " + ", ".join("{:.3f}".format(x) for x in mod))
    print("      raices unitarias esperadas con r={}: {}   observadas (>0.95): {}   {}".format(
        RANGO, k - RANGO, unit,
        "coherente" if unit == k - RANGO else "REVISAR"))
    print("      mayor raiz no unitaria: {:.3f}   {}".format(
        mod[unit] if unit < len(mod) else np.nan,
        "sistema estable" if unit < len(mod) and mod[unit] < 0.95 else "revisar estabilidad"))
    for j, x in enumerate(mod):
        raices.append({"Pais": nombre, "Sistema": clave, "Raiz": j + 1,
                       "Modulo": round(float(x), 4),
                       "Unitaria": bool(x > 0.95)})

    resumen.append({"Pais": nombre, "Sistema": clave, "n": T, "k": k, "p": p,
                    "Rango impuesto": RANGO,
                    "Raices unitarias esperadas": k - RANGO,
                    "Raices unitarias observadas": unit,
                    "Mayor raiz no unitaria": round(float(mod[unit]), 4) if unit < len(mod) else None,
                    "Normalidad del sistema (p)": round(float(res.test_normality().pvalue), 4)})

for archivo, contenido in (("cuadro15_residuos_por_ecuacion.csv", residuos),
                           ("cuadro15_criticos_simulados.csv", criticos),
                           ("cuadro15_raices_companera.csv", raices),
                           ("cuadro15_resumen.csv", resumen)):
    pd.DataFrame(contenido).to_csv(os.path.join(SAL, archivo), index=False)
    print("\nGuardado: {}".format(os.path.join(SAL, archivo)))

Validacion: discrepancia maxima = 2.166e-11  correcta

China  S-A   n=34 (1990-2023)   p=2  rezagos en diferencias=1

   1. Diagnostico de residuos por ecuacion
   ecuacion                               asimetria  curtosis    JB (p) mayores residuos tipificados
   PIB per capita real (log)                   0.33      5.64     0.007   2022 (-2.94), 1992 (+2.79), 2007 (+2.12)
   Formacion bruta de capital fijo             0.04      3.12     0.986   1994 (-2.37), 1993 (+2.31), 2019 (-1.75)
   Credito al sector privado (BM)             -0.20      2.74     0.857   2008 (-2.17), 1992 (-1.88), 2015 (+1.81)
   Apertura comercial                         -0.49      3.53     0.434   2009 (-2.53), 1994 (-2.35), 2003 (+1.88)

   2. Valores criticos simulados con la ficticia en su posicion real
      (5000 replicas, caminatas aleatorias independientes)
   hipotesis     traza  VC sim 90  VC sim 95  VC sim 99 p simulada     veredicto 5%
   r <= 0       63.728      59.03      63.56      73.59     0.048

Aquí está el paso 4e. Recorre una rejilla de seis configuraciones por sistema, tres órdenes de rezago por dos tratamientos de la ficticia, y contrasta cada una contra sus propios valores críticos simulados. Para cada configuración reporta el rango, la velocidad de ajuste de la ecuación del producto, el contraste conjunto de exogeneidad débil y los dos diagnósticos del sistema, de modo que puedas ver de un vistazo si las conclusiones dependen de decisiones de especificación.

Las simulaciones se guardan en memoria intermedia por combinación de longitud, dimensión, rezagos y ficticia, así que las veinticuatro configuraciones no cuestan veinticuatro tandas.

In [41]:
# -*- coding: utf-8 -*-
"""
Paso 4e. Sensibilidad del analisis de sistema al orden de rezagos y a la
ficticia de 2020, con valores criticos simulados para cada configuracion.
"""

import os
import time
import warnings
import numpy as np
import pandas as pd
from scipy import linalg, stats
from statsmodels.tsa.vector_ar.vecm import coint_johansen, VECM

warnings.filterwarnings("ignore")

PANEL = "/content/drive/MyDrive/tesis_china_eeuu/data_clean/china_us_panel_analisis_v6_1990_2023.csv"
SAL = "/content/drive/MyDrive/tesis_china_eeuu/outputs"
os.makedirs(SAL, exist_ok=True)

DET, RANGO = "co", 1
NSIM, SEMILLA = 3000, 20260731
REZAGOS = [0, 1, 2]          # rezagos en diferencias; orden en niveles = rezagos + 1

ETIQUETAS = {
    "ln_gdp_pc_pwt": "PIB per capita real (log)",
    "inv_gfcf_gdp": "Formacion bruta de capital fijo",
    "credit_priv_gdp": "Credito al sector privado (BM)",
    "bis_pvt_credit_gdp": "Credito al sector privado (BIS)",
    "priv_debt_ls_gdp": "Deuda privada (FMI)",
    "trade_gdp": "Apertura comercial",
}

SISTEMAS = [
    ("CN", "China", "S-A", ["ln_gdp_pc_pwt", "inv_gfcf_gdp", "credit_priv_gdp", "trade_gdp"]),
    ("CN", "China", "S-B", ["ln_gdp_pc_pwt", "inv_gfcf_gdp", "bis_pvt_credit_gdp", "trade_gdp"]),
    ("CN", "China", "S-C", ["ln_gdp_pc_pwt", "inv_gfcf_gdp", "priv_debt_ls_gdp", "trade_gdp"]),
    ("US", "Estados Unidos", "S-A", ["ln_gdp_pc_pwt", "inv_gfcf_gdp", "credit_priv_gdp", "trade_gdp"]),
]

df = pd.read_csv(PANEL)
if "d2020" not in df.columns:
    df["d2020"] = (df["year"] == 2020).astype(int)


def valores_propios(Y, kdiff, D=None, ecuaciones=None):
    Y = np.asarray(Y, dtype=float)
    T, k = Y.shape
    dY = np.diff(Y, axis=0)
    idx = np.arange(kdiff + 1, T)
    n = len(idx)
    Z0 = dY[idx - 1, :]
    Z1 = Y[idx - 1, :]
    partes = [np.ones((n, 1))] + [dY[idx - 1 - j, :] for j in range(1, kdiff + 1)]
    if D is not None:
        partes.append(np.asarray(D, dtype=float)[idx].reshape(n, -1))
    if ecuaciones is not None:
        cond = [i for i in range(k) if i not in ecuaciones]
        if cond:
            partes.append(Z0[:, cond])
        Z0 = Z0[:, list(ecuaciones)]
    Z2 = np.column_stack(partes)
    P = Z2 @ linalg.pinv(Z2)
    R0, R1 = Z0 - P @ Z0, Z1 - P @ Z1
    S00, S11, S01 = R0.T @ R0 / n, R1.T @ R1 / n, R0.T @ R1 / n
    M = linalg.pinv(S11) @ S01.T @ linalg.pinv(S00) @ S01
    val = np.sort(np.clip(np.real(linalg.eigvals(M)), 0.0, 1 - 1e-12))[::-1]
    return val[:min(k, Z0.shape[1])], n


def traza(val, n):
    return np.array([-n * np.sum(np.log(1 - val[i:])) for i in range(len(val))])


# validacion
_Y = df[df["country"] == "CN"].sort_values("year")[
    ["ln_gdp_pc_pwt", "inv_gfcf_gdp", "credit_priv_gdp", "trade_gdp"]].dropna().values
_v, _n = valores_propios(_Y, 1)
_r = coint_johansen(_Y, det_order=0, k_ar_diff=1)
_d = max(float(np.max(np.abs(_v - _r.eig))), float(np.max(np.abs(traza(_v, _n) - _r.lr1))))
print("Validacion: discrepancia maxima = {:.3e}  {}\n".format(
    _d, "correcta" if _d < 1e-6 else "REVISAR"))
if _d >= 1e-6:
    raise SystemExit("Implementacion no validada.")

CACHE = {}


def criticos_simulados(T, m, kdiff, patron):
    """Distribucion nula de la traza para m tendencias comunes."""
    clave = (T, m, kdiff, None if patron is None else int(np.argmax(patron)))
    if clave in CACHE:
        return CACHE[clave]
    rng = np.random.default_rng(SEMILLA + 1000 * m + 10 * kdiff + (0 if patron is None else 7))
    sims = np.empty(NSIM)
    for s in range(NSIM):
        W = np.cumsum(rng.standard_normal((T, m)), axis=0)
        vs, ns = valores_propios(W, kdiff, patron)
        sims[s] = -ns * np.sum(np.log(1 - vs))
    CACHE[clave] = sims
    return sims


filas = []
t_ini = time.time()

for pais, nombre, clave, variables in SISTEMAS:
    g = df[df["country"] == pais].sort_values("year").reset_index(drop=True)
    if any(v not in g.columns for v in variables):
        continue
    base = g[["year"] + variables + ["d2020"]].dropna().reset_index(drop=True)
    Y, Dfull = base[variables].astype(float), base[["d2020"]].astype(float)
    T, k = Y.shape

    print("=" * 104)
    print("{}  {}   n={} ({}-{})   variables: {}".format(
        nombre, clave, T, int(base["year"].min()), int(base["year"].max()),
        ", ".join(ETIQUETAS[v] for v in variables)))
    print("   {:<6} {:<9} {:>9} {:>10} {:>7} {:>12} {:>9} {:>10} {:>8} {:>8}".format(
        "orden", "ficticia", "traza r=0", "VC sim 95", "rango", "alfa producto", "t", "LR exog.",
        "norm.", "autoc."))

    for kdiff in REZAGOS:
        for usa_d in (False, True):
            D = Dfull if usa_d else None
            patron = np.asarray(Dfull).ravel() if usa_d else None
            try:
                val, n_ef = valores_propios(Y.values, kdiff, None if D is None else D.values)
                tr = traza(val, n_ef)
            except Exception as exc:
                print("   p={} {}: fallo el calculo ({})".format(kdiff + 1, usa_d, exc))
                continue

            rango, cv95_0, p0 = 0, np.nan, np.nan
            for i in range(k):
                sims = criticos_simulados(T, k - i, kdiff, patron)
                q95 = float(np.percentile(sims, 95))
                pv = float(np.mean(sims >= tr[i]))
                if i == 0:
                    cv95_0, p0 = q95, pv
                if tr[i] > q95 and rango == i:
                    rango = i + 1
                else:
                    if rango == i:
                        break

            try:
                res = VECM(Y, exog=D, k_ar_diff=kdiff, coint_rank=RANGO,
                           deterministic=DET).fit()
                alfa = float(np.asarray(res.alpha)[0, 0])
                talfa = float(np.asarray(res.tvalues_alpha)[0, 0])
                lam_c, _ = valores_propios(Y.values, kdiff,
                                           None if D is None else D.values, ecuaciones=[0])
                lam_f, _ = valores_propios(Y.values, kdiff,
                                           None if D is None else D.values)
                lr = max(n_ef * float(np.sum(np.log(1 - lam_c[:RANGO])
                                             - np.log(1 - lam_f[:RANGO]))), 0.0)
                p_ex = float(stats.chi2.sf(lr, RANGO * (k - 1)))
                try:
                    p_no = float(res.test_normality().pvalue)
                except Exception:
                    p_no = np.nan
                try:
                    p_au = float(res.test_whiteness(nlags=max(kdiff + 2, 4)).pvalue)
                except Exception:
                    p_au = np.nan
            except Exception as exc:
                alfa = talfa = lr = p_ex = p_no = p_au = np.nan

            sig = "***" if abs(talfa) > 2.75 else "**" if abs(talfa) > 2.05 else \
                  "*" if abs(talfa) > 1.70 else ""
            print("   p={:<4} {:<9} {:>9.3f} {:>10.2f} {:>7} {:>9.4f}{:<3} {:>7.2f} {:>10.4f} "
                  "{:>8.3f} {:>8.3f}".format(
                      kdiff + 1, "con d2020" if usa_d else "sin", tr[0], cv95_0, rango,
                      alfa, sig, talfa, p_ex, p_no, p_au))

            filas.append({
                "Pais": nombre, "Sistema": clave, "Orden en niveles": kdiff + 1,
                "Rezagos en diferencias": kdiff, "Ficticia 2020": "Si" if usa_d else "No",
                "n efectiva": n_ef, "Traza r<=0": round(float(tr[0]), 3),
                "VC simulado 95%": round(cv95_0, 2),
                "VC tabulado 95%": float(coint_johansen(
                    Y.values, det_order=0, k_ar_diff=kdiff).cvt[0][1]),
                "p simulada": round(p0, 4), "Rango": rango,
                "Alfa producto": round(alfa, 4) if alfa == alfa else None,
                "t alfa producto": round(talfa, 2) if talfa == talfa else None,
                "Signo del ajuste": ("corrector" if alfa < 0 else "divergente")
                                    if alfa == alfa else None,
                "LR exogeneidad debil conjunta": round(lr, 3) if lr == lr else None,
                "p exogeneidad": round(p_ex, 4) if p_ex == p_ex else None,
                "Normalidad (p)": round(p_no, 3) if p_no == p_no else None,
                "Autocorrelacion (p)": round(p_au, 3) if p_au == p_au else None})
    print()

res_df = pd.DataFrame(filas)
ruta = os.path.join(SAL, "cuadro16_sensibilidad_sistema.csv")
res_df.to_csv(ruta, index=False)

print("=" * 104)
print("Resumen: proporcion de configuraciones con rango mayor o igual a uno")
for (pa, si), sub in res_df.groupby(["Pais", "Sistema"], sort=False):
    con = int((sub["Rango"] >= 1).sum())
    div = int((sub["Signo del ajuste"] == "divergente").sum())
    print("   {:<16} {:<5}  rango>=1 en {} de {}   ajuste divergente del producto en {} de {}".format(
        pa, si, con, len(sub), div, len(sub)))
print("\nTiempo total: {:.0f} s".format(time.time() - t_ini))
print("Guardado: {}".format(ruta))

Validacion: discrepancia maxima = 2.166e-11  correcta

China  S-A   n=34 (1990-2023)   variables: PIB per capita real (log), Formacion bruta de capital fijo, Credito al sector privado (BM), Apertura comercial
   orden  ficticia  traza r=0  VC sim 95   rango alfa producto         t   LR exog.    norm.   autoc.
   p=1    sin          77.003      53.00       2   -0.0673***   -7.33     0.0123    0.446    0.265
   p=1    con d2020    75.650      54.70       2   -0.0657***   -6.54     0.0204    0.351    0.173
   p=2    sin          64.682      61.60       1   -0.0378      -1.56     0.0022    0.552    0.219
   p=2    con d2020    63.728      63.50       1   -0.0315      -1.43     0.0016    0.165    0.296
   p=3    sin          63.775      74.42       0   -0.0994***   -3.93     0.0007    0.714    0.111
   p=3    con d2020    60.393      77.62       0   -0.0701***   -2.96     0.0014    0.294    0.096

China  S-B   n=32 (1990-2021)   variables: PIB per capita real (log), Formacion bruta de capit

In [42]:
# -*- coding: utf-8 -*-
"""
Paso 2. Procedencia de los cuadros 3 y 6.
Compara las celdas 25 y 26, revisa las marcas de tiempo de las salidas y
determina si el cuadro de raices unitarias incluye pruebas KPSS.
"""

import os
import re
import glob
import time
import difflib
import pandas as pd

BASE = "/content/drive/MyDrive/tesis_china_eeuu"
CELDAS = os.path.join(BASE, "scripts/celdas")
OUT = os.path.join(BASE, "outputs")

# ==========================================================================
# 1. Comparacion de las celdas 25 y 26
# ==========================================================================
print("=" * 78)
print("1. CELDAS 25 Y 26")
print("=" * 78)

txt = {}
for etiq in ("25", "26"):
    ruta = os.path.join(CELDAS, "celda_{}.py".format(etiq))
    if not os.path.exists(ruta):
        print("celda_{}.py  NO ENCONTRADA en {}".format(etiq, CELDAS))
        continue
    txt[etiq] = open(ruta, encoding="utf-8").read()
    print("celda_{}.py  {:,} bytes  {} lineas".format(
        etiq, len(txt[etiq]), txt[etiq].count("\n")))

if len(txt) == 2:
    print("\n" + "-" * 78)
    print("ARCHIVOS QUE ESCRIBE CADA CELDA")
    print("-" * 78)
    for etiq in ("25", "26"):
        salidas = re.findall(r'to_csv\(\s*[^)]*?["\']([^"\']+\.csv)', txt[etiq])
        salidas += re.findall(r'\.save\(\s*[^)]*?["\']([^"\']+\.docx)', txt[etiq])
        salidas += re.findall(r'["\']([^"\']*cuadro[^"\']*\.(?:csv|docx))["\']',
                              txt[etiq], re.I)
        print("\ncelda_{}:".format(etiq))
        vistos = sorted(set(os.path.basename(x) for x in salidas))
        for s in vistos:
            print("   ", s)
        if not vistos:
            print("    (ninguno detectado)")

    print("\n" + "-" * 78)
    print("PRUEBAS Y FUNCIONES INVOCADAS")
    print("-" * 78)
    CLAVES = ["adfuller", "kpss", "PhillipsPerron", "phillips", "zivot_andrews",
              "breakvar", "variance_inflation_factor", "corr(", "Document(",
              "add_table", "describe(", "skew", "kurtosis"]
    print("{:<32} {:>6} {:>6}".format("clave", "c25", "c26"))
    for k in CLAVES:
        a, b = txt["25"].lower().count(k.lower()), txt["26"].lower().count(k.lower())
        if a or b:
            print("{:<32} {:>6} {:>6}".format(k, a, b))

    print("\n" + "-" * 78)
    print("CUADROS MENCIONADOS EN EL CODIGO")
    print("-" * 78)
    for etiq in ("25", "26"):
        m = sorted(set(re.findall(r'[Cc]uadro\s*\d+[a-z]?', txt[etiq])))
        print("celda_{}: {}".format(etiq, ", ".join(m) if m else "ninguno"))

    print("\n" + "-" * 78)
    print("DIFERENCIAS (primeras 120 lineas del diff unificado)")
    print("-" * 78)
    d = difflib.unified_diff(txt["25"].splitlines(), txt["26"].splitlines(),
                             "celda_25", "celda_26", lineterm="", n=1)
    for i, ln in enumerate(d):
        if i >= 120:
            print("... (truncado)")
            break
        print(ln)

# ==========================================================================
# 2. Marcas de tiempo de las salidas
# ==========================================================================
print("\n" + "=" * 78)
print("2. MARCAS DE TIEMPO")
print("=" * 78)

patrones = [os.path.join(CELDAS, "celda_2*.py"),
            os.path.join(OUT, "cuadro3*"), os.path.join(OUT, "cuadro6*"),
            os.path.join(OUT, "Cuadros_resultados_APA*.docx"),
            "/content/outputs/cuadro3*", "/content/outputs/cuadro6*"]
encontrados = sorted({f for p in patrones for f in glob.glob(p)},
                     key=lambda f: os.path.getmtime(f))
if not encontrados:
    print("   no se encontraron archivos con esos patrones")
for f in encontrados:
    print("   {}   {:>10,} bytes   {}".format(
        time.strftime("%Y-%m-%d %H:%M", time.localtime(os.path.getmtime(f))),
        os.path.getsize(f), os.path.basename(f)))

# ==========================================================================
# 3. Contenido del cuadro de raices unitarias: hubo KPSS?
# ==========================================================================
print("\n" + "=" * 78)
print("3. CONTENIDO DEL CUADRO 3 (RAICES UNITARIAS)")
print("=" * 78)

candidatos = sorted(glob.glob(os.path.join(OUT, "cuadro3*.csv")))
if not candidatos:
    print("   no se encontro ningun cuadro3*.csv")
for ruta in candidatos:
    print("\n--- {}".format(os.path.basename(ruta)))
    c3 = pd.read_csv(ruta)
    print("   dimensiones: {} filas x {} columnas".format(*c3.shape))
    print("   columnas: {}".format(list(c3.columns)))
    texto = " ".join(map(str, c3.columns)).lower() + " " + \
            " ".join(c3.astype(str).values.ravel()).lower()
    for prueba, marcas in (("Dickey-Fuller aumentada", ["adf", "dickey"]),
                           ("Phillips-Perron", ["pp", "phillips", "perron"]),
                           ("KPSS", ["kpss"]),
                           ("Zivot-Andrews", ["zivot", "andrews"])):
        hay = any(m in texto for m in marcas)
        print("   {:<26} {}".format(prueba, "PRESENTE" if hay else "ausente"))
    print(c3.head(8).to_string(index=False))

# ==========================================================================
# 4. Contenido del cuadro 6 (factores de inflacion de varianza)
# ==========================================================================
print("\n" + "=" * 78)
print("4. CONTENIDO DEL CUADRO 6 (VIF)")
print("=" * 78)
for ruta in sorted(glob.glob(os.path.join(OUT, "cuadro6*.csv"))):
    c6 = pd.read_csv(ruta)
    print("\n--- {}   {} filas x {} columnas".format(
        os.path.basename(ruta), *c6.shape))
    print("   columnas: {}".format(list(c6.columns)))
    print(c6.head(10).to_string(index=False))

1. CELDAS 25 Y 26
celda_25.py  14,026 bytes  374 lineas
celda_26.py  7,780 bytes  208 lineas

------------------------------------------------------------------------------
ARCHIVOS QUE ESCRIBE CADA CELDA
------------------------------------------------------------------------------

celda_25:
    Cuadros_diagnostico_v2_APA.docx
    cuadro3_raiz_unitaria_v2.csv
    cuadro6_vif_v2.csv

celda_26:
    cuadro3b_segunda_ronda.csv
    cuadro6b_vif_por_modelo.csv

------------------------------------------------------------------------------
PRUEBAS Y FUNCIONES INVOCADAS
------------------------------------------------------------------------------
clave                               c25    c26
adfuller                              2      2
PhillipsPerron                        2      2
phillips                              2      2
zivot_andrews                         2      2
variance_inflation_factor             2      2
Document(                             1      0
add_table            

In [45]:
# -*- coding: utf-8 -*-
"""
Cuadro 3 version 3. Raiz unitaria con las tres pruebas.
ADF y PP (nula: raiz unitaria) mas KPSS (nula: estacionariedad),
en constante y en constante con tendencia, sobre niveles y diferencias.
Salida: outputs/cuadro3_raiz_unitaria_v3.csv
"""

import os
import sys
import subprocess
import warnings

# ------------------------------------------------ dependencias y Drive
try:
    import arch  # noqa: F401
except ModuleNotFoundError:
    print("instalando arch ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "arch"],
                   check=True)
    print("arch instalado")

if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

import numpy as np
import pandas as pd
from statsmodels.tsa.stattools import adfuller, kpss
from arch.unitroot import PhillipsPerron, ZivotAndrews

warnings.filterwarnings("ignore")

BASE = "/content/drive/MyDrive/tesis_china_eeuu"
PANEL = os.path.join(BASE, "data_clean/china_us_panel_analisis_v6_1990_2023.csv")
OUT = os.path.join(BASE, "outputs")
SALIDA = os.path.join(OUT, "cuadro3_raiz_unitaria_v3.csv")
V2 = os.path.join(OUT, "cuadro3_raiz_unitaria_v2.csv")

SERIES = [
    ("ln_gdp_pc_pwt",      "log PIB per capita real",             True),
    ("gdp_growth_pct",     "Crecimiento del PIB (%)",             False),
    ("inv_gfcf_gdp",       "FBCF (% del PIB)",                    False),
    ("trade_gdp",          "Apertura comercial (% del PIB)",      True),
    ("inflation_cpi_pct",  "Inflacion IPC (%)",                   False),
    ("credit_priv_gdp",    "Credito privado, WDI (% del PIB)",    True),
    ("bis_pvt_credit_gdp", "Credito privado, BIS (% del PIB)",    True),
    ("priv_debt_ls_gdp",   "Deuda privada, FMI GDD (% del PIB)",  True),
    ("pub_debt_gdp",       "Deuda publica (% del PIB)",           True),
    ("hh_debt_gdp",        "Deuda de hogares (% del PIB)",        True),
    ("corp_debt_gdp",      "Deuda corporativa (% del PIB)",       True),
    ("ratio_ind_agr_rec",  "Brecha productividad ind./agr.",      True),
    ("IIC_alt",            "IIC (indice compuesto)",              True),
    ("hc",                 "Indice de capital humano",            True),
    ("ln_rtfpna",          "log PTF (PWT)",                       True),
]

PAISES = [("CN", "China"), ("US", "Estados Unidos")]
MIN_OBS = 15


# ------------------------------------------------------------ utilidades
def estrellas(p):
    if p is None or not np.isfinite(p):
        return ""
    if p <= 0.01:
        return "***"
    if p <= 0.05:
        return "**"
    if p <= 0.10:
        return "*"
    return ""


def fmt(stat, p):
    if stat is None or not np.isfinite(stat):
        return "—"
    return "{:.2f}{}".format(stat, estrellas(p))


def t_adf(y, reg):
    try:
        r = adfuller(np.asarray(y, float), regression=reg, autolag="AIC")
        return r[0], r[1]
    except Exception:
        return None, None


def t_pp(y, trend):
    try:
        r = PhillipsPerron(np.asarray(y, float), trend=trend)
        return r.stat, r.pvalue
    except Exception:
        return None, None


def t_kpss(y, reg):
    """p acotado a [0.01, 0.10] por interpolacion de la tabla."""
    try:
        s, p, _, _ = kpss(np.asarray(y, float), regression=reg, nlags="auto")
        return s, p
    except Exception:
        return None, None


def t_za(y, trend):
    try:
        r = ZivotAndrews(np.asarray(y, float), trend=trend)
        return r.stat, r.pvalue
    except Exception:
        return None, None


def veredicto(pa_n, pp_n, pk_n, pa_d, pp_d, pk_d):
    """
    I(0)  : ADF o PP rechazan en nivel Y KPSS no rechaza en nivel
    I(1)  : no hay rechazo firme en nivel, si en la diferencia,
            y KPSS no rechaza en la diferencia
    """
    def v(x):
        return x if (x is not None and np.isfinite(x)) else np.nan

    rn = (v(pa_n) <= .05) or (v(pp_n) <= .05)
    kn = v(pk_n) > .05
    rd = (v(pa_d) <= .05) or (v(pp_d) <= .05)
    kd = v(pk_d) > .05

    if rn and kn:
        return "I(0)"
    if rn and not kn:
        return "I(0)/I(1) no concluyente"
    if rd and kd:
        return "I(1)"
    if rd and not kd:
        return "I(1) con reservas"
    return "I(2) o superior"


# ------------------------------------------- anios de quiebre de la v2
anio_quiebre = {}
if os.path.exists(V2):
    v2 = pd.read_csv(V2)
    col_za = [c for c in v2.columns if "Zivot" in c]
    if col_za:
        for _, r in v2.iterrows():
            txt = str(r[col_za[0]])
            if "[" in txt and "]" in txt:
                a = txt.split("[")[1].split("]")[0].strip()
                if a.isdigit():
                    anio_quiebre[(str(r["Pais"]).strip(),
                                  str(r["Serie"]).strip())] = int(a)
    print("anios de quiebre recuperados de la v2: {}".format(len(anio_quiebre)))
else:
    print("aviso: no se encontro la v2, la columna de quiebre ira sin anio")


# --------------------------------------------------------------- calculo
df = pd.read_csv(PANEL)
print("panel: {} filas x {} columnas".format(*df.shape))

filas = []
for cod, nombre in PAISES:
    sub = df[df["country"] == cod].sort_values("year")
    for var, etiq, con_tend in SERIES:
        if var not in sub.columns:
            continue
        s = sub[[var]].dropna()
        y = s[var].astype(float)
        n = len(y)

        if n < MIN_OBS:
            filas.append({"Pais": nombre, "Serie": etiq, "n": n,
                          "ADF (c)": "—", "ADF (c+t)": "—",
                          "PP (c)": "—", "PP (c+t)": "—",
                          "KPSS (c)": "—", "KPSS (c+t)": "—",
                          "ADF D": "—", "PP D": "—", "KPSS D": "—",
                          "Zivot-Andrews": "—",
                          "Orden": "muestra insuficiente"})
            continue

        dy = y.diff().dropna()
        tr = "ct" if con_tend else "c"

        a_c, pa_c = t_adf(y, "c")
        a_t, pa_t = t_adf(y, "ct")
        p_c, pp_c = t_pp(y, "c")
        p_t, pp_t = t_pp(y, "ct")
        k_c, pk_c = t_kpss(y, "c")
        k_t, pk_t = t_kpss(y, "ct")

        a_d, pa_d = t_adf(dy, "c")
        p_d, pp_d = t_pp(dy, "c")
        k_d, pk_d = t_kpss(dy, "c")

        za_s, za_p = t_za(y, tr)
        anio = anio_quiebre.get((nombre, etiq))
        za_txt = "—" if za_s is None else "{:.2f}{}{}".format(
            za_s, estrellas(za_p), " [{}]".format(anio) if anio else "")

        pa_n = pa_t if con_tend else pa_c
        pp_n = pp_t if con_tend else pp_c
        pk_n = pk_t if con_tend else pk_c

        filas.append({
            "Pais": nombre, "Serie": etiq, "n": n,
            "ADF (c)": fmt(a_c, pa_c), "ADF (c+t)": fmt(a_t, pa_t),
            "PP (c)": fmt(p_c, pp_c), "PP (c+t)": fmt(p_t, pp_t),
            "KPSS (c)": fmt(k_c, pk_c), "KPSS (c+t)": fmt(k_t, pk_t),
            "ADF D": fmt(a_d, pa_d), "PP D": fmt(p_d, pp_d),
            "KPSS D": fmt(k_d, pk_d),
            "Zivot-Andrews": za_txt,
            "Orden": veredicto(pa_n, pp_n, pk_n, pa_d, pp_d, pk_d),
        })

res = pd.DataFrame(filas)
os.makedirs(OUT, exist_ok=True)
res.to_csv(SALIDA, index=False, encoding="utf-8-sig")

pd.set_option("display.width", 260)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 40)

print("\n" + "=" * 78)
print("CUADRO 3 v3")
print("=" * 78)
print(res.to_string(index=False))

print("\n" + "=" * 78)
print("DISTRIBUCION DE VEREDICTOS")
print("=" * 78)
print(res["Orden"].value_counts().to_string())

print("\n" + "=" * 78)
print("SERIES DE LOS MODELOS PRINCIPALES")
print("=" * 78)
CLAVE = ["log PIB per capita real", "FBCF (% del PIB)",
         "Apertura comercial (% del PIB)", "Credito privado, WDI (% del PIB)",
         "Credito privado, BIS (% del PIB)",
         "Deuda privada, FMI GDD (% del PIB)"]
print(res[res["Serie"].isin(CLAVE)][
    ["Pais", "Serie", "ADF (c+t)", "PP (c+t)", "KPSS (c+t)",
     "ADF D", "PP D", "KPSS D", "Orden"]].to_string(index=False))

print("\nguardado en:", SALIDA)

instalando arch ...
arch instalado
anios de quiebre recuperados de la v2: 26
panel: 68 filas x 148 columnas

CUADRO 3 v3
          Pais                              Serie  n  ADF (c) ADF (c+t)   PP (c) PP (c+t) KPSS (c) KPSS (c+t)    ADF D      PP D  KPSS D   Zivot-Andrews                    Orden
         China            log PIB per capita real 34    -1.62     -0.73  -3.13**     0.47  0.79***     0.19**    -1.37    -2.65*  0.53**           -3.67          I(2) o superior
         China            Crecimiento del PIB (%) 34    -1.72    -3.33*  -3.23** -4.67***   0.48**      0.14* -7.31***  -9.97***   0.15*  -5.25** [2003] I(0)/I(1) no concluyente
         China                   FBCF (% del PIB) 34   -2.64*    -3.25*  -2.98**    -2.36  0.83***     0.16** -4.60***  -3.73***   0.33*  -4.83** [2002] I(0)/I(1) no concluyente
         China     Apertura comercial (% del PIB) 34    -2.01     -1.76    -1.89    -1.35    0.25*     0.20** -3.88***  -3.76***   0.28*    -4.83 [2002]               

In [46]:
# -*- coding: utf-8 -*-
"""
Cuadro 3 definitivo. Raiz unitaria con ADF, PP y KPSS.
Corrige el tope de probabilidad de KPSS, agrega la configuracion con
tendencia sobre las diferencias y documenta dos comprobaciones criticas.
Salidas: outputs/cuadro3_raiz_unitaria_v4.csv
         outputs/cuadro3c_comprobaciones.csv
"""

import os
import sys
import subprocess
import warnings

try:
    import arch  # noqa: F401
except ModuleNotFoundError:
    print("instalando arch ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "arch"],
                   check=True)

if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, kpss
from arch.unitroot import PhillipsPerron, ZivotAndrews

warnings.filterwarnings("ignore")

BASE = "/content/drive/MyDrive/tesis_china_eeuu"
PANEL = os.path.join(BASE, "data_clean/china_us_panel_analisis_v6_1990_2023.csv")
OUT = os.path.join(BASE, "outputs")
V2 = os.path.join(OUT, "cuadro3_raiz_unitaria_v2.csv")
SAL_A = os.path.join(OUT, "cuadro3_raiz_unitaria_v4.csv")
SAL_B = os.path.join(OUT, "cuadro3c_comprobaciones.csv")

# Valores criticos de KPSS (Kwiatkowski et al., 1992)
KPSS_VC = {"c":  {0.10: 0.347, 0.05: 0.463, 0.01: 0.739},
           "ct": {0.10: 0.119, 0.05: 0.146, 0.01: 0.216}}

SERIES = [
    ("ln_gdp_pc_pwt",      "log PIB per capita real",             True),
    ("gdp_growth_pct",     "Crecimiento del PIB (%)",             False),
    ("inv_gfcf_gdp",       "FBCF (% del PIB)",                    False),
    ("trade_gdp",          "Apertura comercial (% del PIB)",      True),
    ("inflation_cpi_pct",  "Inflacion IPC (%)",                   False),
    ("credit_priv_gdp",    "Credito privado, WDI (% del PIB)",    True),
    ("bis_pvt_credit_gdp", "Credito privado, BIS (% del PIB)",    True),
    ("priv_debt_ls_gdp",   "Deuda privada, FMI GDD (% del PIB)",  True),
    ("pub_debt_gdp",       "Deuda publica (% del PIB)",           True),
    ("hh_debt_gdp",        "Deuda de hogares (% del PIB)",        True),
    ("corp_debt_gdp",      "Deuda corporativa (% del PIB)",       True),
    ("ratio_ind_agr_rec",  "Brecha productividad ind./agr.",      True),
    ("IIC_alt",            "IIC (indice compuesto)",              True),
    ("hc",                 "Indice de capital humano",            True),
    ("ln_rtfpna",          "log PTF (PWT)",                       True),
]

PAISES = [("CN", "China"), ("US", "Estados Unidos")]
MIN_OBS = 15


# ---------------------------------------------------------- formateo
def est_p(p):
    """Asteriscos para pruebas con probabilidad exacta (ADF, PP, ZA)."""
    if p is None or not np.isfinite(p):
        return ""
    if p <= 0.01:
        return "***"
    if p <= 0.05:
        return "**"
    if p < 0.10:
        return "*"
    return ""


def est_kpss(stat, reg):
    """Asteriscos de KPSS por comparacion directa con la tabla."""
    if stat is None or not np.isfinite(stat):
        return ""
    vc = KPSS_VC[reg]
    if stat >= vc[0.01]:
        return "***"
    if stat >= vc[0.05]:
        return "**"
    if stat >= vc[0.10]:
        return "*"
    return ""


def fmt_p(stat, p):
    return "—" if (stat is None or not np.isfinite(stat)) \
        else "{:.2f}{}".format(stat, est_p(p))


def fmt_k(stat, reg):
    return "—" if (stat is None or not np.isfinite(stat)) \
        else "{:.2f}{}".format(stat, est_kpss(stat, reg))


# ---------------------------------------------------------- pruebas
def t_adf(y, reg):
    try:
        r = adfuller(np.asarray(y, float), regression=reg, autolag="AIC")
        return r[0], r[1]
    except Exception:
        return None, None


def t_pp(y, trend):
    try:
        r = PhillipsPerron(np.asarray(y, float), trend=trend)
        return r.stat, r.pvalue
    except Exception:
        return None, None


def t_kpss(y, reg):
    try:
        s, _, _, _ = kpss(np.asarray(y, float), regression=reg, nlags="auto")
        return s
    except Exception:
        return None


def rechaza_kpss(stat, reg, nivel=0.05):
    if stat is None or not np.isfinite(stat):
        return None
    return bool(stat >= KPSS_VC[reg][nivel])


def veredicto(pa_n, pp_n, k_n, reg_n,
              pa_d, pp_d, k_d, pa_dt, pp_dt, k_dt):
    """
    Nivel : ADF o PP rechazan al 5% Y KPSS no rechaza al 5%   -> I(0)
    Dif.  : ADF o PP rechazan al 5% Y KPSS no rechaza al 5%   -> I(1)
            si solo se logra con tendencia en la diferencia,
            se marca como I(1) con tendencia en la deriva
    """
    def v(x):
        return x if (x is not None and np.isfinite(x)) else np.nan

    rn = (v(pa_n) <= .05) or (v(pp_n) <= .05)
    kn = not rechaza_kpss(k_n, reg_n) if k_n is not None else False
    if rn and kn:
        return "I(0)"
    if rn and not kn:
        return "I(0)/I(1) no concluyente"

    rd = (v(pa_d) <= .05) or (v(pp_d) <= .05)
    kd = not rechaza_kpss(k_d, "c") if k_d is not None else False
    if rd and kd:
        return "I(1)"

    rdt = (v(pa_dt) <= .05) or (v(pp_dt) <= .05)
    kdt = not rechaza_kpss(k_dt, "ct") if k_dt is not None else False
    if rdt and kdt:
        return "I(1) con deriva en tendencia"
    if rd or rdt:
        return "I(1) con reservas"
    return "I(2) o superior"


# ------------------------------------------- anios de quiebre de la v2
anio_q = {}
if os.path.exists(V2):
    v2 = pd.read_csv(V2)
    cz = [c for c in v2.columns if "Zivot" in c]
    if cz:
        for _, r in v2.iterrows():
            t = str(r[cz[0]])
            if "[" in t and "]" in t:
                a = t.split("[")[1].split("]")[0].strip()
                if a.isdigit():
                    anio_q[(str(r["Pais"]).strip(), str(r["Serie"]).strip())] = int(a)

# ------------------------------------------------------------- calculo
df = pd.read_csv(PANEL)
print("panel: {} filas x {} columnas".format(*df.shape))

filas = []
for cod, nombre in PAISES:
    sub = df[df["country"] == cod].sort_values("year")
    for var, etiq, con_tend in SERIES:
        if var not in sub.columns:
            continue
        y = sub[var].dropna().astype(float)
        n = len(y)
        if n < MIN_OBS:
            filas.append({"Pais": nombre, "Serie": etiq, "n": n,
                          "Orden": "muestra insuficiente"})
            continue

        dy = y.diff().dropna()
        reg_n = "ct" if con_tend else "c"

        a_c, pa_c = t_adf(y, "c")
        a_t, pa_t = t_adf(y, "ct")
        p_c, pp_c = t_pp(y, "c")
        p_t, pp_t = t_pp(y, "ct")
        k_c = t_kpss(y, "c")
        k_t = t_kpss(y, "ct")

        a_d, pa_d = t_adf(dy, "c")
        p_d, pp_d = t_pp(dy, "c")
        k_d = t_kpss(dy, "c")
        a_dt, pa_dt = t_adf(dy, "ct")
        p_dt, pp_dt = t_pp(dy, "ct")
        k_dt = t_kpss(dy, "ct")

        za_s, za_p = None, None
        try:
            za = ZivotAndrews(np.asarray(y, float), trend=reg_n)
            za_s, za_p = za.stat, za.pvalue
        except Exception:
            pass
        anio = anio_q.get((nombre, etiq))
        za_txt = "—" if za_s is None else "{:.2f}{}{}".format(
            za_s, est_p(za_p), " [{}]".format(anio) if anio else "")

        pa_n = pa_t if con_tend else pa_c
        pp_n = pp_t if con_tend else pp_c
        k_n = k_t if con_tend else k_c

        filas.append({
            "Pais": nombre, "Serie": etiq, "n": n,
            "ADF (c)": fmt_p(a_c, pa_c), "ADF (c+t)": fmt_p(a_t, pa_t),
            "PP (c)": fmt_p(p_c, pp_c), "PP (c+t)": fmt_p(p_t, pp_t),
            "KPSS (c)": fmt_k(k_c, "c"), "KPSS (c+t)": fmt_k(k_t, "ct"),
            "ADF D (c)": fmt_p(a_d, pa_d), "PP D (c)": fmt_p(p_d, pp_d),
            "KPSS D (c)": fmt_k(k_d, "c"),
            "ADF D (c+t)": fmt_p(a_dt, pa_dt), "PP D (c+t)": fmt_p(p_dt, pp_dt),
            "KPSS D (c+t)": fmt_k(k_dt, "ct"),
            "Zivot-Andrews": za_txt,
            "Orden": veredicto(pa_n, pp_n, k_n, reg_n,
                               pa_d, pp_d, k_d, pa_dt, pp_dt, k_dt),
        })

res = pd.DataFrame(filas)
os.makedirs(OUT, exist_ok=True)
res.to_csv(SAL_A, index=False, encoding="utf-8-sig")

pd.set_option("display.width", 300)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 42)

print("\n" + "=" * 78)
print("CUADRO 3 v4 — NIVELES")
print("=" * 78)
print(res[["Pais", "Serie", "n", "ADF (c)", "ADF (c+t)", "PP (c)",
           "PP (c+t)", "KPSS (c)", "KPSS (c+t)"]].to_string(index=False))

print("\n" + "=" * 78)
print("CUADRO 3 v4 — DIFERENCIAS Y VEREDICTO")
print("=" * 78)
print(res[["Pais", "Serie", "ADF D (c)", "PP D (c)", "KPSS D (c)",
           "ADF D (c+t)", "PP D (c+t)", "KPSS D (c+t)",
           "Zivot-Andrews", "Orden"]].to_string(index=False))

print("\n" + "=" * 78)
print("DISTRIBUCION DE VEREDICTOS")
print("=" * 78)
print(res["Orden"].value_counts().to_string())

# =====================================================================
# COMPROBACIONES CRITICAS
# =====================================================================
comp = []

print("\n" + "=" * 78)
print("A. DESACELERACION CHINA: hay tendencia en la tasa de crecimiento?")
print("=" * 78)
cn = df[df["country"] == "CN"].sort_values("year")
g = cn["ln_gdp_pc_pwt"].diff().dropna()
tt = np.arange(len(g), dtype=float)
X = sm.add_constant(np.column_stack([tt]))
m = sm.OLS(np.asarray(g, float), X).fit(cov_type="HAC",
                                        cov_kwds={"maxlags": 3})
print("D log PIB pc = {:.5f} {:+.6f} t".format(m.params[0], m.params[1]))
print("   t de la tendencia = {:.3f}   p = {:.4f}   R2 = {:.3f}".format(
    m.tvalues[1], m.pvalues[1], m.rsquared))
print("   media 1991-2000 = {:.4f}   media 2014-2023 = {:.4f}".format(
    cn.loc[cn.year.between(1991, 2000), "ln_gdp_pc_pwt"].diff().mean(),
    cn.loc[cn.year.between(2014, 2023), "ln_gdp_pc_pwt"].diff().mean()))
comp.append({"Comprobacion": "Tendencia en D log PIB pc (China)",
             "Estadistico": round(float(m.tvalues[1]), 3),
             "p": round(float(m.pvalues[1]), 4),
             "Lectura": "tendencia significativa en la deriva"
             if m.pvalues[1] < .05 else "sin tendencia en la deriva"})

print("\n" + "=" * 78)
print("B. FBCF ESTADOUNIDENSE: es realmente I(0)?")
print("=" * 78)
us = df[df["country"] == "US"].sort_values("year")
f = us["inv_gfcf_gdp"].dropna().astype(float)
for reg, etq in (("c", "constante"), ("ct", "constante y tendencia")):
    a, pa = t_adf(f, reg)
    p_, pp_ = t_pp(f, reg)
    k_ = t_kpss(f, reg)
    print("{:<24} ADF {:>7.2f} (p {:.3f})   PP {:>7.2f} (p {:.3f})   "
          "KPSS {:.3f} [{}5% = {}]".format(
              etq, a, pa, p_, pp_, k_,
              "vc ", KPSS_VC[reg][0.05]))
    comp.append({"Comprobacion": "FBCF EE. UU., {}".format(etq),
                 "Estadistico": round(float(a), 3), "p": round(float(pa), 4),
                 "Lectura": "estacionaria" if (pa < .05 and not
                            rechaza_kpss(k_, reg)) else "no concluyente"})

print("\nSubmuestra sin la crisis (1990-2007):")
f2 = us.loc[us.year <= 2007, "inv_gfcf_gdp"].dropna().astype(float)
a2, pa2 = t_adf(f2, "c")
print("   ADF (c) = {:.2f}   p = {:.3f}   n = {}".format(a2, pa2, len(f2)))

print("\n" + "=" * 78)
print("C. ORDENES DE INTEGRACION DE CADA SISTEMA DE JOHANSEN")
print("=" * 78)
SIST = {
    "China S-A":  ("CN", ["ln_gdp_pc_pwt", "inv_gfcf_gdp",
                          "credit_priv_gdp", "trade_gdp"]),
    "China S-B":  ("CN", ["ln_gdp_pc_pwt", "inv_gfcf_gdp",
                          "bis_pvt_credit_gdp", "trade_gdp"]),
    "China S-C":  ("CN", ["ln_gdp_pc_pwt", "inv_gfcf_gdp",
                          "priv_debt_ls_gdp", "trade_gdp"]),
    "EE. UU. S-A": ("US", ["ln_gdp_pc_pwt", "inv_gfcf_gdp",
                           "credit_priv_gdp", "trade_gdp"]),
}
mapa = {(r["Pais"], r["Serie"]): r["Orden"] for _, r in res.iterrows()}
etiqueta = {v[0]: v[1] for v in [(s[0], s[1]) for s in SERIES]}
pais_nom = {"CN": "China", "US": "Estados Unidos"}
for nom, (cod, vs) in SIST.items():
    print("\n{}".format(nom))
    ok = True
    for v_ in vs:
        o = mapa.get((pais_nom[cod], etiqueta.get(v_, v_)), "?")
        print("   {:<22} {}".format(v_, o))
        if o.startswith("I(0)") and o == "I(0)":
            ok = False
    print("   -> {}".format("todas I(1), Johansen aplicable" if ok else
                            "MEZCLA DE ORDENES, rango no interpretable"))
    comp.append({"Comprobacion": "Sistema {}".format(nom),
                 "Estadistico": np.nan, "p": np.nan,
                 "Lectura": "todas I(1)" if ok else "mezcla I(0)/I(1)"})

pd.DataFrame(comp).to_csv(SAL_B, index=False, encoding="utf-8-sig")
print("\nguardado en:\n  {}\n  {}".format(SAL_A, SAL_B))

panel: 68 filas x 148 columnas

CUADRO 3 v4 — NIVELES
          Pais                              Serie  n  ADF (c) ADF (c+t)   PP (c) PP (c+t) KPSS (c) KPSS (c+t)
         China            log PIB per capita real 34    -1.62     -0.73  -3.13**     0.47  0.79***     0.19**
         China            Crecimiento del PIB (%) 34    -1.72    -3.33*  -3.23** -4.67***   0.48**      0.14*
         China                   FBCF (% del PIB) 34   -2.64*    -3.25*  -2.98**    -2.36  0.83***     0.16**
         China     Apertura comercial (% del PIB) 34    -2.01     -1.76    -1.89    -1.35     0.25     0.20**
         China                  Inflacion IPC (%) 34    -2.28     -2.86    -1.75    -2.17     0.29       0.08
         China   Credito privado, WDI (% del PIB) 34     0.69     -1.53     1.96    -1.30   0.73**     0.18**
         China   Credito privado, BIS (% del PIB) 32     2.59     -0.73     0.86    -2.25  0.85***     0.17**
         China Deuda privada, FMI GDD (% del PIB) 34     2.06     

In [47]:
# -*- coding: utf-8 -*-
"""
Paso 4f. Especificacion deterministica de los sistemas chinos.
Compara el caso III (constante irrestricta) con el caso IV (tendencia
lineal restringida al espacio de cointegracion) y contrasta por razon
de verosimilitud si la tendencia es necesaria.
Salida: outputs/cuadro17_deterministicos.csv
"""

import os
import warnings
import numpy as np
import pandas as pd
from numpy.linalg import pinv, eig
from scipy import stats
from statsmodels.tsa.vector_ar.vecm import coint_johansen

warnings.filterwarnings("ignore")

if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

BASE = "/content/drive/MyDrive/tesis_china_eeuu"
PANEL = os.path.join(BASE, "data_clean/china_us_panel_analisis_v6_1990_2023.csv")
OUT = os.path.join(BASE, "outputs")
SALIDA = os.path.join(OUT, "cuadro17_deterministicos.csv")

SISTEMAS = {
    "S-A": (["ln_gdp_pc_pwt", "inv_gfcf_gdp", "credit_priv_gdp", "trade_gdp"], 2),
    "S-B": (["ln_gdp_pc_pwt", "inv_gfcf_gdp", "bis_pvt_credit_gdp", "trade_gdp"], 3),
    "S-C": (["ln_gdp_pc_pwt", "inv_gfcf_gdp", "priv_debt_ls_gdp", "trade_gdp"], 2),
}


def rrr(Y, kdiff, dummy, con_tendencia):
    """
    Regresion de rango reducido de Johansen.
    Caso III : Z1 = Y[t-1]            Z2 = [1, dY rezagadas, dummy]
    Caso IV  : Z1 = [Y[t-1], t]       Z2 = [1, dY rezagadas, dummy]
    Devuelve autovalores, n efectivo, y las matrices para el ajuste.
    """
    T, k = Y.shape
    dY = np.diff(Y, axis=0)
    ini = kdiff + 1
    n = T - ini

    Z0 = dY[kdiff:, :]
    Z1 = Y[ini - 1: T - 1, :]
    if con_tendencia:
        t_col = np.arange(ini, T, dtype=float).reshape(-1, 1)
        Z1 = np.hstack([Z1, t_col])

    Z2 = [np.ones((n, 1))]
    for j in range(1, kdiff + 1):
        Z2.append(dY[kdiff - j: T - 1 - j, :])
    if dummy is not None:
        Z2.append(dummy[ini:T].reshape(-1, 1))
    Z2 = np.hstack(Z2)

    M = np.eye(n) - Z2 @ pinv(Z2)
    R0, R1 = M @ Z0, M @ Z1

    S00 = R0.T @ R0 / n
    S01 = R0.T @ R1 / n
    S11 = R1.T @ R1 / n

    A = pinv(S11) @ S01.T @ pinv(S00) @ S01
    val, vec = eig(A)
    val = np.real(val)
    orden = np.argsort(val)[::-1]
    val = np.clip(val[orden], 0.0, 0.999999)
    vec = np.real(vec[:, orden])
    return val, n, k, vec, S11, R0, R1


def traza(val, n, k):
    return np.array([-n * np.sum(np.log(1.0 - val[i:k])) for i in range(k)])


df = pd.read_csv(PANEL)
cn = df[df["country"] == "CN"].sort_values("year").reset_index(drop=True)

filas = []
print("=" * 78)
print("PASO 4F. ESPECIFICACION DETERMINISTICA, SISTEMAS CHINOS")
print("=" * 78)

for nom, (vars_, p) in SISTEMAS.items():
    kdiff = p - 1
    sub = cn[["year", "d2020"] + vars_].dropna().reset_index(drop=True)
    Y = sub[vars_].to_numpy(float)
    d = sub["d2020"].to_numpy(float)
    k = Y.shape[1]

    print("\n" + "-" * 78)
    print("{}   variables: {}".format(nom, ", ".join(vars_)))
    print("   periodo {}-{}   T = {}   orden VAR = {}".format(
        int(sub.year.min()), int(sub.year.max()), len(sub), p))

    v3, n3, _, _, _, _, _ = rrr(Y, kdiff, d, False)   # caso III
    v4, n4, _, _, _, _, _ = rrr(Y, kdiff, d, True)    # caso IV

    tr3, tr4 = traza(v3, n3, k), traza(v4, n4, k)

    # validacion contra la implementacion de referencia (sin ficticia)
    try:
        ref0 = coint_johansen(Y, det_order=0, k_ar_diff=kdiff)
        ref1 = coint_johansen(Y, det_order=1, k_ar_diff=kdiff)
        w3, _, _, _, _, _, _ = rrr(Y, kdiff, None, False)
        w4, _, _, _, _, _, _ = rrr(Y, kdiff, None, True)
        e3 = np.max(np.abs(np.sort(w3)[::-1] - np.sort(ref0.eig)[::-1]))
        e4 = np.max(np.abs(np.sort(w4)[::-1][:k] - np.sort(ref1.eig)[::-1][:k]))
        print("   validacion sin ficticia: caso III {:.2e}   caso IV {:.2e}"
              .format(e3, e4))
    except Exception as exc:
        e3 = e4 = np.nan
        print("   validacion no disponible: {}".format(exc))

    print("\n   {:<6} {:>12} {:>12} {:>12} {:>12}".format(
        "H0", "traza III", "VC 95 III", "traza IV", "VC 95 IV"))
    vc3 = ref0.cvt[:, 1] if not np.isnan(e3) else [np.nan] * k
    vc4 = ref1.cvt[:, 1] if not np.isnan(e4) else [np.nan] * k
    for i in range(k):
        print("   r<={:<3} {:>12.3f} {:>12.2f} {:>12.3f} {:>12.2f}".format(
            i, tr3[i], vc3[i], tr4[i], vc4[i]))

    # ---- contraste de exclusion de la tendencia del espacio de cointegracion
    print("\n   Exclusion de la tendencia (H0: coeficiente de t nulo en beta)")
    for r in (1, 2):
        if r > k:
            continue
        lr = n4 * np.sum(np.log(1.0 - v3[:r]) - np.log(1.0 - v4[:r]))
        lr = float(abs(lr))
        pv = 1.0 - stats.chi2.cdf(lr, r)
        veredicto = ("tendencia NECESARIA" if pv < .05
                     else "tendencia prescindible")
        print("      r = {}   LR = {:8.3f}   gl = {}   p = {:.4f}   {}".format(
            r, lr, r, pv, veredicto))
        filas.append({"Sistema": nom, "Orden VAR": p, "r supuesto": r,
                      "LR tendencia": round(lr, 3), "gl": r,
                      "p": round(pv, 4), "Veredicto": veredicto,
                      "Traza III r<=0": round(float(tr3[0]), 3),
                      "Traza IV r<=0": round(float(tr4[0]), 3),
                      "Error validacion III": e3,
                      "Error validacion IV": e4})

    # ---- vector normalizado bajo cada caso, para comparar magnitudes
    for etiq, val, con_t in (("III", v3, False), ("IV", v4, True)):
        vv, nn, _, vec, S11, R0, R1 = rrr(Y, kdiff, d, con_t)
        beta = vec[:, 0] / vec[0, 0]
        nombres = vars_ + (["tendencia"] if con_t else [])
        txt = "   ".join("{} = {:+.4f}".format(nm, -b)
                         for nm, b in zip(nombres[1:], beta[1:]))
        print("   beta caso {} (normalizado en el producto): {}".format(etiq, txt))

res = pd.DataFrame(filas)
os.makedirs(OUT, exist_ok=True)
res.to_csv(SALIDA, index=False, encoding="utf-8-sig")

print("\n" + "=" * 78)
print("RESUMEN")
print("=" * 78)
print(res.to_string(index=False))
print("\nguardado en:", SALIDA)

PASO 4F. ESPECIFICACION DETERMINISTICA, SISTEMAS CHINOS

------------------------------------------------------------------------------
S-A   variables: ln_gdp_pc_pwt, inv_gfcf_gdp, credit_priv_gdp, trade_gdp
   periodo 1990-2023   T = 34   orden VAR = 2
   validacion sin ficticia: caso III 2.67e-13   caso IV 2.32e-01

   H0        traza III    VC 95 III     traza IV     VC 95 IV
   r<=0         63.728        47.85       88.503        55.25
   r<=1         29.571        29.80       50.154        35.01
   r<=2         10.947        15.49       17.383        18.40
   r<=3          2.463         3.84        3.292         3.84

   Exclusion de la tendencia (H0: coeficiente de t nulo en beta)
      r = 1   LR =    4.192   gl = 1   p = 0.0406   tendencia NECESARIA
      r = 2   LR =   18.339   gl = 2   p = 0.0001   tendencia NECESARIA
   beta caso III (normalizado en el producto): inv_gfcf_gdp = +0.0634   credit_priv_gdp = +0.0159   trade_gdp = +0.0090
   beta caso IV (normalizado en el prod

In [48]:
# -*- coding: utf-8 -*-
"""
Paso 4f corregido. Especificacion deterministica de los sistemas chinos.
Caso III : constante irrestricta
Caso IV  : constante irrestricta + tendencia restringida a beta
Caso V   : constante y tendencia irrestrictas
Valida cada caso contra su referencia correcta, simula valores criticos
y aplica el principio de Pantula.
Salidas: outputs/cuadro17_deterministicos_v2.csv
         outputs/cuadro17_pantula.csv
"""

import os
import time
import warnings
import numpy as np
import pandas as pd
from numpy.linalg import pinv, eig
from scipy import stats
from statsmodels.tsa.vector_ar.vecm import coint_johansen, VECM

warnings.filterwarnings("ignore")

if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

BASE = "/content/drive/MyDrive/tesis_china_eeuu"
PANEL = os.path.join(BASE, "data_clean/china_us_panel_analisis_v6_1990_2023.csv")
OUT = os.path.join(BASE, "outputs")
SAL_A = os.path.join(OUT, "cuadro17_deterministicos_v2.csv")
SAL_B = os.path.join(OUT, "cuadro17_pantula.csv")

SISTEMAS = {
    "S-A": (["ln_gdp_pc_pwt", "inv_gfcf_gdp", "credit_priv_gdp", "trade_gdp"], 2),
    "S-B": (["ln_gdp_pc_pwt", "inv_gfcf_gdp", "bis_pvt_credit_gdp", "trade_gdp"], 3),
    "S-C": (["ln_gdp_pc_pwt", "inv_gfcf_gdp", "priv_debt_ls_gdp", "trade_gdp"], 2),
}

NSIM = 3000
TOL = 1e-6
SEMILLA = 20260801


# ===================================================== nucleo del calculo
def rrr(Y, kdiff, dummy, caso):
    """
    caso 3 : Z1 = Y[t-1]           Z2 = [1, dY rezagadas, dummy]
    caso 4 : Z1 = [Y[t-1], t]      Z2 = [1, dY rezagadas, dummy]
    caso 5 : Z1 = Y[t-1]           Z2 = [1, t, dY rezagadas, dummy]
    """
    T, k = Y.shape
    dY = np.diff(Y, axis=0)
    ini = kdiff + 1
    n = T - ini
    t_col = np.arange(ini, T, dtype=float).reshape(-1, 1)

    Z0 = dY[kdiff:, :]
    Z1 = Y[ini - 1: T - 1, :]
    if caso == 4:
        Z1 = np.hstack([Z1, t_col])

    Z2 = [np.ones((n, 1))]
    if caso == 5:
        Z2.append(t_col)
    for j in range(1, kdiff + 1):
        Z2.append(dY[kdiff - j: T - 1 - j, :])
    if dummy is not None:
        Z2.append(dummy[ini:T].reshape(-1, 1))
    Z2 = np.hstack(Z2)

    M = np.eye(n) - Z2 @ pinv(Z2)
    R0, R1 = M @ Z0, M @ Z1

    S00 = R0.T @ R0 / n
    S01 = R0.T @ R1 / n
    S11 = R1.T @ R1 / n

    A = pinv(S11) @ S01.T @ pinv(S00) @ S01
    val, vec = eig(A)
    val = np.real(val)
    o = np.argsort(val)[::-1]
    val = np.clip(val[o], 0.0, 0.999999)
    vec = np.real(vec[:, o])
    return val, n, vec


def traza(val, n, k):
    """Estadistico de traza para H0: r<=i, i=0..k-1."""
    return np.array([-n * np.sum(np.log(1.0 - val[i:k])) for i in range(k)])


# ===================================================== validaciones
def validar(Y, kdiff, caso, k):
    """Compara contra la referencia que corresponde a cada caso."""
    if caso == 3:
        ref = coint_johansen(Y, det_order=0, k_ar_diff=kdiff)
        mio, _, _ = rrr(Y, kdiff, None, 3)
        e = np.max(np.abs(np.sort(mio)[::-1][:k] - np.sort(ref.eig)[::-1][:k]))
        return e, "coint_johansen det_order=0"
    if caso == 5:
        ref = coint_johansen(Y, det_order=1, k_ar_diff=kdiff)
        mio, _, _ = rrr(Y, kdiff, None, 5)
        e = np.max(np.abs(np.sort(mio)[::-1][:k] - np.sort(ref.eig)[::-1][:k]))
        return e, "coint_johansen det_order=1"
    # caso 4: se compara el vector beta con el del estimador VECM
    try:
        m = VECM(Y, k_ar_diff=kdiff, coint_rank=1, deterministic="coli").fit()
        b_ref = np.asarray(m.beta).ravel()
        b_ref = b_ref / b_ref[0]
        _, _, vec = rrr(Y, kdiff, None, 4)
        b_mio = vec[:, 0] / vec[0, 0]
        e = np.max(np.abs(b_mio - b_ref))
        return e, "VECM deterministic='coli' (beta)"
    except Exception as exc:
        return np.nan, "no disponible: {}".format(exc)


# ===================================================== valores criticos
def cv_simulados(k, i, T, kdiff, caso, pos_dummy, nsim=NSIM, semilla=SEMILLA):
    """
    Distribucion de la traza bajo H0: r<=i, con m=k-i paseos aleatorios
    independientes y la misma estructura deterministica y de ficticia.
    """
    rng = np.random.default_rng(semilla + 1000 * caso + i)
    m = k - i
    if m <= 0:
        return np.nan, np.nan
    d = np.zeros(T)
    if 0 <= pos_dummy < T:
        d[pos_dummy] = 1.0
    out = np.empty(nsim)
    for s in range(nsim):
        W = np.cumsum(rng.standard_normal((T, m)), axis=0)
        val, n, _ = rrr(W, kdiff, d, caso)
        out[s] = -n * np.sum(np.log(1.0 - val[0:m]))
    return float(np.percentile(out, 95)), out


# ===================================================== ejecucion
df = pd.read_csv(PANEL)
cn = df[df["country"] == "CN"].sort_values("year").reset_index(drop=True)

filas, pantula = [], []
t0 = time.time()

print("=" * 78)
print("PASO 4F CORREGIDO. ESPECIFICACION DETERMINISTICA, SISTEMAS CHINOS")
print("=" * 78)

for nom, (vars_, p) in SISTEMAS.items():
    kdiff = p - 1
    sub = cn[["year", "d2020"] + vars_].dropna().reset_index(drop=True)
    Y = sub[vars_].to_numpy(float)
    d = sub["d2020"].to_numpy(float)
    T, k = Y.shape
    pos_d = int(np.argmax(d)) if d.max() > 0 else -1

    print("\n" + "=" * 78)
    print("{}   {}".format(nom, ", ".join(vars_)))
    print("   {}-{}   T = {}   orden VAR = {}   ficticia en la posicion {}"
          .format(int(sub.year.min()), int(sub.year.max()), T, p, pos_d))

    guardado = {}
    for caso in (3, 4, 5):
        e, ref_txt = validar(Y, kdiff, caso, k)
        ok = (e is not None) and np.isfinite(e) and (e < TOL)
        print("\n   Caso {}   validacion = {:.2e}   [{}]   {}".format(
            caso, e if np.isfinite(e) else np.nan, ref_txt,
            "OK" if ok else "NO SUPERADA"))

        val, n, vec = rrr(Y, kdiff, d, caso)
        tr = traza(val, n, k)
        guardado[caso] = (val, n, vec, tr, e, ok)

        if not ok:
            print("      resultados del caso {} NO utilizables".format(caso))
            continue

        print("      {:<7} {:>10} {:>12} {:>10}".format(
            "H0", "traza", "VC 95 sim", "p sim"))
        for i in range(k):
            vc, dist = cv_simulados(k, i, T, kdiff, caso, pos_d)
            psim = float(np.mean(dist >= tr[i])) if dist is not None else np.nan
            print("      r<={:<4} {:>10.3f} {:>12.2f} {:>10.4f}".format(
                i, tr[i], vc, psim))
            filas.append({"Sistema": nom, "Caso": caso, "H0": "r<={}".format(i),
                          "Traza": round(float(tr[i]), 3),
                          "VC 95 simulado": round(float(vc), 2),
                          "p simulada": round(psim, 4),
                          "Validacion": e,
                          "Rechaza": bool(psim < .05)})
            pantula.append({"Sistema": nom, "Caso": caso, "r": i,
                            "p simulada": round(psim, 4),
                            "Rechaza": bool(psim < .05)})

        nombres = list(vars_) + (["tendencia"] if caso == 4 else [])
        b = vec[:, 0] / vec[0, 0]
        print("      beta: " + "   ".join(
            "{} = {:+.4f}".format(nm, -bb)
            for nm, bb in zip(nombres[1:], b[1:])))

    # ---- contraste de exclusion de la tendencia, solo si ambos validan
    if guardado[3][5] and guardado[4][5]:
        v3, n3 = guardado[3][0], guardado[3][1]
        v4, n4 = guardado[4][0], guardado[4][1]
        print("\n   Exclusion de la tendencia restringida (caso IV frente a III)")
        for r in (1, 2):
            lr = abs(float(n4 * np.sum(np.log(1 - v3[:r]) - np.log(1 - v4[:r]))))
            pv = 1.0 - stats.chi2.cdf(lr, r)
            print("      r = {}   LR = {:8.3f}   p = {:.4f}   {}".format(
                r, lr, pv, "tendencia necesaria" if pv < .05 else "prescindible"))
            filas.append({"Sistema": nom, "Caso": "IV vs III",
                          "H0": "beta_t = 0, r={}".format(r),
                          "Traza": round(lr, 3), "VC 95 simulado": np.nan,
                          "p simulada": round(pv, 4),
                          "Validacion": np.nan, "Rechaza": bool(pv < .05)})
    else:
        print("\n   Exclusion de la tendencia: omitida, falta validacion")

# ===================================================== Pantula
print("\n" + "=" * 78)
print("PRINCIPIO DE PANTULA")
print("=" * 78)
pt = pd.DataFrame(pantula)
if not pt.empty:
    for nom in SISTEMAS:
        s = pt[pt["Sistema"] == nom].sort_values(["r", "Caso"])
        elegido = None
        for _, row in s.iterrows():
            if not row["Rechaza"]:
                elegido = (int(row["r"]), int(row["Caso"]), row["p simulada"])
                break
        print("{}: {}".format(nom, "primer no rechazo en r = {}, caso {} "
              "(p = {:.4f})".format(*elegido) if elegido
              else "rechaza en toda la secuencia"))

pd.DataFrame(filas).to_csv(SAL_A, index=False, encoding="utf-8-sig")
pt.to_csv(SAL_B, index=False, encoding="utf-8-sig")
print("\ntiempo: {:.0f} s".format(time.time() - t0))
print("guardado en:\n  {}\n  {}".format(SAL_A, SAL_B))

PASO 4F CORREGIDO. ESPECIFICACION DETERMINISTICA, SISTEMAS CHINOS

S-A   ln_gdp_pc_pwt, inv_gfcf_gdp, credit_priv_gdp, trade_gdp
   1990-2023   T = 34   orden VAR = 2   ficticia en la posicion 30

   Caso 3   validacion = 2.67e-13   [coint_johansen det_order=0]   OK
      H0           traza    VC 95 sim      p sim
      r<=0        63.728        63.46     0.0470
      r<=1        29.571        37.91     0.2503
      r<=2        10.947        20.30     0.4573
      r<=3         2.463         8.89     0.5227
      beta: inv_gfcf_gdp = +0.0634   credit_priv_gdp = +0.0159   trade_gdp = +0.0090

   Caso 4   validacion = nan   [no disponible: operands could not be broadcast together with shapes (5,) (4,) ]   NO SUPERADA
      resultados del caso 4 NO utilizables

   Caso 5   validacion = 1.48e-01   [coint_johansen det_order=1]   NO SUPERADA
      resultados del caso 5 NO utilizables

   Exclusion de la tendencia: omitida, falta validacion

S-B   ln_gdp_pc_pwt, inv_gfcf_gdp, bis_pvt_credit_gd

In [50]:
# -*- coding: utf-8 -*-
"""
Paso 4f v3. Necesidad de una tendencia en el espacio de cointegracion.
Contraste por razon de verosimilitud entre el modelo con constante
irrestricta y el mismo modelo con tendencia lineal restringida a beta,
ambos estimados con el estimador VECM de statsmodels.
Salida: outputs/cuadro17_tendencia.csv
"""

import os
import warnings
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.tsa.vector_ar.vecm import VECM

warnings.filterwarnings("ignore")

if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

BASE = "/content/drive/MyDrive/tesis_china_eeuu"
PANEL = os.path.join(BASE, "data_clean/china_us_panel_analisis_v6_1990_2023.csv")
OUT = os.path.join(BASE, "outputs")
SALIDA = os.path.join(OUT, "cuadro17_tendencia.csv")

SISTEMAS = {
    "S-A": (["ln_gdp_pc_pwt", "inv_gfcf_gdp", "credit_priv_gdp", "trade_gdp"], 2, 1),
    "S-B": (["ln_gdp_pc_pwt", "inv_gfcf_gdp", "bis_pvt_credit_gdp", "trade_gdp"], 3, 1),
    "S-C": (["ln_gdp_pc_pwt", "inv_gfcf_gdp", "priv_debt_ls_gdp", "trade_gdp"], 2, 1),
}

df = pd.read_csv(PANEL)
cn = df[df["country"] == "CN"].sort_values("year").reset_index(drop=True)

filas = []
print("=" * 78)
print("NECESIDAD DE TENDENCIA EN EL ESPACIO DE COINTEGRACION")
print("=" * 78)

for nom, (vars_, p, r) in SISTEMAS.items():
    kdiff = p - 1
    sub = cn[["year", "d2020"] + vars_].dropna().reset_index(drop=True)
    Y = sub[vars_].to_numpy(float)
    ex = sub[["d2020"]].to_numpy(float)

    print("\n" + "-" * 78)
    print("{}   {}-{}   T = {}   orden VAR = {}   r = {}".format(
        nom, int(sub.year.min()), int(sub.year.max()), len(sub), p, r))

    res = {}
    for det, etiq in (("co", "constante irrestricta"),
                      ("coli", "constante irrestricta + tendencia en beta")):
        try:
            m = VECM(Y, k_ar_diff=kdiff, coint_rank=r,
                     deterministic=det, exog=ex).fit()
            res[det] = m
            b = np.asarray(m.beta).ravel()
            b = b / b[0]
            partes = ["{} = {:+.4f}".format(v, -bb)
                      for v, bb in zip(vars_[1:], b[1:])]
            if det == "coli" and getattr(m, "det_coef_coint", None) is not None:
                dc = np.asarray(m.det_coef_coint).ravel()
                dc = dc / np.asarray(m.beta).ravel()[0]
                partes.append("tendencia = {:+.4f}".format(-dc[0]))
            print("\n   {:<44} lnL = {:12.4f}".format(etiq, m.llf))
            print("      beta: " + "   ".join(partes))
            a = np.asarray(m.alpha).ravel()
            print("      alfa del producto = {:+.4f}".format(a[0]))
        except Exception as exc:
            print("\n   {} no estimable: {}".format(etiq, exc))

    if "co" in res and "coli" in res:
        lr = 2.0 * (res["coli"].llf - res["co"].llf)
        pv = 1.0 - stats.chi2.cdf(max(lr, 0.0), r)
        ver = "tendencia necesaria" if pv < .05 else "tendencia prescindible"
        print("\n   LR = {:.3f}   gl = {}   p = {:.4f}   ->  {}".format(
            lr, r, pv, ver.upper()))
        filas.append({"Sistema": nom, "Orden VAR": p, "r": r,
                      "lnL constante": round(float(res["co"].llf), 4),
                      "lnL con tendencia": round(float(res["coli"].llf), 4),
                      "LR": round(float(lr), 3), "gl": r,
                      "p": round(float(pv), 4), "Veredicto": ver})

out = pd.DataFrame(filas)
os.makedirs(OUT, exist_ok=True)
out.to_csv(SALIDA, index=False, encoding="utf-8-sig")

print("\n" + "=" * 78)
print("RESUMEN")
print("=" * 78)
print(out.to_string(index=False))
print("\nguardado en:", SALIDA)

NECESIDAD DE TENDENCIA EN EL ESPACIO DE COINTEGRACION

------------------------------------------------------------------------------
S-A   1990-2023   T = 34   orden VAR = 2   r = 1

   constante irrestricta                        lnL =    -130.7786
      beta: inv_gfcf_gdp = +0.0634   credit_priv_gdp = +0.0159   trade_gdp = +0.0090
      alfa del producto = -0.0315

   constante irrestricta + tendencia en beta    lnL =    -128.6824
      beta: inv_gfcf_gdp = +0.0165   credit_priv_gdp = -0.0047   trade_gdp = +0.0012   tendencia = +0.0838
      alfa del producto = -0.3197

   LR = 4.192   gl = 1   p = 0.0406   ->  TENDENCIA NECESARIA

------------------------------------------------------------------------------
S-B   1990-2021   T = 32   orden VAR = 3   r = 1

   constante irrestricta                        lnL =    -107.8727
      beta: inv_gfcf_gdp = -0.0097   bis_pvt_credit_gdp = +0.0149   trade_gdp = +0.0230
      alfa del producto = -0.0594

   constante irrestricta + tendencia e

In [51]:
# -*- coding: utf-8 -*-
"""
Paso 5a. Inventario previo al documento de cuadros v5.
"""

import os
import glob
import pandas as pd
from docx import Document

BASE = "/content/drive/MyDrive/tesis_china_eeuu"
OUT = os.path.join(BASE, "outputs")
DOCV4 = os.path.join(OUT, "Cuadros_resultados_APA_v4.docx")

print("=" * 78)
print("A. ESTRUCTURA DE Cuadros_resultados_APA_v4.docx")
print("=" * 78)
if not os.path.exists(DOCV4):
    print("no encontrado:", DOCV4)
else:
    doc = Document(DOCV4)
    print("parrafos: {}   tablas: {}".format(len(doc.paragraphs), len(doc.tables)))
    print("\n{:<5} {:<10} {:<62}".format("#", "estilo", "texto"))
    for i, p in enumerate(doc.paragraphs):
        t = p.text.strip()
        if t:
            print("{:<5} {:<10} {:<62}".format(
                i, (p.style.name or "")[:10], t[:62]))
    print("\n{:<5} {:>6} {:>6}  {}".format("tabla", "filas", "cols", "encabezado"))
    for i, tb in enumerate(doc.tables):
        enc = " | ".join(c.text.strip()[:16] for c in tb.rows[0].cells)
        print("{:<5} {:>6} {:>6}  {}".format(
            i, len(tb.rows), len(tb.columns), enc[:80]))

print("\n" + "=" * 78)
print("B. ARCHIVOS DISPONIBLES EN outputs")
print("=" * 78)
print("{:<44} {:>7} {:>6}  {}".format("archivo", "filas", "cols", "columnas"))
for f in sorted(glob.glob(os.path.join(OUT, "cuadro*.csv"))):
    try:
        d = pd.read_csv(f)
        cols = ", ".join(map(str, d.columns))
        print("{:<44} {:>7} {:>6}  {}".format(
            os.path.basename(f), d.shape[0], d.shape[1], cols[:100]))
    except Exception as exc:
        print("{:<44} error: {}".format(os.path.basename(f), exc))

print("\n" + "=" * 78)
print("C. OTROS DOCUMENTOS")
print("=" * 78)
for f in sorted(glob.glob(os.path.join(OUT, "*.docx"))):
    print("   {:>9,} bytes   {}".format(
        os.path.getsize(f), os.path.basename(f)))

A. ESTRUCTURA DE Cuadros_resultados_APA_v4.docx
parrafos: 37   tablas: 7

#     estilo     texto                                                         
0     Normal     Resultados de la estimacion. China y Estados Unidos, 1990-2023
2     Normal     Cuadro 4                                                      
3     Normal     Especificaciones estimadas, variables y correspondencia con la
4     Normal     Nota. Los tamanos de muestra corresponden a las observaciones 
6     Normal     Cuadro 5                                                      
7     Normal     Bitacora de reconstruccion de series con cobertura incompleta 
8     Normal     Nota. Cada registro documenta la serie afectada, el diagnostic
10    Normal     Cuadro 7                                                      
11    Normal     Prueba de limites de cointegracion y termino de correccion de 
12    Normal     Nota. Estimacion mediante un modelo autorregresivo de rezagos 
14    Normal     Cuadro 8                     

In [52]:
# -*- coding: utf-8 -*-
"""
Paso 5. Documento de cuadros version 5, formato APA 7.
Reune los cuadros 1 a 17 mas los anexos, desde los CSV de outputs.
Los cuadros 4 y 5 se copian del documento v4 por no existir en CSV.
Salida: outputs/Cuadros_resultados_APA_v5.docx
"""

import os
import numpy as np
import pandas as pd
from docx import Document
from docx.shared import Pt, Cm
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.section import WD_ORIENT
from docx.enum.table import WD_TABLE_ALIGNMENT
from docx.oxml.ns import qn
from docx.oxml import OxmlElement

if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

BASE = "/content/drive/MyDrive/tesis_china_eeuu"
OUT = os.path.join(BASE, "outputs")
DOCV4 = os.path.join(OUT, "Cuadros_resultados_APA_v4.docx")
SALIDA = os.path.join(OUT, "Cuadros_resultados_APA_v5.docx")

FUENTE = "Arial"
PT_TXT = 10
PT_TAB = 8
ANCHO_UTIL = 23.9  # cm disponibles en horizontal con margenes de 2 cm


# ==================================================================
# utilidades de formato
# ==================================================================
def celda_txt(v):
    if v is None:
        return "—"
    if isinstance(v, float) and not np.isfinite(v):
        return "—"
    if isinstance(v, (np.floating, float)):
        a = abs(v)
        if a == 0:
            return "0.000"
        if a >= 1000:
            return "{:,.1f}".format(v)
        if a < 0.001:
            return "{:.2e}".format(v)
        return "{:.3f}".format(v)
    if isinstance(v, (np.integer, int, np.bool_, bool)):
        return str(v)
    s = str(v).strip()
    if s.lower() in ("nan", "none", ""):
        return "—"
    return s


def borde(celda, lado, sz=6):
    tcPr = celda._tc.get_or_add_tcPr()
    b = tcPr.find(qn("w:tcBorders"))
    if b is None:
        b = OxmlElement("w:tcBorders")
        tcPr.append(b)
    e = b.find(qn("w:" + lado))
    if e is None:
        e = OxmlElement("w:" + lado)
        b.append(e)
    e.set(qn("w:val"), "single")
    e.set(qn("w:sz"), str(sz))
    e.set(qn("w:color"), "000000")


def sin_bordes(celda):
    tcPr = celda._tc.get_or_add_tcPr()
    b = tcPr.find(qn("w:tcBorders"))
    if b is None:
        b = OxmlElement("w:tcBorders")
        tcPr.append(b)
    for lado in ("top", "left", "bottom", "right"):
        e = b.find(qn("w:" + lado))
        if e is None:
            e = OxmlElement("w:" + lado)
            b.append(e)
        e.set(qn("w:val"), "none")
        e.set(qn("w:sz"), "0")


def repetir_encabezado(fila):
    trPr = fila._tr.get_or_add_trPr()
    th = OxmlElement("w:tblHeader")
    th.set(qn("w:val"), "true")
    trPr.append(th)


def layout_fijo(tabla):
    tblPr = tabla._tbl.tblPr
    el = OxmlElement("w:tblLayout")
    el.set(qn("w:type"), "fixed")
    tblPr.append(el)


def parrafo(doc, texto, cursiva=False, negrita=False, pt=PT_TXT,
            espacio_despues=6, alineacion=WD_ALIGN_PARAGRAPH.LEFT):
    p = doc.add_paragraph()
    p.alignment = alineacion
    pf = p.paragraph_format
    pf.space_after = Pt(espacio_despues)
    pf.space_before = Pt(0)
    r = p.add_run(texto)
    r.font.name = FUENTE
    r.font.size = Pt(pt)
    r.italic = cursiva
    r.bold = negrita
    r._element.rPr.rFonts.set(qn("w:eastAsia"), FUENTE)
    return p


def anchos(df):
    largos = []
    for c in df.columns:
        l = max([len(str(c))] + [len(celda_txt(v)) for v in df[c].tolist()])
        largos.append(min(max(l, 5), 44))
    tot = float(sum(largos))
    return [max(1.25, ANCHO_UTIL * l / tot) for l in largos]


def agregar_tabla(doc, df):
    df = df.copy()
    nf, nc = df.shape
    tabla = doc.add_table(rows=nf + 1, cols=nc)
    tabla.alignment = WD_TABLE_ALIGNMENT.CENTER
    tabla.autofit = False
    layout_fijo(tabla)
    ws = anchos(df)

    for j, col in enumerate(df.columns):
        c = tabla.cell(0, j)
        c.width = Cm(ws[j])
        p = c.paragraphs[0]
        p.paragraph_format.space_after = Pt(2)
        p.paragraph_format.space_before = Pt(2)
        r = p.add_run(str(col))
        r.font.name = FUENTE
        r.font.size = Pt(PT_TAB)
        r.bold = True
        r._element.rPr.rFonts.set(qn("w:eastAsia"), FUENTE)

    for i in range(nf):
        for j in range(nc):
            c = tabla.cell(i + 1, j)
            c.width = Cm(ws[j])
            p = c.paragraphs[0]
            p.paragraph_format.space_after = Pt(1)
            p.paragraph_format.space_before = Pt(1)
            r = p.add_run(celda_txt(df.iat[i, j]))
            r.font.name = FUENTE
            r.font.size = Pt(PT_TAB)
            r._element.rPr.rFonts.set(qn("w:eastAsia"), FUENTE)

    for fila in tabla.rows:
        for c in fila.cells:
            sin_bordes(c)
    for c in tabla.rows[0].cells:
        borde(c, "top")
        borde(c, "bottom")
    for c in tabla.rows[-1].cells:
        borde(c, "bottom")
    repetir_encabezado(tabla.rows[0])
    return tabla


def copiar_tabla(doc, tabla_origen):
    """Reproduce una tabla del documento v4 como DataFrame y la reinserta."""
    datos = [[c.text.strip() for c in f.cells] for f in tabla_origen.rows]
    df = pd.DataFrame(datos[1:], columns=datos[0])
    return agregar_tabla(doc, df)


def bloque(doc, numero, titulo, df, nota, panel=None):
    parrafo(doc, "Cuadro {}".format(numero), negrita=True, espacio_despues=0)
    parrafo(doc, titulo, cursiva=True, espacio_despues=6)
    if panel:
        parrafo(doc, panel, negrita=True, pt=PT_TAB + 1, espacio_despues=3)
    agregar_tabla(doc, df)
    parrafo(doc, "", espacio_despues=0, pt=4)
    parrafo(doc, nota, pt=PT_TAB, espacio_despues=16)


def leer(nombre, cols=None, filtro=None, orden=None):
    ruta = os.path.join(OUT, nombre)
    if not os.path.exists(ruta):
        print("   FALTA: {}".format(nombre))
        return None
    d = pd.read_csv(ruta)
    if filtro is not None:
        d = d[filtro(d)]
    if cols:
        pres = [c for c in cols if c in d.columns]
        if len(pres) >= 2:
            d = d[pres]
    if orden:
        pres = [c for c in orden if c in d.columns]
        if pres:
            d = d.sort_values(pres)
    return d.reset_index(drop=True)


# ==================================================================
# documento
# ==================================================================
doc = Document()
sec = doc.sections[0]
sec.orientation = WD_ORIENT.LANDSCAPE
sec.page_width, sec.page_height = Cm(27.94), Cm(21.59)
for m in ("left_margin", "right_margin", "top_margin", "bottom_margin"):
    setattr(sec, m, Cm(2))

est = doc.styles["Normal"]
est.font.name = FUENTE
est.font.size = Pt(PT_TXT)
est.element.rPr.rFonts.set(qn("w:eastAsia"), FUENTE)

parrafo(doc, "Cuadros de resultados", negrita=True, pt=13, espacio_despues=2)
parrafo(doc, "Trayectorias de crecimiento en China y Estados Unidos, "
             "1990–2023", cursiva=True, espacio_despues=18)

faltan = []

# ---------------------------------------------------------- Cuadro 1
d = leer("cuadro1_cobertura.csv")
if d is not None:
    bloque(doc, "1",
           "Cobertura temporal y completitud de las series por país",
           d,
           "Nota. CN = China; US = Estados Unidos; ini y fin indican el primer "
           "y el último año con dato; n es el número de observaciones válidas "
           "y na el de valores ausentes dentro del intervalo. Fuentes: Banco "
           "Mundial, World Development Indicators (https://data.worldbank.org); "
           "Penn World Table 10.01 (Feenstra et al., 2015); Bank for "
           "International Settlements, Credit to the non-financial sector "
           "(https://www.bis.org/statistics/totcredit.htm); Fondo Monetario "
           "Internacional, Global Debt Database "
           "(https://www.imf.org/external/datamapper/datasets/GDD).")
else:
    faltan.append("cuadro1")

# ---------------------------------------------------------- Cuadro 2
d = leer("cuadro2_descriptivos.csv",
         cols=["Pais", "Variable", "n", "M", "Mdn", "DE", "Min", "Max", "CV"])
if d is not None:
    bloque(doc, "2",
           "Estadísticos descriptivos de las variables por país",
           d,
           "Nota. M = media; Mdn = mediana; DE = desviación estándar; "
           "CV = coeficiente de variación. Todas las razones se expresan como "
           "porcentaje del producto interno bruto, salvo indicación en "
           "contrario.")
d = leer("cuadro2_diferencia_medias.csv")
if d is not None:
    bloque(doc, "2b",
           "Contraste de diferencia de medias entre China y Estados Unidos",
           d,
           "Nota. Prueba t de Welch para varianzas desiguales. Valores "
           "positivos de la diferencia indican una media superior en China.")

# ---------------------------------------------------------- Cuadro 3
d = leer("cuadro3_raiz_unitaria_v4.csv")
if d is not None:
    pa = [c for c in ["Pais", "Serie", "n", "ADF (c)", "ADF (c+t)", "PP (c)",
                      "PP (c+t)", "KPSS (c)", "KPSS (c+t)"] if c in d.columns]
    pb = [c for c in ["Pais", "Serie", "ADF D (c)", "PP D (c)", "KPSS D (c)",
                      "ADF D (c+t)", "PP D (c+t)", "KPSS D (c+t)",
                      "Zivot-Andrews", "Orden"] if c in d.columns]
    nota3 = ("Nota. c = constante; c+t = constante y tendencia; D = primera "
             "diferencia. En las pruebas de Dickey–Fuller aumentada y de "
             "Phillips–Perron la hipótesis nula es la existencia de una raíz "
             "unitaria, de modo que el rechazo indica estacionariedad. En la "
             "prueba KPSS la hipótesis nula es la estacionariedad y el rechazo "
             "indica lo contrario; sus asteriscos se asignan por comparación "
             "directa con los valores críticos tabulados (con constante: .347, "
             ".463 y .739; con constante y tendencia: .119, .146 y .216). El "
             "orden de integración se declara cuando al menos una prueba de "
             "raíz unitaria rechaza al 5 % y KPSS no rechaza al mismo nivel. "
             "*p < .10. **p < .05. ***p < .01. Pruebas: Dickey y Fuller (1979), "
             "Phillips y Perron (1988), Kwiatkowski et al. (1992) y "
             "Zivot y Andrews (1992).")
    bloque(doc, "3", "Pruebas de raíz unitaria y orden de integración de las "
                     "series por país", d[pa],
           "Nota. Panel A. Series en nivel. Véanse las notas del panel B.",
           panel="Panel A. Series en nivel")
    bloque(doc, "3 (cont.)", "Pruebas de raíz unitaria y orden de integración "
                             "de las series por país", d[pb], nota3,
           panel="Panel B. Series en primera diferencia y veredicto")
else:
    faltan.append("cuadro3_v4")

d = leer("cuadro3b_segunda_ronda.csv")
if d is not None:
    bloque(doc, "3b",
           "Segunda ronda de contrastes para las series no concluyentes",
           d,
           "Nota. Se aplican pruebas sobre la primera y la segunda diferencia "
           "para discriminar entre integración de orden uno y de orden "
           "superior. Las series clasificadas como I(2) se excluyen de las "
           "especificaciones en niveles. *p < .10. **p < .05. ***p < .01.")

d = leer("cuadro3c_comprobaciones.csv")
if d is not None:
    bloque(doc, "3c",
           "Comprobaciones sobre la estructura determinista y el orden de "
           "integración de las variables críticas",
           d,
           "Nota. La primera comprobación contrasta la presencia de una "
           "tendencia determinista en la tasa de crecimiento china mediante "
           "mínimos cuadrados con errores estándar consistentes ante "
           "heterocedasticidad y autocorrelación (Newey y West, 1987). Las "
           "restantes evalúan la estacionariedad de la formación bruta de "
           "capital fijo estadounidense y el orden de integración conjunto de "
           "cada sistema.")

# ------------------------------------------------- Cuadros 4 y 5 del v4
if os.path.exists(DOCV4):
    v4 = Document(DOCV4)
    if len(v4.tables) >= 2:
        parrafo(doc, "Cuadro 4", negrita=True, espacio_despues=0)
        parrafo(doc, "Especificaciones estimadas, variables y correspondencia "
                     "con las hipótesis", cursiva=True, espacio_despues=6)
        copiar_tabla(doc, v4.tables[0])
        parrafo(doc, "", pt=4, espacio_despues=0)
        parrafo(doc, "Nota. Los tamaños de muestra corresponden a las "
                     "observaciones disponibles para cada país tras la "
                     "eliminación por casos completos.", pt=PT_TAB,
                espacio_despues=16)

        parrafo(doc, "Cuadro 5", negrita=True, espacio_despues=0)
        parrafo(doc, "Bitácora de reconstrucción de series con cobertura "
                     "incompleta", cursiva=True, espacio_despues=6)
        copiar_tabla(doc, v4.tables[1])
        parrafo(doc, "", pt=4, espacio_despues=0)
        parrafo(doc, "Nota. Cada registro documenta la serie afectada, el "
                     "diagnóstico, la decisión adoptada y su justificación, "
                     "con el fin de garantizar la reproducibilidad del panel.",
                pt=PT_TAB, espacio_despues=16)
else:
    faltan.append("Cuadros_resultados_APA_v4.docx")

# ---------------------------------------------------------- Cuadro 6
d = leer("cuadro6b_vif_por_modelo.csv")
if d is not None:
    bloque(doc, "6",
           "Factores de inflación de la varianza por especificación estimada",
           d,
           "Nota. VIF máx = mayor factor de inflación de la varianza entre los "
           "regresores de cada especificación. Se reporta por modelo y no "
           "sobre un conjunto agregado de regresores, dado que ninguna "
           "especificación estimada incluye simultáneamente todas las "
           "variables del panel. Se considera aceptable un valor inferior a 5 "
           "y severo uno superior a 10.")

# ---------------------------------------------------------- Cuadro 7
d = leer("cuadro7_cointegracion.csv")
if d is not None:
    bloque(doc, "7",
           "Prueba de límites de cointegración y término de corrección de "
           "error por modelo",
           d,
           "Nota. Estimación mediante un modelo autorregresivo de rezagos "
           "distribuidos con selección del orden por criterio de información "
           "de Akaike. Los valores críticos corresponden al caso III de la "
           "tabla CI(iii) de Pesaran et al. (2001) al 5 %, con los pares "
           "3.79 y 4.85 para dos regresores, 3.23 y 4.35 para tres, y 2.86 y "
           "4.01 para cuatro. La regla de decisión exige simultáneamente que "
           "el estadístico F supere el límite superior y que el término de "
           "corrección de error sea negativo y significativo. La vida media se "
           "obtiene como ln(0.5)/ln(1 + ECT).")

# ---------------------------------------------------------- Cuadro 8
d = leer("cuadro8_largo_plazo.csv")
if d is not None:
    bloque(doc, "8",
           "Coeficientes de largo plazo de las relaciones cointegrantes "
           "identificadas",
           d,
           "Nota. La variable dependiente es el logaritmo del producto interno "
           "bruto por habitante. Los errores estándar de los coeficientes de "
           "largo plazo se obtienen por el método delta. Solo se interpretan "
           "los modelos que superan conjuntamente la prueba de límites y el "
           "signo del término de corrección de error.")

# ---------------------------------------------------------- Cuadro 9
d = leer("cuadro9_corto_plazo.csv")
if d is not None:
    bloque(doc, "9",
           "Dinámica de corto plazo y multiplicadores netos",
           d,
           "Nota. El multiplicador neto acumula los coeficientes "
           "contemporáneos y rezagados de cada regresor, dividido por uno "
           "menos la suma de los coeficientes autorregresivos.")

# --------------------------------------------------------- Cuadro 10
d = leer("cuadro10_diagnosticos.csv")
if d is not None:
    bloque(doc, "10",
           "Pruebas de diagnóstico de los residuos de las ecuaciones de "
           "corrección de error",
           d,
           "Nota. Se reportan valores de probabilidad. Breusch–Godfrey y "
           "Ljung–Box contrastan ausencia de autocorrelación; Breusch–Pagan, "
           "homocedasticidad; Jarque–Bera, normalidad. Valores superiores a "
           ".05 indican que no se rechaza el supuesto correspondiente.")

# --------------------------------------------------------- Cuadro 11
d = leer("cuadro11_dols.csv")
if d is not None:
    bloque(doc, "11",
           "Estimación de los coeficientes de largo plazo por mínimos "
           "cuadrados dinámicos",
           d,
           "Nota. Estimador de mínimos cuadrados dinámicos de Stock y Watson "
           "(1993) con adelantos y rezagos de las primeras diferencias de los "
           "regresores y errores estándar consistentes ante heterocedasticidad "
           "y autocorrelación (Newey y West, 1987), con ancho de banda "
           "seleccionado por el procedimiento automático de Andrews (1991). La "
           "columna de diferencia compara el coeficiente con el obtenido por "
           "el modelo de rezagos distribuidos.")

# --------------------------------------------------------- Cuadro 12
for suf, tit in (("cointegracion", "Prueba de límites"),
                 ("largo_plazo", "Coeficientes de largo plazo"),
                 ("corto_plazo", "Dinámica de corto plazo"),
                 ("diagnosticos", "Diagnósticos de los residuos")):
    d = leer("cuadro12_m10_{}.csv".format(suf))
    if d is not None:
        bloque(doc, "12" + {"cointegracion": "", "largo_plazo": "b",
                            "corto_plazo": "c", "diagnosticos": "d"}[suf],
               "Modelo de deuda privada y apertura comercial (M10). {}".format(tit),
               d,
               "Nota. La deuda privada procede de la Global Debt Database del "
               "Fondo Monetario Internacional. Esta especificación es la única "
               "cuyos regresores satisfacen conjuntamente la condición de "
               "exogeneidad débil en el sistema correspondiente.")

# --------------------------------------------------------- Cuadro 13
for suf, tit, nota in (
    ("cointegracion", "Prueba de límites",
     "Nota. Se contrastan tres especificaciones auxiliares: la inversión como "
     "único regresor del producto (M11), la inversión junto con la apertura "
     "comercial sin variable financiera (M12) y la inversión como variable "
     "dependiente de la apertura y el crédito (MA). Se reportan los veredictos "
     "bajo los valores críticos correctos y bajo los empleados originalmente."),
    ("largo_plazo", "Coeficientes de largo plazo",
     "Nota. La columna de interpretabilidad indica si la especificación supera "
     "los diagnósticos de residuos exigidos para atribuir contenido "
     "estructural al coeficiente."),
    ("diagnosticos", "Diagnósticos de los residuos",
     "Nota. Valores de probabilidad. La especificación M11 para China no "
     "supera los contrastes de autocorrelación ni de normalidad, por lo que su "
     "coeficiente no admite lectura estructural."),
    ("dols", "Contraste por mínimos cuadrados dinámicos",
     "Nota. Estimación por el procedimiento de Stock y Watson (1993) como "
     "verificación independiente de los coeficientes de largo plazo.")):
    d = leer("cuadro13_mediacion_{}.csv".format(suf))
    if d is not None:
        bloque(doc, "13" + {"cointegracion": "", "largo_plazo": "b",
                            "diagnosticos": "c", "dols": "d"}[suf],
               "Contraste de la hipótesis de mediación de la inversión. {}".format(tit),
               d, nota)

# --------------------------------------------------------- Cuadro 14
for arch, sufijo, tit, nota in (
    ("cuadro14_johansen_rango_v3.csv", "", "Estadísticos de traza y rango",
     "Nota. Procedimiento de Johansen (1988, 1991) con constante irrestricta. "
     "La traza corregida aplica el factor de corrección por muestra finita de "
     "Reinsel y Ahn (1992). Los valores críticos tabulados resultan "
     "excesivamente permisivos en muestras de esta magnitud; véase el "
     "Cuadro 15."),
    ("cuadro14_johansen_vectores_v3.csv", "b", "Vectores de cointegración",
     "Nota. Coeficientes normalizados sobre el logaritmo del producto por "
     "habitante y expresados con el signo de la relación de largo plazo."),
    ("cuadro14_johansen_ajustes_v3.csv", "c",
     "Velocidades de ajuste y exogeneidad débil por ecuación",
     "Nota. El contraste de exogeneidad débil sigue el procedimiento de "
     "sistema parcial de Johansen (1992). Un valor de probabilidad superior a "
     ".05 indica que la ecuación correspondiente no responde a las "
     "desviaciones respecto de la relación de largo plazo."),
    ("cuadro14_johansen_exogeneidad_v3.csv", "d",
     "Contraste conjunto de exogeneidad débil de los regresores",
     "Nota. El no rechazo valida formalmente la estimación uniecuacional de la "
     "relación de largo plazo, que resulta eficiente en ese caso."),
    ("cuadro14_johansen_resumen_v3.csv", "e",
     "Resumen de la especificación y los diagnósticos de cada sistema",
     "Nota. Todos los sistemas incluyen una variable indicadora para 2020.")):
    d = leer(arch)
    if d is not None:
        bloque(doc, "14" + sufijo,
               "Análisis de cointegración multivariante. {}".format(tit),
               d, nota)

# --------------------------------------------------------- Cuadro 15
d = leer("cuadro15_criticos_simulados.csv")
if d is not None:
    bloque(doc, "15",
           "Valores críticos simulados de la prueba de traza",
           d,
           "Nota. Distribución obtenida por simulación de paseos aleatorios "
           "independientes bajo la hipótesis nula, con la misma longitud de "
           "muestra, el mismo orden de rezagos y la variable indicadora de "
           "2020 en su posición efectiva. La simulación sustituye a la "
           "corrección por muestra finita y no se acumula con ella. Los "
           "valores tabulados resultan sistemáticamente permisivos, lo que "
           "invertiría el veredicto de rango en varias configuraciones.")

d = leer("cuadro15_resumen.csv")
if d is not None:
    bloque(doc, "15b",
           "Estabilidad de los sistemas: raíces de la matriz compañera",
           d,
           "Nota. Impuesto un rango de cointegración de uno en un sistema de "
           "cuatro variables, cabe esperar tres raíces unitarias. La mayor "
           "raíz no unitaria por debajo de la unidad confirma la estabilidad "
           "dinámica del sistema.")

d = leer("cuadro15_residuos_por_ecuacion.csv",
         filtro=lambda x: x["Residuo tipificado"].abs() >= 2.0
         if "Residuo tipificado" in x.columns else x.index == x.index)
if d is not None and len(d):
    bloque(doc, "15c",
           "Observaciones atípicas de los residuos por ecuación",
           d,
           "Nota. Se listan las observaciones cuyo residuo tipificado alcanza "
           "o supera dos desviaciones estándar en valor absoluto. Su "
           "dispersión entre ecuaciones y años distintos desaconseja "
           "incorporar variables indicadoras adicionales, cuya inclusión "
           "elevaría los valores críticos simulados sin corregir una fuente "
           "común de perturbación.")

# --------------------------------------------------------- Cuadro 16
d = leer("cuadro16_sensibilidad_sistema.csv")
if d is not None:
    bloque(doc, "16",
           "Sensibilidad de los resultados de sistema al orden de rezagos y al "
           "tratamiento de 2020",
           d,
           "Nota. Cada configuración se contrasta contra sus propios valores "
           "críticos simulados. El signo de la velocidad de ajuste de la "
           "ecuación del producto es el resultado de interés: negativo indica "
           "ajuste corrector hacia la relación de largo plazo.")

# --------------------------------------------------------- Cuadro 17
d = leer("cuadro17_tendencia.csv")
if d is not None:
    bloque(doc, "17",
           "Contraste de necesidad de una tendencia lineal en el espacio de "
           "cointegración",
           d,
           "Nota. Razón de verosimilitud entre el modelo con constante "
           "irrestricta y el mismo modelo con tendencia lineal restringida al "
           "espacio de cointegración, con distribución ji cuadrada y grados de "
           "libertad iguales al rango. El rechazo no conduce a adoptar la "
           "especificación con tendencia: la desaceleración del crecimiento "
           "chino implica un componente cóncavo en el nivel logarítmico que "
           "una tendencia lineal no representa, y su inclusión desplaza la "
           "relación de largo plazo hacia un ajuste de tendencia en el que los "
           "coeficientes de los regresores se aproximan a cero. Véase el "
           "apartado de limitaciones.")

d = leer("cuadro17_pantula.csv")
if d is not None:
    bloque(doc, "17b",
           "Secuencia de Pantula para la determinación conjunta del rango y de "
           "la especificación determinista",
           d,
           "Nota. Procedimiento secuencial de Pantula (1989) evaluado con "
           "valores críticos simulados. El primer no rechazo determina "
           "simultáneamente el rango de cointegración y el modelo "
           "determinista.")

# ---------------------------------------------------------- Anexos
d = leer("cuadro11b_dols_sensibilidad.csv")
if d is not None:
    bloque(doc, "A1",
           "Sensibilidad de la estimación por mínimos cuadrados dinámicos al "
           "número de adelantos y rezagos y al ancho de banda",
           d,
           "Nota. Se reportan todas las combinaciones evaluadas. La "
           "estabilidad de los coeficientes a lo largo de la rejilla respalda "
           "la robustez de la estimación principal.")

if os.path.exists(DOCV4) and len(Document(DOCV4).tables) >= 7:
    v4 = Document(DOCV4)
    parrafo(doc, "Cuadro A2", negrita=True, espacio_despues=0)
    parrafo(doc, "Estimaciones de largo plazo no identificadas. "
                 "Estados Unidos", cursiva=True, espacio_despues=6)
    copiar_tabla(doc, v4.tables[6])
    parrafo(doc, "", pt=4, espacio_despues=0)
    parrafo(doc, "Nota. Se presentan por transparencia y no admiten "
                 "interpretación estructural, al no satisfacer las "
                 "condiciones de identificación.", pt=PT_TAB,
            espacio_despues=16)

# ------------------------------------------------------ Referencias
parrafo(doc, "Referencias", negrita=True, pt=12, espacio_despues=8)
REFS = [
    "Andrews, D. W. K. (1991). Heteroskedasticity and autocorrelation "
    "consistent covariance matrix estimation. Econometrica, 59(3), 817–858. "
    "https://doi.org/10.2307/2938229",
    "Bank for International Settlements. (s. f.). Credit to the non-financial "
    "sector [Conjunto de datos]. https://www.bis.org/statistics/totcredit.htm",
    "Bureau of Economic Analysis. (s. f.). National income and product "
    "accounts [Conjunto de datos]. https://www.bea.gov/data",
    "Dickey, D. A., y Fuller, W. A. (1979). Distribution of the estimators for "
    "autoregressive time series with a unit root. Journal of the American "
    "Statistical Association, 74(366), 427–431. "
    "https://doi.org/10.2307/2286348",
    "Feenstra, R. C., Inklaar, R., y Timmer, M. P. (2015). The next generation "
    "of the Penn World Table. American Economic Review, 105(10), 3150–3182. "
    "https://doi.org/10.1257/aer.20130954",
    "Fondo Monetario Internacional. (s. f.). Global Debt Database [Conjunto de "
    "datos]. https://www.imf.org/external/datamapper/datasets/GDD",
    "Johansen, S. (1988). Statistical analysis of cointegration vectors. "
    "Journal of Economic Dynamics and Control, 12(2–3), 231–254. "
    "https://doi.org/10.1016/0165-1889(88)90041-3",
    "Johansen, S. (1991). Estimation and hypothesis testing of cointegration "
    "vectors in Gaussian vector autoregressive models. Econometrica, 59(6), "
    "1551–1580. https://doi.org/10.2307/2938278",
    "Johansen, S. (1992a). Cointegration in partial systems and the efficiency "
    "of single-equation analysis. Journal of Econometrics, 52(3), 389–402.",
    "Johansen, S. (1992b). Determination of cointegration rank in the presence "
    "of a linear trend. Oxford Bulletin of Economics and Statistics, 54(3), "
    "383–397.",
    "Johansen, S., Mosconi, R., y Nielsen, B. (2000). Cointegration analysis "
    "in the presence of structural breaks in the deterministic trend. "
    "Econometrics Journal, 3(2), 216–249. "
    "https://doi.org/10.1111/1368-423X.00047",
    "Kwiatkowski, D., Phillips, P. C. B., Schmidt, P., y Shin, Y. (1992). "
    "Testing the null hypothesis of stationarity against the alternative of a "
    "unit root. Journal of Econometrics, 54(1–3), 159–178.",
    "Mbaye, S., Moreno-Badia, M., y Chae, K. (2018). Global debt database: "
    "Methodology and sources (Working Paper N.º 18/111). Fondo Monetario "
    "Internacional.",
    "Newey, W. K., y West, K. D. (1987). A simple, positive semi-definite, "
    "heteroskedasticity and autocorrelation consistent covariance matrix. "
    "Econometrica, 55(3), 703–708. https://doi.org/10.2307/1913610",
    "Pantula, S. G. (1989). Testing for unit roots in time series data. "
    "Econometric Theory, 5(2), 256–271.",
    "Pesaran, M. H., Shin, Y., y Smith, R. J. (2001). Bounds testing "
    "approaches to the analysis of level relationships. Journal of Applied "
    "Econometrics, 16(3), 289–326. https://doi.org/10.1002/jae.616",
    "Phillips, P. C. B., y Perron, P. (1988). Testing for a unit root in time "
    "series regression. Biometrika, 75(2), 335–346. "
    "https://doi.org/10.1093/biomet/75.2.335",
    "Reinsel, G. C., y Ahn, S. K. (1992). Vector autoregressive models with "
    "unit roots and reduced rank structure: Estimation, likelihood ratio test, "
    "and forecasting. Journal of Time Series Analysis, 13(4), 353–375. "
    "https://doi.org/10.1111/j.1467-9892.1992.tb00113.x",
    "Stock, J. H., y Watson, M. W. (1993). A simple estimator of cointegrating "
    "vectors in higher order integrated systems. Econometrica, 61(4), 783–820. "
    "https://doi.org/10.2307/2951763",
    "Zivot, E., y Andrews, D. W. K. (1992). Further evidence on the great "
    "crash, the oil-price shock, and the unit-root hypothesis. Journal of "
    "Business & Economic Statistics, 10(3), 251–270. "
    "https://doi.org/10.1080/07350015.1992.10509904",
]
for r in REFS:
    p = doc.add_paragraph()
    pf = p.paragraph_format
    pf.left_indent = Cm(1.25)
    pf.first_line_indent = Cm(-1.25)
    pf.space_after = Pt(6)
    run = p.add_run(r)
    run.font.name = FUENTE
    run.font.size = Pt(PT_TXT)
    run._element.rPr.rFonts.set(qn("w:eastAsia"), FUENTE)

doc.save(SALIDA)

print("=" * 78)
print("DOCUMENTO GENERADO")
print("=" * 78)
print("archivo: {}".format(SALIDA))
print("tamano : {:,} bytes".format(os.path.getsize(SALIDA)))
d2 = Document(SALIDA)
print("tablas : {}   parrafos: {}".format(len(d2.tables), len(d2.paragraphs)))
if faltan:
    print("\nARCHIVOS AUSENTES: {}".format(", ".join(faltan)))
else:
    print("\nsin archivos ausentes")

DOCUMENTO GENERADO
archivo: /content/drive/MyDrive/tesis_china_eeuu/outputs/Cuadros_resultados_APA_v5.docx
tamano : 97,070 bytes
tablas : 36   parrafos: 169

sin archivos ausentes


In [53]:
# -*- coding: utf-8 -*-
"""
Paso 5, version 6. Documento de cuadros en formato APA 7.
Corrige: desborde de anchos, particion de cifras y codigos, acentuacion,
booleanos, formato numerico, notas y correspondencia cita-referencia.
Salida: outputs/Cuadros_resultados_APA_v6.docx
"""

import os
import re
import numpy as np
import pandas as pd
from docx import Document
from docx.shared import Pt, Cm
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.section import WD_ORIENT
from docx.enum.table import WD_TABLE_ALIGNMENT
from docx.oxml.ns import qn
from docx.oxml import OxmlElement

if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

BASE = "/content/drive/MyDrive/tesis_china_eeuu"
OUT = os.path.join(BASE, "outputs")
DOCV4 = os.path.join(OUT, "Cuadros_resultados_APA_v4.docx")
SALIDA = os.path.join(OUT, "Cuadros_resultados_APA_v6.docx")

FUENTE = "Arial"
PT_TXT = 10
PT_NOTA = 8
ESCALA = [8.0, 7.5, 7.0, 6.5]      # cuerpo de tabla, de mayor a menor
ANCHO_UTIL = 23.90                  # cm
PAD = 0.16                          # margen interno de celda, ambos lados
ZWSP = "\u200b"                     # espacio de ancho cero

# ------------------------------------------------------------------
# metrica tipografica (anchos AFM de Helvetica, milesimas de em)
# ------------------------------------------------------------------
_W = {}
for _c in "0123456789":
    _W[_c] = 556
for _c, _v in {
    " ": 278, "!": 278, '"': 355, "#": 556, "$": 556, "%": 889, "&": 667,
    "'": 191, "(": 333, ")": 333, "*": 389, "+": 584, ",": 278, "-": 333,
    ".": 278, "/": 278, ":": 278, ";": 278, "<": 584, "=": 584, ">": 584,
    "?": 556, "@": 1015, "[": 278, "\\": 278, "]": 278, "^": 469, "_": 556,
    "`": 333, "{": 334, "|": 260, "}": 334, "~": 584,
    "A": 667, "B": 667, "C": 722, "D": 722, "E": 667, "F": 611, "G": 778,
    "H": 722, "I": 278, "J": 500, "K": 667, "L": 556, "M": 833, "N": 722,
    "O": 778, "P": 667, "Q": 778, "R": 722, "S": 667, "T": 611, "U": 722,
    "V": 667, "W": 944, "X": 667, "Y": 667, "Z": 611,
    "a": 556, "b": 556, "c": 500, "d": 556, "e": 556, "f": 278, "g": 556,
    "h": 556, "i": 222, "j": 222, "k": 500, "l": 222, "m": 833, "n": 556,
    "o": 556, "p": 556, "q": 556, "r": 333, "s": 500, "t": 278, "u": 556,
    "v": 500, "w": 722, "x": 500, "y": 500, "z": 500,
}.items():
    _W[_c] = _v

PT_A_CM = 0.0352778


def medir(s, pt, negrita=False):
    """Ancho de una cadena en centimetros."""
    u = 0
    for ch in str(s):
        if ch == ZWSP:
            continue
        u += _W.get(ch, 667 if ch.isupper() else 556)
    cm = u / 1000.0 * pt * PT_A_CM
    if negrita:
        cm *= 1.07
    return cm * 1.03      # margen de seguridad


# ------------------------------------------------------------------
# normalizacion de contenido
# ------------------------------------------------------------------
TILDES = {
    "Pais": "País", "PAIS": "PAÍS", "Codigo": "Código", "Anio": "Año",
    "Ano": "Año", "Hipotesis": "Hipótesis", "Diagnostico": "Diagnóstico",
    "Ordenes": "Órdenes", "Razon": "Razón", "Estandar": "Estándar",
    "estandar": "estándar", "Valido": "Válido", "valido": "válido",
    "Ecuacion": "Ecuación", "Asimetria": "Asimetría",
    "Cointegracion": "Cointegración", "cointegracion": "cointegración",
    "Comprobacion": "Comprobación", "Estadistico": "Estadístico",
    "Decision": "Decisión", "Justificacion": "Justificación",
    "Proporcion": "Proporción", "Autocorrelacion": "Autocorrelación",
    "autocorrelacion": "autocorrelación", "Raices": "Raíces",
    "raices": "raíces", "debil": "débil", "Debil": "Débil",
    "Formacion": "Formación", "formacion": "formación",
    "Inflacion": "Inflación", "inflacion": "inflación",
    "Credito": "Crédito", "credito": "crédito",
    "Publica": "Pública", "publica": "pública",
    "Indice": "Índice", "Poblacion": "Población",
    "Especificacion": "Especificación", "Restriccion": "Restricción",
    "Simulacion": "Simulación", "Version": "Versión",
    "Companera": "Compañera", "companera": "compañera",
    "Deteccion": "Detección", "Interpretacion": "Interpretación",
    "Numero": "Número", "Practica": "Práctica", "practica": "práctica",
    "Analisis": "Análisis", "analisis": "análisis",
    "Aritmetica": "Aritmética", "Geometrica": "Geométrica",
    "Ficticia": "Ficticia", "Interpolacion": "Interpolación",
    "interpolacion": "interpolación", "Empalme": "Empalme",
    "Exclusion": "Exclusión", "exclusion": "exclusión",
    "Sustitucion": "Sustitución", "sustitucion": "sustitución",
    "Correccion": "Corrección", "correccion": "corrección",
    "Estimacion": "Estimación", "estimacion": "estimación",
    "Rechaza": "Rechaza", "Traza": "Traza",
}
_PAT = re.compile(r"\b(" + "|".join(sorted(map(re.escape, TILDES), key=len,
                                            reverse=True)) + r")\b")

SIMBOLOS = [("r<=0", "r ≤ 0"), ("r<=1", "r ≤ 1"), ("r<=2", "r ≤ 2"),
            ("r<=3", "r ≤ 3"), ("<=", " ≤ "), (">=", " ≥ "),
            ("R2", "R²"), ("Chi2", "χ²"), ("chi2", "χ²")]


def acentuar(s):
    if "_" in s:
        return s
    return _PAT.sub(lambda m: TILDES[m.group(0)], s)


def simbolos(s):
    for a, b in SIMBOLOS:
        s = s.replace(a, b)
    return re.sub(r"\s+", " ", s).strip()


def quebrable(s):
    """Permite corte de linea tras guion bajo en codigos largos."""
    if "_" in s and len(s) > 12:
        return s.replace("_", "_" + ZWSP)
    return s


def formatear_columna(serie):
    """Devuelve la columna como texto, con criterio uniforme."""
    vals = serie.tolist()
    num = pd.to_numeric(serie, errors="coerce")
    es_num = num.notna().sum() >= max(1, int(0.8 * len(vals)))
    entera = False
    if es_num:
        fin = num.dropna()
        entera = len(fin) > 0 and np.all(np.isclose(fin, np.round(fin)))
    out = []
    for v, nv in zip(vals, num.tolist()):
        if isinstance(v, (bool, np.bool_)):
            out.append("Sí" if v else "No")
            continue
        s = "" if v is None else str(v).strip()
        if s.lower() in ("nan", "none", "", "<na>"):
            out.append("—")
            continue
        if s == "True":
            out.append("Sí")
            continue
        if s == "False":
            out.append("No")
            continue
        if es_num and nv is not None and np.isfinite(nv):
            if entera:
                out.append("{:,.0f}".format(nv).replace(",", " "))
            elif abs(nv) >= 10000:
                out.append("{:,.1f}".format(nv).replace(",", " "))
            elif abs(nv) < 0.0005 and nv != 0:
                out.append("{:.1e}".format(nv))
            else:
                out.append("{:.3f}".format(nv))
            continue
        out.append(quebrable(simbolos(acentuar(s))))
    return out


def preparar(df):
    d = pd.DataFrame()
    for c in df.columns:
        d[simbolos(acentuar(str(c)))] = formatear_columna(df[c])
    return d


# ------------------------------------------------------------------
# calculo de anchos
# ------------------------------------------------------------------
def _tokens(s):
    return [t for t in re.split(r"[ \u200b]+", str(s)) if t]


def anchos_columna(dft, pt):
    """Devuelve (minimos, deseados) en cm, ya con margen interno."""
    mn, ds = [], []
    for c in dft.columns:
        cab = str(c)
        tmin = max([medir(t, pt, True) for t in _tokens(cab)] or [0.4])
        tful = medir(cab, pt, True)
        for v in dft[c]:
            for t in _tokens(v):
                tmin = max(tmin, medir(t, pt))
            tful = max(tful, medir(v, pt))
        mn.append(tmin + PAD)
        ds.append(min(tful + PAD, 7.0))
    return mn, ds


def repartir(mn, ds, disponible):
    total = sum(mn)
    if total > disponible:
        return None
    resto = disponible - total
    extra = [max(0.0, d - m) for m, d in zip(mn, ds)]
    se = sum(extra)
    if se <= 0:
        return list(mn)
    return [m + resto * e / se if resto < se else d
            for m, d, e in zip(mn, ds, extra)]


def ajustar(dft, disponible=ANCHO_UTIL):
    """Busca el mayor cuerpo que quepa. Devuelve (pt, anchos) o None."""
    for pt in ESCALA:
        mn, ds = anchos_columna(dft, pt)
        w = repartir(mn, ds, disponible)
        if w is not None:
            return pt, w
    return None


CLAVES = {"País", "Sistema", "Modelo", "Variable", "Serie", "Ecuación",
          "Hipótesis", "Caso", "Comprobación", "Forma", "Dependiente"}


def partir_en_paneles(dft, disponible=ANCHO_UTIL):
    """Divide un cuadro ancho repitiendo las columnas identificadoras."""
    cols = list(dft.columns)
    claves = []
    for c in cols:
        if c in CLAVES and len(claves) < 3:
            claves.append(c)
        else:
            break
    if not claves:
        claves = cols[:1]
    resto = [c for c in cols if c not in claves]
    pt = ESCALA[-1]
    mn, _ = anchos_columna(dft, pt)
    ancho = dict(zip(cols, mn))
    base = sum(ancho[c] for c in claves)
    paneles, actual = [], []
    for c in resto:
        if actual and base + sum(ancho[x] for x in actual) + ancho[c] > disponible:
            paneles.append(claves + actual)
            actual = [c]
        else:
            actual.append(c)
    if actual:
        paneles.append(claves + actual)
    return paneles


# ------------------------------------------------------------------
# construccion de tablas
# ------------------------------------------------------------------
def borde(celda, lado, sz=6):
    tcPr = celda._tc.get_or_add_tcPr()
    b = tcPr.find(qn("w:tcBorders"))
    if b is None:
        b = OxmlElement("w:tcBorders")
        tcPr.append(b)
    e = b.find(qn("w:" + lado))
    if e is None:
        e = OxmlElement("w:" + lado)
        b.append(e)
    e.set(qn("w:val"), "single")
    e.set(qn("w:sz"), str(sz))
    e.set(qn("w:color"), "000000")


def sin_bordes(celda):
    tcPr = celda._tc.get_or_add_tcPr()
    b = tcPr.find(qn("w:tcBorders"))
    if b is None:
        b = OxmlElement("w:tcBorders")
        tcPr.append(b)
    for lado in ("top", "left", "bottom", "right"):
        e = b.find(qn("w:" + lado))
        if e is None:
            e = OxmlElement("w:" + lado)
            b.append(e)
        e.set(qn("w:val"), "none")
        e.set(qn("w:sz"), "0")


def margenes_celda(tabla, cm=PAD / 2):
    tblPr = tabla._tbl.tblPr
    mar = OxmlElement("w:tblCellMar")
    for lado, val in (("top", 0.03), ("left", cm), ("bottom", 0.03),
                      ("right", cm)):
        e = OxmlElement("w:" + lado)
        e.set(qn("w:w"), str(int(val * 567)))
        e.set(qn("w:type"), "dxa")
        mar.append(e)
    tblPr.append(mar)


def layout_fijo(tabla):
    el = OxmlElement("w:tblLayout")
    el.set(qn("w:type"), "fixed")
    tabla._tbl.tblPr.append(el)


def repetir_encabezado(fila):
    trPr = fila._tr.get_or_add_trPr()
    th = OxmlElement("w:tblHeader")
    th.set(qn("w:val"), "true")
    trPr.append(th)


def escribir(celda, texto, pt, negrita=False):
    p = celda.paragraphs[0]
    pf = p.paragraph_format
    pf.space_before = Pt(1)
    pf.space_after = Pt(1)
    pf.line_spacing = 1.0
    r = p.add_run(str(texto))
    r.font.name = FUENTE
    r.font.size = Pt(pt)
    r.bold = negrita
    r._element.rPr.rFonts.set(qn("w:eastAsia"), FUENTE)


def insertar_tabla(doc, dft, pt, ws):
    nf, nc = dft.shape
    t = doc.add_table(rows=nf + 1, cols=nc)
    t.alignment = WD_TABLE_ALIGNMENT.CENTER
    t.autofit = False
    layout_fijo(t)
    margenes_celda(t)
    for j, col in enumerate(dft.columns):
        c = t.cell(0, j)
        c.width = Cm(ws[j])
        escribir(c, col, pt, True)
    for i in range(nf):
        for j in range(nc):
            c = t.cell(i + 1, j)
            c.width = Cm(ws[j])
            escribir(c, dft.iat[i, j], pt)
    for f in t.rows:
        for c in f.cells:
            sin_bordes(c)
    for c in t.rows[0].cells:
        borde(c, "top")
        borde(c, "bottom")
    for c in t.rows[-1].cells:
        borde(c, "bottom")
    repetir_encabezado(t.rows[0])
    return t


def parrafo(doc, texto, cursiva=False, negrita=False, pt=PT_TXT,
            despues=6, junto=False):
    p = doc.add_paragraph()
    pf = p.paragraph_format
    pf.space_after = Pt(despues)
    pf.space_before = Pt(0)
    pf.keep_with_next = junto
    r = p.add_run(texto)
    r.font.name = FUENTE
    r.font.size = Pt(pt)
    r.italic = cursiva
    r.bold = negrita
    r._element.rPr.rFonts.set(qn("w:eastAsia"), FUENTE)
    return p


REGISTRO = []


def bloque(doc, numero, titulo, df, nota, subtitulo=None):
    """Inserta un cuadro completo, partiendolo en paneles si no cabe."""
    if df is None or len(df) == 0:
        REGISTRO.append(("VACIO", numero, "", 0, 0))
        return
    dft = preparar(df)
    r = ajustar(dft)
    if r is not None:
        grupos = [list(dft.columns)]
        ptl, wsl = [r[0]], [r[1]]
    else:
        grupos = partir_en_paneles(dft)
        ptl, wsl = [], []
        for g in grupos:
            rg = ajustar(dft[g])
            if rg is None:
                mn, _ = anchos_columna(dft[g], ESCALA[-1])
                f = ANCHO_UTIL / sum(mn)
                rg = (ESCALA[-1], [m * f for m in mn])
            ptl.append(rg[0])
            wsl.append(rg[1])

    letras = "ABCDEFGH"
    for k, g in enumerate(grupos):
        etq = "" if len(grupos) == 1 else " (cont.)" if k else ""
        parrafo(doc, "Cuadro {}{}".format(numero, etq), negrita=True,
                despues=0, junto=True)
        parrafo(doc, titulo, cursiva=True, despues=4, junto=True)
        if subtitulo and len(grupos) == 1:
            parrafo(doc, subtitulo, negrita=True, pt=PT_NOTA + 1, despues=3,
                    junto=True)
        if len(grupos) > 1:
            parrafo(doc, "Panel {}".format(letras[k]), negrita=True,
                    pt=PT_NOTA + 1, despues=3, junto=True)
        insertar_tabla(doc, dft[g], ptl[k], wsl[k])
        parrafo(doc, "", pt=4, despues=0)
        if k == len(grupos) - 1:
            parrafo(doc, nota, pt=PT_NOTA, despues=16)
        else:
            parrafo(doc, "Nota. Continúa en el panel siguiente.", pt=PT_NOTA,
                    despues=14)
        REGISTRO.append(("OK", numero, letras[k] if len(grupos) > 1 else "-",
                         ptl[k], round(sum(wsl[k]), 2)))


def leer(nombre, cols=None, filtro=None):
    ruta = os.path.join(OUT, nombre)
    if not os.path.exists(ruta):
        REGISTRO.append(("FALTA", nombre, "", 0, 0))
        return None
    d = pd.read_csv(ruta)
    if filtro is not None:
        try:
            d = d[filtro(d)]
        except Exception:
            pass
    if cols:
        pres = [c for c in cols if c in d.columns]
        if len(pres) >= 2:
            d = d[pres]
    return d.reset_index(drop=True)


def tabla_docx_a_df(t):
    datos = [[c.text.strip() for c in f.cells] for f in t.rows]
    return pd.DataFrame(datos[1:], columns=datos[0])


# ==================================================================
# documento
# ==================================================================
doc = Document()
sec = doc.sections[0]
sec.orientation = WD_ORIENT.LANDSCAPE
sec.page_width, sec.page_height = Cm(27.94), Cm(21.59)
for m in ("left_margin", "right_margin", "top_margin", "bottom_margin"):
    setattr(sec, m, Cm(2))

est = doc.styles["Normal"]
est.font.name = FUENTE
est.font.size = Pt(PT_TXT)
est.element.rPr.rFonts.set(qn("w:eastAsia"), FUENTE)

parrafo(doc, "Cuadros de resultados", negrita=True, pt=13, despues=2)
parrafo(doc, "Trayectorias de crecimiento en China y Estados Unidos, "
             "1990–2023", cursiva=True, despues=18)

# ---------------------------------------------------------- Cuadro 1
bloque(doc, "1",
       "Cobertura temporal y completitud de las series por país",
       leer("cuadro1_cobertura.csv"),
       "Nota. CN = China; US = Estados Unidos. Las columnas ini y fin indican "
       "el primer y el último año con dato; n es el número de observaciones "
       "válidas y na el porcentaje de valores ausentes dentro del intervalo "
       "1990–2023. Fuentes: Banco Mundial, World Development Indicators "
       "(https://data.worldbank.org); Penn World Table (Feenstra et al., "
       "2015; https://www.rug.nl/ggdc/productivity/pwt); Bank for "
       "International Settlements, Credit to the non-financial sector "
       "(https://www.bis.org/statistics/totcredit.htm); Fondo Monetario "
       "Internacional, Global Debt Database (Mbaye et al., 2018) y World "
       "Economic Outlook (https://www.imf.org/en/Publications/WEO).")

# ---------------------------------------------------------- Cuadro 2
bloque(doc, "2",
       "Estadísticos descriptivos de las variables por país",
       leer("cuadro2_descriptivos.csv"),
       "Nota. M = media; Mdn = mediana; DE = desviación estándar; "
       "CV = coeficiente de variación. Las razones se expresan como "
       "porcentaje del producto interno bruto, salvo indicación en contrario.")

bloque(doc, "2b",
       "Contraste de diferencia de medias entre China y Estados Unidos",
       leer("cuadro2_diferencia_medias.csv"),
       "Nota. Prueba t de Welch para varianzas desiguales. Un valor positivo "
       "de la diferencia indica una media superior en China.")

# ---------------------------------------------------------- Cuadro 3
d3 = leer("cuadro3_raiz_unitaria_v4.csv")
NOTA3 = ("Nota. c = constante; c+t = constante y tendencia; Δ = primera "
         "diferencia. En las pruebas de Dickey–Fuller aumentada y de "
         "Phillips–Perron la hipótesis nula es la existencia de una raíz "
         "unitaria, de modo que el rechazo indica estacionariedad. En la "
         "prueba KPSS la hipótesis nula es la estacionariedad y el rechazo "
         "indica lo contrario; sus asteriscos se asignan por comparación "
         "directa con los valores críticos tabulados, iguales a .347, .463 y "
         ".739 con constante y a .119, .146 y .216 con constante y tendencia. "
         "El orden de integración se declara cuando al menos una prueba de "
         "raíz unitaria rechaza al 5 % y KPSS no rechaza al mismo nivel. "
         "*p < .10. **p < .05. ***p < .01. Pruebas: Dickey y Fuller (1979), "
         "Phillips y Perron (1988), Kwiatkowski et al. (1992) y Zivot y "
         "Andrews (1992).")
if d3 is not None:
    pa = [c for c in ["Pais", "Serie", "n", "ADF (c)", "ADF (c+t)", "PP (c)",
                      "PP (c+t)", "KPSS (c)", "KPSS (c+t)"] if c in d3.columns]
    pb = [c for c in ["Pais", "Serie", "ADF D (c)", "PP D (c)", "KPSS D (c)",
                      "ADF D (c+t)", "PP D (c+t)", "KPSS D (c+t)",
                      "Zivot-Andrews", "Orden"] if c in d3.columns]
    bloque(doc, "3", "Pruebas de raíz unitaria y orden de integración de las "
           "series por país", d3[pa],
           "Nota. Panel A, series en nivel. Las notas completas figuran al pie "
           "del panel B.", subtitulo="Panel A. Series en nivel")
    bloque(doc, "3 (cont.)", "Pruebas de raíz unitaria y orden de integración "
           "de las series por país", d3[pb], NOTA3,
           subtitulo="Panel B. Series en primera diferencia y veredicto")

bloque(doc, "3b",
       "Segunda ronda de contrastes para las series no concluyentes",
       leer("cuadro3b_segunda_ronda.csv"),
       "Nota. Se contrasta la primera y la segunda diferencia para discriminar "
       "entre integración de orden uno y de orden superior. Las series "
       "clasificadas como I(2) se excluyen de las especificaciones en niveles. "
       "*p < .10. **p < .05. ***p < .01.")

bloque(doc, "3c",
       "Comprobaciones sobre la estructura determinista y el orden de "
       "integración de las variables críticas",
       leer("cuadro3c_comprobaciones.csv"),
       "Nota. La primera comprobación contrasta la presencia de una tendencia "
       "determinista en la tasa de crecimiento china mediante mínimos "
       "cuadrados con errores estándar consistentes ante heterocedasticidad y "
       "autocorrelación (Newey y West, 1987). Las restantes evalúan la "
       "estacionariedad de la formación bruta de capital fijo estadounidense y "
       "el orden de integración conjunto de cada sistema.")

# --------------------------------------------------- Cuadros 4, 5 y A2
v4 = Document(DOCV4) if os.path.exists(DOCV4) else None
if v4 is not None and len(v4.tables) >= 7:
    bloque(doc, "4",
           "Especificaciones estimadas, variables y correspondencia con las "
           "hipótesis", tabla_docx_a_df(v4.tables[0]),
           "Nota. Los tamaños de muestra corresponden a las observaciones "
           "disponibles para cada país tras la eliminación por casos "
           "completos.")
    bloque(doc, "5",
           "Bitácora de reconstrucción de series con cobertura incompleta",
           tabla_docx_a_df(v4.tables[1]),
           "Nota. Cada registro documenta la serie afectada, el diagnóstico, "
           "la decisión adoptada y su justificación, con el fin de garantizar "
           "la reproducibilidad del panel.")
else:
    REGISTRO.append(("FALTA", "Cuadros_resultados_APA_v4.docx", "", 0, 0))

# ---------------------------------------------------------- Cuadro 6
bloque(doc, "6",
       "Factores de inflación de la varianza por especificación estimada",
       leer("cuadro6b_vif_por_modelo.csv"),
       "Nota. VIF máx = mayor factor de inflación de la varianza entre los "
       "regresores de cada especificación. El cálculo se realiza por modelo y "
       "no sobre un conjunto agregado de regresores, dado que ninguna "
       "especificación estimada incluye simultáneamente todas las variables "
       "del panel. Se considera aceptable un valor inferior a 5 y severo uno "
       "superior a 10.")

# --------------------------------------------------- Cuadros 7 a 10
bloque(doc, "7",
       "Prueba de límites de cointegración y término de corrección de error "
       "por modelo", leer("cuadro7_cointegracion.csv"),
       "Nota. Estimación mediante un modelo autorregresivo de rezagos "
       "distribuidos con selección del orden por criterio de información de "
       "Akaike. Los valores críticos corresponden al caso III de la tabla "
       "CI(iii) de Pesaran et al. (2001) al 5 %, con los pares 3.79 y 4.85 "
       "para dos regresores, 3.23 y 4.35 para tres, y 2.86 y 4.01 para cuatro. "
       "La regla de decisión exige simultáneamente que el estadístico F supere "
       "el límite superior y que el término de corrección de error sea "
       "negativo y significativo. La vida media se obtiene como "
       "ln(0.5)/ln(1 + ECT).")

bloque(doc, "8",
       "Coeficientes de largo plazo de las relaciones cointegrantes "
       "identificadas", leer("cuadro8_largo_plazo.csv"),
       "Nota. La variable dependiente es el logaritmo del producto interno "
       "bruto por habitante. Los errores estándar de los coeficientes de largo "
       "plazo se obtienen por el método delta. Solo se interpretan los modelos "
       "que superan conjuntamente la prueba de límites y el signo del término "
       "de corrección de error. *p < .10. **p < .05. ***p < .01.")

bloque(doc, "9", "Dinámica de corto plazo y multiplicadores netos",
       leer("cuadro9_corto_plazo.csv"),
       "Nota. El multiplicador neto acumula los coeficientes contemporáneos y "
       "rezagados de cada regresor, dividido por uno menos la suma de los "
       "coeficientes autorregresivos.")

bloque(doc, "10",
       "Pruebas de diagnóstico de los residuos de las ecuaciones de corrección "
       "de error", leer("cuadro10_diagnosticos.csv"),
       "Nota. Se reportan valores de probabilidad. Breusch–Godfrey y Ljung–Box "
       "contrastan la ausencia de autocorrelación; Breusch–Pagan, la "
       "homocedasticidad; Jarque–Bera, la normalidad. Valores superiores a .05 "
       "indican que no se rechaza el supuesto correspondiente.")

# --------------------------------------------------------- Cuadro 11
bloque(doc, "11",
       "Estimación de los coeficientes de largo plazo por mínimos cuadrados "
       "dinámicos", leer("cuadro11_dols.csv"),
       "Nota. Estimador de mínimos cuadrados dinámicos de Stock y Watson "
       "(1993) con adelantos y rezagos de las primeras diferencias de los "
       "regresores y errores estándar consistentes ante heterocedasticidad y "
       "autocorrelación (Newey y West, 1987), con ancho de banda seleccionado "
       "por el procedimiento automático de Andrews (1991). La columna de "
       "diferencia compara el coeficiente con el obtenido por el modelo de "
       "rezagos distribuidos. *p < .10. **p < .05. ***p < .01.")

# --------------------------------------------------------- Cuadro 12
for suf, sub, tit in (
        ("cointegracion", "", "Prueba de límites"),
        ("largo_plazo", "b", "Coeficientes de largo plazo"),
        ("corto_plazo", "c", "Dinámica de corto plazo"),
        ("diagnosticos", "d", "Diagnósticos de los residuos")):
    bloque(doc, "12" + sub,
           "Modelo de deuda privada y apertura comercial (M10). " + tit,
           leer("cuadro12_m10_{}.csv".format(suf)),
           "Nota. La deuda privada procede de la Global Debt Database del "
           "Fondo Monetario Internacional (Mbaye et al., 2018). Esta "
           "especificación es la única cuyos regresores satisfacen "
           "conjuntamente la condición de exogeneidad débil en el sistema "
           "correspondiente.")

# --------------------------------------------------------- Cuadro 13
for suf, sub, tit, nota in (
    ("cointegracion", "", "Prueba de límites",
     "Nota. Se contrastan tres especificaciones auxiliares: la inversión como "
     "único regresor del producto (M11), la inversión junto con la apertura "
     "comercial sin variable financiera (M12) y la inversión como variable "
     "dependiente de la apertura y el crédito (MA). Se reportan los veredictos "
     "bajo los valores críticos correctos y bajo los empleados originalmente."),
    ("largo_plazo", "b", "Coeficientes de largo plazo",
     "Nota. La columna de interpretabilidad indica si la especificación supera "
     "los diagnósticos de residuos exigidos para atribuir contenido "
     "estructural al coeficiente."),
    ("diagnosticos", "c", "Diagnósticos de los residuos",
     "Nota. Valores de probabilidad. La especificación M11 para China no supera "
     "los contrastes de autocorrelación ni de normalidad, por lo que su "
     "coeficiente no admite lectura estructural."),
    ("dols", "d", "Contraste por mínimos cuadrados dinámicos",
     "Nota. Estimación por el procedimiento de Stock y Watson (1993) como "
     "verificación independiente de los coeficientes de largo plazo.")):
    bloque(doc, "13" + sub,
           "Contraste de la hipótesis de mediación de la inversión. " + tit,
           leer("cuadro13_mediacion_{}.csv".format(suf)), nota)

# --------------------------------------------------------- Cuadro 14
for arch, sub, tit, nota in (
    ("cuadro14_johansen_rango_v3.csv", "", "Estadísticos de traza y rango",
     "Nota. Procedimiento de Johansen (1988, 1991) con constante irrestricta. "
     "La traza corregida aplica el factor de corrección por muestra finita de "
     "Reinsel y Ahn (1992). Los valores críticos tabulados resultan "
     "excesivamente permisivos en muestras de esta magnitud; véase el "
     "Cuadro 15."),
    ("cuadro14_johansen_vectores_v3.csv", "b", "Vectores de cointegración",
     "Nota. Coeficientes normalizados sobre el logaritmo del producto por "
     "habitante y expresados con el signo de la relación de largo plazo."),
    ("cuadro14_johansen_ajustes_v3.csv", "c",
     "Velocidades de ajuste y exogeneidad débil por ecuación",
     "Nota. El contraste de exogeneidad débil sigue el procedimiento de sistema "
     "parcial de Johansen (1992a). Un valor de probabilidad superior a .05 "
     "indica que la ecuación correspondiente no responde a las desviaciones "
     "respecto de la relación de largo plazo."),
    ("cuadro14_johansen_exogeneidad_v3.csv", "d",
     "Contraste conjunto de exogeneidad débil de los regresores",
     "Nota. El no rechazo valida formalmente la estimación uniecuacional de la "
     "relación de largo plazo, que resulta eficiente en ese caso."),
    ("cuadro14_johansen_resumen_v3.csv", "e",
     "Resumen de la especificación y los diagnósticos de cada sistema",
     "Nota. Todos los sistemas incluyen una variable indicadora para 2020.")):
    bloque(doc, "14" + sub,
           "Análisis de cointegración multivariante. " + tit, leer(arch), nota)

# --------------------------------------------------------- Cuadro 15
bloque(doc, "15", "Valores críticos simulados de la prueba de traza",
       leer("cuadro15_criticos_simulados.csv"),
       "Nota. Distribución obtenida por simulación de paseos aleatorios "
       "independientes bajo la hipótesis nula, con la misma longitud de "
       "muestra, el mismo orden de rezagos y la variable indicadora de 2020 en "
       "su posición efectiva, dado que la presencia de variables deterministas "
       "de quiebre altera la distribución asintótica del estadístico "
       "(Johansen et al., 2000). La simulación sustituye a la corrección por "
       "muestra finita y no se acumula con ella. Los valores tabulados "
       "resultan sistemáticamente permisivos, lo que invertiría el veredicto "
       "de rango en varias configuraciones.")

bloque(doc, "15b",
       "Estabilidad de los sistemas: raíces de la matriz compañera",
       leer("cuadro15_resumen.csv"),
       "Nota. Impuesto un rango de cointegración de uno en un sistema de "
       "cuatro variables, cabe esperar tres raíces unitarias. La mayor raíz no "
       "unitaria por debajo de la unidad confirma la estabilidad dinámica del "
       "sistema.")

bloque(doc, "15c", "Observaciones atípicas de los residuos por ecuación",
       leer("cuadro15_residuos_por_ecuacion.csv",
            filtro=lambda x: x["Residuo tipificado"].abs() >= 2.0
            if "Residuo tipificado" in x.columns else x.index == x.index),
       "Nota. Se listan las observaciones cuyo residuo tipificado alcanza o "
       "supera dos desviaciones estándar en valor absoluto. Su dispersión "
       "entre ecuaciones y años distintos desaconseja incorporar variables "
       "indicadoras adicionales, cuya inclusión elevaría los valores críticos "
       "simulados sin corregir una fuente común de perturbación.")

# --------------------------------------------------- Cuadros 16 y 17
bloque(doc, "16",
       "Sensibilidad de los resultados de sistema al orden de rezagos y al "
       "tratamiento de 2020", leer("cuadro16_sensibilidad_sistema.csv"),
       "Nota. Cada configuración se contrasta contra sus propios valores "
       "críticos simulados. El signo de la velocidad de ajuste de la ecuación "
       "del producto es el resultado de interés: negativo indica ajuste "
       "corrector hacia la relación de largo plazo.")

bloque(doc, "17",
       "Contraste de necesidad de una tendencia lineal en el espacio de "
       "cointegración", leer("cuadro17_tendencia.csv"),
       "Nota. Razón de verosimilitud entre el modelo con constante irrestricta "
       "y el mismo modelo con tendencia lineal restringida al espacio de "
       "cointegración, con distribución ji cuadrada y grados de libertad "
       "iguales al rango (Johansen, 1992b). El rechazo no conduce a adoptar la "
       "especificación con tendencia: la desaceleración del crecimiento chino "
       "implica un componente cóncavo en el nivel logarítmico que una "
       "tendencia lineal no representa, y su inclusión desplaza la relación de "
       "largo plazo hacia un ajuste de tendencia en el que los coeficientes de "
       "los regresores se aproximan a cero. Véase el apartado de limitaciones.")

bloque(doc, "17b",
       "Secuencia de Pantula para la determinación conjunta del rango y de la "
       "especificación determinista", leer("cuadro17_pantula.csv"),
       "Nota. Procedimiento secuencial de Pantula (1989) evaluado con valores "
       "críticos simulados. El primer no rechazo determina simultáneamente el "
       "rango de cointegración y el modelo determinista.")

# ---------------------------------------------------------- Anexos
bloque(doc, "A1",
       "Sensibilidad de la estimación por mínimos cuadrados dinámicos al "
       "número de adelantos y rezagos y al ancho de banda",
       leer("cuadro11b_dols_sensibilidad.csv"),
       "Nota. Se reportan todas las combinaciones evaluadas. La estabilidad de "
       "los coeficientes a lo largo de la rejilla respalda la robustez de la "
       "estimación principal. *p < .10. **p < .05. ***p < .01.")

if v4 is not None and len(v4.tables) >= 7:
    bloque(doc, "A2",
           "Estimaciones de largo plazo no identificadas. Estados Unidos",
           tabla_docx_a_df(v4.tables[6]),
           "Nota. Se presentan por transparencia y no admiten interpretación "
           "estructural, al no satisfacer las condiciones de identificación.")

# ------------------------------------------------------ Referencias
parrafo(doc, "Referencias", negrita=True, pt=12, despues=8)
REFS = [
    "Andrews, D. W. K. (1991). Heteroskedasticity and autocorrelation "
    "consistent covariance matrix estimation. Econometrica, 59(3), 817–858. "
    "https://doi.org/10.2307/2938229",
    "Bank for International Settlements. (s. f.). Credit to the non-financial "
    "sector [Conjunto de datos]. https://www.bis.org/statistics/totcredit.htm",
    "Dickey, D. A., y Fuller, W. A. (1979). Distribution of the estimators for "
    "autoregressive time series with a unit root. Journal of the American "
    "Statistical Association, 74(366), 427–431. "
    "https://doi.org/10.2307/2286348",
    "Feenstra, R. C., Inklaar, R., y Timmer, M. P. (2015). The next generation "
    "of the Penn World Table. American Economic Review, 105(10), 3150–3182. "
    "https://doi.org/10.1257/aer.20130954",
    "Fondo Monetario Internacional. (s. f.-a). Global Debt Database [Conjunto "
    "de datos]. https://www.imf.org/external/datamapper/datasets/GDD",
    "Fondo Monetario Internacional. (s. f.-b). World Economic Outlook database "
    "[Conjunto de datos]. https://www.imf.org/en/Publications/WEO",
    "Johansen, S. (1988). Statistical analysis of cointegration vectors. "
    "Journal of Economic Dynamics and Control, 12(2–3), 231–254. "
    "https://doi.org/10.1016/0165-1889(88)90041-3",
    "Johansen, S. (1991). Estimation and hypothesis testing of cointegration "
    "vectors in Gaussian vector autoregressive models. Econometrica, 59(6), "
    "1551–1580. https://doi.org/10.2307/2938278",
    "Johansen, S. (1992a). Cointegration in partial systems and the efficiency "
    "of single-equation analysis. Journal of Econometrics, 52(3), 389–402.",
    "Johansen, S. (1992b). Determination of cointegration rank in the presence "
    "of a linear trend. Oxford Bulletin of Economics and Statistics, 54(3), "
    "383–397.",
    "Johansen, S., Mosconi, R., y Nielsen, B. (2000). Cointegration analysis "
    "in the presence of structural breaks in the deterministic trend. "
    "Econometrics Journal, 3(2), 216–249. "
    "https://doi.org/10.1111/1368-423X.00047",
    "Kwiatkowski, D., Phillips, P. C. B., Schmidt, P., y Shin, Y. (1992). "
    "Testing the null hypothesis of stationarity against the alternative of a "
    "unit root. Journal of Econometrics, 54(1–3), 159–178.",
    "Mbaye, S., Moreno-Badia, M., y Chae, K. (2018). Global debt database: "
    "Methodology and sources (Working Paper N.º 18/111). Fondo Monetario "
    "Internacional.",
    "Newey, W. K., y West, K. D. (1987). A simple, positive semi-definite, "
    "heteroskedasticity and autocorrelation consistent covariance matrix. "
    "Econometrica, 55(3), 703–708. https://doi.org/10.2307/1913610",
    "Pantula, S. G. (1989). Testing for unit roots in time series data. "
    "Econometric Theory, 5(2), 256–271.",
    "Pesaran, M. H., Shin, Y., y Smith, R. J. (2001). Bounds testing "
    "approaches to the analysis of level relationships. Journal of Applied "
    "Econometrics, 16(3), 289–326. https://doi.org/10.1002/jae.616",
    "Phillips, P. C. B., y Perron, P. (1988). Testing for a unit root in time "
    "series regression. Biometrika, 75(2), 335–346. "
    "https://doi.org/10.1093/biomet/75.2.335",
    "Reinsel, G. C., y Ahn, S. K. (1992). Vector autoregressive models with "
    "unit roots and reduced rank structure: Estimation, likelihood ratio test, "
    "and forecasting. Journal of Time Series Analysis, 13(4), 353–375. "
    "https://doi.org/10.1111/j.1467-9892.1992.tb00113.x",
    "Stock, J. H., y Watson, M. W. (1993). A simple estimator of cointegrating "
    "vectors in higher order integrated systems. Econometrica, 61(4), 783–820. "
    "https://doi.org/10.2307/2951763",
    "Zivot, E., y Andrews, D. W. K. (1992). Further evidence on the great "
    "crash, the oil-price shock, and the unit-root hypothesis. Journal of "
    "Business & Economic Statistics, 10(3), 251–270. "
    "https://doi.org/10.1080/07350015.1992.10509904",
]
for r in REFS:
    p = doc.add_paragraph()
    pf = p.paragraph_format
    pf.left_indent = Cm(1.25)
    pf.first_line_indent = Cm(-1.25)
    pf.space_after = Pt(6)
    run = p.add_run(r)
    run.font.name = FUENTE
    run.font.size = Pt(PT_TXT)
    run._element.rPr.rFonts.set(qn("w:eastAsia"), FUENTE)

doc.save(SALIDA)

# ------------------------------------------------------ verificacion
print("=" * 78)
print("DOCUMENTO v6 GENERADO")
print("=" * 78)
print("archivo: {}".format(SALIDA))
print("tamano : {:,} bytes".format(os.path.getsize(SALIDA)))
d2 = Document(SALIDA)
print("tablas : {}   parrafos: {}".format(len(d2.tables), len(d2.paragraphs)))

print("\n{:<8} {:<12} {:<7} {:>6} {:>9}".format(
    "estado", "cuadro", "panel", "pt", "ancho cm"))
print("-" * 78)
for e in REGISTRO:
    print("{:<8} {:<12} {:<7} {:>6} {:>9}".format(*[str(x) for x in e]))

mal = [e for e in REGISTRO if e[0] != "OK"]
anch = [e for e in REGISTRO if e[0] == "OK" and e[4] > ANCHO_UTIL + 0.05]
print("\nincidencias: {}".format(len(mal)))
print("cuadros que desbordan: {}".format(len(anch)))
paneles = sum(1 for e in REGISTRO if e[0] == "OK" and e[2] != "-")
print("paneles generados por particion automatica: {}".format(paneles))

DOCUMENTO v6 GENERADO
archivo: /content/drive/MyDrive/tesis_china_eeuu/outputs/Cuadros_resultados_APA_v6.docx
tamano : 101,619 bytes
tablas : 36   parrafos: 169

estado   cuadro       panel       pt  ancho cm
------------------------------------------------------------------------------
OK       1            -          8.0     23.06
OK       2            -          8.0      18.0
OK       2b           -          8.0     11.52
OK       3            -          8.0      16.0
OK       3 (cont.)    -          8.0     23.16
OK       3b           -          8.0     14.42
OK       3c           -          8.0     12.44
OK       4            -          8.0     21.75
OK       5            -          8.0      23.9
OK       6            -          8.0     13.74
OK       7            -          8.0     14.24
OK       8            -          8.0      9.95
OK       9            -          8.0     14.98
OK       10           -          8.0     16.83
OK       11           -          8.0      23.9
OK     

In [54]:
# -*- coding: utf-8 -*-
"""
Verificacion de la version de la Penn World Table efectivamente utilizada.
No modifica ningun archivo. Solo diagnostica.
"""

import os
import re
import glob
import numpy as np
import pandas as pd

try:
    import requests
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "requests"])
    import requests

if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

BASE = "/content/drive/MyDrive/tesis_china_eeuu"
MAESTRO = os.path.join(BASE, "data_clean",
                       "china_us_super_panel_1990_2025.csv")
PANEL = os.path.join(BASE, "data_clean",
                     "china_us_panel_analisis_v6_1990_2023.csv")

lin = "=" * 78


# ------------------------------------------------------------------
# 1. Archivos de PWT presentes en Drive
# ------------------------------------------------------------------
print(lin)
print("1. ARCHIVOS DE PENN WORLD TABLE EN EL PROYECTO")
print(lin)

patrones = ["**/pwt*.xlsx", "**/pwt*.dta", "**/pwt*.csv",
            "**/PWT*.xlsx", "**/PWT*.dta", "**/PWT*.csv"]
encontrados = []
for p in patrones:
    encontrados += glob.glob(os.path.join(BASE, p), recursive=True)
encontrados = sorted(set(encontrados))

if not encontrados:
    print("No hay ningun archivo crudo de PWT guardado en el proyecto.")
    print("Es decir, la descarga fue en memoria y no quedo copia.")
else:
    for f in encontrados:
        print("\narchivo: {}".format(f))
        print("tamano : {:,} bytes".format(os.path.getsize(f)))
        try:
            if f.endswith(".xlsx"):
                hojas = pd.ExcelFile(f).sheet_names
                print("hojas  : {}".format(hojas))
                hoja = "Data" if "Data" in hojas else hojas[-1]
                d = pd.read_excel(f, sheet_name=hoja)
            elif f.endswith(".dta"):
                d = pd.read_stata(f)
            else:
                d = pd.read_csv(f)
            ymin, ymax = int(d["year"].min()), int(d["year"].max())
            npais = d["countrycode"].nunique()
            print("periodo: {}-{}   paises: {}   filas: {:,}".format(
                ymin, ymax, npais, len(d)))
            if ymax >= 2023 and npais >= 185:
                print("VEREDICTO -> PWT 11.0")
            elif ymax <= 2019 and npais <= 183:
                print("VEREDICTO -> PWT 10.01 o anterior")
            else:
                print("VEREDICTO -> no coincide con ninguna version conocida")
            cn = d[(d["countrycode"] == "CHN") & (d["year"].isin(
                [1990, 2000, 2010, 2019]))][["year", "rgdpna", "pop", "hc"]]
            print("China, valores de referencia:")
            print(cn.to_string(index=False))
        except Exception as e:
            print("no se pudo leer: {}".format(e))


# ------------------------------------------------------------------
# 2. URL de descarga registrada en los scripts
# ------------------------------------------------------------------
print("\n" + lin)
print("2. URL DE DESCARGA REGISTRADA EN LOS SCRIPTS")
print(lin)

urls = set()
for f in glob.glob(os.path.join(BASE, "scripts", "**", "*.py"),
                   recursive=True) + glob.glob(
                       os.path.join(BASE, "scripts", "**", "*.md"),
                       recursive=True):
    try:
        txt = open(f, encoding="utf-8", errors="ignore").read()
    except Exception:
        continue
    for u in re.findall(r"https?://[^\s'\"<>)]+", txt):
        if "dataverse" in u or "pwt" in u.lower() or "ggdc" in u:
            urls.add((os.path.basename(f), u))

if not urls:
    print("No se encontro ninguna URL de PWT en scripts/.")
for f, u in sorted(urls):
    print("\n{}\n  {}".format(f, u))
    m = re.search(r"datafile/(\d+)", u)
    if m:
        try:
            r = requests.head(u, allow_redirects=False, timeout=30)
            loc = r.headers.get("Location", "")
            nom = re.search(r"filename%2A%3DUTF-8%27%27([^&]+)", loc)
            doi = re.search(r"/store/(10\.\d+/[A-Z0-9]+)/", loc)
            print("  fichero real : {}".format(
                nom.group(1) if nom else "no identificado"))
            print("  DOI del conj.: {}".format(
                doi.group(1) if doi else "no identificado"))
            print("  10.34894/QT5BCC = PWT 10.01 | "
                  "10.34894/FABVLR = PWT 11.0")
        except Exception as e:
            print("  no se pudo resolver: {}".format(e))


# ------------------------------------------------------------------
# 3. Coherencia interna del panel
# ------------------------------------------------------------------
print("\n" + lin)
print("3. COHERENCIA DEL PANEL CON CADA VERSION")
print(lin)

ruta = PANEL if os.path.exists(PANEL) else MAESTRO
print("panel examinado: {}".format(ruta))
p = pd.read_csv(ruta)
pwt_cols = [c for c in ["rgdpna", "rkna", "rtfpna", "hc", "pop",
                        "ln_gdp_pc_pwt", "ln_rtfpna"] if c in p.columns]

print("\n{:<16} {:>6} {:>8} {:>8} {:>7} {:>7}".format(
    "variable", "pais", "ini", "fin", "n", "n>2019"))
print("-" * 60)
for c in pwt_cols:
    for pais in sorted(p["country"].unique()):
        s = p[(p["country"] == pais)][["year", c]].dropna()
        if len(s) == 0:
            continue
        post = int((s["year"] > 2019).sum())
        print("{:<16} {:>6} {:>8} {:>8} {:>7} {:>7}".format(
            c, pais, int(s["year"].min()), int(s["year"].max()),
            len(s), post))

print("\nLectura: PWT 10.01 termina en 2019 y PWT 11.0 en 2023.")
print("Si la columna 'n>2019' es mayor que cero, el dato NO puede")
print("provenir directamente de PWT 10.01. O es PWT 11.0, o la serie")
print("fue prolongada por encadenamiento con otra fuente.")


# ------------------------------------------------------------------
# 4. Deteccion del punto de empalme
# ------------------------------------------------------------------
print("\n" + lin)
print("4. DETECCION DE EMPALME EN LA SERIE DEL PRODUCTO")
print(lin)

if "ln_gdp_pc_pwt" in p.columns and "gdp_growth_pct" in p.columns:
    for pais in sorted(p["country"].unique()):
        s = p[p["country"] == pais].sort_values("year")
        d = s["ln_gdp_pc_pwt"].diff() * 100.0
        g = s["gdp_growth_pct"]
        # el crecimiento per capita implicito frente al del PIB total
        comp = pd.DataFrame({"year": s["year"].astype(int),
                             "d_ln_pwt_pct": d.round(3),
                             "gdp_growth_pct": g.round(3)})
        comp["brecha"] = (comp["d_ln_pwt_pct"] -
                          comp["gdp_growth_pct"]).round(3)
        sub = comp[comp["year"] >= 2015]
        print("\n{}".format(pais))
        print(sub.to_string(index=False))
    print("\nLectura: si a partir de cierto ano la brecha se estabiliza")
    print("en el valor del crecimiento demografico, la serie posterior")
    print("procede del encadenamiento con el Banco Mundial y no de PWT.")
else:
    print("No estan las columnas necesarias en el panel.")


# ------------------------------------------------------------------
# 5. Que dice el cuadro 1
# ------------------------------------------------------------------
print("\n" + lin)
print("5. ETIQUETA DECLARADA EN EL CUADRO 1")
print(lin)

c1 = os.path.join(BASE, "outputs", "cuadro1_cobertura.csv")
if os.path.exists(c1):
    d = pd.read_csv(c1)
    col_f = [c for c in d.columns if c.lower().startswith("fuente")]
    if col_f:
        f = col_f[0]
        sel = d[d[f].astype(str).str.contains("PWT", case=False, na=False)]
        print(sel.to_string(index=False))
        print("\netiquetas distintas usadas: {}".format(
            sorted(d[f].astype(str).unique())))
else:
    print("No existe cuadro1_cobertura.csv")

print("\n" + lin)
print("FIN DE LA VERIFICACION")
print(lin)

1. ARCHIVOS DE PENN WORLD TABLE EN EL PROYECTO
No hay ningun archivo crudo de PWT guardado en el proyecto.
Es decir, la descarga fue en memoria y no quedo copia.

2. URL DE DESCARGA REGISTRADA EN LOS SCRIPTS
No se encontro ninguna URL de PWT en scripts/.

3. COHERENCIA DEL PANEL CON CADA VERSION
panel examinado: /content/drive/MyDrive/tesis_china_eeuu/data_clean/china_us_panel_analisis_v6_1990_2023.csv

variable           pais      ini      fin       n  n>2019
------------------------------------------------------------
rgdpna               CN     1990     2023      34       4
rgdpna               US     1990     2023      34       4
rkna                 CN     1990     2023      34       4
rkna                 US     1990     2023      34       4
rtfpna               CN     1990     2023      34       4
rtfpna               US     1990     2023      34       4
hc                   CN     1990     2023      34       4
hc                   US     1990     2023      34       4
pop       

In [55]:
# -*- coding: utf-8 -*-
"""
Contraste definitivo de la version de la Penn World Table utilizada.
Descarga PWT 10.01 y PWT 11.0, las archiva y las compara con el panel.
No modifica el panel ni el maestro.
"""

import os
import sys
import subprocess
import numpy as np
import pandas as pd

for pkg in ("openpyxl", "requests"):
    try:
        __import__(pkg)
    except ModuleNotFoundError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])
import requests

if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

BASE = "/content/drive/MyDrive/tesis_china_eeuu"
CRUDO = os.path.join(BASE, "data_raw")
os.makedirs(CRUDO, exist_ok=True)
PANEL = os.path.join(BASE, "data_clean",
                     "china_us_panel_analisis_v6_1990_2023.csv")

VERSIONES = {
    "PWT 10.01": {"id": 354095, "archivo": "pwt1001.xlsx",
                  "doi": "10.34894/QT5BCC"},
    "PWT 11.0":  {"id": 554105, "archivo": "pwt110.xlsx",
                  "doi": "10.34894/FABVLR"},
}
VARS = ["rgdpna", "rkna", "rtfpna", "hc", "pop"]
PAISES = {"CN": "CHN", "US": "USA"}
lin = "=" * 78


def descargar(meta):
    destino = os.path.join(CRUDO, meta["archivo"])
    if os.path.exists(destino) and os.path.getsize(destino) > 1_000_000:
        return destino
    url = "https://dataverse.nl/api/access/datafile/{}".format(meta["id"])
    print("  descargando {} ...".format(meta["archivo"]))
    r = requests.get(url, timeout=600)
    r.raise_for_status()
    with open(destino, "wb") as fh:
        fh.write(r.content)
    return destino


print(lin)
print("1. DESCARGA Y ARCHIVO DE LAS VERSIONES OFICIALES")
print(lin)
crudos = {}
for nom, meta in VERSIONES.items():
    ruta = descargar(meta)
    x = pd.ExcelFile(ruta)
    hoja = "Data" if "Data" in x.sheet_names else x.sheet_names[-1]
    d = pd.read_excel(ruta, sheet_name=hoja)
    crudos[nom] = d
    print("{:<10} {:<14} paises {:>4}  periodo {}-{}  DOI {}".format(
        nom, meta["archivo"], d["countrycode"].nunique(),
        int(d["year"].min()), int(d["year"].max()), meta["doi"]))
    print("           archivado en {}".format(ruta))


print("\n" + lin)
print("2. COMPARACION DEL PANEL CONTRA CADA VERSION")
print(lin)
p = pd.read_csv(PANEL)

resumen = []
for nom, d in crudos.items():
    for cod, iso in PAISES.items():
        ref = d[d["countrycode"] == iso].set_index("year")
        obs = p[p["country"] == cod].set_index("year")
        for v in VARS:
            if v not in obs.columns or v not in ref.columns:
                continue
            comun = sorted(set(obs.index) & set(ref.index))
            a = pd.to_numeric(obs.loc[comun, v], errors="coerce")
            b = pd.to_numeric(ref.loc[comun, v], errors="coerce")
            m = a.notna() & b.notna() & (b != 0)
            if m.sum() == 0:
                continue
            rel = ((a[m] - b[m]).abs() / b[m].abs() * 100.0)
            resumen.append({
                "version": nom, "pais": cod, "variable": v,
                "anios comparados": int(m.sum()),
                "desv media %": round(float(rel.mean()), 4),
                "desv maxima %": round(float(rel.max()), 4),
                "coincide": "SI" if rel.max() < 0.01 else "no",
            })

res = pd.DataFrame(resumen)
print(res.to_string(index=False))


print("\n" + lin)
print("3. VALORES DE ANCLAJE")
print(lin)
for cod, iso in PAISES.items():
    print("\n{}".format("China" if cod == "CN" else "Estados Unidos"))
    t = pd.DataFrame(index=[1990, 2000, 2010, 2019, 2023])
    obs = p[p["country"] == cod].set_index("year")
    if "rgdpna" in obs.columns:
        t["panel"] = pd.to_numeric(
            obs["rgdpna"], errors="coerce").reindex(t.index)
    for nom, d in crudos.items():
        r = d[d["countrycode"] == iso].set_index("year")["rgdpna"]
        t[nom] = r.reindex(t.index)
    print(t.round(0).to_string())


print("\n" + lin)
print("4. VEREDICTO")
print(lin)
if len(res):
    agg = (res.groupby("version")["desv maxima %"].max()
           .sort_values())
    print(agg.to_string())
    mejor = agg.index[0]
    if agg.iloc[0] < 0.01:
        print("\nEl panel reproduce exactamente {}.".format(mejor))
        print("La etiqueta del cuadro 1 debe decir esa version.")
    else:
        print("\nEl panel NO reproduce exactamente ninguna de las dos.")
        print("La version mas proxima es {} con una desviacion".format(mejor))
        print("maxima de {:.4f} %.".format(agg.iloc[0]))
        print("Esto indica que las series fueron transformadas o")
        print("prolongadas, y que la fuente correcta del cuadro 1 no es")
        print("la Penn World Table sin mas, sino la construccion derivada.")
else:
    print("No se pudo comparar ninguna variable.")

print("\n" + lin)

1. DESCARGA Y ARCHIVO DE LAS VERSIONES OFICIALES
  descargando pwt1001.xlsx ...
PWT 10.01  pwt1001.xlsx   paises  183  periodo 1950-2019  DOI 10.34894/QT5BCC
           archivado en /content/drive/MyDrive/tesis_china_eeuu/data_raw/pwt1001.xlsx
  descargando pwt110.xlsx ...
PWT 11.0   pwt110.xlsx    paises  185  periodo 1950-2023  DOI 10.34894/FABVLR
           archivado en /content/drive/MyDrive/tesis_china_eeuu/data_raw/pwt110.xlsx

2. COMPARACION DEL PANEL CONTRA CADA VERSION
  version pais variable  anios comparados  desv media %  desv maxima % coincide
PWT 10.01   CN   rgdpna                30       16.6952        41.3275       no
PWT 10.01   CN     rkna                30       27.1269        27.2928       no
PWT 10.01   CN   rtfpna                30       31.5819        56.2814       no
PWT 10.01   CN       hc                30        0.0000         0.0000       SI
PWT 10.01   CN      pop                30        1.3811         1.9799       no
PWT 10.01   US   rgdpna              

In [56]:
# -*- coding: utf-8 -*-
"""
Registro de la verificacion de procedencia de la Penn World Table.
Anade una entrada a la bitacora sin sobrescribir nada anterior.
"""

import os
from datetime import datetime
import pandas as pd

if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

BASE = "/content/drive/MyDrive/tesis_china_eeuu"
BIT = os.path.join(BASE, "bitacora_reparacion.csv")

entrada = {
    "fecha": datetime.now().strftime("%Y-%m-%d %H:%M"),
    "paso": "5-verificacion",
    "serie": "rgdpna, rkna, rtfpna, hc, pop",
    "incidencia": (
        "La bitacora registraba el archivo de Dataverse 354095, "
        "correspondiente a pwt1001.xlsx (PWT 10.01), mientras que el "
        "cuadro 1 declaraba PWT 11.0."),
    "diagnostico": (
        "Comparacion numerica del panel contra ambas versiones oficiales. "
        "Desviacion maxima frente a PWT 11.0 de 0.0000 % en 5 variables, "
        "2 paises y 34 anios. Desviacion frente a PWT 10.01 de hasta "
        "56.28 %."),
    "decision": (
        "Se confirma PWT 11.0 (DOI 10.34894/FABVLR, archivo pwt110.xlsx, "
        "id 554105). Se corrige el registro previo del identificador. "
        "Se archivan ambas versiones en data_raw/ como respaldo."),
    "justificacion": (
        "Se verifica ademas que ln_gdp_pc_pwt = ln(rgdpna/pop) sobre PWT "
        "11.0 sin encadenamiento externo, lo que anula el pendiente de "
        "correccion de la atribucion de fuente de la variable dependiente. "
        "Queda pendiente declarar en el articulo que la version 11.0 usa "
        "la serie oficial de China y no la ajustada de Maddison y Wu, lo "
        "que impide la comparacion directa con estudios basados en "
        "versiones anteriores."),
}

if os.path.exists(BIT):
    b = pd.read_csv(BIT)
    for k in entrada:
        if k not in b.columns:
            b[k] = pd.NA
    b = pd.concat([b, pd.DataFrame([entrada])], ignore_index=True)
else:
    b = pd.DataFrame([entrada])

b.to_csv(BIT, index=False, encoding="utf-8")
print("bitacora actualizada: {}".format(BIT))
print("registros totales: {}".format(len(b)))
print(b.tail(1).to_string(index=False))

bitacora actualizada: /content/drive/MyDrive/tesis_china_eeuu/bitacora_reparacion.csv
registros totales: 1
           fecha           paso                         serie                                                                                                                                      incidencia                                                                                                                                                                                           diagnostico                                                                                                                                                                             decision                                                                                                                                                                                                                                                                                                                     

In [57]:
# -*- coding: utf-8 -*-
"""
Registro de la verificacion de procedencia de la Penn World Table.
Anade una entrada a la bitacora sin sobrescribir nada anterior.
"""

import os
from datetime import datetime
import pandas as pd

if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

BASE = "/content/drive/MyDrive/tesis_china_eeuu"
BIT = os.path.join(BASE, "bitacora_reparacion.csv")

entrada = {
    "fecha": datetime.now().strftime("%Y-%m-%d %H:%M"),
    "paso": "5-verificacion",
    "serie": "rgdpna, rkna, rtfpna, hc, pop",
    "incidencia": (
        "La bitacora registraba el archivo de Dataverse 354095, "
        "correspondiente a pwt1001.xlsx (PWT 10.01), mientras que el "
        "cuadro 1 declaraba PWT 11.0."),
    "diagnostico": (
        "Comparacion numerica del panel contra ambas versiones oficiales. "
        "Desviacion maxima frente a PWT 11.0 de 0.0000 % en 5 variables, "
        "2 paises y 34 anios. Desviacion frente a PWT 10.01 de hasta "
        "56.28 %."),
    "decision": (
        "Se confirma PWT 11.0 (DOI 10.34894/FABVLR, archivo pwt110.xlsx, "
        "id 554105). Se corrige el registro previo del identificador. "
        "Se archivan ambas versiones en data_raw/ como respaldo."),
    "justificacion": (
        "Se verifica ademas que ln_gdp_pc_pwt = ln(rgdpna/pop) sobre PWT "
        "11.0 sin encadenamiento externo, lo que anula el pendiente de "
        "correccion de la atribucion de fuente de la variable dependiente. "
        "Queda pendiente declarar en el articulo que la version 11.0 usa "
        "la serie oficial de China y no la ajustada de Maddison y Wu, lo "
        "que impide la comparacion directa con estudios basados en "
        "versiones anteriores."),
}

if os.path.exists(BIT):
    b = pd.read_csv(BIT)
    for k in entrada:
        if k not in b.columns:
            b[k] = pd.NA
    b = pd.concat([b, pd.DataFrame([entrada])], ignore_index=True)
else:
    b = pd.DataFrame([entrada])

b.to_csv(BIT, index=False, encoding="utf-8")
print("bitacora actualizada: {}".format(BIT))
print("registros totales: {}".format(len(b)))
print(b.tail(1).to_string(index=False))

bitacora actualizada: /content/drive/MyDrive/tesis_china_eeuu/bitacora_reparacion.csv
registros totales: 2
           fecha           paso                         serie                                                                                                                                      incidencia                                                                                                                                                                                           diagnostico                                                                                                                                                                             decision                                                                                                                                                                                                                                                                                                                     

In [58]:
# -*- coding: utf-8 -*-
"""
Correccion final del Paso 5.
Actualiza la nota del Cuadro 1 y la lista de referencias del documento
Cuadros_resultados_APA_v6.docx con la procedencia verificada de la
Penn World Table 11.0. No modifica ningun otro contenido.
"""

import os
import copy
from docx import Document
from docx.shared import Pt, Cm
from docx.oxml.ns import qn

if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

BASE = "/content/drive/MyDrive/tesis_china_eeuu"
DOCU = os.path.join(BASE, "outputs", "Cuadros_resultados_APA_v6.docx")
FUENTE = "Arial"

NOTA1 = (
    "Nota. CN = China; US = Estados Unidos. Las columnas ini y fin indican "
    "el primer y el último año con dato; n es el número de observaciones "
    "válidas y na el porcentaje de valores ausentes dentro del intervalo "
    "1990–2023. Las series identificadas como PWT proceden de la Penn World "
    "Table 11.0 (Feenstra et al., 2015, 2025), verificadas mediante "
    "comparación numérica con el archivo pwt110.xlsx, con una desviación "
    "máxima de 0.0000 % en las cinco variables, los dos países y los 34 "
    "años. La versión 11.0 emplea la serie oficial de China y no la serie "
    "ajustada de Maddison y Wu de las versiones anteriores. Fuentes "
    "restantes: Banco Mundial, World Development Indicators "
    "(https://data.worldbank.org); Bank for International Settlements, "
    "Credit to the non-financial sector "
    "(https://www.bis.org/statistics/totcredit.htm); Fondo Monetario "
    "Internacional, Global Debt Database (Mbaye et al., 2018) y World "
    "Economic Outlook (https://www.imf.org/en/Publications/WEO)."
)

REF_ART = (
    "Feenstra, R. C., Inklaar, R., y Timmer, M. P. (2015). The next "
    "generation of the Penn World Table. American Economic Review, 105(10), "
    "3150–3182. https://doi.org/10.1257/aer.20130954"
)
REF_DAT = (
    "Feenstra, R. C., Inklaar, R., y Timmer, M. P. (2025). Penn World Table "
    "version 11.0 [Conjunto de datos]. Groningen Growth and Development "
    "Centre. https://doi.org/10.34894/FABVLR"
)

if not os.path.exists(DOCU):
    raise SystemExit(
        "No existe {}. Ejecuta antes el generador de la version 6.".format(
            DOCU))

doc = Document(DOCU)


def reescribir(p, texto):
    """Sustituye el texto de un parrafo conservando su formato."""
    if not p.runs:
        r = p.add_run(texto)
        r.font.name = FUENTE
        r.font.size = Pt(8)
        r._element.rPr.rFonts.set(qn("w:eastAsia"), FUENTE)
        return
    p.runs[0].text = texto
    for r in p.runs[1:]:
        r.text = ""


cambios = []

# --- nota del Cuadro 1 -------------------------------------------
for p in doc.paragraphs:
    t = p.text.strip()
    if t.startswith("Nota. CN = China") and "PWT" in t.upper():
        reescribir(p, NOTA1)
        cambios.append("nota del Cuadro 1 actualizada")
        break
else:
    cambios.append("AVISO: no se localizo la nota del Cuadro 1")

# --- referencias --------------------------------------------------
ref_feenstra = None
for p in doc.paragraphs:
    if p.text.strip().startswith("Feenstra, R. C."):
        ref_feenstra = p
        break

if ref_feenstra is None:
    cambios.append("AVISO: no se localizo la referencia de Feenstra")
else:
    reescribir(ref_feenstra, REF_ART)
    cambios.append("referencia del articulo normalizada")
    ya = any(p.text.strip().startswith("Feenstra, R. C., Inklaar, R., y "
                                       "Timmer, M. P. (2025)")
             for p in doc.paragraphs)
    if ya:
        cambios.append("la entrada del conjunto de datos ya existia")
    else:
        nuevo = copy.deepcopy(ref_feenstra._p)
        ref_feenstra._p.addnext(nuevo)
        from docx.text.paragraph import Paragraph
        pn = Paragraph(nuevo, ref_feenstra._parent)
        reescribir(pn, REF_DAT)
        pf = pn.paragraph_format
        pf.left_indent = Cm(1.25)
        pf.first_line_indent = Cm(-1.25)
        pf.space_after = Pt(6)
        cambios.append("entrada del conjunto de datos insertada")

doc.save(DOCU)

print("=" * 70)
print("CORRECCION APLICADA")
print("=" * 70)
for c in cambios:
    print(" -", c)
d = Document(DOCU)
refs = [p.text.strip() for p in d.paragraphs
        if p.text.strip().startswith(("Andrews", "Bank", "Dickey", "Feenstra",
                                      "Fondo", "Johansen", "Kwiatkowski",
                                      "Mbaye", "Newey", "Pantula", "Pesaran",
                                      "Phillips", "Reinsel", "Stock", "Zivot"))]
print("\nreferencias en el documento: {}".format(len(refs)))
for r in refs:
    print("  {}".format(r[:88]))
print("\ntablas: {}   tamano: {:,} bytes".format(
    len(d.tables), os.path.getsize(DOCU)))

CORRECCION APLICADA
 - nota del Cuadro 1 actualizada
 - referencia del articulo normalizada
 - entrada del conjunto de datos insertada

referencias en el documento: 21
  Andrews, D. W. K. (1991). Heteroskedasticity and autocorrelation consistent covariance m
  Bank for International Settlements. (s. f.). Credit to the non-financial sector [Conjunt
  Dickey, D. A., y Fuller, W. A. (1979). Distribution of the estimators for autoregressive
  Feenstra, R. C., Inklaar, R., y Timmer, M. P. (2015). The next generation of the Penn Wo
  Feenstra, R. C., Inklaar, R., y Timmer, M. P. (2025). Penn World Table version 11.0 [Con
  Fondo Monetario Internacional. (s. f.-a). Global Debt Database [Conjunto de datos]. http
  Fondo Monetario Internacional. (s. f.-b). World Economic Outlook database [Conjunto de d
  Johansen, S. (1988). Statistical analysis of cointegration vectors. Journal of Economic 
  Johansen, S. (1991). Estimation and hypothesis testing of cointegration vectors in Gauss
  Johansen, S

In [59]:
# -*- coding: utf-8 -*-
"""
Paso 5, version 7. Documento de cuadros en formato APA 7.
Version 6: corrige desborde de anchos, particion de cifras y codigos,
acentuacion, booleanos, formato numerico, notas y correspondencia
cita-referencia.
Version 7: anade tres correcciones detectadas en la revision visual.
  1. Los anios dejan de imprimirse con separador de millares.
  2. Se elimina la notacion cientifica; los valores menores que 0.0005
     se imprimen como < 0.001 en columnas de probabilidad y como 0.000
     en las demas.
  3. Ningun encabezado se parte a mitad de palabra: cuando un panel no
     cabe con el cuerpo mas pequeno de la escala se subdivide en dos
     paneles en lugar de comprimir las columnas por debajo de su ancho
     minimo. El registro final avisa si aun asi hubo compresion.
Se incorporan tambien la nota corregida del Cuadro 1 y la entrada de
referencia del conjunto de datos PWT 11.0.
Salida: outputs/Cuadros_resultados_APA_v7.docx
"""

import os
import re
import numpy as np
import pandas as pd
from docx import Document
from docx.shared import Pt, Cm
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.section import WD_ORIENT
from docx.enum.table import WD_TABLE_ALIGNMENT
from docx.oxml.ns import qn
from docx.oxml import OxmlElement

if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

BASE = "/content/drive/MyDrive/tesis_china_eeuu"
OUT = os.path.join(BASE, "outputs")
DOCV4 = os.path.join(OUT, "Cuadros_resultados_APA_v4.docx")
SALIDA = os.path.join(OUT, "Cuadros_resultados_APA_v7.docx")

FUENTE = "Arial"
PT_TXT = 10
PT_NOTA = 8
ESCALA = [8.0, 7.5, 7.0, 6.5]      # cuerpo de tabla, de mayor a menor
ANCHO_UTIL = 23.90                  # cm
PAD = 0.16                          # margen interno de celda, ambos lados
ZWSP = "\u200b"                     # espacio de ancho cero

# ------------------------------------------------------------------
# metrica tipografica (anchos AFM de Helvetica, milesimas de em)
# ------------------------------------------------------------------
_W = {}
for _c in "0123456789":
    _W[_c] = 556
for _c, _v in {
    " ": 278, "!": 278, '"': 355, "#": 556, "$": 556, "%": 889, "&": 667,
    "'": 191, "(": 333, ")": 333, "*": 389, "+": 584, ",": 278, "-": 333,
    ".": 278, "/": 278, ":": 278, ";": 278, "<": 584, "=": 584, ">": 584,
    "?": 556, "@": 1015, "[": 278, "\\": 278, "]": 278, "^": 469, "_": 556,
    "`": 333, "{": 334, "|": 260, "}": 334, "~": 584,
    "A": 667, "B": 667, "C": 722, "D": 722, "E": 667, "F": 611, "G": 778,
    "H": 722, "I": 278, "J": 500, "K": 667, "L": 556, "M": 833, "N": 722,
    "O": 778, "P": 667, "Q": 778, "R": 722, "S": 667, "T": 611, "U": 722,
    "V": 667, "W": 944, "X": 667, "Y": 667, "Z": 611,
    "a": 556, "b": 556, "c": 500, "d": 556, "e": 556, "f": 278, "g": 556,
    "h": 556, "i": 222, "j": 222, "k": 500, "l": 222, "m": 833, "n": 556,
    "o": 556, "p": 556, "q": 556, "r": 333, "s": 500, "t": 278, "u": 556,
    "v": 500, "w": 722, "x": 500, "y": 500, "z": 500,
}.items():
    _W[_c] = _v

PT_A_CM = 0.0352778


def medir(s, pt, negrita=False):
    """Ancho de una cadena en centimetros."""
    u = 0
    for ch in str(s):
        if ch == ZWSP:
            continue
        u += _W.get(ch, 667 if ch.isupper() else 556)
    cm = u / 1000.0 * pt * PT_A_CM
    if negrita:
        cm *= 1.07
    return cm * 1.03      # margen de seguridad


# ------------------------------------------------------------------
# normalizacion de contenido
# ------------------------------------------------------------------
TILDES = {
    "Pais": "País", "PAIS": "PAÍS", "Codigo": "Código", "Anio": "Año",
    "Ano": "Año", "Hipotesis": "Hipótesis", "Diagnostico": "Diagnóstico",
    "Ordenes": "Órdenes", "Razon": "Razón", "Estandar": "Estándar",
    "estandar": "estándar", "Valido": "Válido", "valido": "válido",
    "Ecuacion": "Ecuación", "Asimetria": "Asimetría",
    "Cointegracion": "Cointegración", "cointegracion": "cointegración",
    "Comprobacion": "Comprobación", "Estadistico": "Estadístico",
    "Decision": "Decisión", "Justificacion": "Justificación",
    "Proporcion": "Proporción", "Autocorrelacion": "Autocorrelación",
    "autocorrelacion": "autocorrelación", "Raices": "Raíces",
    "raices": "raíces", "debil": "débil", "Debil": "Débil",
    "Formacion": "Formación", "formacion": "formación",
    "Inflacion": "Inflación", "inflacion": "inflación",
    "Credito": "Crédito", "credito": "crédito",
    "Publica": "Pública", "publica": "pública",
    "Indice": "Índice", "Poblacion": "Población",
    "Especificacion": "Especificación", "Restriccion": "Restricción",
    "Simulacion": "Simulación", "Version": "Versión",
    "Companera": "Compañera", "companera": "compañera",
    "Deteccion": "Detección", "Interpretacion": "Interpretación",
    "Numero": "Número", "Practica": "Práctica", "practica": "práctica",
    "Analisis": "Análisis", "analisis": "análisis",
    "Aritmetica": "Aritmética", "Geometrica": "Geométrica",
    "Ficticia": "Ficticia", "Interpolacion": "Interpolación",
    "interpolacion": "interpolación", "Empalme": "Empalme",
    "Exclusion": "Exclusión", "exclusion": "exclusión",
    "Sustitucion": "Sustitución", "sustitucion": "sustitución",
    "Correccion": "Corrección", "correccion": "corrección",
    "Estimacion": "Estimación", "estimacion": "estimación",
    "Rechaza": "Rechaza", "Traza": "Traza",
}
_PAT = re.compile(r"\b(" + "|".join(sorted(map(re.escape, TILDES), key=len,
                                            reverse=True)) + r")\b")

SIMBOLOS = [("r<=0", "r ≤ 0"), ("r<=1", "r ≤ 1"), ("r<=2", "r ≤ 2"),
            ("r<=3", "r ≤ 3"), ("<=", " ≤ "), (">=", " ≥ "),
            ("R2", "R²"), ("Chi2", "χ²"), ("chi2", "χ²")]


def acentuar(s):
    if "_" in s:
        return s
    return _PAT.sub(lambda m: TILDES[m.group(0)], s)


def simbolos(s):
    for a, b in SIMBOLOS:
        s = s.replace(a, b)
    return re.sub(r"\s+", " ", s).strip()


def quebrable(s):
    """Permite corte de linea tras guion bajo en codigos largos."""
    if "_" in s and len(s) > 12:
        return s.replace("_", "_" + ZWSP)
    return s


ANIO_PAT = re.compile(r"(^|_)(a[nñ]io|a[nñ]o|year|ini|fin)($|_)", re.I)
P_PAT = re.compile(r"(^p$|^p[ _(]|\(p\)|valor[ _]p|p[- _]valor|prob)", re.I)


def formatear_columna(serie):
    """Devuelve la columna como texto, con criterio uniforme."""
    vals = serie.tolist()
    nombre = str(getattr(serie, "name", "") or "")
    num = pd.to_numeric(serie, errors="coerce")
    es_num = num.notna().sum() >= max(1, int(0.8 * len(vals)))
    es_p = bool(P_PAT.search(nombre))
    entera = False
    if es_num:
        fin = num.dropna()
        entera = len(fin) > 0 and np.all(np.isclose(fin, np.round(fin)))
    es_anio = False
    if es_num and entera:
        fin = num.dropna()
        es_anio = bool(ANIO_PAT.search(nombre)) or (
            len(fin) > 0 and fin.min() >= 1500 and fin.max() <= 2200)
    out = []
    for v, nv in zip(vals, num.tolist()):
        if isinstance(v, (bool, np.bool_)):
            out.append("Sí" if v else "No")
            continue
        s = "" if v is None else str(v).strip()
        if s.lower() in ("nan", "none", "", "<na>"):
            out.append("—")
            continue
        if s == "True":
            out.append("Sí")
            continue
        if s == "False":
            out.append("No")
            continue
        if es_num and nv is not None and np.isfinite(nv):
            if es_anio:
                out.append("{:.0f}".format(nv))
            elif entera:
                out.append("{:,.0f}".format(nv).replace(",", " "))
            elif abs(nv) >= 10000:
                out.append("{:,.1f}".format(nv).replace(",", " "))
            elif abs(nv) < 0.0005 and nv != 0:
                out.append("< 0.001" if es_p else "0.000")
            else:
                out.append("{:.3f}".format(nv))
            continue
        out.append(quebrable(simbolos(acentuar(s))))
    return out


def preparar(df):
    d = pd.DataFrame()
    for c in df.columns:
        d[simbolos(acentuar(str(c)))] = formatear_columna(df[c])
    return d


# ------------------------------------------------------------------
# calculo de anchos
# ------------------------------------------------------------------
def _tokens(s):
    return [t for t in re.split(r"[ \u200b]+", str(s)) if t]


def anchos_columna(dft, pt):
    """Devuelve (minimos, deseados) en cm, ya con margen interno."""
    mn, ds = [], []
    for c in dft.columns:
        cab = str(c)
        tmin = max([medir(t, pt, True) for t in _tokens(cab)] or [0.4])
        tful = medir(cab, pt, True)
        for v in dft[c]:
            for t in _tokens(v):
                tmin = max(tmin, medir(t, pt))
            tful = max(tful, medir(v, pt))
        mn.append(tmin + PAD)
        ds.append(min(tful + PAD, 7.0))
    return mn, ds


def repartir(mn, ds, disponible):
    total = sum(mn)
    if total > disponible:
        return None
    resto = disponible - total
    extra = [max(0.0, d - m) for m, d in zip(mn, ds)]
    se = sum(extra)
    if se <= 0:
        return list(mn)
    return [m + resto * e / se if resto < se else d
            for m, d, e in zip(mn, ds, extra)]


def ajustar(dft, disponible=ANCHO_UTIL):
    """Busca el mayor cuerpo que quepa. Devuelve (pt, anchos) o None."""
    for pt in ESCALA:
        mn, ds = anchos_columna(dft, pt)
        w = repartir(mn, ds, disponible)
        if w is not None:
            return pt, w
    return None


CLAVES = {"País", "Sistema", "Modelo", "Variable", "Serie", "Ecuación",
          "Hipótesis", "Caso", "Comprobación", "Forma", "Dependiente"}


def partir_en_paneles(dft, disponible=ANCHO_UTIL):
    """Divide un cuadro ancho repitiendo las columnas identificadoras."""
    cols = list(dft.columns)
    claves = []
    for c in cols:
        if c in CLAVES and len(claves) < 3:
            claves.append(c)
        else:
            break
    if not claves:
        claves = cols[:1]
    resto = [c for c in cols if c not in claves]
    pt = ESCALA[-1]
    mn, _ = anchos_columna(dft, pt)
    ancho = dict(zip(cols, mn))
    base = sum(ancho[c] for c in claves)
    paneles, actual = [], []
    for c in resto:
        if actual and base + sum(ancho[x] for x in actual) + ancho[c] > disponible:
            paneles.append(claves + actual)
            actual = [c]
        else:
            actual.append(c)
    if actual:
        paneles.append(claves + actual)
    return paneles


def subdividir(dft, cols, claves=None, prof=0):
    """Garantiza que cada grupo quepa; parte por la mitad si hace falta.

    Nunca comprime una columna por debajo de su ancho minimo: si un
    panel no cabe con el cuerpo mas pequeno de la escala, se divide en
    dos paneles repitiendo las columnas identificadoras.
    """
    cols = list(cols)
    if ajustar(dft[cols]) is not None or prof >= 4:
        return [cols]
    if claves is None:
        claves = [c for c in cols if c in CLAVES][:3] or cols[:1]
    resto = [c for c in cols if c not in claves]
    if len(resto) < 2:
        return [cols]
    m = len(resto) // 2
    izq = subdividir(dft, claves + resto[:m], claves, prof + 1)
    der = subdividir(dft, claves + resto[m:], claves, prof + 1)
    return izq + der


# ------------------------------------------------------------------
# construccion de tablas
# ------------------------------------------------------------------
def borde(celda, lado, sz=6):
    tcPr = celda._tc.get_or_add_tcPr()
    b = tcPr.find(qn("w:tcBorders"))
    if b is None:
        b = OxmlElement("w:tcBorders")
        tcPr.append(b)
    e = b.find(qn("w:" + lado))
    if e is None:
        e = OxmlElement("w:" + lado)
        b.append(e)
    e.set(qn("w:val"), "single")
    e.set(qn("w:sz"), str(sz))
    e.set(qn("w:color"), "000000")


def sin_bordes(celda):
    tcPr = celda._tc.get_or_add_tcPr()
    b = tcPr.find(qn("w:tcBorders"))
    if b is None:
        b = OxmlElement("w:tcBorders")
        tcPr.append(b)
    for lado in ("top", "left", "bottom", "right"):
        e = b.find(qn("w:" + lado))
        if e is None:
            e = OxmlElement("w:" + lado)
            b.append(e)
        e.set(qn("w:val"), "none")
        e.set(qn("w:sz"), "0")


def margenes_celda(tabla, cm=PAD / 2):
    tblPr = tabla._tbl.tblPr
    mar = OxmlElement("w:tblCellMar")
    for lado, val in (("top", 0.03), ("left", cm), ("bottom", 0.03),
                      ("right", cm)):
        e = OxmlElement("w:" + lado)
        e.set(qn("w:w"), str(int(val * 567)))
        e.set(qn("w:type"), "dxa")
        mar.append(e)
    tblPr.append(mar)


def layout_fijo(tabla):
    el = OxmlElement("w:tblLayout")
    el.set(qn("w:type"), "fixed")
    tabla._tbl.tblPr.append(el)


def repetir_encabezado(fila):
    trPr = fila._tr.get_or_add_trPr()
    th = OxmlElement("w:tblHeader")
    th.set(qn("w:val"), "true")
    trPr.append(th)


def escribir(celda, texto, pt, negrita=False):
    p = celda.paragraphs[0]
    pf = p.paragraph_format
    pf.space_before = Pt(1)
    pf.space_after = Pt(1)
    pf.line_spacing = 1.0
    r = p.add_run(str(texto))
    r.font.name = FUENTE
    r.font.size = Pt(pt)
    r.bold = negrita
    r._element.rPr.rFonts.set(qn("w:eastAsia"), FUENTE)


def insertar_tabla(doc, dft, pt, ws):
    nf, nc = dft.shape
    t = doc.add_table(rows=nf + 1, cols=nc)
    t.alignment = WD_TABLE_ALIGNMENT.CENTER
    t.autofit = False
    layout_fijo(t)
    margenes_celda(t)
    for j, col in enumerate(dft.columns):
        c = t.cell(0, j)
        c.width = Cm(ws[j])
        escribir(c, col, pt, True)
    for i in range(nf):
        for j in range(nc):
            c = t.cell(i + 1, j)
            c.width = Cm(ws[j])
            escribir(c, dft.iat[i, j], pt)
    for f in t.rows:
        for c in f.cells:
            sin_bordes(c)
    for c in t.rows[0].cells:
        borde(c, "top")
        borde(c, "bottom")
    for c in t.rows[-1].cells:
        borde(c, "bottom")
    repetir_encabezado(t.rows[0])
    return t


def parrafo(doc, texto, cursiva=False, negrita=False, pt=PT_TXT,
            despues=6, junto=False):
    p = doc.add_paragraph()
    pf = p.paragraph_format
    pf.space_after = Pt(despues)
    pf.space_before = Pt(0)
    pf.keep_with_next = junto
    r = p.add_run(texto)
    r.font.name = FUENTE
    r.font.size = Pt(pt)
    r.italic = cursiva
    r.bold = negrita
    r._element.rPr.rFonts.set(qn("w:eastAsia"), FUENTE)
    return p


REGISTRO = []


def bloque(doc, numero, titulo, df, nota, subtitulo=None):
    """Inserta un cuadro completo, partiendolo en paneles si no cabe."""
    if df is None or len(df) == 0:
        REGISTRO.append(("VACIO", numero, "", 0, 0))
        return
    dft = preparar(df)
    r = ajustar(dft)
    if r is not None:
        grupos = [list(dft.columns)]
        ptl, wsl = [r[0]], [r[1]]
    else:
        grupos = []
        for g0 in partir_en_paneles(dft):
            grupos.extend(subdividir(dft, g0))
        ptl, wsl = [], []
        for g in grupos:
            rg = ajustar(dft[g])
            if rg is None:
                mn, _ = anchos_columna(dft[g], ESCALA[-1])
                f = ANCHO_UTIL / sum(mn)
                rg = (ESCALA[-1], [m * f for m in mn])
                REGISTRO.append(("COMPRIME", numero, ",".join(map(str, g))[:40],
                                 ESCALA[-1], round(sum(mn), 2)))
            ptl.append(rg[0])
            wsl.append(rg[1])

    letras = "ABCDEFGH"
    for k, g in enumerate(grupos):
        etq = "" if len(grupos) == 1 else " (cont.)" if k else ""
        parrafo(doc, "Cuadro {}{}".format(numero, etq), negrita=True,
                despues=0, junto=True)
        parrafo(doc, titulo, cursiva=True, despues=4, junto=True)
        if subtitulo and len(grupos) == 1:
            parrafo(doc, subtitulo, negrita=True, pt=PT_NOTA + 1, despues=3,
                    junto=True)
        if len(grupos) > 1:
            parrafo(doc, "Panel {}".format(letras[k]), negrita=True,
                    pt=PT_NOTA + 1, despues=3, junto=True)
        insertar_tabla(doc, dft[g], ptl[k], wsl[k])
        parrafo(doc, "", pt=4, despues=0)
        if k == len(grupos) - 1:
            parrafo(doc, nota, pt=PT_NOTA, despues=16)
        else:
            parrafo(doc, "Nota. Continúa en el panel siguiente.", pt=PT_NOTA,
                    despues=14)
        REGISTRO.append(("OK", numero, letras[k] if len(grupos) > 1 else "-",
                         ptl[k], round(sum(wsl[k]), 2)))


def leer(nombre, cols=None, filtro=None):
    ruta = os.path.join(OUT, nombre)
    if not os.path.exists(ruta):
        REGISTRO.append(("FALTA", nombre, "", 0, 0))
        return None
    d = pd.read_csv(ruta)
    if filtro is not None:
        try:
            d = d[filtro(d)]
        except Exception:
            pass
    if cols:
        pres = [c for c in cols if c in d.columns]
        if len(pres) >= 2:
            d = d[pres]
    return d.reset_index(drop=True)


def tabla_docx_a_df(t):
    datos = [[c.text.strip() for c in f.cells] for f in t.rows]
    return pd.DataFrame(datos[1:], columns=datos[0])


# ==================================================================
# documento
# ==================================================================
doc = Document()
sec = doc.sections[0]
sec.orientation = WD_ORIENT.LANDSCAPE
sec.page_width, sec.page_height = Cm(27.94), Cm(21.59)
for m in ("left_margin", "right_margin", "top_margin", "bottom_margin"):
    setattr(sec, m, Cm(2))

est = doc.styles["Normal"]
est.font.name = FUENTE
est.font.size = Pt(PT_TXT)
est.element.rPr.rFonts.set(qn("w:eastAsia"), FUENTE)

parrafo(doc, "Cuadros de resultados", negrita=True, pt=13, despues=2)
parrafo(doc, "Trayectorias de crecimiento en China y Estados Unidos, "
             "1990–2023", cursiva=True, despues=18)

# ---------------------------------------------------------- Cuadro 1
bloque(doc, "1",
       "Cobertura temporal y completitud de las series por país",
       leer("cuadro1_cobertura.csv"),
       "Nota. CN = China; US = Estados Unidos. Las columnas ini y fin indican "
       "el primer y el último año con dato; n es el número de observaciones "
       "válidas y na el porcentaje de valores ausentes dentro del intervalo "
       "1990–2023. Las series identificadas como PWT proceden de la Penn "
       "World Table 11.0 (Feenstra et al., 2015, 2025), verificadas mediante "
       "comparación numérica con el archivo pwt110.xlsx, con una desviación "
       "máxima de 0.0000 % en las cinco variables, los dos países y los 34 "
       "años. La versión 11.0 emplea la serie oficial de China y no la serie "
       "ajustada de Maddison y Wu de las versiones anteriores. Fuentes "
       "restantes: Banco Mundial, World Development Indicators "
       "(https://data.worldbank.org); Bank for "
       "International Settlements, Credit to the non-financial sector "
       "(https://www.bis.org/statistics/totcredit.htm); Fondo Monetario "
       "Internacional, Global Debt Database (Mbaye et al., 2018) y World "
       "Economic Outlook (https://www.imf.org/en/Publications/WEO).")

# ---------------------------------------------------------- Cuadro 2
bloque(doc, "2",
       "Estadísticos descriptivos de las variables por país",
       leer("cuadro2_descriptivos.csv"),
       "Nota. M = media; Mdn = mediana; DE = desviación estándar; "
       "CV = coeficiente de variación. Las razones se expresan como "
       "porcentaje del producto interno bruto, salvo indicación en contrario.")

bloque(doc, "2b",
       "Contraste de diferencia de medias entre China y Estados Unidos",
       leer("cuadro2_diferencia_medias.csv"),
       "Nota. Prueba t de Welch para varianzas desiguales. Un valor positivo "
       "de la diferencia indica una media superior en China.")

# ---------------------------------------------------------- Cuadro 3
d3 = leer("cuadro3_raiz_unitaria_v4.csv")
NOTA3 = ("Nota. c = constante; c+t = constante y tendencia; Δ = primera "
         "diferencia. En las pruebas de Dickey–Fuller aumentada y de "
         "Phillips–Perron la hipótesis nula es la existencia de una raíz "
         "unitaria, de modo que el rechazo indica estacionariedad. En la "
         "prueba KPSS la hipótesis nula es la estacionariedad y el rechazo "
         "indica lo contrario; sus asteriscos se asignan por comparación "
         "directa con los valores críticos tabulados, iguales a .347, .463 y "
         ".739 con constante y a .119, .146 y .216 con constante y tendencia. "
         "El orden de integración se declara cuando al menos una prueba de "
         "raíz unitaria rechaza al 5 % y KPSS no rechaza al mismo nivel. "
         "*p < .10. **p < .05. ***p < .01. Pruebas: Dickey y Fuller (1979), "
         "Phillips y Perron (1988), Kwiatkowski et al. (1992) y Zivot y "
         "Andrews (1992).")
if d3 is not None:
    pa = [c for c in ["Pais", "Serie", "n", "ADF (c)", "ADF (c+t)", "PP (c)",
                      "PP (c+t)", "KPSS (c)", "KPSS (c+t)"] if c in d3.columns]
    pb = [c for c in ["Pais", "Serie", "ADF D (c)", "PP D (c)", "KPSS D (c)",
                      "ADF D (c+t)", "PP D (c+t)", "KPSS D (c+t)",
                      "Zivot-Andrews", "Orden"] if c in d3.columns]
    bloque(doc, "3", "Pruebas de raíz unitaria y orden de integración de las "
           "series por país", d3[pa],
           "Nota. Panel A, series en nivel. Las notas completas figuran al pie "
           "del panel B.", subtitulo="Panel A. Series en nivel")
    bloque(doc, "3 (cont.)", "Pruebas de raíz unitaria y orden de integración "
           "de las series por país", d3[pb], NOTA3,
           subtitulo="Panel B. Series en primera diferencia y veredicto")

bloque(doc, "3b",
       "Segunda ronda de contrastes para las series no concluyentes",
       leer("cuadro3b_segunda_ronda.csv"),
       "Nota. Se contrasta la primera y la segunda diferencia para discriminar "
       "entre integración de orden uno y de orden superior. Las series "
       "clasificadas como I(2) se excluyen de las especificaciones en niveles. "
       "*p < .10. **p < .05. ***p < .01.")

bloque(doc, "3c",
       "Comprobaciones sobre la estructura determinista y el orden de "
       "integración de las variables críticas",
       leer("cuadro3c_comprobaciones.csv"),
       "Nota. La primera comprobación contrasta la presencia de una tendencia "
       "determinista en la tasa de crecimiento china mediante mínimos "
       "cuadrados con errores estándar consistentes ante heterocedasticidad y "
       "autocorrelación (Newey y West, 1987). Las restantes evalúan la "
       "estacionariedad de la formación bruta de capital fijo estadounidense y "
       "el orden de integración conjunto de cada sistema.")

# --------------------------------------------------- Cuadros 4, 5 y A2
v4 = Document(DOCV4) if os.path.exists(DOCV4) else None
if v4 is not None and len(v4.tables) >= 7:
    bloque(doc, "4",
           "Especificaciones estimadas, variables y correspondencia con las "
           "hipótesis", tabla_docx_a_df(v4.tables[0]),
           "Nota. Los tamaños de muestra corresponden a las observaciones "
           "disponibles para cada país tras la eliminación por casos "
           "completos.")
    bloque(doc, "5",
           "Bitácora de reconstrucción de series con cobertura incompleta",
           tabla_docx_a_df(v4.tables[1]),
           "Nota. Cada registro documenta la serie afectada, el diagnóstico, "
           "la decisión adoptada y su justificación, con el fin de garantizar "
           "la reproducibilidad del panel.")
else:
    REGISTRO.append(("FALTA", "Cuadros_resultados_APA_v4.docx", "", 0, 0))

# ---------------------------------------------------------- Cuadro 6
bloque(doc, "6",
       "Factores de inflación de la varianza por especificación estimada",
       leer("cuadro6b_vif_por_modelo.csv"),
       "Nota. VIF máx = mayor factor de inflación de la varianza entre los "
       "regresores de cada especificación. El cálculo se realiza por modelo y "
       "no sobre un conjunto agregado de regresores, dado que ninguna "
       "especificación estimada incluye simultáneamente todas las variables "
       "del panel. Se considera aceptable un valor inferior a 5 y severo uno "
       "superior a 10.")

# --------------------------------------------------- Cuadros 7 a 10
bloque(doc, "7",
       "Prueba de límites de cointegración y término de corrección de error "
       "por modelo", leer("cuadro7_cointegracion.csv"),
       "Nota. Estimación mediante un modelo autorregresivo de rezagos "
       "distribuidos con selección del orden por criterio de información de "
       "Akaike. Los valores críticos corresponden al caso III de la tabla "
       "CI(iii) de Pesaran et al. (2001) al 5 %, con los pares 3.79 y 4.85 "
       "para dos regresores, 3.23 y 4.35 para tres, y 2.86 y 4.01 para cuatro. "
       "La regla de decisión exige simultáneamente que el estadístico F supere "
       "el límite superior y que el término de corrección de error sea "
       "negativo y significativo. La vida media se obtiene como "
       "ln(0.5)/ln(1 + ECT).")

bloque(doc, "8",
       "Coeficientes de largo plazo de las relaciones cointegrantes "
       "identificadas", leer("cuadro8_largo_plazo.csv"),
       "Nota. La variable dependiente es el logaritmo del producto interno "
       "bruto por habitante. Los errores estándar de los coeficientes de largo "
       "plazo se obtienen por el método delta. Solo se interpretan los modelos "
       "que superan conjuntamente la prueba de límites y el signo del término "
       "de corrección de error. *p < .10. **p < .05. ***p < .01.")

bloque(doc, "9", "Dinámica de corto plazo y multiplicadores netos",
       leer("cuadro9_corto_plazo.csv"),
       "Nota. El multiplicador neto acumula los coeficientes contemporáneos y "
       "rezagados de cada regresor, dividido por uno menos la suma de los "
       "coeficientes autorregresivos.")

bloque(doc, "10",
       "Pruebas de diagnóstico de los residuos de las ecuaciones de corrección "
       "de error", leer("cuadro10_diagnosticos.csv"),
       "Nota. Se reportan valores de probabilidad. Breusch–Godfrey y Ljung–Box "
       "contrastan la ausencia de autocorrelación; Breusch–Pagan, la "
       "homocedasticidad; Jarque–Bera, la normalidad. Valores superiores a .05 "
       "indican que no se rechaza el supuesto correspondiente.")

# --------------------------------------------------------- Cuadro 11
bloque(doc, "11",
       "Estimación de los coeficientes de largo plazo por mínimos cuadrados "
       "dinámicos", leer("cuadro11_dols.csv"),
       "Nota. Estimador de mínimos cuadrados dinámicos de Stock y Watson "
       "(1993) con adelantos y rezagos de las primeras diferencias de los "
       "regresores y errores estándar consistentes ante heterocedasticidad y "
       "autocorrelación (Newey y West, 1987), con ancho de banda seleccionado "
       "por el procedimiento automático de Andrews (1991). La columna de "
       "diferencia compara el coeficiente con el obtenido por el modelo de "
       "rezagos distribuidos. *p < .10. **p < .05. ***p < .01.")

# --------------------------------------------------------- Cuadro 12
for suf, sub, tit in (
        ("cointegracion", "", "Prueba de límites"),
        ("largo_plazo", "b", "Coeficientes de largo plazo"),
        ("corto_plazo", "c", "Dinámica de corto plazo"),
        ("diagnosticos", "d", "Diagnósticos de los residuos")):
    bloque(doc, "12" + sub,
           "Modelo de deuda privada y apertura comercial (M10). " + tit,
           leer("cuadro12_m10_{}.csv".format(suf)),
           "Nota. La deuda privada procede de la Global Debt Database del "
           "Fondo Monetario Internacional (Mbaye et al., 2018). Esta "
           "especificación es la única cuyos regresores satisfacen "
           "conjuntamente la condición de exogeneidad débil en el sistema "
           "correspondiente.")

# --------------------------------------------------------- Cuadro 13
for suf, sub, tit, nota in (
    ("cointegracion", "", "Prueba de límites",
     "Nota. Se contrastan tres especificaciones auxiliares: la inversión como "
     "único regresor del producto (M11), la inversión junto con la apertura "
     "comercial sin variable financiera (M12) y la inversión como variable "
     "dependiente de la apertura y el crédito (MA). Se reportan los veredictos "
     "bajo los valores críticos correctos y bajo los empleados originalmente."),
    ("largo_plazo", "b", "Coeficientes de largo plazo",
     "Nota. La columna de interpretabilidad indica si la especificación supera "
     "los diagnósticos de residuos exigidos para atribuir contenido "
     "estructural al coeficiente."),
    ("diagnosticos", "c", "Diagnósticos de los residuos",
     "Nota. Valores de probabilidad. La especificación M11 para China no supera "
     "los contrastes de autocorrelación ni de normalidad, por lo que su "
     "coeficiente no admite lectura estructural."),
    ("dols", "d", "Contraste por mínimos cuadrados dinámicos",
     "Nota. Estimación por el procedimiento de Stock y Watson (1993) como "
     "verificación independiente de los coeficientes de largo plazo.")):
    bloque(doc, "13" + sub,
           "Contraste de la hipótesis de mediación de la inversión. " + tit,
           leer("cuadro13_mediacion_{}.csv".format(suf)), nota)

# --------------------------------------------------------- Cuadro 14
for arch, sub, tit, nota in (
    ("cuadro14_johansen_rango_v3.csv", "", "Estadísticos de traza y rango",
     "Nota. Procedimiento de Johansen (1988, 1991) con constante irrestricta. "
     "La traza corregida aplica el factor de corrección por muestra finita de "
     "Reinsel y Ahn (1992). Los valores críticos tabulados resultan "
     "excesivamente permisivos en muestras de esta magnitud; véase el "
     "Cuadro 15."),
    ("cuadro14_johansen_vectores_v3.csv", "b", "Vectores de cointegración",
     "Nota. Coeficientes normalizados sobre el logaritmo del producto por "
     "habitante y expresados con el signo de la relación de largo plazo."),
    ("cuadro14_johansen_ajustes_v3.csv", "c",
     "Velocidades de ajuste y exogeneidad débil por ecuación",
     "Nota. El contraste de exogeneidad débil sigue el procedimiento de sistema "
     "parcial de Johansen (1992a). Un valor de probabilidad superior a .05 "
     "indica que la ecuación correspondiente no responde a las desviaciones "
     "respecto de la relación de largo plazo."),
    ("cuadro14_johansen_exogeneidad_v3.csv", "d",
     "Contraste conjunto de exogeneidad débil de los regresores",
     "Nota. El no rechazo valida formalmente la estimación uniecuacional de la "
     "relación de largo plazo, que resulta eficiente en ese caso."),
    ("cuadro14_johansen_resumen_v3.csv", "e",
     "Resumen de la especificación y los diagnósticos de cada sistema",
     "Nota. Todos los sistemas incluyen una variable indicadora para 2020.")):
    bloque(doc, "14" + sub,
           "Análisis de cointegración multivariante. " + tit, leer(arch), nota)

# --------------------------------------------------------- Cuadro 15
bloque(doc, "15", "Valores críticos simulados de la prueba de traza",
       leer("cuadro15_criticos_simulados.csv"),
       "Nota. Distribución obtenida por simulación de paseos aleatorios "
       "independientes bajo la hipótesis nula, con la misma longitud de "
       "muestra, el mismo orden de rezagos y la variable indicadora de 2020 en "
       "su posición efectiva, dado que la presencia de variables deterministas "
       "de quiebre altera la distribución asintótica del estadístico "
       "(Johansen et al., 2000). La simulación sustituye a la corrección por "
       "muestra finita y no se acumula con ella. Los valores tabulados "
       "resultan sistemáticamente permisivos, lo que invertiría el veredicto "
       "de rango en varias configuraciones.")

bloque(doc, "15b",
       "Estabilidad de los sistemas: raíces de la matriz compañera",
       leer("cuadro15_resumen.csv"),
       "Nota. Impuesto un rango de cointegración de uno en un sistema de "
       "cuatro variables, cabe esperar tres raíces unitarias. La mayor raíz no "
       "unitaria por debajo de la unidad confirma la estabilidad dinámica del "
       "sistema.")

bloque(doc, "15c", "Observaciones atípicas de los residuos por ecuación",
       leer("cuadro15_residuos_por_ecuacion.csv",
            filtro=lambda x: x["Residuo tipificado"].abs() >= 2.0
            if "Residuo tipificado" in x.columns else x.index == x.index),
       "Nota. Se listan las observaciones cuyo residuo tipificado alcanza o "
       "supera dos desviaciones estándar en valor absoluto. Su dispersión "
       "entre ecuaciones y años distintos desaconseja incorporar variables "
       "indicadoras adicionales, cuya inclusión elevaría los valores críticos "
       "simulados sin corregir una fuente común de perturbación.")

# --------------------------------------------------- Cuadros 16 y 17
bloque(doc, "16",
       "Sensibilidad de los resultados de sistema al orden de rezagos y al "
       "tratamiento de 2020", leer("cuadro16_sensibilidad_sistema.csv"),
       "Nota. Cada configuración se contrasta contra sus propios valores "
       "críticos simulados. El signo de la velocidad de ajuste de la ecuación "
       "del producto es el resultado de interés: negativo indica ajuste "
       "corrector hacia la relación de largo plazo.")

bloque(doc, "17",
       "Contraste de necesidad de una tendencia lineal en el espacio de "
       "cointegración", leer("cuadro17_tendencia.csv"),
       "Nota. Razón de verosimilitud entre el modelo con constante irrestricta "
       "y el mismo modelo con tendencia lineal restringida al espacio de "
       "cointegración, con distribución ji cuadrada y grados de libertad "
       "iguales al rango (Johansen, 1992b). El rechazo no conduce a adoptar la "
       "especificación con tendencia: la desaceleración del crecimiento chino "
       "implica un componente cóncavo en el nivel logarítmico que una "
       "tendencia lineal no representa, y su inclusión desplaza la relación de "
       "largo plazo hacia un ajuste de tendencia en el que los coeficientes de "
       "los regresores se aproximan a cero. Véase el apartado de limitaciones.")

bloque(doc, "17b",
       "Secuencia de Pantula para la determinación conjunta del rango y de la "
       "especificación determinista", leer("cuadro17_pantula.csv"),
       "Nota. Procedimiento secuencial de Pantula (1989) evaluado con valores "
       "críticos simulados. El primer no rechazo determina simultáneamente el "
       "rango de cointegración y el modelo determinista.")

# ---------------------------------------------------------- Anexos
bloque(doc, "A1",
       "Sensibilidad de la estimación por mínimos cuadrados dinámicos al "
       "número de adelantos y rezagos y al ancho de banda",
       leer("cuadro11b_dols_sensibilidad.csv"),
       "Nota. Se reportan todas las combinaciones evaluadas. La estabilidad de "
       "los coeficientes a lo largo de la rejilla respalda la robustez de la "
       "estimación principal. *p < .10. **p < .05. ***p < .01.")

if v4 is not None and len(v4.tables) >= 7:
    bloque(doc, "A2",
           "Estimaciones de largo plazo no identificadas. Estados Unidos",
           tabla_docx_a_df(v4.tables[6]),
           "Nota. Se presentan por transparencia y no admiten interpretación "
           "estructural, al no satisfacer las condiciones de identificación.")

# ------------------------------------------------------ Referencias
parrafo(doc, "Referencias", negrita=True, pt=12, despues=8)
REFS = [
    "Andrews, D. W. K. (1991). Heteroskedasticity and autocorrelation "
    "consistent covariance matrix estimation. Econometrica, 59(3), 817–858. "
    "https://doi.org/10.2307/2938229",
    "Bank for International Settlements. (s. f.). Credit to the non-financial "
    "sector [Conjunto de datos]. https://www.bis.org/statistics/totcredit.htm",
    "Dickey, D. A., y Fuller, W. A. (1979). Distribution of the estimators for "
    "autoregressive time series with a unit root. Journal of the American "
    "Statistical Association, 74(366), 427–431. "
    "https://doi.org/10.2307/2286348",
    "Feenstra, R. C., Inklaar, R., y Timmer, M. P. (2015). The next generation "
    "of the Penn World Table. American Economic Review, 105(10), 3150–3182. "
    "https://doi.org/10.1257/aer.20130954",
    "Feenstra, R. C., Inklaar, R., y Timmer, M. P. (2025). Penn World Table "
    "version 11.0 [Conjunto de datos]. Groningen Growth and Development "
    "Centre. https://doi.org/10.34894/FABVLR",
    "Fondo Monetario Internacional. (s. f.-a). Global Debt Database [Conjunto "
    "de datos]. https://www.imf.org/external/datamapper/datasets/GDD",
    "Fondo Monetario Internacional. (s. f.-b). World Economic Outlook database "
    "[Conjunto de datos]. https://www.imf.org/en/Publications/WEO",
    "Johansen, S. (1988). Statistical analysis of cointegration vectors. "
    "Journal of Economic Dynamics and Control, 12(2–3), 231–254. "
    "https://doi.org/10.1016/0165-1889(88)90041-3",
    "Johansen, S. (1991). Estimation and hypothesis testing of cointegration "
    "vectors in Gaussian vector autoregressive models. Econometrica, 59(6), "
    "1551–1580. https://doi.org/10.2307/2938278",
    "Johansen, S. (1992a). Cointegration in partial systems and the efficiency "
    "of single-equation analysis. Journal of Econometrics, 52(3), 389–402.",
    "Johansen, S. (1992b). Determination of cointegration rank in the presence "
    "of a linear trend. Oxford Bulletin of Economics and Statistics, 54(3), "
    "383–397.",
    "Johansen, S., Mosconi, R., y Nielsen, B. (2000). Cointegration analysis "
    "in the presence of structural breaks in the deterministic trend. "
    "Econometrics Journal, 3(2), 216–249. "
    "https://doi.org/10.1111/1368-423X.00047",
    "Kwiatkowski, D., Phillips, P. C. B., Schmidt, P., y Shin, Y. (1992). "
    "Testing the null hypothesis of stationarity against the alternative of a "
    "unit root. Journal of Econometrics, 54(1–3), 159–178.",
    "Mbaye, S., Moreno-Badia, M., y Chae, K. (2018). Global debt database: "
    "Methodology and sources (Working Paper N.º 18/111). Fondo Monetario "
    "Internacional.",
    "Newey, W. K., y West, K. D. (1987). A simple, positive semi-definite, "
    "heteroskedasticity and autocorrelation consistent covariance matrix. "
    "Econometrica, 55(3), 703–708. https://doi.org/10.2307/1913610",
    "Pantula, S. G. (1989). Testing for unit roots in time series data. "
    "Econometric Theory, 5(2), 256–271.",
    "Pesaran, M. H., Shin, Y., y Smith, R. J. (2001). Bounds testing "
    "approaches to the analysis of level relationships. Journal of Applied "
    "Econometrics, 16(3), 289–326. https://doi.org/10.1002/jae.616",
    "Phillips, P. C. B., y Perron, P. (1988). Testing for a unit root in time "
    "series regression. Biometrika, 75(2), 335–346. "
    "https://doi.org/10.1093/biomet/75.2.335",
    "Reinsel, G. C., y Ahn, S. K. (1992). Vector autoregressive models with "
    "unit roots and reduced rank structure: Estimation, likelihood ratio test, "
    "and forecasting. Journal of Time Series Analysis, 13(4), 353–375. "
    "https://doi.org/10.1111/j.1467-9892.1992.tb00113.x",
    "Stock, J. H., y Watson, M. W. (1993). A simple estimator of cointegrating "
    "vectors in higher order integrated systems. Econometrica, 61(4), 783–820. "
    "https://doi.org/10.2307/2951763",
    "Zivot, E., y Andrews, D. W. K. (1992). Further evidence on the great "
    "crash, the oil-price shock, and the unit-root hypothesis. Journal of "
    "Business & Economic Statistics, 10(3), 251–270. "
    "https://doi.org/10.1080/07350015.1992.10509904",
]
for r in REFS:
    p = doc.add_paragraph()
    pf = p.paragraph_format
    pf.left_indent = Cm(1.25)
    pf.first_line_indent = Cm(-1.25)
    pf.space_after = Pt(6)
    run = p.add_run(r)
    run.font.name = FUENTE
    run.font.size = Pt(PT_TXT)
    run._element.rPr.rFonts.set(qn("w:eastAsia"), FUENTE)

doc.save(SALIDA)

# ------------------------------------------------------ verificacion
print("=" * 78)
print("DOCUMENTO v6 GENERADO")
print("=" * 78)
print("archivo: {}".format(SALIDA))
print("tamano : {:,} bytes".format(os.path.getsize(SALIDA)))
d2 = Document(SALIDA)
print("tablas : {}   parrafos: {}".format(len(d2.tables), len(d2.paragraphs)))

print("\n{:<8} {:<12} {:<7} {:>6} {:>9}".format(
    "estado", "cuadro", "panel", "pt", "ancho cm"))
print("-" * 78)
for e in REGISTRO:
    print("{:<8} {:<12} {:<7} {:>6} {:>9}".format(*[str(x) for x in e]))

mal = [e for e in REGISTRO if e[0] != "OK"]
anch = [e for e in REGISTRO if e[0] == "OK" and e[4] > ANCHO_UTIL + 0.05]
print("\nincidencias: {}".format(len(mal)))
print("cuadros que desbordan: {}".format(len(anch)))
paneles = sum(1 for e in REGISTRO if e[0] == "OK" and e[2] != "-")
print("paneles generados por particion automatica: {}".format(paneles))


DOCUMENTO v6 GENERADO
archivo: /content/drive/MyDrive/tesis_china_eeuu/outputs/Cuadros_resultados_APA_v7.docx
tamano : 101,843 bytes
tablas : 36   parrafos: 170

estado   cuadro       panel       pt  ancho cm
------------------------------------------------------------------------------
OK       1            -          8.0     23.06
OK       2            -          8.0      18.0
OK       2b           -          8.0     11.52
OK       3            -          8.0      16.0
OK       3 (cont.)    -          8.0     23.16
OK       3b           -          8.0     14.42
OK       3c           -          8.0     12.44
OK       4            -          8.0     21.75
OK       5            -          8.0      23.9
OK       6            -          8.0     13.74
OK       7            -          8.0     14.24
OK       8            -          8.0      9.95
OK       9            -          8.0     14.98
OK       10           -          8.0     16.83
OK       11           -          8.0      23.9
OK     

In [60]:
# -*- coding: utf-8 -*-
"""
Paso 5, version 8. Documento de cuadros en formato APA 7.
Version 6: corrige desborde de anchos, particion de cifras y codigos,
acentuacion, booleanos, formato numerico, notas y correspondencia
cita-referencia.
Version 7: anade tres correcciones detectadas en la revision visual.
  1. Los anios dejan de imprimirse con separador de millares.
  2. Se elimina la notacion cientifica; los valores menores que 0.0005
     se imprimen como < 0.001 en columnas de probabilidad y como 0.000
     en las demas.
  3. Ningun encabezado se parte a mitad de palabra: cuando un panel no
     cabe con el cuerpo mas pequeno de la escala se subdivide en dos
     paneles en lugar de comprimir las columnas por debajo de su ancho
     minimo. El registro final avisa si aun asi hubo compresion.
Version 8: corrige la causa raiz del desajuste de columnas. Con
w:tblLayout fijo, Word y LibreOffice leen los anchos de w:tblGrid y no
de w:tcW. python-docx crea la rejilla con columnas iguales y no la
actualiza al asignar celda.width, de modo que en las versiones
anteriores todo el calculo de anchos se descartaba en la composicion y
las columnas salian uniformes. Ahora se escribe w:tblGrid y w:tblW.
Se incorporan tambien la nota corregida del Cuadro 1 y la entrada de
referencia del conjunto de datos PWT 11.0.
Salida: outputs/Cuadros_resultados_APA_v8.docx
"""

import os
import re
import numpy as np
import pandas as pd
from docx import Document
from docx.shared import Pt, Cm
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.section import WD_ORIENT
from docx.enum.table import WD_TABLE_ALIGNMENT
from docx.oxml.ns import qn
from docx.oxml import OxmlElement

if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

BASE = "/content/drive/MyDrive/tesis_china_eeuu"
OUT = os.path.join(BASE, "outputs")
DOCV4 = os.path.join(OUT, "Cuadros_resultados_APA_v4.docx")
SALIDA = os.path.join(OUT, "Cuadros_resultados_APA_v8.docx")

FUENTE = "Arial"
PT_TXT = 10
PT_NOTA = 8
ESCALA = [8.0, 7.5, 7.0, 6.5]      # cuerpo de tabla, de mayor a menor
ANCHO_UTIL = 23.90                  # cm
PAD = 0.16                          # margen interno de celda, ambos lados
ZWSP = "\u200b"                     # espacio de ancho cero

# ------------------------------------------------------------------
# metrica tipografica (anchos AFM de Helvetica, milesimas de em)
# ------------------------------------------------------------------
_W = {}
for _c in "0123456789":
    _W[_c] = 556
for _c, _v in {
    " ": 278, "!": 278, '"': 355, "#": 556, "$": 556, "%": 889, "&": 667,
    "'": 191, "(": 333, ")": 333, "*": 389, "+": 584, ",": 278, "-": 333,
    ".": 278, "/": 278, ":": 278, ";": 278, "<": 584, "=": 584, ">": 584,
    "?": 556, "@": 1015, "[": 278, "\\": 278, "]": 278, "^": 469, "_": 556,
    "`": 333, "{": 334, "|": 260, "}": 334, "~": 584,
    "A": 667, "B": 667, "C": 722, "D": 722, "E": 667, "F": 611, "G": 778,
    "H": 722, "I": 278, "J": 500, "K": 667, "L": 556, "M": 833, "N": 722,
    "O": 778, "P": 667, "Q": 778, "R": 722, "S": 667, "T": 611, "U": 722,
    "V": 667, "W": 944, "X": 667, "Y": 667, "Z": 611,
    "a": 556, "b": 556, "c": 500, "d": 556, "e": 556, "f": 278, "g": 556,
    "h": 556, "i": 222, "j": 222, "k": 500, "l": 222, "m": 833, "n": 556,
    "o": 556, "p": 556, "q": 556, "r": 333, "s": 500, "t": 278, "u": 556,
    "v": 500, "w": 722, "x": 500, "y": 500, "z": 500,
}.items():
    _W[_c] = _v

PT_A_CM = 0.0352778


def medir(s, pt, negrita=False):
    """Ancho de una cadena en centimetros."""
    u = 0
    for ch in str(s):
        if ch == ZWSP:
            continue
        u += _W.get(ch, 667 if ch.isupper() else 556)
    cm = u / 1000.0 * pt * PT_A_CM
    if negrita:
        cm *= 1.07
    return cm * 1.03      # margen de seguridad


# ------------------------------------------------------------------
# normalizacion de contenido
# ------------------------------------------------------------------
TILDES = {
    "Pais": "País", "PAIS": "PAÍS", "Codigo": "Código", "Anio": "Año",
    "Ano": "Año", "Hipotesis": "Hipótesis", "Diagnostico": "Diagnóstico",
    "Ordenes": "Órdenes", "Razon": "Razón", "Estandar": "Estándar",
    "estandar": "estándar", "Valido": "Válido", "valido": "válido",
    "Ecuacion": "Ecuación", "Asimetria": "Asimetría",
    "Cointegracion": "Cointegración", "cointegracion": "cointegración",
    "Comprobacion": "Comprobación", "Estadistico": "Estadístico",
    "Decision": "Decisión", "Justificacion": "Justificación",
    "Proporcion": "Proporción", "Autocorrelacion": "Autocorrelación",
    "autocorrelacion": "autocorrelación", "Raices": "Raíces",
    "raices": "raíces", "debil": "débil", "Debil": "Débil",
    "Formacion": "Formación", "formacion": "formación",
    "Inflacion": "Inflación", "inflacion": "inflación",
    "Credito": "Crédito", "credito": "crédito",
    "Publica": "Pública", "publica": "pública",
    "Indice": "Índice", "Poblacion": "Población",
    "Especificacion": "Especificación", "Restriccion": "Restricción",
    "Simulacion": "Simulación", "Version": "Versión",
    "Companera": "Compañera", "companera": "compañera",
    "Deteccion": "Detección", "Interpretacion": "Interpretación",
    "Numero": "Número", "Practica": "Práctica", "practica": "práctica",
    "Analisis": "Análisis", "analisis": "análisis",
    "Aritmetica": "Aritmética", "Geometrica": "Geométrica",
    "Ficticia": "Ficticia", "Interpolacion": "Interpolación",
    "interpolacion": "interpolación", "Empalme": "Empalme",
    "Exclusion": "Exclusión", "exclusion": "exclusión",
    "Sustitucion": "Sustitución", "sustitucion": "sustitución",
    "Correccion": "Corrección", "correccion": "corrección",
    "Estimacion": "Estimación", "estimacion": "estimación",
    "Rechaza": "Rechaza", "Traza": "Traza",
}
_PAT = re.compile(r"\b(" + "|".join(sorted(map(re.escape, TILDES), key=len,
                                            reverse=True)) + r")\b")

SIMBOLOS = [("r<=0", "r ≤ 0"), ("r<=1", "r ≤ 1"), ("r<=2", "r ≤ 2"),
            ("r<=3", "r ≤ 3"), ("<=", " ≤ "), (">=", " ≥ "),
            ("R2", "R²"), ("Chi2", "χ²"), ("chi2", "χ²")]


def acentuar(s):
    if "_" in s:
        return s
    return _PAT.sub(lambda m: TILDES[m.group(0)], s)


def simbolos(s):
    for a, b in SIMBOLOS:
        s = s.replace(a, b)
    return re.sub(r"\s+", " ", s).strip()


def quebrable(s):
    """Permite corte de linea tras guion bajo en codigos largos."""
    if "_" in s and len(s) > 12:
        return s.replace("_", "_" + ZWSP)
    return s


ANIO_PAT = re.compile(r"(^|_)(a[nñ]io|a[nñ]o|year|ini|fin)($|_)", re.I)
P_PAT = re.compile(r"(^p$|^p[ _(]|\(p\)|valor[ _]p|p[- _]valor|prob)", re.I)


def formatear_columna(serie):
    """Devuelve la columna como texto, con criterio uniforme."""
    vals = serie.tolist()
    nombre = str(getattr(serie, "name", "") or "")
    num = pd.to_numeric(serie, errors="coerce")
    es_num = num.notna().sum() >= max(1, int(0.8 * len(vals)))
    es_p = bool(P_PAT.search(nombre))
    entera = False
    if es_num:
        fin = num.dropna()
        entera = len(fin) > 0 and np.all(np.isclose(fin, np.round(fin)))
    es_anio = False
    if es_num and entera:
        fin = num.dropna()
        es_anio = bool(ANIO_PAT.search(nombre)) or (
            len(fin) > 0 and fin.min() >= 1500 and fin.max() <= 2200)
    out = []
    for v, nv in zip(vals, num.tolist()):
        if isinstance(v, (bool, np.bool_)):
            out.append("Sí" if v else "No")
            continue
        s = "" if v is None else str(v).strip()
        if s.lower() in ("nan", "none", "", "<na>"):
            out.append("—")
            continue
        if s == "True":
            out.append("Sí")
            continue
        if s == "False":
            out.append("No")
            continue
        if es_num and nv is not None and np.isfinite(nv):
            if es_anio:
                out.append("{:.0f}".format(nv))
            elif entera:
                out.append("{:,.0f}".format(nv).replace(",", " "))
            elif abs(nv) >= 10000:
                out.append("{:,.1f}".format(nv).replace(",", " "))
            elif abs(nv) < 0.0005 and nv != 0:
                out.append("< 0.001" if es_p else "0.000")
            else:
                out.append("{:.3f}".format(nv))
            continue
        out.append(quebrable(simbolos(acentuar(s))))
    return out


def preparar(df):
    d = pd.DataFrame()
    for c in df.columns:
        d[simbolos(acentuar(str(c)))] = formatear_columna(df[c])
    return d


# ------------------------------------------------------------------
# calculo de anchos
# ------------------------------------------------------------------
def _tokens(s):
    return [t for t in re.split(r"[ \u200b]+", str(s)) if t]


def anchos_columna(dft, pt):
    """Devuelve (minimos, deseados) en cm, ya con margen interno."""
    mn, ds = [], []
    for c in dft.columns:
        cab = str(c)
        tmin = max([medir(t, pt, True) for t in _tokens(cab)] or [0.4])
        tful = medir(cab, pt, True)
        for v in dft[c]:
            for t in _tokens(v):
                tmin = max(tmin, medir(t, pt))
            tful = max(tful, medir(v, pt))
        mn.append(tmin + PAD)
        ds.append(min(tful + PAD, 7.0))
    return mn, ds


def repartir(mn, ds, disponible):
    total = sum(mn)
    if total > disponible:
        return None
    resto = disponible - total
    extra = [max(0.0, d - m) for m, d in zip(mn, ds)]
    se = sum(extra)
    if se <= 0:
        return list(mn)
    return [m + resto * e / se if resto < se else d
            for m, d, e in zip(mn, ds, extra)]


def ajustar(dft, disponible=ANCHO_UTIL):
    """Busca el mayor cuerpo que quepa. Devuelve (pt, anchos) o None."""
    for pt in ESCALA:
        mn, ds = anchos_columna(dft, pt)
        w = repartir(mn, ds, disponible)
        if w is not None:
            return pt, w
    return None


CLAVES = {"País", "Sistema", "Modelo", "Variable", "Serie", "Ecuación",
          "Hipótesis", "Caso", "Comprobación", "Forma", "Dependiente"}


def partir_en_paneles(dft, disponible=ANCHO_UTIL):
    """Divide un cuadro ancho repitiendo las columnas identificadoras."""
    cols = list(dft.columns)
    claves = []
    for c in cols:
        if c in CLAVES and len(claves) < 3:
            claves.append(c)
        else:
            break
    if not claves:
        claves = cols[:1]
    resto = [c for c in cols if c not in claves]
    pt = ESCALA[-1]
    mn, _ = anchos_columna(dft, pt)
    ancho = dict(zip(cols, mn))
    base = sum(ancho[c] for c in claves)
    paneles, actual = [], []
    for c in resto:
        if actual and base + sum(ancho[x] for x in actual) + ancho[c] > disponible:
            paneles.append(claves + actual)
            actual = [c]
        else:
            actual.append(c)
    if actual:
        paneles.append(claves + actual)
    return paneles


def subdividir(dft, cols, claves=None, prof=0):
    """Garantiza que cada grupo quepa; parte por la mitad si hace falta.

    Nunca comprime una columna por debajo de su ancho minimo: si un
    panel no cabe con el cuerpo mas pequeno de la escala, se divide en
    dos paneles repitiendo las columnas identificadoras.
    """
    cols = list(cols)
    if ajustar(dft[cols]) is not None or prof >= 4:
        return [cols]
    if claves is None:
        claves = [c for c in cols if c in CLAVES][:3] or cols[:1]
    resto = [c for c in cols if c not in claves]
    if len(resto) < 2:
        return [cols]
    m = len(resto) // 2
    izq = subdividir(dft, claves + resto[:m], claves, prof + 1)
    der = subdividir(dft, claves + resto[m:], claves, prof + 1)
    return izq + der


# ------------------------------------------------------------------
# construccion de tablas
# ------------------------------------------------------------------
def borde(celda, lado, sz=6):
    tcPr = celda._tc.get_or_add_tcPr()
    b = tcPr.find(qn("w:tcBorders"))
    if b is None:
        b = OxmlElement("w:tcBorders")
        tcPr.append(b)
    e = b.find(qn("w:" + lado))
    if e is None:
        e = OxmlElement("w:" + lado)
        b.append(e)
    e.set(qn("w:val"), "single")
    e.set(qn("w:sz"), str(sz))
    e.set(qn("w:color"), "000000")


def sin_bordes(celda):
    tcPr = celda._tc.get_or_add_tcPr()
    b = tcPr.find(qn("w:tcBorders"))
    if b is None:
        b = OxmlElement("w:tcBorders")
        tcPr.append(b)
    for lado in ("top", "left", "bottom", "right"):
        e = b.find(qn("w:" + lado))
        if e is None:
            e = OxmlElement("w:" + lado)
            b.append(e)
        e.set(qn("w:val"), "none")
        e.set(qn("w:sz"), "0")


def margenes_celda(tabla, cm=PAD / 2):
    tblPr = tabla._tbl.tblPr
    mar = OxmlElement("w:tblCellMar")
    for lado, val in (("top", 0.03), ("left", cm), ("bottom", 0.03),
                      ("right", cm)):
        e = OxmlElement("w:" + lado)
        e.set(qn("w:w"), str(int(val * 567)))
        e.set(qn("w:type"), "dxa")
        mar.append(e)
    tblPr.append(mar)


def layout_fijo(tabla):
    el = OxmlElement("w:tblLayout")
    el.set(qn("w:type"), "fixed")
    tabla._tbl.tblPr.append(el)


def repetir_encabezado(fila):
    trPr = fila._tr.get_or_add_trPr()
    th = OxmlElement("w:tblHeader")
    th.set(qn("w:val"), "true")
    trPr.append(th)


def escribir(celda, texto, pt, negrita=False):
    p = celda.paragraphs[0]
    pf = p.paragraph_format
    pf.space_before = Pt(1)
    pf.space_after = Pt(1)
    pf.line_spacing = 1.0
    r = p.add_run(str(texto))
    r.font.name = FUENTE
    r.font.size = Pt(pt)
    r.bold = negrita
    r._element.rPr.rFonts.set(qn("w:eastAsia"), FUENTE)


def fijar_rejilla(tabla, ws):
    """Escribe los anchos calculados en la rejilla de la tabla.

    Es el paso decisivo: con w:tblLayout fijo, Word y LibreOffice leen
    los anchos de w:tblGrid, no de w:tcW. Si la rejilla conserva los
    valores uniformes que crea python-docx al construir la tabla, todo
    el calculo de anchos se pierde y las columnas salen iguales.
    """
    grid = tabla._tbl.find(qn("w:tblGrid"))
    if grid is not None:
        for col, w in zip(grid.findall(qn("w:gridCol")), ws):
            col.set(qn("w:w"), str(int(round(w * 567))))
    tblPr = tabla._tbl.tblPr
    ant = tblPr.find(qn("w:tblW"))
    if ant is not None:
        tblPr.remove(ant)
    e = OxmlElement("w:tblW")
    e.set(qn("w:w"), str(int(round(sum(ws) * 567))))
    e.set(qn("w:type"), "dxa")
    tblPr.append(e)


def insertar_tabla(doc, dft, pt, ws):
    nf, nc = dft.shape
    t = doc.add_table(rows=nf + 1, cols=nc)
    t.alignment = WD_TABLE_ALIGNMENT.CENTER
    t.autofit = False
    layout_fijo(t)
    margenes_celda(t)
    fijar_rejilla(t, ws)
    for j, col in enumerate(dft.columns):
        c = t.cell(0, j)
        c.width = Cm(ws[j])
        escribir(c, col, pt, True)
    for i in range(nf):
        for j in range(nc):
            c = t.cell(i + 1, j)
            c.width = Cm(ws[j])
            escribir(c, dft.iat[i, j], pt)
    for f in t.rows:
        for c in f.cells:
            sin_bordes(c)
    for c in t.rows[0].cells:
        borde(c, "top")
        borde(c, "bottom")
    for c in t.rows[-1].cells:
        borde(c, "bottom")
    repetir_encabezado(t.rows[0])
    fijar_rejilla(t, ws)
    return t


def parrafo(doc, texto, cursiva=False, negrita=False, pt=PT_TXT,
            despues=6, junto=False):
    p = doc.add_paragraph()
    pf = p.paragraph_format
    pf.space_after = Pt(despues)
    pf.space_before = Pt(0)
    pf.keep_with_next = junto
    r = p.add_run(texto)
    r.font.name = FUENTE
    r.font.size = Pt(pt)
    r.italic = cursiva
    r.bold = negrita
    r._element.rPr.rFonts.set(qn("w:eastAsia"), FUENTE)
    return p


REGISTRO = []


def bloque(doc, numero, titulo, df, nota, subtitulo=None):
    """Inserta un cuadro completo, partiendolo en paneles si no cabe."""
    if df is None or len(df) == 0:
        REGISTRO.append(("VACIO", numero, "", 0, 0))
        return
    dft = preparar(df)
    r = ajustar(dft)
    if r is not None:
        grupos = [list(dft.columns)]
        ptl, wsl = [r[0]], [r[1]]
    else:
        grupos = []
        for g0 in partir_en_paneles(dft):
            grupos.extend(subdividir(dft, g0))
        ptl, wsl = [], []
        for g in grupos:
            rg = ajustar(dft[g])
            if rg is None:
                mn, _ = anchos_columna(dft[g], ESCALA[-1])
                f = ANCHO_UTIL / sum(mn)
                rg = (ESCALA[-1], [m * f for m in mn])
                REGISTRO.append(("COMPRIME", numero, ",".join(map(str, g))[:40],
                                 ESCALA[-1], round(sum(mn), 2)))
            ptl.append(rg[0])
            wsl.append(rg[1])

    letras = "ABCDEFGH"
    for k, g in enumerate(grupos):
        etq = "" if len(grupos) == 1 else " (cont.)" if k else ""
        parrafo(doc, "Cuadro {}{}".format(numero, etq), negrita=True,
                despues=0, junto=True)
        parrafo(doc, titulo, cursiva=True, despues=4, junto=True)
        if subtitulo and len(grupos) == 1:
            parrafo(doc, subtitulo, negrita=True, pt=PT_NOTA + 1, despues=3,
                    junto=True)
        if len(grupos) > 1:
            parrafo(doc, "Panel {}".format(letras[k]), negrita=True,
                    pt=PT_NOTA + 1, despues=3, junto=True)
        insertar_tabla(doc, dft[g], ptl[k], wsl[k])
        parrafo(doc, "", pt=4, despues=0)
        if k == len(grupos) - 1:
            parrafo(doc, nota, pt=PT_NOTA, despues=16)
        else:
            parrafo(doc, "Nota. Continúa en el panel siguiente.", pt=PT_NOTA,
                    despues=14)
        REGISTRO.append(("OK", numero, letras[k] if len(grupos) > 1 else "-",
                         ptl[k], round(sum(wsl[k]), 2)))


def leer(nombre, cols=None, filtro=None):
    ruta = os.path.join(OUT, nombre)
    if not os.path.exists(ruta):
        REGISTRO.append(("FALTA", nombre, "", 0, 0))
        return None
    d = pd.read_csv(ruta)
    if filtro is not None:
        try:
            d = d[filtro(d)]
        except Exception:
            pass
    if cols:
        pres = [c for c in cols if c in d.columns]
        if len(pres) >= 2:
            d = d[pres]
    return d.reset_index(drop=True)


def tabla_docx_a_df(t):
    datos = [[c.text.strip() for c in f.cells] for f in t.rows]
    return pd.DataFrame(datos[1:], columns=datos[0])


# ==================================================================
# documento
# ==================================================================
doc = Document()
sec = doc.sections[0]
sec.orientation = WD_ORIENT.LANDSCAPE
sec.page_width, sec.page_height = Cm(27.94), Cm(21.59)
for m in ("left_margin", "right_margin", "top_margin", "bottom_margin"):
    setattr(sec, m, Cm(2))

est = doc.styles["Normal"]
est.font.name = FUENTE
est.font.size = Pt(PT_TXT)
est.element.rPr.rFonts.set(qn("w:eastAsia"), FUENTE)

parrafo(doc, "Cuadros de resultados", negrita=True, pt=13, despues=2)
parrafo(doc, "Trayectorias de crecimiento en China y Estados Unidos, "
             "1990–2023", cursiva=True, despues=18)

# ---------------------------------------------------------- Cuadro 1
bloque(doc, "1",
       "Cobertura temporal y completitud de las series por país",
       leer("cuadro1_cobertura.csv"),
       "Nota. CN = China; US = Estados Unidos. Las columnas ini y fin indican "
       "el primer y el último año con dato; n es el número de observaciones "
       "válidas y na el porcentaje de valores ausentes dentro del intervalo "
       "1990–2023. Las series identificadas como PWT proceden de la Penn "
       "World Table 11.0 (Feenstra et al., 2015, 2025), verificadas mediante "
       "comparación numérica con el archivo pwt110.xlsx, con una desviación "
       "máxima de 0.0000 % en las cinco variables, los dos países y los 34 "
       "años. La versión 11.0 emplea la serie oficial de China y no la serie "
       "ajustada de Maddison y Wu de las versiones anteriores. Fuentes "
       "restantes: Banco Mundial, World Development Indicators "
       "(https://data.worldbank.org); Bank for "
       "International Settlements, Credit to the non-financial sector "
       "(https://www.bis.org/statistics/totcredit.htm); Fondo Monetario "
       "Internacional, Global Debt Database (Mbaye et al., 2018) y World "
       "Economic Outlook (https://www.imf.org/en/Publications/WEO).")

# ---------------------------------------------------------- Cuadro 2
bloque(doc, "2",
       "Estadísticos descriptivos de las variables por país",
       leer("cuadro2_descriptivos.csv"),
       "Nota. M = media; Mdn = mediana; DE = desviación estándar; "
       "CV = coeficiente de variación. Las razones se expresan como "
       "porcentaje del producto interno bruto, salvo indicación en contrario.")

bloque(doc, "2b",
       "Contraste de diferencia de medias entre China y Estados Unidos",
       leer("cuadro2_diferencia_medias.csv"),
       "Nota. Prueba t de Welch para varianzas desiguales. Un valor positivo "
       "de la diferencia indica una media superior en China.")

# ---------------------------------------------------------- Cuadro 3
d3 = leer("cuadro3_raiz_unitaria_v4.csv")
NOTA3 = ("Nota. c = constante; c+t = constante y tendencia; Δ = primera "
         "diferencia. En las pruebas de Dickey–Fuller aumentada y de "
         "Phillips–Perron la hipótesis nula es la existencia de una raíz "
         "unitaria, de modo que el rechazo indica estacionariedad. En la "
         "prueba KPSS la hipótesis nula es la estacionariedad y el rechazo "
         "indica lo contrario; sus asteriscos se asignan por comparación "
         "directa con los valores críticos tabulados, iguales a .347, .463 y "
         ".739 con constante y a .119, .146 y .216 con constante y tendencia. "
         "El orden de integración se declara cuando al menos una prueba de "
         "raíz unitaria rechaza al 5 % y KPSS no rechaza al mismo nivel. "
         "*p < .10. **p < .05. ***p < .01. Pruebas: Dickey y Fuller (1979), "
         "Phillips y Perron (1988), Kwiatkowski et al. (1992) y Zivot y "
         "Andrews (1992).")
if d3 is not None:
    pa = [c for c in ["Pais", "Serie", "n", "ADF (c)", "ADF (c+t)", "PP (c)",
                      "PP (c+t)", "KPSS (c)", "KPSS (c+t)"] if c in d3.columns]
    pb = [c for c in ["Pais", "Serie", "ADF D (c)", "PP D (c)", "KPSS D (c)",
                      "ADF D (c+t)", "PP D (c+t)", "KPSS D (c+t)",
                      "Zivot-Andrews", "Orden"] if c in d3.columns]
    bloque(doc, "3", "Pruebas de raíz unitaria y orden de integración de las "
           "series por país", d3[pa],
           "Nota. Panel A, series en nivel. Las notas completas figuran al pie "
           "del panel B.", subtitulo="Panel A. Series en nivel")
    bloque(doc, "3 (cont.)", "Pruebas de raíz unitaria y orden de integración "
           "de las series por país", d3[pb], NOTA3,
           subtitulo="Panel B. Series en primera diferencia y veredicto")

bloque(doc, "3b",
       "Segunda ronda de contrastes para las series no concluyentes",
       leer("cuadro3b_segunda_ronda.csv"),
       "Nota. Se contrasta la primera y la segunda diferencia para discriminar "
       "entre integración de orden uno y de orden superior. Las series "
       "clasificadas como I(2) se excluyen de las especificaciones en niveles. "
       "*p < .10. **p < .05. ***p < .01.")

bloque(doc, "3c",
       "Comprobaciones sobre la estructura determinista y el orden de "
       "integración de las variables críticas",
       leer("cuadro3c_comprobaciones.csv"),
       "Nota. La primera comprobación contrasta la presencia de una tendencia "
       "determinista en la tasa de crecimiento china mediante mínimos "
       "cuadrados con errores estándar consistentes ante heterocedasticidad y "
       "autocorrelación (Newey y West, 1987). Las restantes evalúan la "
       "estacionariedad de la formación bruta de capital fijo estadounidense y "
       "el orden de integración conjunto de cada sistema.")

# --------------------------------------------------- Cuadros 4, 5 y A2
v4 = Document(DOCV4) if os.path.exists(DOCV4) else None
if v4 is not None and len(v4.tables) >= 7:
    bloque(doc, "4",
           "Especificaciones estimadas, variables y correspondencia con las "
           "hipótesis", tabla_docx_a_df(v4.tables[0]),
           "Nota. Los tamaños de muestra corresponden a las observaciones "
           "disponibles para cada país tras la eliminación por casos "
           "completos.")
    bloque(doc, "5",
           "Bitácora de reconstrucción de series con cobertura incompleta",
           tabla_docx_a_df(v4.tables[1]),
           "Nota. Cada registro documenta la serie afectada, el diagnóstico, "
           "la decisión adoptada y su justificación, con el fin de garantizar "
           "la reproducibilidad del panel.")
else:
    REGISTRO.append(("FALTA", "Cuadros_resultados_APA_v4.docx", "", 0, 0))

# ---------------------------------------------------------- Cuadro 6
bloque(doc, "6",
       "Factores de inflación de la varianza por especificación estimada",
       leer("cuadro6b_vif_por_modelo.csv"),
       "Nota. VIF máx = mayor factor de inflación de la varianza entre los "
       "regresores de cada especificación. El cálculo se realiza por modelo y "
       "no sobre un conjunto agregado de regresores, dado que ninguna "
       "especificación estimada incluye simultáneamente todas las variables "
       "del panel. Se considera aceptable un valor inferior a 5 y severo uno "
       "superior a 10.")

# --------------------------------------------------- Cuadros 7 a 10
bloque(doc, "7",
       "Prueba de límites de cointegración y término de corrección de error "
       "por modelo", leer("cuadro7_cointegracion.csv"),
       "Nota. Estimación mediante un modelo autorregresivo de rezagos "
       "distribuidos con selección del orden por criterio de información de "
       "Akaike. Los valores críticos corresponden al caso III de la tabla "
       "CI(iii) de Pesaran et al. (2001) al 5 %, con los pares 3.79 y 4.85 "
       "para dos regresores, 3.23 y 4.35 para tres, y 2.86 y 4.01 para cuatro. "
       "La regla de decisión exige simultáneamente que el estadístico F supere "
       "el límite superior y que el término de corrección de error sea "
       "negativo y significativo. La vida media se obtiene como "
       "ln(0.5)/ln(1 + ECT).")

bloque(doc, "8",
       "Coeficientes de largo plazo de las relaciones cointegrantes "
       "identificadas", leer("cuadro8_largo_plazo.csv"),
       "Nota. La variable dependiente es el logaritmo del producto interno "
       "bruto por habitante. Los errores estándar de los coeficientes de largo "
       "plazo se obtienen por el método delta. Solo se interpretan los modelos "
       "que superan conjuntamente la prueba de límites y el signo del término "
       "de corrección de error. *p < .10. **p < .05. ***p < .01.")

bloque(doc, "9", "Dinámica de corto plazo y multiplicadores netos",
       leer("cuadro9_corto_plazo.csv"),
       "Nota. El multiplicador neto acumula los coeficientes contemporáneos y "
       "rezagados de cada regresor, dividido por uno menos la suma de los "
       "coeficientes autorregresivos.")

bloque(doc, "10",
       "Pruebas de diagnóstico de los residuos de las ecuaciones de corrección "
       "de error", leer("cuadro10_diagnosticos.csv"),
       "Nota. Se reportan valores de probabilidad. Breusch–Godfrey y Ljung–Box "
       "contrastan la ausencia de autocorrelación; Breusch–Pagan, la "
       "homocedasticidad; Jarque–Bera, la normalidad. Valores superiores a .05 "
       "indican que no se rechaza el supuesto correspondiente.")

# --------------------------------------------------------- Cuadro 11
bloque(doc, "11",
       "Estimación de los coeficientes de largo plazo por mínimos cuadrados "
       "dinámicos", leer("cuadro11_dols.csv"),
       "Nota. Estimador de mínimos cuadrados dinámicos de Stock y Watson "
       "(1993) con adelantos y rezagos de las primeras diferencias de los "
       "regresores y errores estándar consistentes ante heterocedasticidad y "
       "autocorrelación (Newey y West, 1987), con ancho de banda seleccionado "
       "por el procedimiento automático de Andrews (1991). La columna de "
       "diferencia compara el coeficiente con el obtenido por el modelo de "
       "rezagos distribuidos. *p < .10. **p < .05. ***p < .01.")

# --------------------------------------------------------- Cuadro 12
for suf, sub, tit in (
        ("cointegracion", "", "Prueba de límites"),
        ("largo_plazo", "b", "Coeficientes de largo plazo"),
        ("corto_plazo", "c", "Dinámica de corto plazo"),
        ("diagnosticos", "d", "Diagnósticos de los residuos")):
    bloque(doc, "12" + sub,
           "Modelo de deuda privada y apertura comercial (M10). " + tit,
           leer("cuadro12_m10_{}.csv".format(suf)),
           "Nota. La deuda privada procede de la Global Debt Database del "
           "Fondo Monetario Internacional (Mbaye et al., 2018). Esta "
           "especificación es la única cuyos regresores satisfacen "
           "conjuntamente la condición de exogeneidad débil en el sistema "
           "correspondiente.")

# --------------------------------------------------------- Cuadro 13
for suf, sub, tit, nota in (
    ("cointegracion", "", "Prueba de límites",
     "Nota. Se contrastan tres especificaciones auxiliares: la inversión como "
     "único regresor del producto (M11), la inversión junto con la apertura "
     "comercial sin variable financiera (M12) y la inversión como variable "
     "dependiente de la apertura y el crédito (MA). Se reportan los veredictos "
     "bajo los valores críticos correctos y bajo los empleados originalmente."),
    ("largo_plazo", "b", "Coeficientes de largo plazo",
     "Nota. La columna de interpretabilidad indica si la especificación supera "
     "los diagnósticos de residuos exigidos para atribuir contenido "
     "estructural al coeficiente."),
    ("diagnosticos", "c", "Diagnósticos de los residuos",
     "Nota. Valores de probabilidad. La especificación M11 para China no supera "
     "los contrastes de autocorrelación ni de normalidad, por lo que su "
     "coeficiente no admite lectura estructural."),
    ("dols", "d", "Contraste por mínimos cuadrados dinámicos",
     "Nota. Estimación por el procedimiento de Stock y Watson (1993) como "
     "verificación independiente de los coeficientes de largo plazo.")):
    bloque(doc, "13" + sub,
           "Contraste de la hipótesis de mediación de la inversión. " + tit,
           leer("cuadro13_mediacion_{}.csv".format(suf)), nota)

# --------------------------------------------------------- Cuadro 14
for arch, sub, tit, nota in (
    ("cuadro14_johansen_rango_v3.csv", "", "Estadísticos de traza y rango",
     "Nota. Procedimiento de Johansen (1988, 1991) con constante irrestricta. "
     "La traza corregida aplica el factor de corrección por muestra finita de "
     "Reinsel y Ahn (1992). Los valores críticos tabulados resultan "
     "excesivamente permisivos en muestras de esta magnitud; véase el "
     "Cuadro 15."),
    ("cuadro14_johansen_vectores_v3.csv", "b", "Vectores de cointegración",
     "Nota. Coeficientes normalizados sobre el logaritmo del producto por "
     "habitante y expresados con el signo de la relación de largo plazo."),
    ("cuadro14_johansen_ajustes_v3.csv", "c",
     "Velocidades de ajuste y exogeneidad débil por ecuación",
     "Nota. El contraste de exogeneidad débil sigue el procedimiento de sistema "
     "parcial de Johansen (1992a). Un valor de probabilidad superior a .05 "
     "indica que la ecuación correspondiente no responde a las desviaciones "
     "respecto de la relación de largo plazo."),
    ("cuadro14_johansen_exogeneidad_v3.csv", "d",
     "Contraste conjunto de exogeneidad débil de los regresores",
     "Nota. El no rechazo valida formalmente la estimación uniecuacional de la "
     "relación de largo plazo, que resulta eficiente en ese caso."),
    ("cuadro14_johansen_resumen_v3.csv", "e",
     "Resumen de la especificación y los diagnósticos de cada sistema",
     "Nota. Todos los sistemas incluyen una variable indicadora para 2020.")):
    bloque(doc, "14" + sub,
           "Análisis de cointegración multivariante. " + tit, leer(arch), nota)

# --------------------------------------------------------- Cuadro 15
bloque(doc, "15", "Valores críticos simulados de la prueba de traza",
       leer("cuadro15_criticos_simulados.csv"),
       "Nota. Distribución obtenida por simulación de paseos aleatorios "
       "independientes bajo la hipótesis nula, con la misma longitud de "
       "muestra, el mismo orden de rezagos y la variable indicadora de 2020 en "
       "su posición efectiva, dado que la presencia de variables deterministas "
       "de quiebre altera la distribución asintótica del estadístico "
       "(Johansen et al., 2000). La simulación sustituye a la corrección por "
       "muestra finita y no se acumula con ella. Los valores tabulados "
       "resultan sistemáticamente permisivos, lo que invertiría el veredicto "
       "de rango en varias configuraciones.")

bloque(doc, "15b",
       "Estabilidad de los sistemas: raíces de la matriz compañera",
       leer("cuadro15_resumen.csv"),
       "Nota. Impuesto un rango de cointegración de uno en un sistema de "
       "cuatro variables, cabe esperar tres raíces unitarias. La mayor raíz no "
       "unitaria por debajo de la unidad confirma la estabilidad dinámica del "
       "sistema.")

bloque(doc, "15c", "Observaciones atípicas de los residuos por ecuación",
       leer("cuadro15_residuos_por_ecuacion.csv",
            filtro=lambda x: x["Residuo tipificado"].abs() >= 2.0
            if "Residuo tipificado" in x.columns else x.index == x.index),
       "Nota. Se listan las observaciones cuyo residuo tipificado alcanza o "
       "supera dos desviaciones estándar en valor absoluto. Su dispersión "
       "entre ecuaciones y años distintos desaconseja incorporar variables "
       "indicadoras adicionales, cuya inclusión elevaría los valores críticos "
       "simulados sin corregir una fuente común de perturbación.")

# --------------------------------------------------- Cuadros 16 y 17
bloque(doc, "16",
       "Sensibilidad de los resultados de sistema al orden de rezagos y al "
       "tratamiento de 2020", leer("cuadro16_sensibilidad_sistema.csv"),
       "Nota. Cada configuración se contrasta contra sus propios valores "
       "críticos simulados. El signo de la velocidad de ajuste de la ecuación "
       "del producto es el resultado de interés: negativo indica ajuste "
       "corrector hacia la relación de largo plazo.")

bloque(doc, "17",
       "Contraste de necesidad de una tendencia lineal en el espacio de "
       "cointegración", leer("cuadro17_tendencia.csv"),
       "Nota. Razón de verosimilitud entre el modelo con constante irrestricta "
       "y el mismo modelo con tendencia lineal restringida al espacio de "
       "cointegración, con distribución ji cuadrada y grados de libertad "
       "iguales al rango (Johansen, 1992b). El rechazo no conduce a adoptar la "
       "especificación con tendencia: la desaceleración del crecimiento chino "
       "implica un componente cóncavo en el nivel logarítmico que una "
       "tendencia lineal no representa, y su inclusión desplaza la relación de "
       "largo plazo hacia un ajuste de tendencia en el que los coeficientes de "
       "los regresores se aproximan a cero. Véase el apartado de limitaciones.")

bloque(doc, "17b",
       "Secuencia de Pantula para la determinación conjunta del rango y de la "
       "especificación determinista", leer("cuadro17_pantula.csv"),
       "Nota. Procedimiento secuencial de Pantula (1989) evaluado con valores "
       "críticos simulados. El primer no rechazo determina simultáneamente el "
       "rango de cointegración y el modelo determinista.")

# ---------------------------------------------------------- Anexos
bloque(doc, "A1",
       "Sensibilidad de la estimación por mínimos cuadrados dinámicos al "
       "número de adelantos y rezagos y al ancho de banda",
       leer("cuadro11b_dols_sensibilidad.csv"),
       "Nota. Se reportan todas las combinaciones evaluadas. La estabilidad de "
       "los coeficientes a lo largo de la rejilla respalda la robustez de la "
       "estimación principal. *p < .10. **p < .05. ***p < .01.")

if v4 is not None and len(v4.tables) >= 7:
    bloque(doc, "A2",
           "Estimaciones de largo plazo no identificadas. Estados Unidos",
           tabla_docx_a_df(v4.tables[6]),
           "Nota. Se presentan por transparencia y no admiten interpretación "
           "estructural, al no satisfacer las condiciones de identificación.")

# ------------------------------------------------------ Referencias
parrafo(doc, "Referencias", negrita=True, pt=12, despues=8)
REFS = [
    "Andrews, D. W. K. (1991). Heteroskedasticity and autocorrelation "
    "consistent covariance matrix estimation. Econometrica, 59(3), 817–858. "
    "https://doi.org/10.2307/2938229",
    "Bank for International Settlements. (s. f.). Credit to the non-financial "
    "sector [Conjunto de datos]. https://www.bis.org/statistics/totcredit.htm",
    "Dickey, D. A., y Fuller, W. A. (1979). Distribution of the estimators for "
    "autoregressive time series with a unit root. Journal of the American "
    "Statistical Association, 74(366), 427–431. "
    "https://doi.org/10.2307/2286348",
    "Feenstra, R. C., Inklaar, R., y Timmer, M. P. (2015). The next generation "
    "of the Penn World Table. American Economic Review, 105(10), 3150–3182. "
    "https://doi.org/10.1257/aer.20130954",
    "Feenstra, R. C., Inklaar, R., y Timmer, M. P. (2025). Penn World Table "
    "version 11.0 [Conjunto de datos]. Groningen Growth and Development "
    "Centre. https://doi.org/10.34894/FABVLR",
    "Fondo Monetario Internacional. (s. f.-a). Global Debt Database [Conjunto "
    "de datos]. https://www.imf.org/external/datamapper/datasets/GDD",
    "Fondo Monetario Internacional. (s. f.-b). World Economic Outlook database "
    "[Conjunto de datos]. https://www.imf.org/en/Publications/WEO",
    "Johansen, S. (1988). Statistical analysis of cointegration vectors. "
    "Journal of Economic Dynamics and Control, 12(2–3), 231–254. "
    "https://doi.org/10.1016/0165-1889(88)90041-3",
    "Johansen, S. (1991). Estimation and hypothesis testing of cointegration "
    "vectors in Gaussian vector autoregressive models. Econometrica, 59(6), "
    "1551–1580. https://doi.org/10.2307/2938278",
    "Johansen, S. (1992a). Cointegration in partial systems and the efficiency "
    "of single-equation analysis. Journal of Econometrics, 52(3), 389–402.",
    "Johansen, S. (1992b). Determination of cointegration rank in the presence "
    "of a linear trend. Oxford Bulletin of Economics and Statistics, 54(3), "
    "383–397.",
    "Johansen, S., Mosconi, R., y Nielsen, B. (2000). Cointegration analysis "
    "in the presence of structural breaks in the deterministic trend. "
    "Econometrics Journal, 3(2), 216–249. "
    "https://doi.org/10.1111/1368-423X.00047",
    "Kwiatkowski, D., Phillips, P. C. B., Schmidt, P., y Shin, Y. (1992). "
    "Testing the null hypothesis of stationarity against the alternative of a "
    "unit root. Journal of Econometrics, 54(1–3), 159–178.",
    "Mbaye, S., Moreno-Badia, M., y Chae, K. (2018). Global debt database: "
    "Methodology and sources (Working Paper N.º 18/111). Fondo Monetario "
    "Internacional.",
    "Newey, W. K., y West, K. D. (1987). A simple, positive semi-definite, "
    "heteroskedasticity and autocorrelation consistent covariance matrix. "
    "Econometrica, 55(3), 703–708. https://doi.org/10.2307/1913610",
    "Pantula, S. G. (1989). Testing for unit roots in time series data. "
    "Econometric Theory, 5(2), 256–271.",
    "Pesaran, M. H., Shin, Y., y Smith, R. J. (2001). Bounds testing "
    "approaches to the analysis of level relationships. Journal of Applied "
    "Econometrics, 16(3), 289–326. https://doi.org/10.1002/jae.616",
    "Phillips, P. C. B., y Perron, P. (1988). Testing for a unit root in time "
    "series regression. Biometrika, 75(2), 335–346. "
    "https://doi.org/10.1093/biomet/75.2.335",
    "Reinsel, G. C., y Ahn, S. K. (1992). Vector autoregressive models with "
    "unit roots and reduced rank structure: Estimation, likelihood ratio test, "
    "and forecasting. Journal of Time Series Analysis, 13(4), 353–375. "
    "https://doi.org/10.1111/j.1467-9892.1992.tb00113.x",
    "Stock, J. H., y Watson, M. W. (1993). A simple estimator of cointegrating "
    "vectors in higher order integrated systems. Econometrica, 61(4), 783–820. "
    "https://doi.org/10.2307/2951763",
    "Zivot, E., y Andrews, D. W. K. (1992). Further evidence on the great "
    "crash, the oil-price shock, and the unit-root hypothesis. Journal of "
    "Business & Economic Statistics, 10(3), 251–270. "
    "https://doi.org/10.1080/07350015.1992.10509904",
]
for r in REFS:
    p = doc.add_paragraph()
    pf = p.paragraph_format
    pf.left_indent = Cm(1.25)
    pf.first_line_indent = Cm(-1.25)
    pf.space_after = Pt(6)
    run = p.add_run(r)
    run.font.name = FUENTE
    run.font.size = Pt(PT_TXT)
    run._element.rPr.rFonts.set(qn("w:eastAsia"), FUENTE)

doc.save(SALIDA)

# ------------------------------------------------------ verificacion
print("=" * 78)
print("DOCUMENTO v8 GENERADO")
print("=" * 78)
print("archivo: {}".format(SALIDA))
print("tamano : {:,} bytes".format(os.path.getsize(SALIDA)))
d2 = Document(SALIDA)
print("tablas : {}   parrafos: {}".format(len(d2.tables), len(d2.paragraphs)))

print("\n{:<8} {:<12} {:<7} {:>6} {:>9}".format(
    "estado", "cuadro", "panel", "pt", "ancho cm"))
print("-" * 78)
for e in REGISTRO:
    print("{:<8} {:<12} {:<7} {:>6} {:>9}".format(*[str(x) for x in e]))

mal = [e for e in REGISTRO if e[0] != "OK"]
anch = [e for e in REGISTRO if e[0] == "OK" and e[4] > ANCHO_UTIL + 0.05]
print("\nincidencias: {}".format(len(mal)))
print("cuadros que desbordan: {}".format(len(anch)))
paneles = sum(1 for e in REGISTRO if e[0] == "OK" and e[2] != "-")
print("paneles generados por particion automatica: {}".format(paneles))

# comprobacion de que la rejilla quedo escrita y no uniforme
uniformes, sin_rejilla = [], []
for i, t in enumerate(d2.tables):
    g = t._tbl.find(qn("w:tblGrid"))
    if g is None:
        sin_rejilla.append(i)
        continue
    anchos = [int(c.get(qn("w:w"))) for c in g.findall(qn("w:gridCol"))]
    if len(set(anchos)) <= 1 and len(anchos) > 2:
        uniformes.append(i)
print("tablas sin rejilla: {}".format(len(sin_rejilla)))
print("tablas con rejilla uniforme: {}".format(len(uniformes)))
print("anchos de rejilla del cuadro mas ancho (cm): {}".format(
    [round(int(c.get(qn("w:w"))) / 567, 2)
     for c in max(d2.tables,
                  key=lambda t: len(t.columns))._tbl.find(
         qn("w:tblGrid")).findall(qn("w:gridCol"))]))


DOCUMENTO v8 GENERADO
archivo: /content/drive/MyDrive/tesis_china_eeuu/outputs/Cuadros_resultados_APA_v8.docx
tamano : 103,002 bytes
tablas : 36   parrafos: 170

estado   cuadro       panel       pt  ancho cm
------------------------------------------------------------------------------
OK       1            -          8.0     23.06
OK       2            -          8.0      18.0
OK       2b           -          8.0     11.52
OK       3            -          8.0      16.0
OK       3 (cont.)    -          8.0     23.16
OK       3b           -          8.0     14.42
OK       3c           -          8.0     12.44
OK       4            -          8.0     21.75
OK       5            -          8.0      23.9
OK       6            -          8.0     13.74
OK       7            -          8.0     14.24
OK       8            -          8.0      9.95
OK       9            -          8.0     14.98
OK       10           -          8.0     16.83
OK       11           -          8.0      23.9
OK     

In [61]:
# -*- coding: utf-8 -*-
"""
Paso 5, version 9. Documento de cuadros en formato APA 7.
Version 6: corrige desborde de anchos, particion de cifras y codigos,
acentuacion, booleanos, formato numerico, notas y correspondencia
cita-referencia.
Version 7: anade tres correcciones detectadas en la revision visual.
  1. Los anios dejan de imprimirse con separador de millares.
  2. Se elimina la notacion cientifica; los valores menores que 0.0005
     se imprimen como < 0.001 en columnas de probabilidad y como 0.000
     en las demas.
  3. Ningun encabezado se parte a mitad de palabra: cuando un panel no
     cabe con el cuerpo mas pequeno de la escala se subdivide en dos
     paneles en lugar de comprimir las columnas por debajo de su ancho
     minimo. El registro final avisa si aun asi hubo compresion.
Version 9: eleva el margen de seguridad de la medicion del 3 % al 6 %
y redondea al alza los anchos de la rejilla, porque el truncamiento al
escribir w:gridCol dejaba algunos encabezados un centesimo por debajo
de lo necesario y se partian igualmente.
Version 8: corrige la causa raiz del desajuste de columnas. Con
w:tblLayout fijo, Word y LibreOffice leen los anchos de w:tblGrid y no
de w:tcW. python-docx crea la rejilla con columnas iguales y no la
actualiza al asignar celda.width, de modo que en las versiones
anteriores todo el calculo de anchos se descartaba en la composicion y
las columnas salian uniformes. Ahora se escribe w:tblGrid y w:tblW.
Se incorporan tambien la nota corregida del Cuadro 1 y la entrada de
referencia del conjunto de datos PWT 11.0.
Salida: outputs/Cuadros_resultados_APA_v9.docx
"""

import os
import re
import math
import numpy as np
import pandas as pd
from docx import Document
from docx.shared import Pt, Cm
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.section import WD_ORIENT
from docx.enum.table import WD_TABLE_ALIGNMENT
from docx.oxml.ns import qn
from docx.oxml import OxmlElement

if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

BASE = "/content/drive/MyDrive/tesis_china_eeuu"
OUT = os.path.join(BASE, "outputs")
DOCV4 = os.path.join(OUT, "Cuadros_resultados_APA_v4.docx")
SALIDA = os.path.join(OUT, "Cuadros_resultados_APA_v9.docx")

FUENTE = "Arial"
PT_TXT = 10
PT_NOTA = 8
ESCALA = [8.0, 7.5, 7.0, 6.5]      # cuerpo de tabla, de mayor a menor
ANCHO_UTIL = 23.90                  # cm
PAD = 0.16                          # margen interno de celda, ambos lados
ZWSP = "\u200b"                     # espacio de ancho cero

# ------------------------------------------------------------------
# metrica tipografica (anchos AFM de Helvetica, milesimas de em)
# ------------------------------------------------------------------
_W = {}
for _c in "0123456789":
    _W[_c] = 556
for _c, _v in {
    " ": 278, "!": 278, '"': 355, "#": 556, "$": 556, "%": 889, "&": 667,
    "'": 191, "(": 333, ")": 333, "*": 389, "+": 584, ",": 278, "-": 333,
    ".": 278, "/": 278, ":": 278, ";": 278, "<": 584, "=": 584, ">": 584,
    "?": 556, "@": 1015, "[": 278, "\\": 278, "]": 278, "^": 469, "_": 556,
    "`": 333, "{": 334, "|": 260, "}": 334, "~": 584,
    "A": 667, "B": 667, "C": 722, "D": 722, "E": 667, "F": 611, "G": 778,
    "H": 722, "I": 278, "J": 500, "K": 667, "L": 556, "M": 833, "N": 722,
    "O": 778, "P": 667, "Q": 778, "R": 722, "S": 667, "T": 611, "U": 722,
    "V": 667, "W": 944, "X": 667, "Y": 667, "Z": 611,
    "a": 556, "b": 556, "c": 500, "d": 556, "e": 556, "f": 278, "g": 556,
    "h": 556, "i": 222, "j": 222, "k": 500, "l": 222, "m": 833, "n": 556,
    "o": 556, "p": 556, "q": 556, "r": 333, "s": 500, "t": 278, "u": 556,
    "v": 500, "w": 722, "x": 500, "y": 500, "z": 500,
}.items():
    _W[_c] = _v

PT_A_CM = 0.0352778


def medir(s, pt, negrita=False):
    """Ancho de una cadena en centimetros."""
    u = 0
    for ch in str(s):
        if ch == ZWSP:
            continue
        u += _W.get(ch, 667 if ch.isupper() else 556)
    cm = u / 1000.0 * pt * PT_A_CM
    if negrita:
        cm *= 1.07
    return cm * 1.06      # margen de seguridad


# ------------------------------------------------------------------
# normalizacion de contenido
# ------------------------------------------------------------------
TILDES = {
    "Pais": "País", "PAIS": "PAÍS", "Codigo": "Código", "Anio": "Año",
    "Ano": "Año", "Hipotesis": "Hipótesis", "Diagnostico": "Diagnóstico",
    "Ordenes": "Órdenes", "Razon": "Razón", "Estandar": "Estándar",
    "estandar": "estándar", "Valido": "Válido", "valido": "válido",
    "Ecuacion": "Ecuación", "Asimetria": "Asimetría",
    "Cointegracion": "Cointegración", "cointegracion": "cointegración",
    "Comprobacion": "Comprobación", "Estadistico": "Estadístico",
    "Decision": "Decisión", "Justificacion": "Justificación",
    "Proporcion": "Proporción", "Autocorrelacion": "Autocorrelación",
    "autocorrelacion": "autocorrelación", "Raices": "Raíces",
    "raices": "raíces", "debil": "débil", "Debil": "Débil",
    "Formacion": "Formación", "formacion": "formación",
    "Inflacion": "Inflación", "inflacion": "inflación",
    "Credito": "Crédito", "credito": "crédito",
    "Publica": "Pública", "publica": "pública",
    "Indice": "Índice", "Poblacion": "Población",
    "Especificacion": "Especificación", "Restriccion": "Restricción",
    "Simulacion": "Simulación", "Version": "Versión",
    "Companera": "Compañera", "companera": "compañera",
    "Deteccion": "Detección", "Interpretacion": "Interpretación",
    "Numero": "Número", "Practica": "Práctica", "practica": "práctica",
    "Analisis": "Análisis", "analisis": "análisis",
    "Aritmetica": "Aritmética", "Geometrica": "Geométrica",
    "Ficticia": "Ficticia", "Interpolacion": "Interpolación",
    "interpolacion": "interpolación", "Empalme": "Empalme",
    "Exclusion": "Exclusión", "exclusion": "exclusión",
    "Sustitucion": "Sustitución", "sustitucion": "sustitución",
    "Correccion": "Corrección", "correccion": "corrección",
    "Estimacion": "Estimación", "estimacion": "estimación",
    "Rechaza": "Rechaza", "Traza": "Traza",
}
_PAT = re.compile(r"\b(" + "|".join(sorted(map(re.escape, TILDES), key=len,
                                            reverse=True)) + r")\b")

SIMBOLOS = [("r<=0", "r ≤ 0"), ("r<=1", "r ≤ 1"), ("r<=2", "r ≤ 2"),
            ("r<=3", "r ≤ 3"), ("<=", " ≤ "), (">=", " ≥ "),
            ("R2", "R²"), ("Chi2", "χ²"), ("chi2", "χ²")]


def acentuar(s):
    if "_" in s:
        return s
    return _PAT.sub(lambda m: TILDES[m.group(0)], s)


def simbolos(s):
    for a, b in SIMBOLOS:
        s = s.replace(a, b)
    return re.sub(r"\s+", " ", s).strip()


def quebrable(s):
    """Permite corte de linea tras guion bajo en codigos largos."""
    if "_" in s and len(s) > 12:
        return s.replace("_", "_" + ZWSP)
    return s


ANIO_PAT = re.compile(r"(^|_)(a[nñ]io|a[nñ]o|year|ini|fin)($|_)", re.I)
P_PAT = re.compile(r"(^p$|^p[ _(]|\(p\)|valor[ _]p|p[- _]valor|prob)", re.I)


def formatear_columna(serie):
    """Devuelve la columna como texto, con criterio uniforme."""
    vals = serie.tolist()
    nombre = str(getattr(serie, "name", "") or "")
    num = pd.to_numeric(serie, errors="coerce")
    es_num = num.notna().sum() >= max(1, int(0.8 * len(vals)))
    es_p = bool(P_PAT.search(nombre))
    entera = False
    if es_num:
        fin = num.dropna()
        entera = len(fin) > 0 and np.all(np.isclose(fin, np.round(fin)))
    es_anio = False
    if es_num and entera:
        fin = num.dropna()
        es_anio = bool(ANIO_PAT.search(nombre)) or (
            len(fin) > 0 and fin.min() >= 1500 and fin.max() <= 2200)
    out = []
    for v, nv in zip(vals, num.tolist()):
        if isinstance(v, (bool, np.bool_)):
            out.append("Sí" if v else "No")
            continue
        s = "" if v is None else str(v).strip()
        if s.lower() in ("nan", "none", "", "<na>"):
            out.append("—")
            continue
        if s == "True":
            out.append("Sí")
            continue
        if s == "False":
            out.append("No")
            continue
        if es_num and nv is not None and np.isfinite(nv):
            if es_anio:
                out.append("{:.0f}".format(nv))
            elif entera:
                out.append("{:,.0f}".format(nv).replace(",", " "))
            elif abs(nv) >= 10000:
                out.append("{:,.1f}".format(nv).replace(",", " "))
            elif abs(nv) < 0.0005 and nv != 0:
                out.append("< 0.001" if es_p else "0.000")
            else:
                out.append("{:.3f}".format(nv))
            continue
        out.append(quebrable(simbolos(acentuar(s))))
    return out


def preparar(df):
    d = pd.DataFrame()
    for c in df.columns:
        d[simbolos(acentuar(str(c)))] = formatear_columna(df[c])
    return d


# ------------------------------------------------------------------
# calculo de anchos
# ------------------------------------------------------------------
def _tokens(s):
    return [t for t in re.split(r"[ \u200b]+", str(s)) if t]


def anchos_columna(dft, pt):
    """Devuelve (minimos, deseados) en cm, ya con margen interno."""
    mn, ds = [], []
    for c in dft.columns:
        cab = str(c)
        tmin = max([medir(t, pt, True) for t in _tokens(cab)] or [0.4])
        tful = medir(cab, pt, True)
        for v in dft[c]:
            for t in _tokens(v):
                tmin = max(tmin, medir(t, pt))
            tful = max(tful, medir(v, pt))
        mn.append(tmin + PAD)
        ds.append(min(tful + PAD, 7.0))
    return mn, ds


def repartir(mn, ds, disponible):
    total = sum(mn)
    if total > disponible:
        return None
    resto = disponible - total
    extra = [max(0.0, d - m) for m, d in zip(mn, ds)]
    se = sum(extra)
    if se <= 0:
        return list(mn)
    return [m + resto * e / se if resto < se else d
            for m, d, e in zip(mn, ds, extra)]


def ajustar(dft, disponible=ANCHO_UTIL):
    """Busca el mayor cuerpo que quepa. Devuelve (pt, anchos) o None."""
    for pt in ESCALA:
        mn, ds = anchos_columna(dft, pt)
        w = repartir(mn, ds, disponible)
        if w is not None:
            return pt, w
    return None


CLAVES = {"País", "Sistema", "Modelo", "Variable", "Serie", "Ecuación",
          "Hipótesis", "Caso", "Comprobación", "Forma", "Dependiente"}


def partir_en_paneles(dft, disponible=ANCHO_UTIL):
    """Divide un cuadro ancho repitiendo las columnas identificadoras."""
    cols = list(dft.columns)
    claves = []
    for c in cols:
        if c in CLAVES and len(claves) < 3:
            claves.append(c)
        else:
            break
    if not claves:
        claves = cols[:1]
    resto = [c for c in cols if c not in claves]
    pt = ESCALA[-1]
    mn, _ = anchos_columna(dft, pt)
    ancho = dict(zip(cols, mn))
    base = sum(ancho[c] for c in claves)
    paneles, actual = [], []
    for c in resto:
        if actual and base + sum(ancho[x] for x in actual) + ancho[c] > disponible:
            paneles.append(claves + actual)
            actual = [c]
        else:
            actual.append(c)
    if actual:
        paneles.append(claves + actual)
    return paneles


def subdividir(dft, cols, claves=None, prof=0):
    """Garantiza que cada grupo quepa; parte por la mitad si hace falta.

    Nunca comprime una columna por debajo de su ancho minimo: si un
    panel no cabe con el cuerpo mas pequeno de la escala, se divide en
    dos paneles repitiendo las columnas identificadoras.
    """
    cols = list(cols)
    if ajustar(dft[cols]) is not None or prof >= 4:
        return [cols]
    if claves is None:
        claves = [c for c in cols if c in CLAVES][:3] or cols[:1]
    resto = [c for c in cols if c not in claves]
    if len(resto) < 2:
        return [cols]
    m = len(resto) // 2
    izq = subdividir(dft, claves + resto[:m], claves, prof + 1)
    der = subdividir(dft, claves + resto[m:], claves, prof + 1)
    return izq + der


# ------------------------------------------------------------------
# construccion de tablas
# ------------------------------------------------------------------
def borde(celda, lado, sz=6):
    tcPr = celda._tc.get_or_add_tcPr()
    b = tcPr.find(qn("w:tcBorders"))
    if b is None:
        b = OxmlElement("w:tcBorders")
        tcPr.append(b)
    e = b.find(qn("w:" + lado))
    if e is None:
        e = OxmlElement("w:" + lado)
        b.append(e)
    e.set(qn("w:val"), "single")
    e.set(qn("w:sz"), str(sz))
    e.set(qn("w:color"), "000000")


def sin_bordes(celda):
    tcPr = celda._tc.get_or_add_tcPr()
    b = tcPr.find(qn("w:tcBorders"))
    if b is None:
        b = OxmlElement("w:tcBorders")
        tcPr.append(b)
    for lado in ("top", "left", "bottom", "right"):
        e = b.find(qn("w:" + lado))
        if e is None:
            e = OxmlElement("w:" + lado)
            b.append(e)
        e.set(qn("w:val"), "none")
        e.set(qn("w:sz"), "0")


def margenes_celda(tabla, cm=PAD / 2):
    tblPr = tabla._tbl.tblPr
    mar = OxmlElement("w:tblCellMar")
    for lado, val in (("top", 0.03), ("left", cm), ("bottom", 0.03),
                      ("right", cm)):
        e = OxmlElement("w:" + lado)
        e.set(qn("w:w"), str(int(val * 567)))
        e.set(qn("w:type"), "dxa")
        mar.append(e)
    tblPr.append(mar)


def layout_fijo(tabla):
    el = OxmlElement("w:tblLayout")
    el.set(qn("w:type"), "fixed")
    tabla._tbl.tblPr.append(el)


def repetir_encabezado(fila):
    trPr = fila._tr.get_or_add_trPr()
    th = OxmlElement("w:tblHeader")
    th.set(qn("w:val"), "true")
    trPr.append(th)


def escribir(celda, texto, pt, negrita=False):
    p = celda.paragraphs[0]
    pf = p.paragraph_format
    pf.space_before = Pt(1)
    pf.space_after = Pt(1)
    pf.line_spacing = 1.0
    r = p.add_run(str(texto))
    r.font.name = FUENTE
    r.font.size = Pt(pt)
    r.bold = negrita
    r._element.rPr.rFonts.set(qn("w:eastAsia"), FUENTE)


def fijar_rejilla(tabla, ws):
    """Escribe los anchos calculados en la rejilla de la tabla.

    Es el paso decisivo: con w:tblLayout fijo, Word y LibreOffice leen
    los anchos de w:tblGrid, no de w:tcW. Si la rejilla conserva los
    valores uniformes que crea python-docx al construir la tabla, todo
    el calculo de anchos se pierde y las columnas salen iguales.
    """
    grid = tabla._tbl.find(qn("w:tblGrid"))
    if grid is not None:
        for col, w in zip(grid.findall(qn("w:gridCol")), ws):
            col.set(qn("w:w"), str(int(math.ceil(w * 567))))
    tblPr = tabla._tbl.tblPr
    ant = tblPr.find(qn("w:tblW"))
    if ant is not None:
        tblPr.remove(ant)
    e = OxmlElement("w:tblW")
    e.set(qn("w:w"), str(int(math.ceil(sum(ws) * 567))))
    e.set(qn("w:type"), "dxa")
    tblPr.append(e)


def insertar_tabla(doc, dft, pt, ws):
    nf, nc = dft.shape
    t = doc.add_table(rows=nf + 1, cols=nc)
    t.alignment = WD_TABLE_ALIGNMENT.CENTER
    t.autofit = False
    layout_fijo(t)
    margenes_celda(t)
    fijar_rejilla(t, ws)
    for j, col in enumerate(dft.columns):
        c = t.cell(0, j)
        c.width = Cm(ws[j])
        escribir(c, col, pt, True)
    for i in range(nf):
        for j in range(nc):
            c = t.cell(i + 1, j)
            c.width = Cm(ws[j])
            escribir(c, dft.iat[i, j], pt)
    for f in t.rows:
        for c in f.cells:
            sin_bordes(c)
    for c in t.rows[0].cells:
        borde(c, "top")
        borde(c, "bottom")
    for c in t.rows[-1].cells:
        borde(c, "bottom")
    repetir_encabezado(t.rows[0])
    fijar_rejilla(t, ws)
    return t


def parrafo(doc, texto, cursiva=False, negrita=False, pt=PT_TXT,
            despues=6, junto=False):
    p = doc.add_paragraph()
    pf = p.paragraph_format
    pf.space_after = Pt(despues)
    pf.space_before = Pt(0)
    pf.keep_with_next = junto
    r = p.add_run(texto)
    r.font.name = FUENTE
    r.font.size = Pt(pt)
    r.italic = cursiva
    r.bold = negrita
    r._element.rPr.rFonts.set(qn("w:eastAsia"), FUENTE)
    return p


REGISTRO = []


def bloque(doc, numero, titulo, df, nota, subtitulo=None):
    """Inserta un cuadro completo, partiendolo en paneles si no cabe."""
    if df is None or len(df) == 0:
        REGISTRO.append(("VACIO", numero, "", 0, 0))
        return
    dft = preparar(df)
    r = ajustar(dft)
    if r is not None:
        grupos = [list(dft.columns)]
        ptl, wsl = [r[0]], [r[1]]
    else:
        grupos = []
        for g0 in partir_en_paneles(dft):
            grupos.extend(subdividir(dft, g0))
        ptl, wsl = [], []
        for g in grupos:
            rg = ajustar(dft[g])
            if rg is None:
                mn, _ = anchos_columna(dft[g], ESCALA[-1])
                f = ANCHO_UTIL / sum(mn)
                rg = (ESCALA[-1], [m * f for m in mn])
                REGISTRO.append(("COMPRIME", numero, ",".join(map(str, g))[:40],
                                 ESCALA[-1], round(sum(mn), 2)))
            ptl.append(rg[0])
            wsl.append(rg[1])

    letras = "ABCDEFGH"
    for k, g in enumerate(grupos):
        etq = "" if len(grupos) == 1 else " (cont.)" if k else ""
        parrafo(doc, "Cuadro {}{}".format(numero, etq), negrita=True,
                despues=0, junto=True)
        parrafo(doc, titulo, cursiva=True, despues=4, junto=True)
        if subtitulo and len(grupos) == 1:
            parrafo(doc, subtitulo, negrita=True, pt=PT_NOTA + 1, despues=3,
                    junto=True)
        if len(grupos) > 1:
            parrafo(doc, "Panel {}".format(letras[k]), negrita=True,
                    pt=PT_NOTA + 1, despues=3, junto=True)
        insertar_tabla(doc, dft[g], ptl[k], wsl[k])
        parrafo(doc, "", pt=4, despues=0)
        if k == len(grupos) - 1:
            parrafo(doc, nota, pt=PT_NOTA, despues=16)
        else:
            parrafo(doc, "Nota. Continúa en el panel siguiente.", pt=PT_NOTA,
                    despues=14)
        REGISTRO.append(("OK", numero, letras[k] if len(grupos) > 1 else "-",
                         ptl[k], round(sum(wsl[k]), 2)))


def leer(nombre, cols=None, filtro=None):
    ruta = os.path.join(OUT, nombre)
    if not os.path.exists(ruta):
        REGISTRO.append(("FALTA", nombre, "", 0, 0))
        return None
    d = pd.read_csv(ruta)
    if filtro is not None:
        try:
            d = d[filtro(d)]
        except Exception:
            pass
    if cols:
        pres = [c for c in cols if c in d.columns]
        if len(pres) >= 2:
            d = d[pres]
    return d.reset_index(drop=True)


def tabla_docx_a_df(t):
    datos = [[c.text.strip() for c in f.cells] for f in t.rows]
    return pd.DataFrame(datos[1:], columns=datos[0])


# ==================================================================
# documento
# ==================================================================
doc = Document()
sec = doc.sections[0]
sec.orientation = WD_ORIENT.LANDSCAPE
sec.page_width, sec.page_height = Cm(27.94), Cm(21.59)
for m in ("left_margin", "right_margin", "top_margin", "bottom_margin"):
    setattr(sec, m, Cm(2))

est = doc.styles["Normal"]
est.font.name = FUENTE
est.font.size = Pt(PT_TXT)
est.element.rPr.rFonts.set(qn("w:eastAsia"), FUENTE)

parrafo(doc, "Cuadros de resultados", negrita=True, pt=13, despues=2)
parrafo(doc, "Trayectorias de crecimiento en China y Estados Unidos, "
             "1990–2023", cursiva=True, despues=18)

# ---------------------------------------------------------- Cuadro 1
bloque(doc, "1",
       "Cobertura temporal y completitud de las series por país",
       leer("cuadro1_cobertura.csv"),
       "Nota. CN = China; US = Estados Unidos. Las columnas ini y fin indican "
       "el primer y el último año con dato; n es el número de observaciones "
       "válidas y na el porcentaje de valores ausentes dentro del intervalo "
       "1990–2023. Las series identificadas como PWT proceden de la Penn "
       "World Table 11.0 (Feenstra et al., 2015, 2025), verificadas mediante "
       "comparación numérica con el archivo pwt110.xlsx, con una desviación "
       "máxima de 0.0000 % en las cinco variables, los dos países y los 34 "
       "años. La versión 11.0 emplea la serie oficial de China y no la serie "
       "ajustada de Maddison y Wu de las versiones anteriores. Fuentes "
       "restantes: Banco Mundial, World Development Indicators "
       "(https://data.worldbank.org); Bank for "
       "International Settlements, Credit to the non-financial sector "
       "(https://www.bis.org/statistics/totcredit.htm); Fondo Monetario "
       "Internacional, Global Debt Database (Mbaye et al., 2018) y World "
       "Economic Outlook (https://www.imf.org/en/Publications/WEO).")

# ---------------------------------------------------------- Cuadro 2
bloque(doc, "2",
       "Estadísticos descriptivos de las variables por país",
       leer("cuadro2_descriptivos.csv"),
       "Nota. M = media; Mdn = mediana; DE = desviación estándar; "
       "CV = coeficiente de variación. Las razones se expresan como "
       "porcentaje del producto interno bruto, salvo indicación en contrario.")

bloque(doc, "2b",
       "Contraste de diferencia de medias entre China y Estados Unidos",
       leer("cuadro2_diferencia_medias.csv"),
       "Nota. Prueba t de Welch para varianzas desiguales. Un valor positivo "
       "de la diferencia indica una media superior en China.")

# ---------------------------------------------------------- Cuadro 3
d3 = leer("cuadro3_raiz_unitaria_v4.csv")
NOTA3 = ("Nota. c = constante; c+t = constante y tendencia; Δ = primera "
         "diferencia. En las pruebas de Dickey–Fuller aumentada y de "
         "Phillips–Perron la hipótesis nula es la existencia de una raíz "
         "unitaria, de modo que el rechazo indica estacionariedad. En la "
         "prueba KPSS la hipótesis nula es la estacionariedad y el rechazo "
         "indica lo contrario; sus asteriscos se asignan por comparación "
         "directa con los valores críticos tabulados, iguales a .347, .463 y "
         ".739 con constante y a .119, .146 y .216 con constante y tendencia. "
         "El orden de integración se declara cuando al menos una prueba de "
         "raíz unitaria rechaza al 5 % y KPSS no rechaza al mismo nivel. "
         "*p < .10. **p < .05. ***p < .01. Pruebas: Dickey y Fuller (1979), "
         "Phillips y Perron (1988), Kwiatkowski et al. (1992) y Zivot y "
         "Andrews (1992).")
if d3 is not None:
    pa = [c for c in ["Pais", "Serie", "n", "ADF (c)", "ADF (c+t)", "PP (c)",
                      "PP (c+t)", "KPSS (c)", "KPSS (c+t)"] if c in d3.columns]
    pb = [c for c in ["Pais", "Serie", "ADF D (c)", "PP D (c)", "KPSS D (c)",
                      "ADF D (c+t)", "PP D (c+t)", "KPSS D (c+t)",
                      "Zivot-Andrews", "Orden"] if c in d3.columns]
    bloque(doc, "3", "Pruebas de raíz unitaria y orden de integración de las "
           "series por país", d3[pa],
           "Nota. Panel A, series en nivel. Las notas completas figuran al pie "
           "del panel B.", subtitulo="Panel A. Series en nivel")
    bloque(doc, "3 (cont.)", "Pruebas de raíz unitaria y orden de integración "
           "de las series por país", d3[pb], NOTA3,
           subtitulo="Panel B. Series en primera diferencia y veredicto")

bloque(doc, "3b",
       "Segunda ronda de contrastes para las series no concluyentes",
       leer("cuadro3b_segunda_ronda.csv"),
       "Nota. Se contrasta la primera y la segunda diferencia para discriminar "
       "entre integración de orden uno y de orden superior. Las series "
       "clasificadas como I(2) se excluyen de las especificaciones en niveles. "
       "*p < .10. **p < .05. ***p < .01.")

bloque(doc, "3c",
       "Comprobaciones sobre la estructura determinista y el orden de "
       "integración de las variables críticas",
       leer("cuadro3c_comprobaciones.csv"),
       "Nota. La primera comprobación contrasta la presencia de una tendencia "
       "determinista en la tasa de crecimiento china mediante mínimos "
       "cuadrados con errores estándar consistentes ante heterocedasticidad y "
       "autocorrelación (Newey y West, 1987). Las restantes evalúan la "
       "estacionariedad de la formación bruta de capital fijo estadounidense y "
       "el orden de integración conjunto de cada sistema.")

# --------------------------------------------------- Cuadros 4, 5 y A2
v4 = Document(DOCV4) if os.path.exists(DOCV4) else None
if v4 is not None and len(v4.tables) >= 7:
    bloque(doc, "4",
           "Especificaciones estimadas, variables y correspondencia con las "
           "hipótesis", tabla_docx_a_df(v4.tables[0]),
           "Nota. Los tamaños de muestra corresponden a las observaciones "
           "disponibles para cada país tras la eliminación por casos "
           "completos.")
    bloque(doc, "5",
           "Bitácora de reconstrucción de series con cobertura incompleta",
           tabla_docx_a_df(v4.tables[1]),
           "Nota. Cada registro documenta la serie afectada, el diagnóstico, "
           "la decisión adoptada y su justificación, con el fin de garantizar "
           "la reproducibilidad del panel.")
else:
    REGISTRO.append(("FALTA", "Cuadros_resultados_APA_v4.docx", "", 0, 0))

# ---------------------------------------------------------- Cuadro 6
bloque(doc, "6",
       "Factores de inflación de la varianza por especificación estimada",
       leer("cuadro6b_vif_por_modelo.csv"),
       "Nota. VIF máx = mayor factor de inflación de la varianza entre los "
       "regresores de cada especificación. El cálculo se realiza por modelo y "
       "no sobre un conjunto agregado de regresores, dado que ninguna "
       "especificación estimada incluye simultáneamente todas las variables "
       "del panel. Se considera aceptable un valor inferior a 5 y severo uno "
       "superior a 10.")

# --------------------------------------------------- Cuadros 7 a 10
bloque(doc, "7",
       "Prueba de límites de cointegración y término de corrección de error "
       "por modelo", leer("cuadro7_cointegracion.csv"),
       "Nota. Estimación mediante un modelo autorregresivo de rezagos "
       "distribuidos con selección del orden por criterio de información de "
       "Akaike. Los valores críticos corresponden al caso III de la tabla "
       "CI(iii) de Pesaran et al. (2001) al 5 %, con los pares 3.79 y 4.85 "
       "para dos regresores, 3.23 y 4.35 para tres, y 2.86 y 4.01 para cuatro. "
       "La regla de decisión exige simultáneamente que el estadístico F supere "
       "el límite superior y que el término de corrección de error sea "
       "negativo y significativo. La vida media se obtiene como "
       "ln(0.5)/ln(1 + ECT).")

bloque(doc, "8",
       "Coeficientes de largo plazo de las relaciones cointegrantes "
       "identificadas", leer("cuadro8_largo_plazo.csv"),
       "Nota. La variable dependiente es el logaritmo del producto interno "
       "bruto por habitante. Los errores estándar de los coeficientes de largo "
       "plazo se obtienen por el método delta. Solo se interpretan los modelos "
       "que superan conjuntamente la prueba de límites y el signo del término "
       "de corrección de error. *p < .10. **p < .05. ***p < .01.")

bloque(doc, "9", "Dinámica de corto plazo y multiplicadores netos",
       leer("cuadro9_corto_plazo.csv"),
       "Nota. El multiplicador neto acumula los coeficientes contemporáneos y "
       "rezagados de cada regresor, dividido por uno menos la suma de los "
       "coeficientes autorregresivos.")

bloque(doc, "10",
       "Pruebas de diagnóstico de los residuos de las ecuaciones de corrección "
       "de error", leer("cuadro10_diagnosticos.csv"),
       "Nota. Se reportan valores de probabilidad. Breusch–Godfrey y Ljung–Box "
       "contrastan la ausencia de autocorrelación; Breusch–Pagan, la "
       "homocedasticidad; Jarque–Bera, la normalidad. Valores superiores a .05 "
       "indican que no se rechaza el supuesto correspondiente.")

# --------------------------------------------------------- Cuadro 11
bloque(doc, "11",
       "Estimación de los coeficientes de largo plazo por mínimos cuadrados "
       "dinámicos", leer("cuadro11_dols.csv"),
       "Nota. Estimador de mínimos cuadrados dinámicos de Stock y Watson "
       "(1993) con adelantos y rezagos de las primeras diferencias de los "
       "regresores y errores estándar consistentes ante heterocedasticidad y "
       "autocorrelación (Newey y West, 1987), con ancho de banda seleccionado "
       "por el procedimiento automático de Andrews (1991). La columna de "
       "diferencia compara el coeficiente con el obtenido por el modelo de "
       "rezagos distribuidos. *p < .10. **p < .05. ***p < .01.")

# --------------------------------------------------------- Cuadro 12
for suf, sub, tit in (
        ("cointegracion", "", "Prueba de límites"),
        ("largo_plazo", "b", "Coeficientes de largo plazo"),
        ("corto_plazo", "c", "Dinámica de corto plazo"),
        ("diagnosticos", "d", "Diagnósticos de los residuos")):
    bloque(doc, "12" + sub,
           "Modelo de deuda privada y apertura comercial (M10). " + tit,
           leer("cuadro12_m10_{}.csv".format(suf)),
           "Nota. La deuda privada procede de la Global Debt Database del "
           "Fondo Monetario Internacional (Mbaye et al., 2018). Esta "
           "especificación es la única cuyos regresores satisfacen "
           "conjuntamente la condición de exogeneidad débil en el sistema "
           "correspondiente.")

# --------------------------------------------------------- Cuadro 13
for suf, sub, tit, nota in (
    ("cointegracion", "", "Prueba de límites",
     "Nota. Se contrastan tres especificaciones auxiliares: la inversión como "
     "único regresor del producto (M11), la inversión junto con la apertura "
     "comercial sin variable financiera (M12) y la inversión como variable "
     "dependiente de la apertura y el crédito (MA). Se reportan los veredictos "
     "bajo los valores críticos correctos y bajo los empleados originalmente."),
    ("largo_plazo", "b", "Coeficientes de largo plazo",
     "Nota. La columna de interpretabilidad indica si la especificación supera "
     "los diagnósticos de residuos exigidos para atribuir contenido "
     "estructural al coeficiente."),
    ("diagnosticos", "c", "Diagnósticos de los residuos",
     "Nota. Valores de probabilidad. La especificación M11 para China no supera "
     "los contrastes de autocorrelación ni de normalidad, por lo que su "
     "coeficiente no admite lectura estructural."),
    ("dols", "d", "Contraste por mínimos cuadrados dinámicos",
     "Nota. Estimación por el procedimiento de Stock y Watson (1993) como "
     "verificación independiente de los coeficientes de largo plazo.")):
    bloque(doc, "13" + sub,
           "Contraste de la hipótesis de mediación de la inversión. " + tit,
           leer("cuadro13_mediacion_{}.csv".format(suf)), nota)

# --------------------------------------------------------- Cuadro 14
for arch, sub, tit, nota in (
    ("cuadro14_johansen_rango_v3.csv", "", "Estadísticos de traza y rango",
     "Nota. Procedimiento de Johansen (1988, 1991) con constante irrestricta. "
     "La traza corregida aplica el factor de corrección por muestra finita de "
     "Reinsel y Ahn (1992). Los valores críticos tabulados resultan "
     "excesivamente permisivos en muestras de esta magnitud; véase el "
     "Cuadro 15."),
    ("cuadro14_johansen_vectores_v3.csv", "b", "Vectores de cointegración",
     "Nota. Coeficientes normalizados sobre el logaritmo del producto por "
     "habitante y expresados con el signo de la relación de largo plazo."),
    ("cuadro14_johansen_ajustes_v3.csv", "c",
     "Velocidades de ajuste y exogeneidad débil por ecuación",
     "Nota. El contraste de exogeneidad débil sigue el procedimiento de sistema "
     "parcial de Johansen (1992a). Un valor de probabilidad superior a .05 "
     "indica que la ecuación correspondiente no responde a las desviaciones "
     "respecto de la relación de largo plazo."),
    ("cuadro14_johansen_exogeneidad_v3.csv", "d",
     "Contraste conjunto de exogeneidad débil de los regresores",
     "Nota. El no rechazo valida formalmente la estimación uniecuacional de la "
     "relación de largo plazo, que resulta eficiente en ese caso."),
    ("cuadro14_johansen_resumen_v3.csv", "e",
     "Resumen de la especificación y los diagnósticos de cada sistema",
     "Nota. Todos los sistemas incluyen una variable indicadora para 2020.")):
    bloque(doc, "14" + sub,
           "Análisis de cointegración multivariante. " + tit, leer(arch), nota)

# --------------------------------------------------------- Cuadro 15
bloque(doc, "15", "Valores críticos simulados de la prueba de traza",
       leer("cuadro15_criticos_simulados.csv"),
       "Nota. Distribución obtenida por simulación de paseos aleatorios "
       "independientes bajo la hipótesis nula, con la misma longitud de "
       "muestra, el mismo orden de rezagos y la variable indicadora de 2020 en "
       "su posición efectiva, dado que la presencia de variables deterministas "
       "de quiebre altera la distribución asintótica del estadístico "
       "(Johansen et al., 2000). La simulación sustituye a la corrección por "
       "muestra finita y no se acumula con ella. Los valores tabulados "
       "resultan sistemáticamente permisivos, lo que invertiría el veredicto "
       "de rango en varias configuraciones.")

bloque(doc, "15b",
       "Estabilidad de los sistemas: raíces de la matriz compañera",
       leer("cuadro15_resumen.csv"),
       "Nota. Impuesto un rango de cointegración de uno en un sistema de "
       "cuatro variables, cabe esperar tres raíces unitarias. La mayor raíz no "
       "unitaria por debajo de la unidad confirma la estabilidad dinámica del "
       "sistema.")

bloque(doc, "15c", "Observaciones atípicas de los residuos por ecuación",
       leer("cuadro15_residuos_por_ecuacion.csv",
            filtro=lambda x: x["Residuo tipificado"].abs() >= 2.0
            if "Residuo tipificado" in x.columns else x.index == x.index),
       "Nota. Se listan las observaciones cuyo residuo tipificado alcanza o "
       "supera dos desviaciones estándar en valor absoluto. Su dispersión "
       "entre ecuaciones y años distintos desaconseja incorporar variables "
       "indicadoras adicionales, cuya inclusión elevaría los valores críticos "
       "simulados sin corregir una fuente común de perturbación.")

# --------------------------------------------------- Cuadros 16 y 17
bloque(doc, "16",
       "Sensibilidad de los resultados de sistema al orden de rezagos y al "
       "tratamiento de 2020", leer("cuadro16_sensibilidad_sistema.csv"),
       "Nota. Cada configuración se contrasta contra sus propios valores "
       "críticos simulados. El signo de la velocidad de ajuste de la ecuación "
       "del producto es el resultado de interés: negativo indica ajuste "
       "corrector hacia la relación de largo plazo.")

bloque(doc, "17",
       "Contraste de necesidad de una tendencia lineal en el espacio de "
       "cointegración", leer("cuadro17_tendencia.csv"),
       "Nota. Razón de verosimilitud entre el modelo con constante irrestricta "
       "y el mismo modelo con tendencia lineal restringida al espacio de "
       "cointegración, con distribución ji cuadrada y grados de libertad "
       "iguales al rango (Johansen, 1992b). El rechazo no conduce a adoptar la "
       "especificación con tendencia: la desaceleración del crecimiento chino "
       "implica un componente cóncavo en el nivel logarítmico que una "
       "tendencia lineal no representa, y su inclusión desplaza la relación de "
       "largo plazo hacia un ajuste de tendencia en el que los coeficientes de "
       "los regresores se aproximan a cero. Véase el apartado de limitaciones.")

bloque(doc, "17b",
       "Secuencia de Pantula para la determinación conjunta del rango y de la "
       "especificación determinista", leer("cuadro17_pantula.csv"),
       "Nota. Procedimiento secuencial de Pantula (1989) evaluado con valores "
       "críticos simulados. El primer no rechazo determina simultáneamente el "
       "rango de cointegración y el modelo determinista.")

# ---------------------------------------------------------- Anexos
bloque(doc, "A1",
       "Sensibilidad de la estimación por mínimos cuadrados dinámicos al "
       "número de adelantos y rezagos y al ancho de banda",
       leer("cuadro11b_dols_sensibilidad.csv"),
       "Nota. Se reportan todas las combinaciones evaluadas. La estabilidad de "
       "los coeficientes a lo largo de la rejilla respalda la robustez de la "
       "estimación principal. *p < .10. **p < .05. ***p < .01.")

if v4 is not None and len(v4.tables) >= 7:
    bloque(doc, "A2",
           "Estimaciones de largo plazo no identificadas. Estados Unidos",
           tabla_docx_a_df(v4.tables[6]),
           "Nota. Se presentan por transparencia y no admiten interpretación "
           "estructural, al no satisfacer las condiciones de identificación.")

# ------------------------------------------------------ Referencias
parrafo(doc, "Referencias", negrita=True, pt=12, despues=8)
REFS = [
    "Andrews, D. W. K. (1991). Heteroskedasticity and autocorrelation "
    "consistent covariance matrix estimation. Econometrica, 59(3), 817–858. "
    "https://doi.org/10.2307/2938229",
    "Bank for International Settlements. (s. f.). Credit to the non-financial "
    "sector [Conjunto de datos]. https://www.bis.org/statistics/totcredit.htm",
    "Dickey, D. A., y Fuller, W. A. (1979). Distribution of the estimators for "
    "autoregressive time series with a unit root. Journal of the American "
    "Statistical Association, 74(366), 427–431. "
    "https://doi.org/10.2307/2286348",
    "Feenstra, R. C., Inklaar, R., y Timmer, M. P. (2015). The next generation "
    "of the Penn World Table. American Economic Review, 105(10), 3150–3182. "
    "https://doi.org/10.1257/aer.20130954",
    "Feenstra, R. C., Inklaar, R., y Timmer, M. P. (2025). Penn World Table "
    "version 11.0 [Conjunto de datos]. Groningen Growth and Development "
    "Centre. https://doi.org/10.34894/FABVLR",
    "Fondo Monetario Internacional. (s. f.-a). Global Debt Database [Conjunto "
    "de datos]. https://www.imf.org/external/datamapper/datasets/GDD",
    "Fondo Monetario Internacional. (s. f.-b). World Economic Outlook database "
    "[Conjunto de datos]. https://www.imf.org/en/Publications/WEO",
    "Johansen, S. (1988). Statistical analysis of cointegration vectors. "
    "Journal of Economic Dynamics and Control, 12(2–3), 231–254. "
    "https://doi.org/10.1016/0165-1889(88)90041-3",
    "Johansen, S. (1991). Estimation and hypothesis testing of cointegration "
    "vectors in Gaussian vector autoregressive models. Econometrica, 59(6), "
    "1551–1580. https://doi.org/10.2307/2938278",
    "Johansen, S. (1992a). Cointegration in partial systems and the efficiency "
    "of single-equation analysis. Journal of Econometrics, 52(3), 389–402.",
    "Johansen, S. (1992b). Determination of cointegration rank in the presence "
    "of a linear trend. Oxford Bulletin of Economics and Statistics, 54(3), "
    "383–397.",
    "Johansen, S., Mosconi, R., y Nielsen, B. (2000). Cointegration analysis "
    "in the presence of structural breaks in the deterministic trend. "
    "Econometrics Journal, 3(2), 216–249. "
    "https://doi.org/10.1111/1368-423X.00047",
    "Kwiatkowski, D., Phillips, P. C. B., Schmidt, P., y Shin, Y. (1992). "
    "Testing the null hypothesis of stationarity against the alternative of a "
    "unit root. Journal of Econometrics, 54(1–3), 159–178.",
    "Mbaye, S., Moreno-Badia, M., y Chae, K. (2018). Global debt database: "
    "Methodology and sources (Working Paper N.º 18/111). Fondo Monetario "
    "Internacional.",
    "Newey, W. K., y West, K. D. (1987). A simple, positive semi-definite, "
    "heteroskedasticity and autocorrelation consistent covariance matrix. "
    "Econometrica, 55(3), 703–708. https://doi.org/10.2307/1913610",
    "Pantula, S. G. (1989). Testing for unit roots in time series data. "
    "Econometric Theory, 5(2), 256–271.",
    "Pesaran, M. H., Shin, Y., y Smith, R. J. (2001). Bounds testing "
    "approaches to the analysis of level relationships. Journal of Applied "
    "Econometrics, 16(3), 289–326. https://doi.org/10.1002/jae.616",
    "Phillips, P. C. B., y Perron, P. (1988). Testing for a unit root in time "
    "series regression. Biometrika, 75(2), 335–346. "
    "https://doi.org/10.1093/biomet/75.2.335",
    "Reinsel, G. C., y Ahn, S. K. (1992). Vector autoregressive models with "
    "unit roots and reduced rank structure: Estimation, likelihood ratio test, "
    "and forecasting. Journal of Time Series Analysis, 13(4), 353–375. "
    "https://doi.org/10.1111/j.1467-9892.1992.tb00113.x",
    "Stock, J. H., y Watson, M. W. (1993). A simple estimator of cointegrating "
    "vectors in higher order integrated systems. Econometrica, 61(4), 783–820. "
    "https://doi.org/10.2307/2951763",
    "Zivot, E., y Andrews, D. W. K. (1992). Further evidence on the great "
    "crash, the oil-price shock, and the unit-root hypothesis. Journal of "
    "Business & Economic Statistics, 10(3), 251–270. "
    "https://doi.org/10.1080/07350015.1992.10509904",
]
for r in REFS:
    p = doc.add_paragraph()
    pf = p.paragraph_format
    pf.left_indent = Cm(1.25)
    pf.first_line_indent = Cm(-1.25)
    pf.space_after = Pt(6)
    run = p.add_run(r)
    run.font.name = FUENTE
    run.font.size = Pt(PT_TXT)
    run._element.rPr.rFonts.set(qn("w:eastAsia"), FUENTE)

doc.save(SALIDA)

# ------------------------------------------------------ verificacion
print("=" * 78)
print("DOCUMENTO v9 GENERADO")
print("=" * 78)
print("archivo: {}".format(SALIDA))
print("tamano : {:,} bytes".format(os.path.getsize(SALIDA)))
d2 = Document(SALIDA)
print("tablas : {}   parrafos: {}".format(len(d2.tables), len(d2.paragraphs)))

print("\n{:<8} {:<12} {:<7} {:>6} {:>9}".format(
    "estado", "cuadro", "panel", "pt", "ancho cm"))
print("-" * 78)
for e in REGISTRO:
    print("{:<8} {:<12} {:<7} {:>6} {:>9}".format(*[str(x) for x in e]))

mal = [e for e in REGISTRO if e[0] != "OK"]
anch = [e for e in REGISTRO if e[0] == "OK" and e[4] > ANCHO_UTIL + 0.05]
print("\nincidencias: {}".format(len(mal)))
print("cuadros que desbordan: {}".format(len(anch)))
paneles = sum(1 for e in REGISTRO if e[0] == "OK" and e[2] != "-")
print("paneles generados por particion automatica: {}".format(paneles))

# comprobacion de que la rejilla quedo escrita y no uniforme
uniformes, sin_rejilla = [], []
for i, t in enumerate(d2.tables):
    g = t._tbl.find(qn("w:tblGrid"))
    if g is None:
        sin_rejilla.append(i)
        continue
    anchos = [int(c.get(qn("w:w"))) for c in g.findall(qn("w:gridCol"))]
    if len(set(anchos)) <= 1 and len(anchos) > 2:
        uniformes.append(i)
print("tablas sin rejilla: {}".format(len(sin_rejilla)))
print("tablas con rejilla uniforme: {}".format(len(uniformes)))
print("anchos de rejilla del cuadro mas ancho (cm): {}".format(
    [round(int(c.get(qn("w:w"))) / 567, 2)
     for c in max(d2.tables,
                  key=lambda t: len(t.columns))._tbl.find(
         qn("w:tblGrid")).findall(qn("w:gridCol"))]))


DOCUMENTO v9 GENERADO
archivo: /content/drive/MyDrive/tesis_china_eeuu/outputs/Cuadros_resultados_APA_v9.docx
tamano : 103,039 bytes
tablas : 36   parrafos: 170

estado   cuadro       panel       pt  ancho cm
------------------------------------------------------------------------------
OK       1            -          8.0     23.49
OK       2            -          8.0     18.49
OK       2b           -          8.0     11.83
OK       3            -          8.0     16.42
OK       3 (cont.)    -          8.0     23.78
OK       3b           -          8.0     14.81
OK       3c           -          8.0     12.79
OK       4            -          8.0     22.16
OK       5            -          8.0      23.9
OK       6            -          8.0     13.92
OK       7            -          8.0     14.61
OK       8            -          8.0     10.21
OK       9            -          8.0     15.37
OK       10           -          8.0     17.28
OK       11           -          8.0      23.9
OK     